# 🚀 실전매매 통합 시스템 v9.7.5.4 — KST 최종저장일 기준 수정판



## v8.9 수정 핵심 — 재무위험 과다 표시 보정

- v8.5에서 PER/PBR/ROE/시가총액 데이터가 비어 있을 때 점수가 한꺼번에 깎여 대부분 `재무위험 관찰`처럼 보일 수 있던 문제를 보정했습니다.
- 이제 KRX 재무지표가 부족하면 `정보부족/재무확인`으로 표시하고, 실제 위험 판정은 DART 영업이익·순이익·부채비율·감사의견을 더 우선합니다.
- DART API 키가 있으면 `OPEN_DART_API_KEY` 또는 `DART_API_KEY` 환경변수/코랩 Secrets에 넣어 사용하세요.
- PER/PBR/ROE는 단기·스윙에서 매수 신호가 아니라 1차 위험 필터로만 사용합니다.


## 0. 사용 방법

1. 아래 셀을 위에서부터 순서대로 실행합니다.
2. 실행이 끝나면 오늘 날짜가 붙은 엑셀 리포트가 생성됩니다.
3. 구글 코랩에서는 자동 다운로드됩니다.
4. 로컬 PC에서는 노트북이 있는 폴더에 `.xlsx` 파일이 저장됩니다.

## 매일 리포트로 쓰는 흐름

```text
아침 장 시작 전/장중 실행
        ↓
시장상태 확인
        ↓
후보소스(AUTO_CONDITION/NAVER_VOLUME/TOSS_CSV/HYBRID) 확인
        ↓
추천 리스트 확인
        ↓
스나이퍼 계산기에서 실제투입금액 조정
        ↓
계좌백테스트요약에서 200만 원 운용 시뮬레이션 확인
        ↓
실제 매매는 뉴스/호가/체결강도 확인 후 최종 판단
```

## v9.1 계좌 백테스트 기본값

| 항목 | 기본값 |
|---|---:|
| 시작일 | 2026-01-01 |
| 초기자금 | 2,000,000원 |
| 동시보유 종목 수 | 최대 4개 |
| 1종목 최대 투입비중 | 평가자산의 25% |
| 기본 익절전략 | [단기스윙] +5%▶60% / +10%▶30% / +20%▶10% |
| 기본 손절전략 | [정석방어] -3%▶70% / -5%▶20% / -10%▶10% |
| 기본 최대 보유기간 | 10거래일 |
| 시장경보 필터 | 투자경고/투자위험/거래정지/관리종목 제외 |
| 기업가치 필터 | PER/PBR/ROE/시가총액 참고, 재무등급 D는 추천 제한 |
| 검증 기준 | 백테스트 실험실 기본값은 1개월이며, 장기 검증은 3개월/6개월/전체 평가기간을 보조로 확인 |


## v9.1 후보소스 선택 — v10.1 기본값은 AUTO_CONDITION

`CONFIG['UNIVERSE_SOURCE']` 값을 바꾸면 후보군 소스를 선택할 수 있습니다.

| 값 | 의미 |
|---|---|
| `TOSS` | 토스 스크리너 URL에서 가져온 종목만 1차 후보로 사용 |
| `NAVER` | 기존 네이버 금융 거래량 상위 종목만 사용 |
| `HYBRID` | 토스 후보와 네이버 거래량 후보를 합쳐 중복 제거 |

토스 수집이 실패하면 `FALLBACK_TO_NAVER_WHEN_TOSS_FAIL=True` 설정에 따라 자동으로 네이버 후보로 대체됩니다.


## v9.5 토스 후보 수집 방식

토스 스크리너는 실행환경에 따라 Selenium 자동수집이 실패할 수 있습니다. v9.5에서는 아래 순서로 후보를 가져옵니다.

```text
1. TOSS API 사용 설정이 있으면 API 후보 시도
2. Selenium으로 토스 스크리너 자동수집 시도
3. 실패하면 TOSS_수동후보.csv 읽기
4. 그래도 없으면 NAVER 후보로 대체
```

`TOSS_수동후보.csv`에는 종목명만 넣어도 되고, 종목코드를 같이 넣으면 더 정확합니다.

| 종목명 | 종목코드 | 스크리너명 | 메모 |
|---|---|---|---|
| 한성크린텍 | 066980 | 토스 스크리너 4 | 예시 |


## 토스 첨부파일 사용법 — v9.6

1. 토스 스크리너에서 후보군을 엑셀로 복사/저장합니다.
2. 파일명을 기본값인 `국내 저평가주식 목록 토스.xlsx`로 두거나, `CONFIG['TOSS_MANUAL_XLSX_FILE']`에 파일명을 적습니다.
3. 노트북을 실행하면 `TOSS_수동후보_정리본.csv`와 `TOSS_매칭실패.csv`가 자동 생성됩니다.
4. 정리된 토스 후보는 1차 후보군으로 들어가고, 기존 안전필터·DART·추세필터·백테스트를 다시 통과해야 최종 후보가 됩니다.


## v9.7.3.3 변경점 — 백테스트 실험실 기본 1개월

- `백테스트_실험실`의 기본 기간을 `1년 전에 샀다면`에서 `1개월 전에 샀다면`으로 변경했습니다.
- 드롭다운에는 `2주`, `1개월`, `3개월`, `6개월`, `전체 평가기간`을 함께 남겨 비교할 수 있게 했습니다.
- 차트 지표 계산을 위해 실제 데이터 조회는 선택기간보다 더 길게 가져오지만, 전략 평가 시작은 선택한 기간 기준으로 적용됩니다.


## v10.1 자동조건검색 기본 방향

- `TOSS_수동후보.csv`는 이제 필수 파일이 아닙니다.
- 기본 후보소스는 `AUTO_CONDITION`이며, 네이버/pykrx 자동 후보를 먼저 만들고 `저평가 성장주` 조건으로 다시 걸러봅니다.
- DART API 키가 있으면 3년 매출/순이익 성장률 보강을 시도합니다.
- DART 키가 없거나 성장률 데이터가 부족하면 PER/거래량/추세/과열필터 중심으로 후보를 유지하고, `저평가성장주_조건검색` 시트의 `조건적용메모`에 부족한 항목을 표시합니다.
- 보유종목 승계 파일은 있으면 자동으로 읽고, 없으면 첫 실행 모드로 빈 입력 시트를 만듭니다.



## v9.7.1 수정 안내
- 전날 리포트 파일이 없어도 멈추지 않고 `첫 실행 모드`로 진행합니다.
- 첫 실행 후 생성된 오늘 날짜 파일의 `보유종목_입력`에 종목을 입력하면, 다음 실행부터 자동 승계됩니다.


## v9.7.3.3.1 수정 안내

- `보유종목_입력` 시트를 항상 생성하고, 첫 실행 때도 빈 입력표/예시/안내가 보이도록 보강했습니다.
- 실제 화면에 보이는 후보 기준을 `통합후보 15개`로 통일했습니다.
- `홈_브리핑`, `모바일_대시보드`, `TOP후보_요약`, `종목카드_TOP15`, `스나이퍼_계산기`, `백테스트_실험실`이 같은 상위 15개 후보군을 기준으로 보이도록 정리했습니다.
- `토스_TOP후보`는 별도 전면 시트에서 제거하고, 토스 후보도 `TOP후보_요약` 안에서 후보출처로 구분되게 했습니다.
- `분야` 표시를 강화했습니다. 토스 엑셀의 카테고리, 기존 업종/분야, 수동 분야 CSV를 우선 사용하고 없으면 확인필요로 표시합니다.
- 데이터 추출/헬퍼 시트는 뒤쪽으로 이동하거나 숨김 처리합니다.


## v9.6.3 패치 내용
- `탈락사유_리포트` 추가 및 주요 화면 종목 리스트 통합
- TOP후보/카드/실험실/스나이퍼가 같은 통합 후보군을 기준으로 표시
- `백테스트_실험실`의 `FILTER`, `XLOOKUP` 의존 수식을 `INDEX/MATCH` 방식으로 교체
- `실험실_기록` helper 열을 만들어 매매기록 표시 안정화
- 엑셀을 열 때 자동 재계산되도록 workbook calculation 옵션 설정

- `스나이퍼_계산기`의 추천 리스트 참조 범위를 실제로 `A:ZZ`로 보정
- `백테스트_실험실`에 선택 종목/기간 기준 최고 수익 전략 표시
- `오늘_체크리스트`를 전부 수동 확인이 아니라 자동판정+근거 형태로 개선
- 토스 후보가 TOP에서 묻히지 않도록 `토스_TOP후보` 시트 추가


In [1]:
# 1. 환경 세팅 — v9.7.5 안정화·과열필터·UI확정
# 코랩에서는 자동 설치됩니다. 로컬 환경에서는 requirements.txt 설치를 권장합니다.
!pip -q install finance-datareader pykrx openpyxl bs4 requests lxml pandas numpy selenium

import warnings, time, math, itertools, os, re, zipfile, io
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
try:
    from zoneinfo import ZoneInfo
except Exception:
    ZoneInfo = None
from copy import copy

import requests
import pandas as pd
import numpy as np
import FinanceDataReader as fdr
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment


# v9.7.3.4 패치: MergedCell 안전 쓰기/읽기 보호
# openpyxl에서 병합셀의 좌상단이 아닌 위치에 값을 쓰면
# AttributeError: 'MergedCell' object attribute 'value' is read-only 가 발생합니다.
# 아래 패치는 ws.cell(row, col, value=...) 또는 ws['H2'].value=... 형태가
# 병합셀 내부를 가리켜도 자동으로 병합영역의 좌상단 셀에 기록되도록 보정합니다.
try:
    from openpyxl.worksheet.worksheet import Worksheet as _OpenpyxlWorksheet
    from openpyxl.cell.cell import MergedCell as _OpenpyxlMergedCell
    if not getattr(_OpenpyxlWorksheet, '_v9734_merged_safe_patch', False):
        _orig_ws_cell_v9734 = _OpenpyxlWorksheet.cell
        _orig_ws_getitem_v9734 = _OpenpyxlWorksheet.__getitem__

        def _v9734_merged_anchor(ws, row, column):
            try:
                coord = f"{get_column_letter(column)}{row}"
                for rng in ws.merged_cells.ranges:
                    if coord in rng:
                        return int(rng.min_row), int(rng.min_col)
            except Exception:
                pass
            return row, column

        def _v9734_safe_cell(self, row=None, column=None, value=None):
            # value가 없을 때는 기존 동작을 최대한 유지
            if value is None:
                return _orig_ws_cell_v9734(self, row=row, column=column, value=value)
            row2, col2 = _v9734_merged_anchor(self, row, column)
            cell = _orig_ws_cell_v9734(self, row=row2, column=col2, value=None)
            cell.value = value
            return cell

        def _v9734_safe_getitem(self, key):
            obj = _orig_ws_getitem_v9734(self, key)
            # ws['H2']처럼 단일 셀을 가져올 때 병합셀 내부면 좌상단 셀을 반환
            if isinstance(obj, _OpenpyxlMergedCell):
                try:
                    row2, col2 = _v9734_merged_anchor(self, obj.row, obj.column)
                    return _orig_ws_cell_v9734(self, row=row2, column=col2, value=None)
                except Exception:
                    return obj
            return obj

        _OpenpyxlWorksheet.cell = _v9734_safe_cell
        _OpenpyxlWorksheet.__getitem__ = _v9734_safe_getitem
        _OpenpyxlWorksheet._v9734_merged_safe_patch = True
except Exception as _e:
    print(f'⚠️ MergedCell 안전 패치 적용 실패: {_e}')


try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def load_dart_api_key():
    """OpenDART API 키를 안전하게 불러옵니다.
    우선순위: 환경변수 OPEN_DART_API_KEY → 환경변수 DART_API_KEY → Colab Secrets.
    키는 노트북에 직접 적지 않는 것을 권장합니다.
    """
    key = (os.getenv('OPEN_DART_API_KEY') or os.getenv('DART_API_KEY') or '').strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = (userdata.get('OPEN_DART_API_KEY') or userdata.get('DART_API_KEY') or '').strip()
    except Exception:
        key = ''
    return key




def load_toss_api_key():
    """향후 토스증권 공식 API 키/토큰이 제공될 경우 안전하게 불러옵니다.
    우선순위: 환경변수 TOSS_API_KEY → TOSS_INVEST_API_KEY → Colab Secrets.
    현재는 API endpoint가 공개/확정되어 있지 않으면 빈 값으로 두면 됩니다.
    """
    key = (os.getenv('TOSS_API_KEY') or os.getenv('TOSS_INVEST_API_KEY') or '').strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = (userdata.get('TOSS_API_KEY') or userdata.get('TOSS_INVEST_API_KEY') or '').strip()
    except Exception:
        key = ''
    return key

def _kst_now_naive():
    """Colab/서버 시간이 UTC여도 한국 시간 기준으로 현재 시각을 반환합니다."""
    if ZoneInfo is not None:
        return datetime.now(ZoneInfo('Asia/Seoul')).replace(tzinfo=None)
    return datetime.utcnow() + timedelta(hours=9)

def refresh_report_context(force_date=None):
    """리포트 기준일을 최종 저장 시점의 KST 기준으로 갱신합니다.
    force_date='20260527' 또는 '2026-05-27'를 넣으면 수동 기준일로 고정됩니다.
    """
    global RUN_AT, REPORT_DATE, REPORT_DATETIME, CONFIG
    if force_date:
        txt = str(force_date).strip().replace('-', '')
        RUN_AT = datetime.strptime(txt, '%Y%m%d')
        REPORT_DATE = RUN_AT.strftime('%Y-%m-%d')
        REPORT_DATETIME = RUN_AT.strftime('%Y-%m-%d 09:00')
    else:
        RUN_AT = _kst_now_naive()
        REPORT_DATE = RUN_AT.strftime('%Y-%m-%d')
        REPORT_DATETIME = RUN_AT.strftime('%Y-%m-%d %H:%M')
    if 'CONFIG' in globals():
        CONFIG['REPORT_DATE'] = REPORT_DATE
        CONFIG['REPORT_DATETIME'] = REPORT_DATETIME
        CONFIG['PORTFOLIO_END'] = REPORT_DATE
        CONFIG['DART_BASE_YEAR'] = max(2024, RUN_AT.year - 1)
        CONFIG.setdefault('OUTPUT_DATE_FORMAT', '%Y%m%d')
    return REPORT_DATE, REPORT_DATETIME

# 최초 실행 시점도 KST로 잡되, 엑셀 생성/최종 저장 직전마다 다시 갱신합니다.
RUN_AT = _kst_now_naive()
REPORT_DATE = RUN_AT.strftime('%Y-%m-%d')
REPORT_DATETIME = RUN_AT.strftime('%Y-%m-%d %H:%M')

CONFIG = {
    # 리포트 날짜 기준
    'FORCE_REPORT_DATE': None,          # 예: '20260527'. None이면 최종 저장 시점 KST 기준
    'OUTPUT_DATE_FORMAT': '%Y%m%d',
    # 후보 필터
    'MIN_PRICE': 1000,                  # 최소 주가
    'MAX_PRICE': 30000,                 # 최대 주가: 단기/스윙 후보용. 필요하면 100000 등으로 조정
    'MIN_TRADING_VALUE_NAVER': 10000,   # 네이버 거래대금 단위 기준. 너무 적으면 잡주가 많이 섞임
    'TOP_N_PER_MARKET': 100,            # 코스피/코스닥 각각 거래량 상위 몇 개 볼지
    'EXCLUDE_KEYWORDS': '레버리지|인버스|2X|ETN|스팩|SPAC|우선주',

    # v9.1 후보소스 통합 설정
    # TOSS: 토스 스크리너 후보만 / NAVER: 네이버 거래량 후보만 / HYBRID: 둘 다 합쳐서 사용
    'UNIVERSE_SOURCE': 'NAVER',
    'TOSS_SCREENER_URL': 'https://www.tossinvest.com/screener/4',
    'TOSS_SCREENER_NAME': '토스 스크리너 4',
    'TOSS_MAX_ITEMS': 80,
    'TOSS_SCROLL_COUNT': 4,
    'TOSS_WAIT_SEC': 4,
    'ENABLE_TOSS_SELENIUM': False,
    'FALLBACK_TO_NAVER_WHEN_TOSS_FAIL': True,
    # v9.5: 토스 자동수집 실패 보완. 토스 페이지에서 직접 복사한 후보를 CSV로 넣으면 안정적으로 사용 가능
    'ENABLE_TOSS_MANUAL_CSV': False,
    'TOSS_MANUAL_FILE': 'TOSS_수동후보.csv',
    'TOSS_MANUAL_FIRST': False,             # True면 Selenium보다 수동 CSV를 우선 사용
    # v9.6: 토스 스크리너 복붙형 엑셀/첨부파일을 직접 읽어 1차 후보군으로 사용
    'ENABLE_TOSS_MANUAL_XLSX': False,
    'TOSS_MANUAL_XLSX_FILE': '국내 저평가주식 목록 토스.xlsx',
    'TOSS_EXCEL_AUTO_DISCOVER': False,         # 지정 파일이 없으면 파일명에 '토스'가 들어간 xlsx를 자동 탐색
    'TOSS_MANUAL_XLSX_FIRST': True,           # 토스 첨부파일 후보를 Selenium보다 먼저 사용
    'TOSS_CLEANED_CSV_FILE': 'TOSS_수동후보_정리본.csv',
    'TOSS_UNMATCHED_CSV_FILE': 'TOSS_매칭실패.csv',
    'TOSS_PARSE_SAVE_CLEANED': True,
    'TOSS_CREATE_TEMPLATE_IF_MISSING': False,
    'TOSS_DEBUG_SAVE_ON_FAIL': True,
    'TOSS_DEBUG_HTML_FILE': 'toss_debug_page.html',

    # v9.5: 향후 공식 Toss API가 열릴 경우를 대비한 설정. 현재 endpoint가 없으면 비워두면 됨
    'ENABLE_TOSS_API': False,
    'TOSS_API_ENDPOINT': '',
    'TOSS_API_KEY': load_toss_api_key(),
    'TOSS_API_TIMEOUT': 10,
    'CANDIDATE_SOURCE_MEMO': '토스 스크리너는 1차 후보군이며, 최종 후보는 안전필터·DART재무·추세필터·백테스트 통과 후 판정',


    # 검증 기간
    'BACKTEST_START': '2025-10-01',     # 지표 계산 포함 전체 조회 시작일
    'EVAL_START': '2026-01-01',         # 실제 전략 평가 시작일
    'INITIAL_BUDGET_FOR_TEST': 300000,

    # v8.9 전략백테스트 실전비용 가정
    'STRATEGY_BUY_COST_RATE': 0.0005,      # 개별 전략 백테스트 매수비용. 증권사 수수료에 맞게 조정
    'STRATEGY_SELL_COST_RATE': 0.0025,     # 개별 전략 백테스트 매도비용. 거래세/수수료 포함 보수적 가정
    'STRATEGY_SLIPPAGE_RATE': 0.0007,      # 개별 전략 백테스트 슬리피지. 0.0007=0.07%

    # 계좌/리스크 관리
    'ACCOUNT_SIZE': 10000000,           # 계좌 기준금액
    'BASE_POSITION_RATIO': 0.10,        # 기본 1종목 투입 비중
    'MAX_POSITION_RATIO': 0.20,         # 한 종목 최대 투입 비중
    'RISK_PER_TRADE_RATIO': 0.01,       # 1회 매매 허용 손실: 계좌의 1%
    'DAILY_LOSS_LIMIT': -3.0,           # 하루 손실률 제한
    'WEEKLY_LOSS_LIMIT': -7.0,          # 주간 손실률 제한
    'MAX_CONSECUTIVE_LOSSES': 4,        # 연속 손절 제한

    # 실행
    'SLEEP_SEC': 0.12,                  # 데이터 요청 간격

    # 계좌 백테스트 설정 v8.4
    'PORTFOLIO_BACKTEST_ENABLED': True,
    'PORTFOLIO_START': '2026-01-01',
    'PORTFOLIO_END': REPORT_DATE,
    'PORTFOLIO_INITIAL_CASH': 2000000,
    'PORTFOLIO_UNIVERSE_LIMIT': 250,       # 정밀도를 높이려면 500~1000, 속도를 원하면 100~250
    'PORTFOLIO_UNIVERSE_MODE': 'KRX_TOP',  # KRX_TOP: KRX 상위 유니버스 / CURRENT_NAVER: 현재 거래량 후보 기반 빠른 검증
    'PORTFOLIO_MAX_POSITIONS': 4,
    'PORTFOLIO_POSITION_RATIO': 0.25,
    'PORTFOLIO_MAX_HOLDING_DAYS': 10,
    'PORTFOLIO_MIN_TURNOVER_EOK': 20,      # 후보 최소 거래대금(억 원)
    'PORTFOLIO_MIN_VOL_RATIO': 1.5,        # 20일 평균 대비 거래량 배율
    'PORTFOLIO_BUY_COST_RATE': 0.0005,     # 매수 비용 가정. 실제 증권사 수수료에 맞게 수정
    'PORTFOLIO_SELL_COST_RATE': 0.0025,    # 매도 비용 가정. 수수료/세금 포함 보수적 기본값, 직접 수정 가능
    'PORTFOLIO_SLIPPAGE_RATE': 0.0007,     # v8.9: 계좌백테스트 체결 불리함 가정. 0.0007=0.07%
    'PORTFOLIO_DEFAULT_TP': '[단기스윙]',
    'PORTFOLIO_DEFAULT_SL': '[정석방어]',

    # v8.9 추세 필터 설정
    'ENABLE_TREND_FILTER': True,
    'TREND_FILTER_MODE': 'BALANCED',       # BALANCED: 현재가>60일선 필수 / STRICT: 20일선>60일선과 60일선 상승까지 요구
    'REQUIRE_CLOSE_ABOVE_MA60_FOR_BUY': True,
    'REQUIRE_MA20_ABOVE_MA60_FOR_BUY': False,
    'REQUIRE_MA60_UPTREND_FOR_BUY': False,
    'MA60_SLOPE_LOOKBACK': 5,

    # v8.5 안전필터 설정
    'ENABLE_SAFETY_FILTER': True,
    'ENABLE_FUNDAMENTAL_FILTER': True,
    'MARKET_ALERT_MANUAL_FILE': '시장경보_수동입력.csv',
    'EXCLUDE_ALERT_KEYWORDS': '투자경고|투자위험|거래정지|관리종목|상장폐지|불성실공시|환기',
    'WATCH_ALERT_KEYWORDS': '투자주의|투자경고 지정예고|지정예고|단기과열',
    'ALLOW_UNKNOWN_ALERT': True,          # 시장경보 자동 확인이 안 될 때 분석은 진행하되 확인필요로 표시
    'MAX_ACCEPTABLE_PER': 80,             # 단기/스윙용 완화 기준. 초과 시 감점
    'MAX_ACCEPTABLE_PBR': 8,              # 과도한 PBR은 감점
    'MIN_ACCEPTABLE_ROE': -5,             # ROE가 크게 낮으면 감점
    'MIN_MARKET_CAP_EOK': 300,            # 너무 작은 종목은 유동성/변동성 위험 감점
    'EXCLUDE_FINANCIAL_GRADE_D': False,   # True로 바꾸면 재무등급 D도 추천 리스트 제외
    'FUNDAMENTAL_UNKNOWN_IS_NOT_RISK': True, # v8.9+: PER/PBR/ROE가 비어 있으면 위험이 아니라 정보부족으로 분리
    'PORTFOLIO_ENABLE_HISTORICAL_FUNDAMENTAL_FILTER': True,
    'PORTFOLIO_ALLOW_FUNDAMENTAL_UNKNOWN': True,
    # v8.6 DART 재무안전 필터 설정
    'ENABLE_DART_FINANCIAL_FILTER': True,
    'DART_API_KEY': load_dart_api_key(),  # OPEN_DART_API_KEY/DART_API_KEY 또는 Colab Secrets에서 자동 인식
    'DART_MANUAL_FILE': 'DART_재무수동입력.csv',
    'DART_CACHE_FILE': 'DART_재무캐시.csv',            # v8.9: DART API 조회 결과 영구 캐시
    'DART_CACHE_TTL_DAYS': 90,                        # 재무제표는 자주 바뀌지 않으므로 90일 캐시 기본
    'DART_USE_CACHE': True,
    'DART_BASE_YEAR': max(2024, RUN_AT.year - 1),        # 기본은 직전 사업연도
    'DART_REPORT_CODE': '11011',                         # 11011=사업보고서, 11013=1분기, 11012=반기, 11014=3분기
    'DART_FS_DIV': 'CFS',                                # CFS=연결, OFS=별도
    'MAX_ACCEPTABLE_DEBT_RATIO': 250,
    'WATCH_DEBT_RATIO': 180,
    'EXCLUDE_BAD_AUDIT_OPINION': True,
    'EXCLUDE_CAPITAL_IMPAIRMENT': True,
    'WATCH_OPERATING_LOSS': True,

    # v8.6 백테스트 검증 신뢰도 기준
    'MIN_RELIABLE_TRADES': 30,
    'PREFERRED_RELIABLE_TRADES': 50,
    'MAX_ACCEPTABLE_MDD_FOR_RELIABLE': -20,
    'MIN_BACKTEST_DAYS_FOR_RELIABLE': 90,

    # v9.3~v9.4 인터페이스/엑셀·HTML UI 설정
    'UX_TOP_N': 15,                         # 홈/요약/카드/실험실 공통 상위 후보 수
    'UX_HIDE_DETAIL_COLUMNS': True,          # 추천 리스트 상세 시트에서 덜 중요한 열은 기본 숨김
    'UX_COMPACT_VIEW_ENABLED': True,         # 홈_브리핑 / TOP후보_요약 시트 생성
    'UX_ZOOM_SCALE': 90,                     # 주요 시트 확대/축소 비율
    'UX_CARD_DASHBOARD_ENABLED': True,       # v9.3: 카드형 대시보드 시트 생성
    'UX_HTML_DASHBOARD_ENABLED': True,       # v9.3: 브라우저용 HTML 대시보드 생성
    'UX_HIDE_RAW_DETAIL_SHEETS': True,       # v9.3: 상세 DB 시트를 기본 숨김 처리

    # v9.6.3: 실제 보는 화면의 종목 기준을 통합하고, 토스 후보는 별도 시트가 아니라 TOP후보_요약에 참고 후보로 합칩니다.
    'UX_UNIFIED_TOP_N': 15,
    'UX_TOSS_REFERENCE_N': 0,
    'UX_HIDE_TOSS_TOP_SHEET': True,
    'UX_ADD_REJECTION_REPORT': True,
    'UX_MASTER_TOP_N': 15,                  # 실제 보는 모든 화면의 기준 후보 수
    'SECTOR_MANUAL_FILE': '종목분야_수동입력.csv',
    'SECTOR_UNKNOWN_LABEL': '분야확인필요',


    # v9.4 백테스트 실험실 설정
    'LAB_BACKTEST_ENABLED': True,
    'LAB_MAX_STOCKS': 15,                  # 실험실에서 미리 계산할 상위 후보 수. 속도가 느리면 5~8로 조정
    'LAB_BASE_BUDGET': 300000,             # 백테스트 기록의 기준 투자금
    'LAB_PERIOD_OPTIONS': {
        '1개월 전에 샀다면': 31,
        '2주 전에 샀다면': 14,
        '3개월 전에 샀다면': 92,
        '6개월 전에 샀다면': 183,
        '전체 평가기간': None
    },
    'LAB_BUDGET_OPTIONS': {
        '30만원': 300000,
        '50만원': 500000,
        '100만원': 1000000,
        '200만원': 2000000
    },


    # v9.7 진입단가/보유종목 자동승계 설정
    'ENABLE_ENTRY_SCENARIO': True,
    'ENTRY_SCENARIO_MAX_ROWS': 15,
    'ENABLE_HOLDINGS_MANAGEMENT': True,
    'HOLDINGS_AUTO_IMPORT_PREVIOUS': True,
    'HOLDINGS_PREVIOUS_FILE': '',                  # 특정 전일 엑셀 파일을 직접 지정하려면 파일명 입력
    'HOLDINGS_INPUT_SHEET': '보유종목_입력',
    'HOLDINGS_DIAGNOSIS_SHEET': '보유종목_진단',
    'HOLDINGS_SEARCH_DATE_REGEX': r'(20\d{6})',
    'OUTPUT_FILENAME_DATE_ONLY': True,              # 최종 엑셀 파일명: 예) 20260522.xlsx
    'OUTPUT_DATE_FORMAT': '%Y%m%d',

    # v9.7.4 급등 과열 필터
    'ENABLE_OVERHEAT_HISTORY_FETCH': False,  # True면 FDR로 52주 고가/당일등락률 보강. 속도는 느려질 수 있음

    'OUTPUT_PREFIX': '실전매매_통합시스템_v10_1_자동조건검색_드라이브운영준비'
}


# v9.7.3.5: 백테스트 실험실은 스나이퍼 계산기와 같은 추천 리스트 전체를 후보로 사용
CONFIG.setdefault('LAB_USE_FULL_RECOMMEND_LIST', True)
CONFIG.setdefault('LAB_SELECTION_STYLE', 'BUTTON_PANEL')
CONFIG.setdefault('LAB_BUTTON_PANEL_ENABLED', True)

# v10.1 기본 후보소스: TOSS 파일 없이 자동조건검색 사용
CONFIG.setdefault('CANDIDATE_SOURCE_MODE', 'AUTO_CONDITION')  # AUTO_CONDITION / NAVER_VOLUME / TOSS_CSV / HYBRID
CONFIG.setdefault('REQUIRE_TOSS_FILE', False)
CONFIG.setdefault('ENABLE_AUTO_CONDITION_SEARCH', True)
CONFIG.setdefault('ENABLE_DART_3Y_GROWTH', True)
CONFIG.setdefault('DART_GROWTH_MAX_CODES', 80)
CONFIG.setdefault('DART_GROWTH_CACHE_FILE', 'DART_3년성장률캐시.csv')
CONFIG.setdefault('DART_GROWTH_TTL_DAYS', 90)
CONFIG.setdefault('AUTO_CONDITION_TEXT', '저평가 성장주: 최근 3년 평균 매출액 증감률 10% 이상, PER 0 초과 20 이하, 최근 3년 평균 순이익 증감률 20% 이상, 과열 제외')


refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))
print(f'✅ v9.7.5.4 환경 세팅 완료 / 리포트 기준일(KST): {REPORT_DATE}')
print('🔐 DART API 키 상태:', '자동조회 가능' if CONFIG.get('DART_API_KEY') else '키 없음 — 수동 CSV 또는 재무 일부 생략')

print('🌐 후보소스 설정:', CONFIG.get('CANDIDATE_SOURCE_MODE'), '/', CONFIG.get('UNIVERSE_SOURCE'))
print('🧩 TOSS 파일 입력:', '필수 아님' if not CONFIG.get('REQUIRE_TOSS_FILE') else '필수', '/', '선택 CSV_ON' if CONFIG.get('ENABLE_TOSS_MANUAL_CSV') else '선택 CSV_OFF')


# v9.7.3.11: 백테스트 실험실 단기 기간 데이터 로드 보정
CONFIG['LAB_MIN_EVAL_ROWS_SHORT'] = 5      # 2주/1개월 구간 최소 거래일
CONFIG['LAB_MIN_EVAL_ROWS_NORMAL'] = 12    # 3개월 이상 구간 최소 거래일
CONFIG['OUTPUT_PREFIX'] = '실전매매_통합시스템_v9_7_5_4_KST최종저장일수정'




✅ v9.7.5.4 환경 세팅 완료 / 리포트 기준일(KST): 2026-06-08
🔐 DART API 키 상태: 자동조회 가능
🌐 후보소스 설정: AUTO_CONDITION / NAVER
🧩 TOSS 파일 입력: 필수 아님 / 선택 CSV_OFF


In [2]:
# === GITHUB_RUNTIME_CONFIG_PATCH_V10_2_HYBRID_TOSS ===
import os
from pathlib import Path

try:
    CONFIG
except NameError:
    CONFIG = {}

# GitHub Actions 실행 기본값
CONFIG['CANDIDATE_SOURCE_MODE'] = os.getenv('CANDIDATE_SOURCE_MODE', 'HYBRID')
CONFIG['UNIVERSE_SOURCE'] = 'HYBRID'  # 구버전 호환
CONFIG['REQUIRE_TOSS_FILE'] = str(os.getenv('REQUIRE_TOSS_FILE', 'false')).lower() == 'true'
CONFIG['ENABLE_AUTO_CONDITION_SEARCH'] = True
CONFIG['FALLBACK_TO_NAVER_WHEN_TOSS_FAIL'] = True

# TOSS 수동 후보 파일
CONFIG['TOSS_MANUAL_CSV_FILE'] = os.getenv('TOSS_MANUAL_CSV_FILE', 'TOSS_수동후보.csv')
CONFIG['TOSS_MANUAL_XLSX_FILE'] = os.getenv('TOSS_MANUAL_XLSX_FILE', '국내 저평가주식 목록 토스.xlsx')

# GitHub에서는 Google Drive 저장을 끄고, 노트북이 생성하는 docs/ 또는 stock_report/ 결과를 커밋합니다.
CONFIG['USE_GOOGLE_DRIVE'] = False
CONFIG['REPORT_ROOT_MODE'] = os.getenv('REPORT_ROOT_MODE', 'LOCAL')
CONFIG['OUTPUT_FILENAME_DATE_ONLY'] = True

# API 키: GitHub Secrets에서 들어오면 CONFIG에도 넣습니다.
_dart_key = os.getenv('OPEN_DART_API_KEY') or os.getenv('DART_API_KEY') or ''
CONFIG['DART_API_KEY'] = _dart_key
CONFIG['OPEN_DART_API_KEY'] = _dart_key
CONFIG['NAVER_CLIENT_ID'] = os.getenv('NAVER_CLIENT_ID', '')
CONFIG['NAVER_CLIENT_SECRET'] = os.getenv('NAVER_CLIENT_SECRET', '')

# TOSS 파일이 없으면 빈 헤더 파일을 만들어 HYBRID 실행이 멈추지 않게 합니다.
_toss_path = Path(CONFIG['TOSS_MANUAL_CSV_FILE'])
if not _toss_path.exists():
    _toss_path.write_text('종목명,종목코드,스크리너명,메모\n', encoding='utf-8-sig')
    print(f'ℹ️ TOSS 수동 후보 파일이 없어 빈 파일을 생성했습니다: {_toss_path}')
else:
    print(f'✅ TOSS 수동 후보 파일 확인: {_toss_path}')

print('✅ GitHub runtime CONFIG 적용 완료')
print('   - CANDIDATE_SOURCE_MODE:', CONFIG.get('CANDIDATE_SOURCE_MODE'))
print('   - TOSS_MANUAL_CSV_FILE:', CONFIG.get('TOSS_MANUAL_CSV_FILE'))


✅ TOSS 수동 후보 파일 확인: TOSS_수동후보.csv
✅ GitHub runtime CONFIG 적용 완료
   - CANDIDATE_SOURCE_MODE: HYBRID
   - TOSS_MANUAL_CSV_FILE: TOSS_수동후보.csv


# v10.0 자동 리포트 운영 준비판

이 섹션은 기존 v9.9 HTML 리포트를 **매일 자동 운영**으로 확장하기 위한 준비 단계입니다.

핵심 기능:
- Google Drive 또는 로컬 폴더에서 전날 보유종목 파일 자동 탐색
- 오늘 결과 엑셀/HTML/ZIP을 날짜별 폴더와 latest 폴더에 저장
- API 키는 Colab Secrets / 환경변수 / GitHub Secrets에서 읽도록 유지
- 자동화용 GitHub Actions 템플릿과 실행 안내 생성

처음에는 Colab 수동 실행 + Drive 저장으로 안정화하고, 이후 GitHub Actions 자동 실행으로 넘어가는 흐름을 권장합니다.


In [3]:
# 1-1. v10.0 자동운영 설정 — Drive/로컬 보유파일 자동탐색 준비
# 이 셀은 CONFIG 생성 이후에 실행됩니다.

from pathlib import Path
from datetime import datetime, timedelta, timezone
import os, re, glob, shutil, json

KST_V10 = timezone(timedelta(hours=9))

# === 자동 운영 기본 설정 ===
CONFIG.setdefault('AUTO_REPORT_MODE', False)        # True: 자동운영 경로 사용. 처음 테스트는 False 권장
CONFIG.setdefault('USE_GOOGLE_DRIVE', False)        # Colab에서 Drive를 mount하려면 True
CONFIG.setdefault('DRIVE_ROOT', '/content/drive/MyDrive/stock_report')
CONFIG.setdefault('LOCAL_REPORT_ROOT', './stock_report')
CONFIG.setdefault('REPORT_ROOT_MODE', 'LOCAL')      # LOCAL / DRIVE
CONFIG.setdefault('AUTO_LOAD_PREVIOUS_HOLDINGS', True)
CONFIG.setdefault('AUTO_SAVE_REPORTS', True)
CONFIG.setdefault('SAVE_LATEST_REPORT', True)
CONFIG.setdefault('AUTO_DOWNLOAD_AT_END', True)     # Colab 수동 실행 시 마지막 파일 다운로드
CONFIG.setdefault('REPORT_OUTPUT_SUBDIR', 'reports')
CONFIG.setdefault('HOLDINGS_SUBDIR', 'holdings')
CONFIG.setdefault('LATEST_SUBDIR', 'latest')
CONFIG.setdefault('DOCS_SUBDIR', 'docs')
CONFIG.setdefault('REPORT_DATE_FORMAT', '%Y%m%d')
CONFIG.setdefault('FORCE_REPORT_DATE', None)


def v100_today_key():
    forced = CONFIG.get('FORCE_REPORT_DATE')
    if forced:
        return str(forced).replace('-', '')
    return datetime.now(KST_V10).strftime('%Y%m%d')


def v100_report_root():
    mode = str(CONFIG.get('REPORT_ROOT_MODE', 'LOCAL')).upper()
    if CONFIG.get('USE_GOOGLE_DRIVE') or mode == 'DRIVE':
        return Path(CONFIG.get('DRIVE_ROOT', '/content/drive/MyDrive/stock_report'))
    return Path(CONFIG.get('LOCAL_REPORT_ROOT', './stock_report'))


def v100_mount_drive_if_needed():
    if not CONFIG.get('USE_GOOGLE_DRIVE', False):
        return False
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return True
    except Exception as e:
        print('⚠️ Google Drive mount 실패 또는 Colab이 아님:', e)
        return False


def v100_prepare_folders():
    root = v100_report_root()
    for sub in [CONFIG['REPORT_OUTPUT_SUBDIR'], CONFIG['HOLDINGS_SUBDIR'], CONFIG['LATEST_SUBDIR'], CONFIG['DOCS_SUBDIR']]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    return root


def v100_find_latest_previous_holding(today_key=None):
    """보고일보다 과거인 날짜형 xlsx 중 가장 최신 파일을 찾아 작업폴더로 복사합니다."""
    today_key = today_key or v100_today_key()
    root = v100_report_root()
    search_dirs = [Path('.'), Path('/content'), Path('/mnt/data'), root / CONFIG['HOLDINGS_SUBDIR'], root / CONFIG['REPORT_OUTPUT_SUBDIR']]
    candidates = []
    for d in search_dirs:
        if not d.exists():
            continue
        files = d.rglob('*.xlsx') if d == root / CONFIG['REPORT_OUTPUT_SUBDIR'] else d.glob('*.xlsx')
        for fp in files:
            name = fp.name
            if name.startswith('~$'):
                continue
            if any(skip in name for skip in ['토스증권', '해외주식리스트', '국내 저평가주식 목록 토스']):
                continue
            m = re.search(r'(20\d{6})', name)
            if not m:
                continue
            key = m.group(1)
            if key >= today_key:
                continue
            candidates.append((key, fp))
    if not candidates:
        return ''
    candidates.sort(key=lambda x: x[0], reverse=True)
    src = candidates[0][1]
    local = Path(src.name)
    try:
        if src.resolve() != local.resolve():
            shutil.copy2(src, local)
    except Exception:
        shutil.copy2(src, local)
    return str(local)


def v100_preflight():
    v100_mount_drive_if_needed()
    root = v100_prepare_folders()
    today = v100_today_key()
    CONFIG['REPORT_DATE'] = today
    CONFIG['PORTFOLIO_END'] = today
    if CONFIG.get('AUTO_LOAD_PREVIOUS_HOLDINGS', True):
        prev = v100_find_latest_previous_holding(today)
        if prev:
            CONFIG['HOLDINGS_PREVIOUS_FILE'] = prev
            CONFIG['HOLDINGS_AUTO_IMPORT_PREVIOUS'] = True
            print(f'📥 전날 보유파일 자동탐색: {prev}')
        else:
            print('ℹ️ 전날 보유파일을 찾지 못했습니다. 첫 실행 모드 또는 보유종목_입력 직접 입력으로 진행합니다.')
    print(f'📁 리포트 루트: {root}')
    print(f'📅 보고 기준일: {today}')
    return root

V10_REPORT_ROOT = v100_preflight()


📥 전날 보유파일 자동탐색: 20260607.xlsx
📁 리포트 루트: stock_report
📅 보고 기준일: 20260608


In [4]:
# 1-2. v10.1 자동조건검색 기본 설정 — TOSS 파일 없이 실행
# 핵심: TOSS_수동후보.csv는 선택사항입니다. 기본은 네이버/pykrx 자동 후보 + 저평가 성장주 조건검색입니다.

CONFIG.setdefault('CANDIDATE_SOURCE_MODE', 'AUTO_CONDITION')
CONFIG.setdefault('AUTO_CONDITION_TEXT', '저평가 성장주: 최근 3년 평균 매출액 증감률 10% 이상, PER 0 초과 20 이하, 최근 3년 평균 순이익 증감률 20% 이상, 과열 제외')
CONFIG.setdefault('ENABLE_DART_3Y_GROWTH', True)
CONFIG.setdefault('DART_GROWTH_MAX_CODES', 80)
CONFIG.setdefault('DART_GROWTH_CACHE_FILE', 'DART_3년성장률캐시.csv')
CONFIG.setdefault('DART_GROWTH_TTL_DAYS', 90)

mode = str(CONFIG.get('CANDIDATE_SOURCE_MODE', 'AUTO_CONDITION')).upper().strip()
if mode == 'AUTO_CONDITION':
    # 토스 파일 없이 네이버/pykrx 후보부터 자동 생성
    CONFIG['UNIVERSE_SOURCE'] = 'NAVER'
    CONFIG['ENABLE_TOSS_API'] = False
    CONFIG['ENABLE_TOSS_SELENIUM'] = False
    CONFIG['ENABLE_TOSS_MANUAL_XLSX'] = False
    CONFIG['ENABLE_TOSS_MANUAL_CSV'] = False
    CONFIG['TOSS_EXCEL_AUTO_DISCOVER'] = False
    CONFIG['TOSS_CREATE_TEMPLATE_IF_MISSING'] = False
elif mode == 'NAVER_VOLUME':
    CONFIG['UNIVERSE_SOURCE'] = 'NAVER'
    CONFIG['ENABLE_TOSS_MANUAL_XLSX'] = False
    CONFIG['ENABLE_TOSS_MANUAL_CSV'] = False
    CONFIG['ENABLE_TOSS_SELENIUM'] = False
elif mode == 'TOSS_CSV':
    CONFIG['UNIVERSE_SOURCE'] = 'TOSS'
    CONFIG['ENABLE_TOSS_MANUAL_CSV'] = True
    CONFIG['TOSS_MANUAL_FIRST'] = True
    CONFIG['ENABLE_TOSS_MANUAL_XLSX'] = False
    CONFIG['ENABLE_TOSS_SELENIUM'] = False
elif mode == 'HYBRID':
    CONFIG['UNIVERSE_SOURCE'] = 'HYBRID'
    CONFIG['ENABLE_TOSS_MANUAL_CSV'] = True
    CONFIG['ENABLE_TOSS_MANUAL_XLSX'] = True

NAVER_CONDITION_TEXT = CONFIG.get('AUTO_CONDITION_TEXT')
print('✅ v10.1 후보소스 모드:', mode)
print('✅ 실제 후보 수집 설정 UNIVERSE_SOURCE:', CONFIG.get('UNIVERSE_SOURCE'))
print('✅ TOSS_수동후보.csv 필수 여부:', '아니오' if mode in ['AUTO_CONDITION','NAVER_VOLUME'] else '선택/사용')
print('✅ 기본 조건검색:', NAVER_CONDITION_TEXT)


✅ v10.1 후보소스 모드: HYBRID
✅ 실제 후보 수집 설정 UNIVERSE_SOURCE: HYBRID
✅ TOSS_수동후보.csv 필수 여부: 선택/사용
✅ 기본 조건검색: 저평가 성장주: 최근 3년 평균 매출액 증감률 10% 이상, PER 0 초과 20 이하, 최근 3년 평균 순이익 증감률 20% 이상, 과열 제외


In [5]:
# 2. 공통 함수

def clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')


def calc_indicators(df):
    """기술 지표 계산: 이동평균, RSI, TRIX, ATR, 20일 수익률"""
    df = df.copy()
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA60'] = df['Close'].rolling(60).mean()
    df['VOL20'] = df['Volume'].rolling(20).mean()

    delta = df['Close'].diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    rs = up.ewm(com=13, adjust=False).mean() / down.ewm(com=13, adjust=False).mean()
    df['RSI'] = 100 - (100 / (1 + rs))

    ema1 = df['Close'].ewm(span=15, adjust=False).mean()
    ema2 = ema1.ewm(span=15, adjust=False).mean()
    ema3 = ema2.ewm(span=15, adjust=False).mean()
    df['TRIX'] = (ema3 - ema3.shift(1)) / ema3.shift(1) * 100

    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(14).mean()
    df['ATR_PCT'] = (df['ATR14'] / df['Close']) * 100
    df['RET20'] = df['Close'].pct_change(20) * 100
    return df


def safe_fdr_reader(code, start):
    try:
        df = fdr.DataReader(code, start)
        if df is None or df.empty:
            return pd.DataFrame()
        return df.dropna().copy()
    except Exception:
        return pd.DataFrame()


def get_quant_stocks(sosok, top_n=100):
    """네이버 금융 거래량 상위 종목 수집. sosok=0 코스피, sosok=1 코스닥"""
    url = f'https://finance.naver.com/sise/sise_quant.naver?sosok={sosok}'
    headers = {'User-Agent': 'Mozilla/5.0'}
    res = requests.get(url, headers=headers, timeout=10)
    res.encoding = 'euc-kr'
    soup = BeautifulSoup(res.text, 'html.parser')
    tables = pd.read_html(res.text)
    if len(tables) < 2:
        return pd.DataFrame()
    df = tables[1].dropna(subset=['종목명']).copy()
    codes = [a.get('href').split('code=')[-1] for a in soup.select('table.type_2 a.tltle')]
    df = df.head(len(codes)).copy()
    df['종목코드'] = codes[:len(df)]
    df['시장구분'] = 'KOSPI' if sosok == 0 else 'KOSDAQ'
    return df.head(top_n)



# ===============================
# v9.6 후보소스 통합 함수 — TOSS API 준비 + 수동 CSV fallback
# ===============================

def normalize_name_for_match(text):
    """종목명 매칭용 정규화"""
    return re.sub(r'\s+', '', str(text or '')).strip()


def normalize_code_v95(value):
    """종목코드를 6자리 문자열로 정리. 기존 normalize_code가 나중 셀에서 정의되어도 안전하게 사용."""
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return ''
        txt = re.sub(r'[^0-9]', '', str(value))
        if not txt:
            return ''
        return txt.zfill(6)[-6:]
    except Exception:
        return ''


def get_krx_listing_for_match():
    """KRX 상장 목록을 종목명/종목코드 매칭용으로 준비"""
    try:
        listing = fdr.StockListing('KRX').copy()
        code_col = 'Code' if 'Code' in listing.columns else 'Symbol'
        name_col = 'Name' if 'Name' in listing.columns else 'Name'
        market_col = 'Market' if 'Market' in listing.columns else None
        out = pd.DataFrame({
            '종목코드': listing[code_col].astype(str).str.zfill(6),
            '종목명': listing[name_col].astype(str),
            '시장구분': listing[market_col].astype(str) if market_col else ''
        })
        out['매칭명'] = out['종목명'].apply(normalize_name_for_match)
        out = out[out['종목코드'].str.match(r'^\d{6}$', na=False)]
        out = out.drop_duplicates(subset=['종목코드'])
        return out
    except Exception as e:
        print(f'⚠️ KRX 상장목록 조회 실패: {e}')
        return pd.DataFrame(columns=['종목코드', '종목명', '시장구분', '매칭명'])


def enrich_candidate_market_data(df):
    """토스처럼 가격/거래대금이 없는 후보에 최근 시세 정보를 보강합니다.
    거래대금은 네이버 거래대금 컬럼과 맞추기 위해 '억 원 × 100' 스케일로 저장합니다.
    """
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    for col in ['현재가', '거래량', '거래대금', '등락률']:
        if col not in df.columns:
            df[col] = np.nan

    for idx, row in df.iterrows():
        curr_val = pd.to_numeric(pd.Series([row.get('현재가')]), errors='coerce').iloc[0]
        turnover_val = pd.to_numeric(pd.Series([row.get('거래대금')]), errors='coerce').iloc[0]
        need_price = pd.isna(curr_val) or curr_val <= 0
        need_turnover = pd.isna(turnover_val) or turnover_val <= 0
        if not (need_price or need_turnover):
            continue

        code = normalize_code_v95(row.get('종목코드'))
        if not code:
            continue
        hist = safe_fdr_reader(code, (RUN_AT - timedelta(days=20)).strftime('%Y-%m-%d'))
        if hist.empty:
            continue

        latest = hist.iloc[-1]
        prev_close = hist['Close'].iloc[-2] if len(hist) >= 2 else np.nan
        close = pd.to_numeric(pd.Series([latest.get('Close')]), errors='coerce').iloc[0]
        volume = pd.to_numeric(pd.Series([latest.get('Volume')]), errors='coerce').iloc[0]
        if not pd.isna(close):
            df.at[idx, '현재가'] = close
        if not pd.isna(volume):
            df.at[idx, '거래량'] = volume
        if not pd.isna(close) and not pd.isna(volume):
            turnover_eok = close * volume / 100000000
            df.at[idx, '거래대금'] = turnover_eok * 100
            df.at[idx, '거래대금(억)'] = round(turnover_eok, 1)
        if not pd.isna(close) and not pd.isna(prev_close) and prev_close != 0:
            df.at[idx, '등락률'] = round((close / prev_close - 1) * 100, 2)

        time.sleep(CONFIG.get('SLEEP_SEC', 0.1))

    return df


def get_naver_quant_universe():
    """기존 네이버 금융 거래량 상위 후보군"""
    print('📡 네이버 금융 거래량 상위 후보 수집')
    df_kospi = get_quant_stocks(0, CONFIG['TOP_N_PER_MARKET'])
    df_kosdaq = get_quant_stocks(1, CONFIG['TOP_N_PER_MARKET'])
    total = pd.concat([df_kospi, df_kosdaq], ignore_index=True)
    if total.empty:
        return total
    total['후보출처'] = 'NAVER'
    total['스크리너명'] = '네이버 거래량 상위'
    total['스크리너URL'] = 'https://finance.naver.com/sise/sise_quant.naver'
    total['후보소스메모'] = '네이버 금융 거래량 상위 종목을 1차 후보로 사용'
    return total


def create_toss_manual_template(path=None):
    """토스 수동 후보 CSV 템플릿을 생성합니다."""
    path = path or CONFIG.get('TOSS_MANUAL_FILE', 'TOSS_수동후보.csv')
    if os.path.exists(path):
        return path
    sample = pd.DataFrame([
        {'종목명': '한성크린텍', '종목코드': '066980', '스크리너명': CONFIG.get('TOSS_SCREENER_NAME', '토스 스크리너'), '메모': '예시. 실제 사용 시 원하는 후보로 교체'},
        {'종목명': '', '종목코드': '', '스크리너명': '', '메모': ''},
    ])
    sample.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'📝 토스 수동후보 템플릿 생성: {path}')
    return path



# ===============================
# v9.6 토스 첨부파일 XLSX 파서
# ===============================

def _v96_num_from_text(value):
    """문자열에서 첫 번째 숫자를 float로 추출"""
    try:
        txt = str(value or '').replace(',', '').replace('−', '-').strip()
        if txt in ['', '-', 'nan', 'None']:
            return np.nan
        m = re.search(r'[+-]?\d+(?:\.\d+)?', txt)
        return float(m.group(0)) if m else np.nan
    except Exception:
        return np.nan


def _v96_parse_percent(value):
    """토스 성장률/등락률을 % 단위 float로 변환. 0.1174는 11.74로 변환."""
    try:
        txt = str(value or '').replace(',', '').replace('−', '-').strip()
        if txt in ['', '-', 'nan', 'None']:
            return np.nan
        m = re.search(r'([+-]?\d+(?:\.\d+)?)\s*%', txt)
        if m:
            return float(m.group(1))
        n = _v96_num_from_text(txt)
        if pd.isna(n):
            return np.nan
        return n * 100 if abs(n) <= 5 else n
    except Exception:
        return np.nan


def _v96_parse_price(value):
    n = _v96_num_from_text(str(value or '').replace('원', ''))
    return int(n) if not pd.isna(n) else np.nan


def _v96_parse_volume(value):
    n = _v96_num_from_text(str(value or '').replace('주', ''))
    return int(n) if not pd.isna(n) else np.nan


def _v96_parse_per(value):
    n = _v96_num_from_text(str(value or '').replace('배', ''))
    return round(float(n), 2) if not pd.isna(n) else np.nan


def _v96_parse_market_cap_eok(value):
    """시가총액 문자열을 억 원 단위로 변환. 예: 1.9조원 -> 19000, 462.8억원 -> 462.8"""
    try:
        txt = str(value or '').replace(',', '').replace(' ', '').strip()
        if txt in ['', '-', 'nan', 'None']:
            return np.nan
        total = 0.0
        jo = re.search(r'([0-9]+(?:\.[0-9]+)?)조', txt)
        eok = re.search(r'([0-9]+(?:\.[0-9]+)?)억', txt)
        if jo:
            total += float(jo.group(1)) * 10000
        if eok:
            total += float(eok.group(1))
        if total > 0:
            return round(total, 1)
        n = _v96_num_from_text(txt)
        return round(float(n), 1) if not pd.isna(n) else np.nan
    except Exception:
        return np.nan


def find_toss_manual_xlsx_file(path=None):
    """토스 수동후보 엑셀 파일을 찾습니다."""
    import glob
    candidates = []
    if path:
        candidates.append(path)
    cfg_path = CONFIG.get('TOSS_MANUAL_XLSX_FILE', '')
    if cfg_path:
        candidates.append(cfg_path)
    if CONFIG.get('TOSS_EXCEL_AUTO_DISCOVER', True):
        candidates += sorted(glob.glob('*토스*.xlsx'))
        candidates += sorted(glob.glob('*TOSS*.xlsx'))
        candidates += sorted(glob.glob('*toss*.xlsx'))
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return ''


def parse_toss_manual_xlsx_raw(path):
    """토스 스크리너 복붙형 XLSX를 정규화합니다.
    현재 토스 복붙 파일은 대체로 `종목명 행` 다음에 `지표 행`이 이어지는 구조입니다.
    """
    raw = pd.read_excel(path, header=None, dtype=str, engine='openpyxl').fillna('')
    rows = raw.values.tolist()
    records = []
    for i, row in enumerate(rows):
        row = [str(x).strip() for x in row]
        a = row[0] if len(row) else ''
        nonempty = [x for x in row if x]
        # 종목명만 있는 행 + 다음 행에 순위/현재가/등락률/카테고리...가 있는 구조
        if not a or re.match(r'^\d+$', a) or len(nonempty) != 1:
            continue
        if i + 1 >= len(rows):
            continue
        nxt = [str(x).strip() for x in rows[i + 1]]
        if len(nxt) < 6 or not re.match(r'^\d+$', nxt[0] or ''):
            continue
        rec = {
            '종목명': a,
            '종목코드': '',
            '토스순위': nxt[0] if len(nxt) > 0 else '',
            '토스현재가원문': nxt[1] if len(nxt) > 1 else '',
            '토스등락률원문': nxt[2] if len(nxt) > 2 else '',
            '토스카테고리': nxt[3] if len(nxt) > 3 else '',
            '토스시가총액원문': nxt[4] if len(nxt) > 4 else '',
            '토스거래량원문': nxt[5] if len(nxt) > 5 else '',
            '토스애널리스트분석': nxt[6] if len(nxt) > 6 else '',
            '토스매출성장률원문': nxt[7] if len(nxt) > 7 else '',
            '토스PER원문': nxt[8] if len(nxt) > 8 else '',
            '토스순이익성장률원문': nxt[9] if len(nxt) > 9 else '',
        }
        rec['토스현재가'] = _v96_parse_price(rec['토스현재가원문'])
        rec['토스등락률'] = _v96_parse_percent(rec['토스등락률원문'])
        rec['토스시가총액(억)'] = _v96_parse_market_cap_eok(rec['토스시가총액원문'])
        rec['토스거래량'] = _v96_parse_volume(rec['토스거래량원문'])
        rec['토스매출성장률(%)'] = _v96_parse_percent(rec['토스매출성장률원문'])
        rec['토스PER'] = _v96_parse_per(rec['토스PER원문'])
        rec['토스순이익성장률(%)'] = _v96_parse_percent(rec['토스순이익성장률원문'])
        rec['후보출처'] = 'TOSS_XLSX'
        rec['스크리너명'] = CONFIG.get('TOSS_SCREENER_NAME', '토스 스크리너')
        rec['스크리너URL'] = CONFIG.get('TOSS_SCREENER_URL', '')
        rec['후보소스메모'] = f"토스 첨부파일({os.path.basename(path)})에서 추출한 1차 후보. 내부 안전/재무/추세/백테스트를 다시 통과해야 함"
        rec['토스원본파일'] = os.path.basename(path)
        rec['토스정리일시'] = REPORT_DATETIME
        records.append(rec)
    return pd.DataFrame(records)


def get_toss_manual_xlsx_candidates(path=None, max_items=None):
    """v9.6: 토스 복붙형 XLSX 첨부파일을 후보군으로 변환합니다.
    - 종목명 기준으로 KRX 종목코드를 매칭합니다.
    - 정리본/매칭실패 파일을 자동 저장합니다.
    """
    global toss_xlsx_cleaned_df, toss_xlsx_unmatched_df
    toss_xlsx_cleaned_df = pd.DataFrame()
    toss_xlsx_unmatched_df = pd.DataFrame()

    if not CONFIG.get('ENABLE_TOSS_MANUAL_XLSX', True):
        return pd.DataFrame()

    path = find_toss_manual_xlsx_file(path)
    max_items = int(max_items or CONFIG.get('TOSS_MAX_ITEMS', 80))
    if not path:
        print('ℹ️ 토스 수동후보 XLSX 파일 없음 → CSV/Selenium/NAVER fallback 확인')
        return pd.DataFrame()

    try:
        raw_df = parse_toss_manual_xlsx_raw(path)
    except Exception as e:
        print(f'⚠️ 토스 수동후보 XLSX 읽기 실패: {e}')
        return pd.DataFrame()

    if raw_df.empty:
        print(f'⚠️ 토스 수동후보 XLSX에서 후보를 찾지 못함: {path}')
        return pd.DataFrame()

    krx = get_krx_listing_for_match()
    if krx.empty:
        print('⚠️ KRX 상장목록을 가져오지 못해 토스 XLSX 후보 매칭 실패')
        return pd.DataFrame()
    krx_by_name = krx.set_index('매칭명')

    matched_rows = []
    unmatched_rows = []
    for _, r in raw_df.iterrows():
        name_norm = normalize_name_for_match(r.get('종목명'))
        matched = None
        if name_norm in krx_by_name.index:
            matched = krx_by_name.loc[name_norm]
            if isinstance(matched, pd.DataFrame):
                matched = matched.iloc[0]
        if matched is None:
            rr = r.to_dict()
            rr['매칭상태'] = '종목코드확인필요'
            unmatched_rows.append(rr)
            continue
        rr = r.to_dict()
        rr['종목코드'] = normalize_code_v95(matched.get('종목코드', ''))
        rr['시장구분'] = matched.get('시장구분', '')
        rr['현재가'] = rr.get('토스현재가', np.nan)
        rr['등락률'] = rr.get('토스등락률', np.nan)
        rr['거래량'] = rr.get('토스거래량', np.nan)
        if not pd.isna(rr.get('토스현재가')) and not pd.isna(rr.get('토스거래량')):
            turnover_eok = float(rr['토스현재가']) * float(rr['토스거래량']) / 100000000
            rr['거래대금(억)'] = round(turnover_eok, 1)
            rr['거래대금'] = turnover_eok * 100
        rr['매칭상태'] = '매칭완료'
        matched_rows.append(rr)

    cleaned = pd.DataFrame(matched_rows)
    unmatched = pd.DataFrame(unmatched_rows)
    if not cleaned.empty:
        cleaned = cleaned.drop_duplicates(subset=['종목코드']).head(max_items).reset_index(drop=True)
    toss_xlsx_cleaned_df = cleaned.copy()
    toss_xlsx_unmatched_df = unmatched.copy()

    if CONFIG.get('TOSS_PARSE_SAVE_CLEANED', True):
        try:
            if not cleaned.empty:
                cleaned.to_csv(CONFIG.get('TOSS_CLEANED_CSV_FILE', 'TOSS_수동후보_정리본.csv'), index=False, encoding='utf-8-sig')
                print(f"🧾 토스 XLSX 정리본 저장: {CONFIG.get('TOSS_CLEANED_CSV_FILE', 'TOSS_수동후보_정리본.csv')} / {len(cleaned)}개")
            if not unmatched.empty:
                unmatched.to_csv(CONFIG.get('TOSS_UNMATCHED_CSV_FILE', 'TOSS_매칭실패.csv'), index=False, encoding='utf-8-sig')
                print(f"⚠️ 토스 XLSX 매칭실패 저장: {CONFIG.get('TOSS_UNMATCHED_CSV_FILE', 'TOSS_매칭실패.csv')} / {len(unmatched)}개")
        except Exception as e:
            print(f'⚠️ 토스 XLSX 정리본 저장 실패: {e}')

    if cleaned.empty:
        print('⚠️ 토스 XLSX 후보가 모두 종목코드 매칭 실패 → CSV/Selenium/NAVER fallback 확인')
        return pd.DataFrame()

    cleaned = enrich_candidate_market_data(cleaned)
    print(f'✅ 토스 XLSX 후보 추출 완료: {len(cleaned)}개 / 파일: {os.path.basename(path)}')
    return cleaned.reset_index(drop=True)

def get_toss_manual_candidates(path=None, max_items=None):
    """토스 스크리너에서 직접 복사한 후보 CSV를 읽어 1차 후보군으로 변환합니다.
    CSV는 종목명만 있어도 되지만 종목코드가 있으면 매칭 정확도가 높습니다.
    """
    if not CONFIG.get('ENABLE_TOSS_MANUAL_CSV', True):
        return pd.DataFrame()
    path = path or CONFIG.get('TOSS_MANUAL_FILE', 'TOSS_수동후보.csv')
    max_items = int(max_items or CONFIG.get('TOSS_MAX_ITEMS', 80))

    if not os.path.exists(path):
        if CONFIG.get('TOSS_CREATE_TEMPLATE_IF_MISSING', True):
            create_toss_manual_template(path)
        print(f'ℹ️ 토스 수동후보 CSV 없음 또는 템플릿만 생성됨: {path}')
        return pd.DataFrame()

    try:
        manual = pd.read_csv(path, dtype=str).fillna('')
    except UnicodeDecodeError:
        manual = pd.read_csv(path, dtype=str, encoding='cp949').fillna('')
    except Exception as e:
        print(f'⚠️ 토스 수동후보 CSV 읽기 실패: {e}')
        return pd.DataFrame()

    if manual.empty:
        return pd.DataFrame()

    # 컬럼명 유연 처리
    colmap = {c.strip(): c for c in manual.columns}
    name_col = colmap.get('종목명') or colmap.get('name') or colmap.get('Name')
    code_col = colmap.get('종목코드') or colmap.get('code') or colmap.get('Code') or colmap.get('symbol') or colmap.get('Symbol')
    screener_col = colmap.get('스크리너명') or colmap.get('screen') or colmap.get('Screener')
    memo_col = colmap.get('메모') or colmap.get('memo') or colmap.get('Memo')

    krx = get_krx_listing_for_match()
    if krx.empty:
        return pd.DataFrame()
    krx_by_code = krx.set_index('종목코드')
    krx_by_name = krx.set_index('매칭명')

    rows = []
    for _, r in manual.iterrows():
        raw_code = r.get(code_col, '') if code_col else ''
        raw_name = r.get(name_col, '') if name_col else ''
        code = normalize_code_v95(raw_code)
        name_norm = normalize_name_for_match(raw_name)

        matched = None
        if code and code in krx_by_code.index:
            matched = krx_by_code.loc[code]
        elif name_norm and name_norm in krx_by_name.index:
            matched = krx_by_name.loc[name_norm]
            if isinstance(matched, pd.DataFrame):
                matched = matched.iloc[0]
            code = matched.name if matched.name and re.match(r'^\d{6}$', str(matched.name)) else matched.get('종목코드', '')
        else:
            # 부분 매칭 보조. 너무 짧은 이름은 오탐 방지를 위해 제외.
            if len(name_norm) >= 3:
                hits = krx[krx['매칭명'].str.contains(re.escape(name_norm), na=False)]
                if len(hits) == 1:
                    matched = hits.iloc[0]
                    code = matched['종목코드']

        if matched is None:
            continue
        if isinstance(matched, pd.Series):
            stock_name = matched.get('종목명', raw_name)
            market = matched.get('시장구분', '')
            if not code:
                code = matched.get('종목코드', '')
        else:
            stock_name = raw_name
            market = ''

        rows.append({
            '종목명': stock_name,
            '종목코드': normalize_code_v95(code),
            '시장구분': market,
            '후보출처': 'TOSS_CSV',
            '스크리너명': (r.get(screener_col, '') if screener_col else '') or CONFIG.get('TOSS_SCREENER_NAME', '토스 수동후보'),
            '스크리너URL': CONFIG.get('TOSS_SCREENER_URL', ''),
            '후보소스메모': (r.get(memo_col, '') if memo_col else '') or '토스 스크리너 수동 입력 후보. 내부 안전/재무/추세/백테스트를 다시 통과해야 함'
        })
        if len(rows) >= max_items:
            break

    out = pd.DataFrame(rows).drop_duplicates(subset=['종목코드']) if rows else pd.DataFrame()
    if out.empty:
        print(f'⚠️ 토스 수동후보 CSV에서 KRX 매칭 후보 없음: {path}')
        return out
    out = enrich_candidate_market_data(out)
    print(f'✅ 토스 수동후보 CSV 반영: {len(out)}개 / 파일: {path}')
    return out.reset_index(drop=True)


def parse_toss_api_json(payload):
    """향후 토스 공식 API 응답을 후보군 형태로 변환하기 위한 유연 파서.
    응답 구조가 확정되면 이 함수의 items/field 매핑만 조정하면 됩니다.
    """
    if payload is None:
        return pd.DataFrame()
    if isinstance(payload, list):
        items = payload
    elif isinstance(payload, dict):
        items = payload.get('stocks') or payload.get('items') or payload.get('data') or payload.get('result') or []
        if isinstance(items, dict):
            items = items.get('stocks') or items.get('items') or []
    else:
        items = []

    rows = []
    for item in items:
        if not isinstance(item, dict):
            continue
        code = normalize_code_v95(item.get('code') or item.get('stockCode') or item.get('symbol') or item.get('ticker') or '')
        name = item.get('name') or item.get('stockName') or item.get('korName') or item.get('displayName') or ''
        if not code and not name:
            continue
        rows.append({'종목명': name, '종목코드': code, '시장구분': item.get('market', ''), '후보출처': 'TOSS_API',
                     '스크리너명': CONFIG.get('TOSS_SCREENER_NAME', '토스 API 후보'), '스크리너URL': CONFIG.get('TOSS_API_ENDPOINT', ''),
                     '후보소스메모': '토스 공식 API 후보. 내부 안전/재무/추세/백테스트를 다시 통과해야 함'})
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    # 코드가 비어 있으면 종목명으로 KRX 매칭
    krx = get_krx_listing_for_match()
    if not krx.empty:
        krx_by_name = krx.set_index('매칭명')
        for idx, r in df.iterrows():
            if normalize_code_v95(r.get('종목코드')):
                df.at[idx, '종목코드'] = normalize_code_v95(r.get('종목코드'))
                continue
            name_norm = normalize_name_for_match(r.get('종목명'))
            if name_norm in krx_by_name.index:
                matched = krx_by_name.loc[name_norm]
                if isinstance(matched, pd.DataFrame):
                    matched = matched.iloc[0]
                df.at[idx, '종목코드'] = matched.get('종목코드', '')
                df.at[idx, '시장구분'] = matched.get('시장구분', '')
    df = df[df['종목코드'].astype(str).str.match(r'^\d{6}$', na=False)].drop_duplicates(subset=['종목코드'])
    return enrich_candidate_market_data(df).reset_index(drop=True)


def get_toss_api_candidates():
    """향후 토스 공식 API가 제공될 때 사용할 후보 수집기.
    CONFIG['ENABLE_TOSS_API']=True, CONFIG['TOSS_API_ENDPOINT'] 설정 시 시도합니다.
    """
    if not CONFIG.get('ENABLE_TOSS_API', False):
        return pd.DataFrame()
    endpoint = str(CONFIG.get('TOSS_API_ENDPOINT', '') or '').strip()
    if not endpoint:
        print('ℹ️ TOSS API endpoint 미설정 → API 수집 생략')
        return pd.DataFrame()
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}
    key = str(CONFIG.get('TOSS_API_KEY', '') or '').strip()
    if key:
        headers['Authorization'] = f'Bearer {key}'
    try:
        print(f'📡 TOSS API 후보 수집 시도: {endpoint}')
        res = requests.get(endpoint, headers=headers, timeout=int(CONFIG.get('TOSS_API_TIMEOUT', 10)))
        res.raise_for_status()
        df = parse_toss_api_json(res.json())
        print(f'✅ TOSS API 후보 수집 완료: {len(df)}개') if not df.empty else print('⚠️ TOSS API 응답에서 후보를 찾지 못함')
        return df
    except Exception as e:
        print(f'⚠️ TOSS API 후보 수집 실패: {e}')
        return pd.DataFrame()


def get_toss_screener_stocks(url=None, max_items=None):
    """토스증권 스크리너 URL에서 표시 종목을 가져와 1차 후보군으로 변환합니다.
    - 토스 후보는 최종 추천이 아니라 1차 후보입니다.
    - Selenium/브라우저 환경이 불안정하면 빈 DataFrame을 반환하고, 수동 CSV/NAVER로 대체할 수 있습니다.
    """
    if not CONFIG.get('ENABLE_TOSS_SELENIUM', True):
        print('ℹ️ 토스 Selenium 수집 비활성화')
        return pd.DataFrame()

    url = url or CONFIG.get('TOSS_SCREENER_URL')
    max_items = int(max_items or CONFIG.get('TOSS_MAX_ITEMS', 80))
    print(f'📡 토스 스크리너 후보 수집: {url}')

    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
    except Exception as e:
        print(f'⚠️ selenium import 실패: {e}')
        return pd.DataFrame()

    driver = None
    try:
        options = webdriver.ChromeOptions()
        options.add_argument('--headless=new')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,2600')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('--lang=ko-KR')
        options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        time.sleep(float(CONFIG.get('TOSS_WAIT_SEC', 4)))

        for _ in range(int(CONFIG.get('TOSS_SCROLL_COUNT', 4))):
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(1.0)

        raw_texts = []
        elements = driver.find_elements(By.XPATH, "//a[contains(@href, '/stocks/')] | //*[@role='link']")
        for el in elements:
            txt = (el.text or '').strip()
            href = el.get_attribute('href') or ''
            if txt:
                raw_texts.append(txt)
            if href and '/stocks/' in href:
                raw_texts.append(href)

        page_text = driver.find_element(By.TAG_NAME, 'body').text
        raw_texts.append(page_text)

        if '지원하지 않는 브라우저' in page_text or '지원하지 않는 브라우저예요' in page_text:
            print('⚠️ 토스 페이지가 현재 실행환경을 지원하지 않는 브라우저로 판정했습니다. 수동 CSV fallback을 사용합니다.')
            if CONFIG.get('TOSS_DEBUG_SAVE_ON_FAIL', True):
                try:
                    with open(CONFIG.get('TOSS_DEBUG_HTML_FILE', 'toss_debug_page.html'), 'w', encoding='utf-8') as f:
                        f.write(driver.page_source)
                    print(f"🧪 토스 디버그 HTML 저장: {CONFIG.get('TOSS_DEBUG_HTML_FILE', 'toss_debug_page.html')}")
                except Exception:
                    pass
            return pd.DataFrame()

    except Exception as e:
        print(f'⚠️ 토스 스크리너 수집 실패: {e}')
        try:
            if driver and CONFIG.get('TOSS_DEBUG_SAVE_ON_FAIL', True):
                with open(CONFIG.get('TOSS_DEBUG_HTML_FILE', 'toss_debug_page.html'), 'w', encoding='utf-8') as f:
                    f.write(driver.page_source)
                print(f"🧪 토스 디버그 HTML 저장: {CONFIG.get('TOSS_DEBUG_HTML_FILE', 'toss_debug_page.html')}")
        except Exception:
            pass
        return pd.DataFrame()
    finally:
        try:
            if driver:
                driver.quit()
        except Exception:
            pass

    krx = get_krx_listing_for_match()
    if krx.empty:
        return pd.DataFrame()

    blob = '\n'.join(raw_texts)
    normalized_blob = normalize_name_for_match(blob)

    # 긴 종목명 우선 매칭해 부분 문자열 오탐을 줄입니다.
    candidates = []
    for _, r in krx.sort_values('종목명', key=lambda s: s.str.len(), ascending=False).iterrows():
        name_norm = r['매칭명']
        code = r['종목코드']
        if not name_norm:
            continue
        # 페이지 본문에 종목명이 있거나 href 등에 6자리 코드가 직접 노출되는 경우를 모두 허용
        if name_norm in normalized_blob or code in blob:
            candidates.append({
                '종목명': r['종목명'],
                '종목코드': code,
                '시장구분': r.get('시장구분', ''),
                '후보출처': 'TOSS_AUTO',
                '스크리너명': CONFIG.get('TOSS_SCREENER_NAME', '토스 스크리너'),
                '스크리너URL': url,
                '후보소스메모': '토스 스크리너 자동수집 1차 후보. 내부 안전/재무/추세/백테스트를 다시 통과해야 함'
            })
        if len(candidates) >= max_items:
            break

    toss_df = pd.DataFrame(candidates).drop_duplicates(subset=['종목코드']) if candidates else pd.DataFrame()
    if toss_df.empty:
        print('⚠️ 토스 스크리너에서 KRX 종목명을 매칭하지 못했습니다. 수동 CSV fallback을 확인합니다.')
        return toss_df

    toss_df = enrich_candidate_market_data(toss_df)
    print(f'✅ 토스 자동 후보 수집 완료: {len(toss_df)}개')
    return toss_df.reset_index(drop=True)


def merge_candidate_sources(dfs):
    """여러 후보군을 합치고 중복 종목의 출처를 병합합니다."""
    valid = [d.copy() for d in dfs if d is not None and not d.empty]
    if not valid:
        return pd.DataFrame()
    merged = pd.concat(valid, ignore_index=True)
    merged['종목코드'] = merged['종목코드'].apply(normalize_code_v95)
    merged = merged[merged['종목코드'].str.match(r'^\d{6}$', na=False)].copy()

    # 숫자 컬럼은 가능한 값 중 첫 번째 유효값을 사용
    rows = []
    for code, g in merged.groupby('종목코드', sort=False):
        base = g.iloc[0].copy()
        base['후보출처'] = ','.join(sorted(set(g['후보출처'].dropna().astype(str))))
        base['스크리너명'] = ','.join(sorted(set(g.get('스크리너명', pd.Series(dtype=str)).dropna().astype(str))))
        base['스크리너URL'] = ','.join(sorted(set(g.get('스크리너URL', pd.Series(dtype=str)).dropna().astype(str))))
        base['후보소스메모'] = ' / '.join(sorted(set(g.get('후보소스메모', pd.Series(dtype=str)).dropna().astype(str))))
        for col in ['현재가', '거래량', '거래대금', '등락률', '거래대금(억)']:
            vals = pd.to_numeric(g[col], errors='coerce') if col in g.columns else pd.Series(dtype=float)
            vals = vals.dropna()
            if len(vals):
                base[col] = vals.iloc[0]
        rows.append(base)
    return pd.DataFrame(rows).reset_index(drop=True)


def collect_toss_candidates_v95():
    """v9.6 토스 후보 수집 순서: API → XLSX/CSV 수동 후보 → Selenium → 보조 fallback.
    토스 페이지 자동수집이 막혀도 첨부파일만 있으면 TOSS_XLSX 후보로 분석을 진행합니다.
    """
    toss_dfs = []

    api_df = get_toss_api_candidates()
    if not api_df.empty:
        toss_dfs.append(api_df)

    xlsx_first = bool(CONFIG.get('TOSS_MANUAL_XLSX_FIRST', True))
    manual_first = bool(CONFIG.get('TOSS_MANUAL_FIRST', False))

    if xlsx_first:
        xlsx_df = get_toss_manual_xlsx_candidates()
        if not xlsx_df.empty:
            toss_dfs.append(xlsx_df)

    if manual_first:
        manual_df = get_toss_manual_candidates()
        if not manual_df.empty:
            toss_dfs.append(manual_df)

    selenium_df = get_toss_screener_stocks(CONFIG.get('TOSS_SCREENER_URL'), CONFIG.get('TOSS_MAX_ITEMS'))
    if not selenium_df.empty:
        toss_dfs.append(selenium_df)

    if not xlsx_first:
        xlsx_df = get_toss_manual_xlsx_candidates()
        if not xlsx_df.empty:
            toss_dfs.append(xlsx_df)

    if not manual_first:
        manual_df = get_toss_manual_candidates()
        if not manual_df.empty:
            toss_dfs.append(manual_df)

    return merge_candidate_sources(toss_dfs)


def build_candidate_universe():
    """v9.6 통합 후보군 생성기.
    CONFIG['UNIVERSE_SOURCE'] = TOSS / NAVER / HYBRID
    TOSS는 API/수동XLSX/Selenium/수동CSV를 순차적으로 시도하고, 실패하면 설정에 따라 NAVER로 대체합니다.
    """
    source = str(CONFIG.get('UNIVERSE_SOURCE', 'HYBRID')).upper().strip()
    dfs = []

    toss_df = pd.DataFrame()
    if source in ['TOSS', 'HYBRID']:
        toss_df = collect_toss_candidates_v95()
        if not toss_df.empty:
            dfs.append(toss_df)
        elif source == 'TOSS' and CONFIG.get('FALLBACK_TO_NAVER_WHEN_TOSS_FAIL', True):
            print('↪️ 토스 후보 수집 실패/API·수동CSV 없음 → 네이버 거래량 후보로 자동 대체')
            dfs.append(get_naver_quant_universe())

    if source in ['NAVER', 'HYBRID']:
        dfs.append(get_naver_quant_universe())

    total = merge_candidate_sources(dfs)
    if total.empty:
        return total

    total = enrich_candidate_market_data(total)
    total['후보소스설정'] = source
    return total

print('✅ 공통 함수 + v9.6 토스 첨부파일 추출/후보소스 통합 함수 준비 완료')


✅ 공통 함수 + v9.6 토스 첨부파일 추출/후보소스 통합 함수 준비 완료


In [6]:
# 2-1. v8.5 안전필터 / 기업가치 필터
# 핵심 철학: “오를 가능성이 있는 종목”을 찾기 전에 “피해야 할 종목”을 먼저 제거합니다.

try:
    from pykrx import stock as krx_stock
    PYKRX_AVAILABLE = True
except Exception as e:
    PYKRX_AVAILABLE = False
    print(f'⚠️ pykrx 사용 불가: {e}')


def safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def normalize_code(code):
    try:
        return str(code).replace('.0', '').zfill(6)
    except Exception:
        return ''


def read_manual_market_alerts(path=None):
    """수동 시장경보 파일을 읽습니다.
    파일명 기본값: 시장경보_수동입력.csv
    컬럼 예시: 종목코드, 시장경보, 사유
    - 한국거래소/KIND에서 직접 확인한 종목을 넣어두면 가장 확실합니다.
    - 파일이 없으면 템플릿을 만들어두고, 분석은 '미확인'으로 진행합니다.
    """
    path = path or CONFIG['MARKET_ALERT_MANUAL_FILE']
    if not os.path.exists(path):
        template = pd.DataFrame([
            {'종목코드': '000000', '시장경보': '예: 투자경고 / 투자주의 / 정상', '사유': 'KRX/KIND에서 확인 후 직접 입력'},
        ])
        try:
            template.to_csv(path, index=False, encoding='utf-8-sig')
            print(f'ℹ️ 시장경보 수동입력 템플릿 생성: {path}')
        except Exception:
            pass
        return pd.DataFrame(columns=['종목코드', '시장경보', '사유'])

    try:
        df = pd.read_csv(path, dtype={'종목코드': str})
        if '종목코드' not in df.columns:
            return pd.DataFrame(columns=['종목코드', '시장경보', '사유'])
        df['종목코드'] = df['종목코드'].apply(normalize_code)
        if '시장경보' not in df.columns:
            df['시장경보'] = '미확인'
        if '사유' not in df.columns:
            df['사유'] = ''
        return df[['종목코드', '시장경보', '사유']].drop_duplicates('종목코드')
    except Exception as e:
        print(f'⚠️ 시장경보 수동파일 읽기 실패: {e}')
        return pd.DataFrame(columns=['종목코드', '시장경보', '사유'])


def find_latest_krx_fundamental_date(max_back_days=20):
    """pykrx가 조회 가능한 최근 날짜를 찾습니다."""
    if not PYKRX_AVAILABLE:
        return None
    for offset in range(max_back_days + 1):
        dt = (RUN_AT - timedelta(days=offset)).strftime('%Y%m%d')
        try:
            df = krx_stock.get_market_fundamental_by_ticker(dt, market='ALL')
            if df is not None and not df.empty:
                return dt
        except Exception:
            continue
    return None


def get_latest_fundamentals(codes):
    """PER/PBR/EPS/BPS/시가총액을 가져옵니다.
    - PER/PBR/EPS/BPS: pykrx get_market_fundamental_by_ticker
    - 시가총액: pykrx get_market_cap_by_ticker
    - ROE는 EPS/BPS로 단순 환산한 참고값입니다.
    """
    codes = [normalize_code(c) for c in codes]
    base = pd.DataFrame({'종목코드': codes}).drop_duplicates()
    if not PYKRX_AVAILABLE:
        base[['PER','PBR','EPS','BPS','ROE(추정)','시가총액(억)']] = np.nan
        base['재무데이터기준일'] = 'pykrx 사용불가'
        return base

    krx_date = find_latest_krx_fundamental_date()
    if krx_date is None:
        base[['PER','PBR','EPS','BPS','ROE(추정)','시가총액(억)']] = np.nan
        base['재무데이터기준일'] = '조회실패'
        return base

    try:
        funda = krx_stock.get_market_fundamental_by_ticker(krx_date, market='ALL').copy()
        funda.index = funda.index.astype(str).str.zfill(6)
        funda = funda.rename_axis('종목코드').reset_index()
        keep_cols = ['종목코드'] + [c for c in ['BPS', 'PER', 'PBR', 'EPS', 'DIV', 'DPS'] if c in funda.columns]
        funda = funda[keep_cols]
    except Exception as e:
        print(f'⚠️ 재무지표 조회 실패: {e}')
        funda = pd.DataFrame({'종목코드': codes})

    try:
        cap = krx_stock.get_market_cap_by_ticker(krx_date, market='ALL').copy()
        cap.index = cap.index.astype(str).str.zfill(6)
        cap = cap.rename_axis('종목코드').reset_index()
        cap_col = '시가총액' if '시가총액' in cap.columns else None
        if cap_col:
            cap = cap[['종목코드', cap_col]].rename(columns={cap_col: '시가총액'})
        else:
            cap = pd.DataFrame({'종목코드': codes, '시가총액': np.nan})
    except Exception:
        cap = pd.DataFrame({'종목코드': codes, '시가총액': np.nan})

    out = base.merge(funda, on='종목코드', how='left').merge(cap, on='종목코드', how='left')
    for c in ['PER', 'PBR', 'EPS', 'BPS', '시가총액']:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out['ROE(추정)'] = np.where(out['BPS'].abs() > 0, out['EPS'] / out['BPS'] * 100, np.nan)
    out['시가총액(억)'] = out['시가총액'] / 100000000
    out['재무데이터기준일'] = f'{krx_date[:4]}-{krx_date[4:6]}-{krx_date[6:]}'
    return out[['종목코드','PER','PBR','EPS','BPS','ROE(추정)','시가총액(억)','재무데이터기준일']]


def classify_alert(alert_text):
    text = str(alert_text) if alert_text is not None else '미확인'
    if text.strip() == '' or text == 'nan':
        text = '미확인'
    if re.search(CONFIG['EXCLUDE_ALERT_KEYWORDS'], text):
        return '매수제외', '시장경보/거래위험'
    if re.search(CONFIG['WATCH_ALERT_KEYWORDS'], text):
        return '소액관찰', '투자주의·지정예고 등 고위험 관찰'
    if '미확인' in text:
        return ('확인필요', '시장경보 자동확인 불가') if CONFIG['ALLOW_UNKNOWN_ALERT'] else ('매수제외', '시장경보 미확인')
    return '정상', ''


def evaluate_financial_grade(row):
    """KRX PER/PBR/ROE 기반 1차 재무판정 v8.9
    v8.5에서는 PER/PBR/ROE/시가총액이 비어 있을 때 모두 감점되어 정상 종목도 재무위험처럼 보일 수 있었습니다.
    v8.9은 데이터부족과 실제 위험을 분리합니다. 데이터가 부족하면 '정보부족'으로 표시하고, DART 재무안전 필터가 있으면 그 판단을 더 우선합니다.
    """
    per = safe_float(row.get('PER'))
    pbr = safe_float(row.get('PBR'))
    roe = safe_float(row.get('ROE(추정)'))
    eps = safe_float(row.get('EPS'))
    mcap = safe_float(row.get('시가총액(억)'))

    known_items = sum([not pd.isna(per), not pd.isna(pbr), not pd.isna(roe), not pd.isna(mcap)])
    if known_items <= 1:
        return pd.Series({
            '재무점수': 55,
            '재무등급': '정보부족',
            '가치부담': '확인필요',
            '재무위험사유': 'KRX PER/PBR/ROE/시가총액 데이터 부족 — 위험 판정이 아니라 추가확인 대상'
        })

    score = 60
    issues = []

    # PER은 단기/스윙에서 매수 신호가 아니라 위험 필터입니다.
    # PER 미제공은 위험으로 단정하지 않고, EPS가 음수/0으로 확인될 때만 적자 가능성으로 감점합니다.
    if pd.isna(per):
        issues.append('PER 확인불가')
    elif per <= 0:
        if not pd.isna(eps) and eps <= 0:
            score -= 12
            issues.append('EPS 음수/0으로 PER 산출불가 — 적자 가능성')
        else:
            score -= 4
            issues.append('PER 산출불가')
    elif per > CONFIG['MAX_ACCEPTABLE_PER']:
        score -= 10
        issues.append(f'PER {per:.1f} 고평가 부담')
    elif per <= 20:
        score += 5

    if pd.isna(pbr):
        issues.append('PBR 확인불가')
    elif pbr <= 0:
        score -= 6
        issues.append('PBR 0 이하')
    elif pbr > CONFIG['MAX_ACCEPTABLE_PBR']:
        score -= 10
        issues.append(f'PBR {pbr:.1f} 가치부담')
    elif pbr <= 2:
        score += 4

    if pd.isna(roe):
        issues.append('ROE 확인불가')
    elif roe < CONFIG['MIN_ACCEPTABLE_ROE']:
        score -= 10
        issues.append(f'ROE {roe:.1f}% 수익성 부담')
    elif roe >= 8:
        score += 7

    if pd.isna(mcap):
        issues.append('시가총액 확인불가')
    elif mcap < CONFIG['MIN_MARKET_CAP_EOK']:
        score -= 10
        issues.append(f'시가총액 {mcap:.0f}억 소형주 위험')
    elif mcap >= 3000:
        score += 4

    score = int(np.clip(score, 0, 100))
    if score >= 75:
        grade = 'A'
    elif score >= 60:
        grade = 'B'
    elif score >= 45:
        grade = 'C'
    else:
        grade = 'D'

    if grade in ['A', 'B']:
        burden = '낮음' if grade == 'A' else '보통'
    elif grade == 'C':
        burden = '높음'
    else:
        burden = '매우높음'

    return pd.Series({
        '재무점수': score,
        '재무등급': grade,
        '가치부담': burden,
        '재무위험사유': ' / '.join(issues) if issues else '특이사항 낮음'
    })


def build_safety_table(codes_df):
    """후보군에 붙일 안전필터 테이블 생성."""
    if codes_df is None or len(codes_df) == 0:
        return pd.DataFrame()
    base = codes_df[['종목코드']].copy()
    base['종목코드'] = base['종목코드'].apply(normalize_code)
    base = base.drop_duplicates('종목코드')

    manual_alerts = read_manual_market_alerts()
    fundamentals = get_latest_fundamentals(base['종목코드'].tolist()) if CONFIG['ENABLE_FUNDAMENTAL_FILTER'] else base.copy()

    out = base.merge(manual_alerts, on='종목코드', how='left').merge(fundamentals, on='종목코드', how='left')
    out['시장경보'] = out['시장경보'].fillna('미확인')
    out['시장경보사유'] = out.get('사유', '').fillna('') if '사유' in out.columns else ''

    alert_eval = out['시장경보'].apply(lambda x: pd.Series(classify_alert(x), index=['안전필터판정', '안전필터사유']))
    out = pd.concat([out, alert_eval], axis=1)

    fin_eval = out.apply(evaluate_financial_grade, axis=1)
    out = pd.concat([out, fin_eval], axis=1)

    # 재무등급 D를 완전 제외할지, 관찰로 남길지 선택 가능
    if CONFIG.get('EXCLUDE_FINANCIAL_GRADE_D', False):
        mask_d = out['재무등급'].eq('D') & ~out['안전필터판정'].isin(['매수제외'])
        out.loc[mask_d, '안전필터판정'] = '추천제한'
        out.loc[mask_d, '안전필터사유'] = out.loc[mask_d, '안전필터사유'].astype(str) + ' / 재무등급 D'

    out['최종위험메모'] = (
        '시장경보=' + out['시장경보'].astype(str) +
        ' / 안전판정=' + out['안전필터판정'].astype(str) +
        ' / 재무등급=' + out['재무등급'].astype(str) +
        ' / 가치부담=' + out['가치부담'].astype(str) +
        ' / 사유=' + out['재무위험사유'].astype(str)
    )
    return out.drop(columns=[c for c in ['사유'] if c in out.columns])


def apply_safety_score_adjustment(base_score, row):
    """기술점수에 시장경보/재무등급/DART 재무안전 보정을 적용합니다."""
    score = float(base_score)
    reasons = []
    verdict = str(row.get('안전필터판정', '확인필요'))
    grade = str(row.get('재무등급', '정보부족'))
    dart_verdict = str(row.get('재무위험판정', ''))
    dart_grade = str(row.get('재무안전등급', ''))

    if verdict in ['매수제외', '추천제외']:
        score -= 100
        reasons.append(verdict)
    elif verdict == '소액관찰':
        score -= 20
        reasons.append('시장경보 소액관찰')
    elif verdict == '확인필요':
        score -= 5
        reasons.append('시장경보 확인필요')

    # KRX PER/PBR/ROE 1차 필터: 정보부족은 위험으로 단정하지 않습니다.
    if grade == 'A':
        score += 6
    elif grade == 'B':
        score += 2
    elif grade == 'C':
        score -= 5
        reasons.append('재무등급 C')
    elif grade == 'D':
        score -= 14
        reasons.append('재무등급 D')
    elif grade in ['정보부족', '데이터부족', 'nan', 'None', '']:
        score -= 1
        reasons.append('KRX 재무정보 부족')

    # DART 필터가 있으면 실제 영업이익/순이익/부채비율 기반 판단을 더 우선 반영합니다.
    if dart_verdict == '매수제외':
        score -= 100
        reasons.append('DART 매수제외')
    elif dart_verdict == '재무위험관찰':
        score -= 18
        reasons.append('DART 재무위험관찰')
    elif dart_verdict == '재무주의':
        score -= 8
        reasons.append('DART 재무주의')
    elif dart_verdict == '정상':
        if dart_grade == 'A':
            score += 5
        elif dart_grade == 'B':
            score += 2

    return int(np.clip(score, 0, 100)), ' / '.join(reasons) if reasons else '보정 특이사항 낮음'


def judgment_text_v85(score, rsi, trix_turning, row):
    verdict = str(row.get('안전필터판정', '확인필요'))
    grade = str(row.get('재무등급', '정보부족'))
    dart_verdict = str(row.get('재무위험판정', ''))

    if verdict in ['매수제외', '추천제외'] or dart_verdict == '매수제외':
        return '⛔ 매수 제외'
    if '관망' in market_mode:
        return '⛔ 관망 우선'
    if verdict == '소액관찰':
        return '⚠️ 소액 관찰'
    if dart_verdict in ['재무위험관찰', '재무주의']:
        return '⚠️ 재무확인 필요'
    if grade == 'D':
        return '⚠️ 재무위험 관찰'
    if score >= 80 and rsi <= 55 and trix_turning and grade in ['A', 'B', 'C', '정보부족', '데이터부족']:
        return '🌟 A급 후보' if grade in ['A', 'B', 'C'] else '✅ 선별 후보(재무확인)'
    if score >= 65 and trix_turning:
        return '✅ 선별 후보' if grade not in ['정보부족', '데이터부족'] else '✅ 선별 후보(재무확인)'
    if rsi >= 70:
        return '🚨 과열 위험'
    return '📊 관망'


# 계좌백테스트용: 과거 특정월 기준 재무 필터 캐시
_fundamental_cache = {}


def get_historical_fundamental_snapshot(signal_date):
    """신호일 이전에 조회 가능한 재무 스냅샷을 월 단위로 캐시합니다.
    6개월 이상 백테스트에서 매일 KRX 요청을 보내지 않도록 월별 첫 조회일만 사용합니다.
    """
    if not (PYKRX_AVAILABLE and CONFIG.get('PORTFOLIO_ENABLE_HISTORICAL_FUNDAMENTAL_FILTER', True)):
        return pd.DataFrame()
    dt = pd.to_datetime(signal_date)
    key = dt.strftime('%Y%m')
    if key in _fundamental_cache:
        return _fundamental_cache[key]
    for offset in range(0, 15):
        q = (dt - timedelta(days=offset)).strftime('%Y%m%d')
        try:
            snap = krx_stock.get_market_fundamental_by_ticker(q, market='ALL').copy()
            if snap is not None and not snap.empty:
                snap.index = snap.index.astype(str).str.zfill(6)
                for c in ['PER', 'PBR', 'EPS', 'BPS']:
                    if c not in snap.columns:
                        snap[c] = np.nan
                snap['ROE(추정)'] = np.where(pd.to_numeric(snap['BPS'], errors='coerce').abs() > 0,
                                            pd.to_numeric(snap['EPS'], errors='coerce') / pd.to_numeric(snap['BPS'], errors='coerce') * 100,
                                            np.nan)
                _fundamental_cache[key] = snap
                return snap
        except Exception:
            continue
    _fundamental_cache[key] = pd.DataFrame()
    return _fundamental_cache[key]


def pass_historical_fundamental_filter(code, signal_date):
    """계좌백테스트 후보가 과거 신호일 기준 재무 최소조건을 통과하는지 확인."""
    if not CONFIG.get('PORTFOLIO_ENABLE_HISTORICAL_FUNDAMENTAL_FILTER', True):
        return True, '재무필터 미사용'
    snap = get_historical_fundamental_snapshot(signal_date)
    code = normalize_code(code)
    if snap.empty or code not in snap.index:
        return CONFIG.get('PORTFOLIO_ALLOW_FUNDAMENTAL_UNKNOWN', True), '재무데이터 부족'
    row = snap.loc[code]
    tmp = pd.Series({
        'PER': row.get('PER', np.nan),
        'PBR': row.get('PBR', np.nan),
        'ROE(추정)': row.get('ROE(추정)', np.nan),
        '시가총액(억)': np.nan
    })
    ev = evaluate_financial_grade(tmp)
    if ev['재무등급'] == 'D':
        return False, f"재무등급 D 제외: {ev['재무위험사유']}"
    if ev['가치부담'] == '매우높음':
        return False, f"가치부담 매우높음 제외: {ev['재무위험사유']}"
    return True, f"재무등급 {ev['재무등급']} / {ev['재무위험사유']}"

print('✅ v8.5 안전필터/기업가치 필터 준비 완료')


KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
✅ v8.5 안전필터/기업가치 필터 준비 완료


In [7]:
# 2-2. v8.9 DART 재무안전 필터 + 영구 캐시 확장
# 이 셀은 v8.5의 build_safety_table/evaluate 계열 함수를 덮어써서 실행됩니다.
# 핵심: PER/PBR/ROE만 보지 않고, DART 기준 영업이익·순이익·부채비율·자본잠식·감사의견을 함께 봅니다.

DART_COLUMNS = [
    '종목코드', 'DART기준연도', '매출액(억)', '영업이익(억)', '당기순이익(억)',
    '자산총계(억)', '부채총계(억)', '자본총계(억)', '부채비율',
    '최근적자여부', '감사의견', 'DART데이터상태', 'DART조회일', 'DART메모'
]


def to_eok_from_dart_amount(value):
    """DART 금액 문자열(원 단위)을 억 원 단위 float로 변환"""
    try:
        if value is None:
            return np.nan
        txt = str(value).replace(',', '').replace(' ', '')
        if txt in ['', '-', 'nan', 'None']:
            return np.nan
        return float(txt) / 100000000
    except Exception:
        return np.nan


def ensure_dart_columns(df):
    df = df.copy()
    for col in DART_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan if col not in ['종목코드', '감사의견', 'DART데이터상태', 'DART조회일', 'DART메모', '최근적자여부'] else ''
    df['종목코드'] = df['종목코드'].apply(normalize_code)
    return df[DART_COLUMNS]


def read_manual_dart_financials(path=None):
    """DART API 키가 없거나 자동조회가 실패할 때 쓰는 수동 재무 입력 파일.
    컬럼 예시:
    종목코드,DART기준연도,매출액(억),영업이익(억),당기순이익(억),자산총계(억),부채총계(억),자본총계(억),부채비율,최근적자여부,감사의견,DART데이터상태,DART메모
    """
    path = path or CONFIG.get('DART_MANUAL_FILE', 'DART_재무수동입력.csv')
    if not os.path.exists(path):
        template = pd.DataFrame([
            {
                '종목코드': '000000',
                'DART기준연도': CONFIG.get('DART_BASE_YEAR', RUN_AT.year - 1),
                '매출액(억)': '',
                '영업이익(억)': '',
                '당기순이익(억)': '',
                '자산총계(억)': '',
                '부채총계(억)': '',
                '자본총계(억)': '',
                '부채비율': '',
                '최근적자여부': '예/아니오',
                '감사의견': '적정/한정/부적정/의견거절',
                'DART데이터상태': '수동입력',
                'DART조회일': REPORT_DATE,
                'DART메모': 'DART 사업보고서 또는 분기보고서 확인 후 직접 입력'
            }
        ])
        try:
            template.to_csv(path, index=False, encoding='utf-8-sig')
            print(f'ℹ️ DART 재무 수동입력 템플릿 생성: {path}')
        except Exception:
            pass
        return ensure_dart_columns(pd.DataFrame(columns=DART_COLUMNS))
    try:
        df = pd.read_csv(path, dtype={'종목코드': str})
        df = ensure_dart_columns(df)
        # 예시 행 제거
        df = df[df['종목코드'].ne('000000')].copy()
        for c in ['매출액(억)', '영업이익(억)', '당기순이익(억)', '자산총계(억)', '부채총계(억)', '자본총계(억)', '부채비율']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        return df.drop_duplicates('종목코드')
    except Exception as e:
        print(f'⚠️ DART 수동파일 읽기 실패: {e}')
        return ensure_dart_columns(pd.DataFrame(columns=DART_COLUMNS))


_dart_corp_cache = None
_dart_fin_cache = {}

_dart_persistent_cache_df = None


def read_dart_persistent_cache(path=None):
    """v8.9: DART API 결과를 CSV로 저장해 반복 호출을 줄입니다."""
    global _dart_persistent_cache_df
    path = path or CONFIG.get('DART_CACHE_FILE', 'DART_재무캐시.csv')
    if _dart_persistent_cache_df is not None:
        return _dart_persistent_cache_df.copy()
    if not CONFIG.get('DART_USE_CACHE', True) or not os.path.exists(path):
        _dart_persistent_cache_df = ensure_dart_columns(pd.DataFrame(columns=DART_COLUMNS))
        return _dart_persistent_cache_df.copy()
    try:
        df = pd.read_csv(path, dtype={'종목코드': str})
        df = ensure_dart_columns(df)
        for c in ['매출액(억)', '영업이익(억)', '당기순이익(억)', '자산총계(억)', '부채총계(억)', '자본총계(억)', '부채비율']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        _dart_persistent_cache_df = df.drop_duplicates(['종목코드', 'DART기준연도'], keep='last')
        return _dart_persistent_cache_df.copy()
    except Exception as e:
        print(f'⚠️ DART 캐시 읽기 실패: {e}')
        _dart_persistent_cache_df = ensure_dart_columns(pd.DataFrame(columns=DART_COLUMNS))
        return _dart_persistent_cache_df.copy()


def get_cached_dart_financial(stock_code, base_year=None):
    """캐시가 유효하면 DART API 호출 없이 반환합니다."""
    if not CONFIG.get('DART_USE_CACHE', True):
        return None
    stock_code = normalize_code(stock_code)
    base_year = int(base_year or CONFIG.get('DART_BASE_YEAR', RUN_AT.year - 1))
    cache = read_dart_persistent_cache()
    if cache.empty:
        return None
    cand = cache[(cache['종목코드'].eq(stock_code)) & (pd.to_numeric(cache['DART기준연도'], errors='coerce').eq(base_year))].copy()
    if cand.empty:
        return None
    row = cand.iloc[-1].to_dict()
    lookup_date = row.get('DART조회일', '')
    try:
        age = (pd.to_datetime(REPORT_DATE) - pd.to_datetime(lookup_date)).days
    except Exception:
        age = 9999
    if age <= int(CONFIG.get('DART_CACHE_TTL_DAYS', 90)):
        row['DART데이터상태'] = 'DART캐시'
        row['DART메모'] = (str(row.get('DART메모', '')) + f' / 캐시사용({lookup_date})').strip(' /')
        return row
    return None


def save_dart_cache_row(row, path=None):
    """성공적으로 조회한 DART 결과를 캐시에 저장합니다."""
    global _dart_persistent_cache_df
    if not CONFIG.get('DART_USE_CACHE', True):
        return
    path = path or CONFIG.get('DART_CACHE_FILE', 'DART_재무캐시.csv')
    try:
        row = dict(row)
        row['종목코드'] = normalize_code(row.get('종목코드', ''))
        row['DART조회일'] = row.get('DART조회일') or REPORT_DATE
        new_df = ensure_dart_columns(pd.DataFrame([row]))
        old_df = read_dart_persistent_cache()
        combined = pd.concat([old_df, new_df], ignore_index=True)
        combined = combined.drop_duplicates(['종목코드', 'DART기준연도'], keep='last')
        combined.to_csv(path, index=False, encoding='utf-8-sig')
        _dart_persistent_cache_df = combined
    except Exception as e:
        print(f'⚠️ DART 캐시 저장 실패: {e}')


def fetch_dart_corp_code_map(api_key=None):
    """OpenDART corpCode.xml을 받아 주식코드→고유번호 매핑 생성.
    API 키가 없으면 빈 DataFrame을 반환합니다.
    """
    global _dart_corp_cache
    api_key = api_key or CONFIG.get('DART_API_KEY', '')
    if _dart_corp_cache is not None:
        return _dart_corp_cache
    if not api_key:
        _dart_corp_cache = pd.DataFrame(columns=['corp_code', 'corp_name', 'stock_code'])
        return _dart_corp_cache
    try:
        url = 'https://opendart.fss.or.kr/api/corpCode.xml'
        r = requests.get(url, params={'crtfc_key': api_key}, timeout=20)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        xml_bytes = z.read('CORPCODE.xml')
        root = ET.fromstring(xml_bytes)
        rows = []
        for item in root.findall('list'):
            stock_code = (item.findtext('stock_code') or '').strip()
            if stock_code:
                rows.append({
                    'corp_code': (item.findtext('corp_code') or '').strip(),
                    'corp_name': (item.findtext('corp_name') or '').strip(),
                    'stock_code': stock_code.zfill(6)
                })
        _dart_corp_cache = pd.DataFrame(rows).drop_duplicates('stock_code')
        print(f'✅ DART 기업코드 매핑 로드: {len(_dart_corp_cache):,}개')
        return _dart_corp_cache
    except Exception as e:
        print(f'⚠️ DART 기업코드 조회 실패: {e}')
        _dart_corp_cache = pd.DataFrame(columns=['corp_code', 'corp_name', 'stock_code'])
        return _dart_corp_cache


def parse_dart_financial_items(items):
    """DART 단일회사 주요계정 목록에서 필요한 항목만 추출"""
    wanted = {
        '매출액': ['매출액', '수익(매출액)', '영업수익'],
        '영업이익': ['영업이익'],
        '당기순이익': ['당기순이익', '당기순이익(손실)', '분기순이익', '반기순이익'],
        '자산총계': ['자산총계'],
        '부채총계': ['부채총계'],
        '자본총계': ['자본총계']
    }
    result = {k: np.nan for k in wanted}
    for it in items:
        name = str(it.get('account_nm', '')).strip()
        amount = it.get('thstrm_amount')
        sj_div = str(it.get('sj_div', ''))
        for key, names in wanted.items():
            if key in result and any(name == n or n in name for n in names):
                # 손익계산서/재무상태표 모두 허용하되 먼저 잡힌 값을 우선
                if pd.isna(result[key]):
                    result[key] = to_eok_from_dart_amount(amount)
    return result


def fetch_dart_financial_one(stock_code, base_year=None):
    """단일 종목 DART 재무 조회. 실패 시 데이터부족으로 반환."""
    api_key = CONFIG.get('DART_API_KEY', '')
    stock_code = normalize_code(stock_code)
    base_year = int(base_year or CONFIG.get('DART_BASE_YEAR', RUN_AT.year - 1))
    cache_key = (stock_code, base_year)
    if cache_key in _dart_fin_cache:
        return _dart_fin_cache[cache_key]

    cached = get_cached_dart_financial(stock_code, base_year)
    if cached is not None:
        _dart_fin_cache[cache_key] = cached
        return cached

    empty = {
        '종목코드': stock_code,
        'DART기준연도': base_year,
        '매출액(억)': np.nan,
        '영업이익(억)': np.nan,
        '당기순이익(억)': np.nan,
        '자산총계(억)': np.nan,
        '부채총계(억)': np.nan,
        '자본총계(억)': np.nan,
        '부채비율': np.nan,
        '최근적자여부': '',
        '감사의견': '',
        'DART데이터상태': 'API키없음' if not api_key else '조회실패',
        'DART조회일': REPORT_DATE,
        'DART메모': ''
    }
    if not api_key:
        _dart_fin_cache[cache_key] = empty
        return empty

    corp_map = fetch_dart_corp_code_map(api_key)
    if corp_map.empty or stock_code not in set(corp_map['stock_code']):
        empty['DART데이터상태'] = '기업코드없음'
        _dart_fin_cache[cache_key] = empty
        return empty
    corp_code = corp_map.loc[corp_map['stock_code'].eq(stock_code), 'corp_code'].iloc[0]

    # 사업보고서 기준. 해당 연도 실패 시 1년 전까지 시도
    for year in [base_year, base_year - 1]:
        try:
            url = 'https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json'
            params = {
                'crtfc_key': api_key,
                'corp_code': corp_code,
                'bsns_year': str(year),
                'reprt_code': CONFIG.get('DART_REPORT_CODE', '11011'),
                'fs_div': CONFIG.get('DART_FS_DIV', 'CFS')
            }
            res = requests.get(url, params=params, timeout=20).json()
            if res.get('status') != '000' or not res.get('list'):
                # 연결 실패 시 별도재무제표도 한번 시도
                params['fs_div'] = 'OFS'
                res = requests.get(url, params=params, timeout=20).json()
            if res.get('status') == '000' and res.get('list'):
                parsed = parse_dart_financial_items(res['list'])
                debt_ratio = np.nan
                if not pd.isna(parsed.get('부채총계')) and not pd.isna(parsed.get('자본총계')) and parsed.get('자본총계') != 0:
                    debt_ratio = parsed.get('부채총계') / parsed.get('자본총계') * 100
                out = {
                    '종목코드': stock_code,
                    'DART기준연도': year,
                    '매출액(억)': parsed.get('매출액'),
                    '영업이익(억)': parsed.get('영업이익'),
                    '당기순이익(억)': parsed.get('당기순이익'),
                    '자산총계(억)': parsed.get('자산총계'),
                    '부채총계(억)': parsed.get('부채총계'),
                    '자본총계(억)': parsed.get('자본총계'),
                    '부채비율': debt_ratio,
                    '최근적자여부': '예' if (safe_float(parsed.get('영업이익')) < 0 or safe_float(parsed.get('당기순이익')) < 0) else '아니오',
                    '감사의견': '',
                    'DART데이터상태': 'DART API',
                    'DART조회일': REPORT_DATE,
                    'DART메모': f"{year}년 {CONFIG.get('DART_FS_DIV','CFS')} 기준"
                }
                _dart_fin_cache[cache_key] = out
                save_dart_cache_row(out)
                return out
        except Exception as e:
            empty['DART메모'] = str(e)[:80]
            continue
    _dart_fin_cache[cache_key] = empty
    return empty


def get_dart_financials(codes):
    """DART API + 수동 CSV를 결합해 재무안전 필터 데이터를 구성합니다.
    수동 CSV에 값이 있으면 API보다 우선합니다.
    """
    codes = [normalize_code(c) for c in codes]
    base = pd.DataFrame({'종목코드': codes}).drop_duplicates()
    manual = read_manual_dart_financials()

    rows = []
    if CONFIG.get('ENABLE_DART_FINANCIAL_FILTER', True):
        for code in base['종목코드']:
            rows.append(fetch_dart_financial_one(code))
            time.sleep(min(CONFIG.get('SLEEP_SEC', 0.12), 0.2))
    api_df = ensure_dart_columns(pd.DataFrame(rows)) if rows else ensure_dart_columns(pd.DataFrame(columns=DART_COLUMNS))

    out = base.merge(api_df, on='종목코드', how='left', suffixes=('', '_api'))
    if manual is not None and not manual.empty:
        out = out.merge(manual, on='종목코드', how='left', suffixes=('', '_manual'))
        for col in DART_COLUMNS:
            if col == '종목코드':
                continue
            manual_col = f'{col}_manual'
            if manual_col in out.columns:
                out[col] = out[manual_col].combine_first(out.get(col))
        out = out[[c for c in out.columns if not c.endswith('_manual') and not c.endswith('_api')]]
    out = ensure_dart_columns(out)
    for c in ['매출액(억)', '영업이익(억)', '당기순이익(억)', '자산총계(억)', '부채총계(억)', '자본총계(억)', '부채비율']:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out.drop_duplicates('종목코드')


def evaluate_dart_financial_safety(row):
    """DART 재무제표 기반 재무안전 점수/등급/판정"""
    status = str(row.get('DART데이터상태', '') or '')
    op = safe_float(row.get('영업이익(억)'))
    ni = safe_float(row.get('당기순이익(억)'))
    equity = safe_float(row.get('자본총계(억)'))
    debt_ratio = safe_float(row.get('부채비율'))
    audit = str(row.get('감사의견', '') or '')

    score = 60
    issues = []
    verdict = '정상'

    if status in ['', 'API키없음', '조회실패', '기업코드없음'] and all(pd.isna(x) for x in [op, ni, equity, debt_ratio]):
        return pd.Series({
            '재무안전점수': 50,
            '재무안전등급': '데이터부족',
            '재무위험판정': '확인필요',
            '재무안전사유': 'DART 재무데이터 부족 또는 API키 미입력'
        })

    if CONFIG.get('EXCLUDE_BAD_AUDIT_OPINION', True) and re.search('한정|부적정|의견거절|범위제한', audit):
        score -= 60
        verdict = '매수제외'
        issues.append(f'감사의견 위험({audit})')

    if CONFIG.get('EXCLUDE_CAPITAL_IMPAIRMENT', True) and not pd.isna(equity) and equity <= 0:
        score -= 50
        verdict = '매수제외'
        issues.append('자본총계 0 이하/자본잠식 위험')

    if not pd.isna(op):
        if op < 0:
            score -= 15
            issues.append(f'영업손실 {op:.0f}억')
        elif op >= 100:
            score += 8
        elif op > 0:
            score += 4
    else:
        score -= 4
        issues.append('영업이익 확인불가')

    if not pd.isna(ni):
        if ni < 0:
            score -= 12
            issues.append(f'순손실 {ni:.0f}억')
        elif ni > 0:
            score += 5
    else:
        score -= 3
        issues.append('당기순이익 확인불가')

    if not pd.isna(debt_ratio):
        if debt_ratio > CONFIG.get('MAX_ACCEPTABLE_DEBT_RATIO', 250):
            score -= 18
            issues.append(f'부채비율 {debt_ratio:.0f}% 과다')
        elif debt_ratio > CONFIG.get('WATCH_DEBT_RATIO', 180):
            score -= 8
            issues.append(f'부채비율 {debt_ratio:.0f}% 주의')
        elif debt_ratio < 100:
            score += 7
    else:
        score -= 3
        issues.append('부채비율 확인불가')

    if verdict != '매수제외':
        if score < 40:
            verdict = '재무위험관찰'
        elif any(('영업손실' in x or '순손실' in x or '부채비율' in x) for x in issues):
            verdict = '재무주의'

    score = int(np.clip(score, 0, 100))
    if score >= 75:
        grade = 'A'
    elif score >= 60:
        grade = 'B'
    elif score >= 45:
        grade = 'C'
    else:
        grade = 'D'

    return pd.Series({
        '재무안전점수': score,
        '재무안전등급': grade,
        '재무위험판정': verdict,
        '재무안전사유': ' / '.join(issues) if issues else 'DART 재무안전 특이사항 낮음'
    })


# v8.9: pykrx 지표 평가도 DART 재무안전 결과와 함께 쓰도록 강화
_original_evaluate_financial_grade_v85 = evaluate_financial_grade


def evaluate_financial_grade(row):
    base = _original_evaluate_financial_grade_v85(row)
    # DART 재무안전 점수를 pykrx 기반 재무점수에 소폭 합산
    dart_score = safe_float(row.get('재무안전점수'))
    if not pd.isna(dart_score):
        score = int(np.clip(base['재무점수'] * 0.65 + dart_score * 0.35, 0, 100))
        if score >= 75:
            grade = 'A'
        elif score >= 60:
            grade = 'B'
        elif score >= 45:
            grade = 'C'
        else:
            grade = 'D'
        base['재무점수'] = score
        base['재무등급'] = grade
        base['가치부담'] = '낮음' if grade == 'A' else '보통' if grade == 'B' else '높음' if grade == 'C' else '매우높음'
        dart_reason = str(row.get('재무안전사유', ''))
        if dart_reason and dart_reason != 'nan':
            base['재무위험사유'] = str(base['재무위험사유']) + ' / ' + dart_reason
    return base


def build_safety_table(codes_df):
    """v8.9 안전필터 테이블: 시장경보 + PER/PBR/ROE + DART 재무안전"""
    if codes_df is None or len(codes_df) == 0:
        return pd.DataFrame()
    base = codes_df[['종목코드']].copy()
    base['종목코드'] = base['종목코드'].apply(normalize_code)
    base = base.drop_duplicates('종목코드')

    manual_alerts = read_manual_market_alerts()
    fundamentals = get_latest_fundamentals(base['종목코드'].tolist()) if CONFIG['ENABLE_FUNDAMENTAL_FILTER'] else base.copy()
    dart_fin = get_dart_financials(base['종목코드'].tolist()) if CONFIG.get('ENABLE_DART_FINANCIAL_FILTER', True) else ensure_dart_columns(pd.DataFrame(columns=DART_COLUMNS))

    out = base.merge(manual_alerts, on='종목코드', how='left').merge(fundamentals, on='종목코드', how='left').merge(dart_fin, on='종목코드', how='left')
    out['시장경보'] = out['시장경보'].fillna('미확인')
    out['시장경보사유'] = out.get('사유', '').fillna('') if '사유' in out.columns else ''

    alert_eval = out['시장경보'].apply(lambda x: pd.Series(classify_alert(x), index=['안전필터판정', '안전필터사유']))
    out = pd.concat([out, alert_eval], axis=1)

    dart_eval = out.apply(evaluate_dart_financial_safety, axis=1)
    out = pd.concat([out, dart_eval], axis=1)

    fin_eval = out.apply(evaluate_financial_grade, axis=1)
    out = pd.concat([out, fin_eval], axis=1)

    # DART에서 명확한 위험이 잡히면 시장경보가 정상이더라도 판정 조정
    bad_dart = out['재무위험판정'].eq('매수제외')
    out.loc[bad_dart, '안전필터판정'] = '매수제외'
    out.loc[bad_dart, '안전필터사유'] = out.loc[bad_dart, '안전필터사유'].astype(str) + ' / DART 재무위험 매수제외'

    watch_dart = out['재무위험판정'].isin(['재무위험관찰', '재무주의']) & ~out['안전필터판정'].isin(['매수제외'])
    out.loc[watch_dart & out['안전필터판정'].eq('정상'), '안전필터판정'] = '추천제한'
    out.loc[watch_dart, '안전필터사유'] = out.loc[watch_dart, '안전필터사유'].astype(str) + ' / DART 재무주의'

    if CONFIG.get('EXCLUDE_FINANCIAL_GRADE_D', False):
        mask_d = out['재무등급'].eq('D') & ~out['안전필터판정'].isin(['매수제외'])
        out.loc[mask_d, '안전필터판정'] = '추천제한'
        out.loc[mask_d, '안전필터사유'] = out.loc[mask_d, '안전필터사유'].astype(str) + ' / 재무등급 D'

    out['최종위험메모'] = (
        '시장경보=' + out['시장경보'].astype(str) +
        ' / 안전판정=' + out['안전필터판정'].astype(str) +
        ' / 재무등급=' + out['재무등급'].astype(str) +
        ' / DART등급=' + out['재무안전등급'].astype(str) +
        ' / DART판정=' + out['재무위험판정'].astype(str) +
        ' / 가치부담=' + out['가치부담'].astype(str) +
        ' / 사유=' + out['재무위험사유'].astype(str)
    )
    return out.drop(columns=[c for c in ['사유'] if c in out.columns])


def apply_safety_score_adjustment(base_score, row):
    """v8.9: 기술점수에 시장경보/pykrx재무/DART재무안전 보정을 적용"""
    score = float(base_score)
    reasons = []
    verdict = str(row.get('안전필터판정', '확인필요'))
    grade = str(row.get('재무등급', '데이터부족'))
    dart_grade = str(row.get('재무안전등급', '데이터부족'))
    dart_verdict = str(row.get('재무위험판정', '확인필요'))

    if verdict in ['매수제외', '추천제외']:
        score -= 100
        reasons.append(verdict)
    elif verdict in ['소액관찰', '추천제한']:
        score -= 20
        reasons.append('시장경보/재무주의 제한')
    elif verdict == '확인필요':
        score -= 5
        reasons.append('시장경보 확인필요')

    if grade == 'A':
        score += 6
    elif grade == 'B':
        score += 2
    elif grade == 'C':
        score -= 6
        reasons.append('재무등급 C')
    elif grade == 'D':
        score -= 16
        reasons.append('재무등급 D')
    else:
        score -= 3
        reasons.append('재무데이터 부족')

    if dart_verdict == '매수제외':
        score -= 80
        reasons.append('DART 매수제외')
    elif dart_verdict == '재무위험관찰':
        score -= 18
        reasons.append('DART 재무위험관찰')
    elif dart_verdict == '재무주의':
        score -= 8
        reasons.append('DART 재무주의')

    if dart_grade == 'A':
        score += 4
    elif dart_grade == 'B':
        score += 2
    elif dart_grade == 'D':
        score -= 8

    return int(np.clip(score, 0, 100)), ' / '.join(reasons) if reasons else '보정 특이사항 낮음'


def judgment_text_v85(score, rsi, trix_turning, row):
    """이름은 v85 호환 유지. 내부 판정은 v8.9 기준."""
    verdict = str(row.get('안전필터판정', '확인필요'))
    grade = str(row.get('재무등급', '데이터부족'))
    dart_verdict = str(row.get('재무위험판정', '확인필요'))
    if verdict in ['매수제외', '추천제외'] or dart_verdict == '매수제외':
        return '⛔ 매수 제외'
    if '관망' in market_mode:
        return '⛔ 관망 우선'
    if verdict in ['소액관찰', '추천제한'] or dart_verdict in ['재무위험관찰', '재무주의']:
        return '⚠️ 소액/재무 관찰'
    if grade == 'D':
        return '⚠️ 재무위험 관찰'
    if score >= 82 and rsi <= 55 and trix_turning and grade in ['A', 'B']:
        return '🌟 A급 후보'
    if score >= 68 and trix_turning and grade in ['A', 'B', 'C']:
        return '✅ 선별 후보'
    if rsi >= 70:
        return '🚨 과열 위험'
    return '📊 관망'

print('✅ v8.9 DART 재무안전 필터 + 영구 캐시 확장 완료')


✅ v8.9 DART 재무안전 필터 + 영구 캐시 확장 완료


In [8]:
# 3. 시장 상태 판단 엔진

def analyze_index(symbol, name, start):
    df = safe_fdr_reader(symbol, start)
    if df.empty or len(df) < 80:
        return {'시장': name, '상태': '데이터부족', '점수': 50, '등락률20일': np.nan, 'MA20위': False, 'MA60위': False, '변동성': np.nan}

    df = calc_indicators(df).dropna()
    latest = df.iloc[-1]
    ret20 = (latest['Close'] / df['Close'].iloc[-21] - 1) * 100
    ma20_up = bool(latest['Close'] > latest['MA20'])
    ma60_up = bool(latest['Close'] > latest['MA60'])
    vol = df['Close'].pct_change().tail(20).std() * math.sqrt(20) * 100

    score = 50
    score += 15 if ma20_up else -15
    score += 15 if ma60_up else -15
    score += np.clip(ret20 * 2, -20, 20)
    score += -10 if vol > 9 else (5 if vol < 5 else 0)
    score = int(np.clip(score, 0, 100))

    if score >= 75:
        state = '공격 가능'
    elif score >= 55:
        state = '선별 매매'
    elif score >= 40:
        state = '방어 매매'
    else:
        state = '관망 우선'

    return {
        '시장': name,
        '상태': state,
        '점수': score,
        '등락률20일': round(ret20, 2),
        'MA20위': ma20_up,
        'MA60위': ma60_up,
        '변동성': round(vol, 2)
    }

market_df = pd.DataFrame([
    analyze_index('KS11', 'KOSPI', CONFIG['BACKTEST_START']),
    analyze_index('KQ11', 'KOSDAQ', CONFIG['BACKTEST_START'])
])

market_score = int(round(market_df['점수'].mean()))
if market_score >= 75:
    market_mode, market_position_factor = '공격 모드', 1.20
elif market_score >= 55:
    market_mode, market_position_factor = '선별 모드', 1.00
elif market_score >= 40:
    market_mode, market_position_factor = '방어 모드', 0.50
else:
    market_mode, market_position_factor = '관망 모드', 0.20

print(f'✅ 종합 시장점수: {market_score} / 100 → {market_mode}')
display(market_df)


✅ 종합 시장점수: 45 / 100 → 방어 모드


,시장,상태,점수,등락률20일,MA20위,MA60위,변동성
0,KOSPI,공격 가능,90,10.51,True,True,15.28
1,KOSDAQ,관망 우선,0,-17.17,False,False,12.27


In [9]:
# 4. 후보군 수집 — v10.1 자동조건검색 기본: TOSS 파일 없이 NAVER 후보 우선
source_mode = str(CONFIG.get('UNIVERSE_SOURCE', 'HYBRID')).upper().strip()
print(f'🚀 후보군 수집 시작 — 후보소스: {source_mode} / 모드: {CONFIG.get("CANDIDATE_SOURCE_MODE")}')

total_df = build_candidate_universe()

if total_df.empty:
    raise RuntimeError('후보군 데이터를 가져오지 못했습니다. 네이버/pykrx 접속 상태, 인터넷 연결, 가격/거래대금 조건을 확인하세요. TOSS_수동후보.csv는 v10.1 기본 모드에서 필수 파일이 아닙니다.')

for col in ['현재가', '거래대금', '등락률', '거래량', '거래대금(억)']:
    if col in total_df.columns:
        total_df[col] = clean_numeric(total_df[col])

for col, default in {
    '후보출처': source_mode,
    '스크리너명': '',
    '스크리너URL': '',
    '후보소스메모': '',
    '후보소스설정': source_mode
}.items():
    if col not in total_df.columns:
        total_df[col] = default

# 가격/거래대금/제외 키워드 필터
filtered_df = total_df[
    (total_df['현재가'] >= CONFIG['MIN_PRICE']) &
    (total_df['현재가'] <= CONFIG['MAX_PRICE']) &
    (total_df['거래대금'] >= CONFIG['MIN_TRADING_VALUE_NAVER']) &
    (~total_df['종목명'].str.contains(CONFIG['EXCLUDE_KEYWORDS'], na=False, regex=True)) &
    (total_df['종목코드'].notna())
].copy()

if '거래대금(억)' not in filtered_df.columns:
    filtered_df['거래대금(억)'] = (filtered_df['거래대금'] / 100).round(1)
else:
    filtered_df['거래대금(억)'] = filtered_df['거래대금(억)'].fillna((filtered_df['거래대금'] / 100).round(1))

filtered_df = filtered_df.drop_duplicates(subset=['종목코드']).reset_index(drop=True)

# v8.6 안전필터/기업가치/DART 재무안전 필터 결합
safety_df = build_safety_table(filtered_df[['종목코드', '종목명', '시장구분']]) if CONFIG.get('ENABLE_SAFETY_FILTER', True) else pd.DataFrame()
if not safety_df.empty:
    filtered_df = filtered_df.merge(safety_df, on='종목코드', how='left')
else:
    filtered_df['시장경보'] = '미확인'
    filtered_df['안전필터판정'] = '확인필요'
    filtered_df['재무등급'] = '데이터부족'
    filtered_df['가치부담'] = '확인필요'
    filtered_df['재무점수'] = 50
    filtered_df['최종위험메모'] = '안전필터 데이터 없음'

# 강한 위험 종목은 별도 시트로 분리하고 분석대상에서 제외
exclude_statuses = ['매수 제외', 'DART 매수제외']
excluded_by_safety_df = filtered_df[filtered_df['안전필터판정'].isin(exclude_statuses)].copy()
filtered_df = filtered_df[~filtered_df['안전필터판정'].isin(exclude_statuses)].reset_index(drop=True)

print(f'✅ 1차 후보군: {len(total_df)}개 / 가격·거래대금 통과: {len(filtered_df) + len(excluded_by_safety_df)}개 / 안전필터 제외: {len(excluded_by_safety_df)}개 / 분석대상: {len(filtered_df)}개')
print('✅ 후보출처 분포:')
display(total_df['후보출처'].value_counts(dropna=False).to_frame('종목수'))

show_cols = [c for c in ['후보출처','스크리너명','시장구분','종목명','종목코드','현재가','등락률','거래대금(억)','시장경보','안전필터판정','재무등급','재무안전등급','재무위험판정','PER','PBR','ROE(추정)','영업이익(억)','당기순이익(억)','부채비율','가치부담','후보소스메모'] if c in filtered_df.columns]
display(filtered_df[show_cols].head(30))



🚀 후보군 수집 시작 — 후보소스: HYBRID / 모드: HYBRID
ℹ️ 토스 수동후보 XLSX 파일 없음 → CSV/Selenium/NAVER fallback 확인
ℹ️ 토스 Selenium 수집 비활성화


⚠️ KRX 상장목록 조회 실패: HTTP Error 404: Not Found
📡 네이버 금융 거래량 상위 후보 수집


ℹ️ 시장경보 수동입력 템플릿 생성: 시장경보_수동입력.csv


Error occurred in get_market_fundamental_by_ticker: Expecting value: line 1 column 1 (char 0)


Error occurred in get_market_fundamental_by_ticker: Expecting value: line 1 column 1 (char 0)


...

Error occurred in get_market_fundamental_by_ticker: Expecting value: line 1 column 1 (char 0)
ℹ️ DART 재무 수동입력 템플릿 생성: DART_재무수동입력.csv


⚠️ DART 기업코드 조회 실패: HTTPSConnectionPool(host='opendart.fss.or.kr', port=443): Max retries exceeded with url: /api/corpCode.xml?crtfc_key=013cd035929dfa8d53bc967b5772de66ac010476 (Caused by ConnectTimeoutError(<HTTPSConnection(host='opendart.fss.or.kr', port=443) at 0x7fd46adcff10>, 'Connection to opendart.fss.or.kr timed out. (connect timeout=20)'))


✅ 1차 후보군: 192개 / 가격·거래대금 통과: 58개 / 안전필터 제외: 0개 / 분석대상: 58개
✅ 후보출처 분포:


,종목수
후보출처,
NAVER,192


,후보출처,스크리너명,시장구분,종목명,종목코드,현재가,등락률,거래대금(억),시장경보,안전필터판정,재무등급,재무안전등급,재무위험판정,PBR,ROE(추정),영업이익(억),당기순이익(억),부채비율,가치부담,후보소스메모
0,NAVER,네이버 거래량 상위,KOSPI,SK네트웍스,001740,13050.0,19.72,2749.8,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
1,NAVER,네이버 거래량 상위,KOSPI,SOL AI반도체TOP2플러스,001670,21200.0,-7.04,3919.6,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
2,NAVER,네이버 거래량 상위,KOSPI,KODEX 코스닥150,229200,16190.0,-7.78,1460.8,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
3,NAVER,네이버 거래량 상위,KOSPI,KODEX 은행,091170,14550.0,-8.81,1316.5,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
4,NAVER,네이버 거래량 상위,KOSPI,RISE 삼성전자SK하이닉스채권혼합50,001620,13850.0,-3.65,1110.9,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
5,NAVER,네이버 거래량 상위,KOSPI,KODEX 200타겟위클리커버드콜,498400,23800.0,-7.73,1835.4,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
6,NAVER,네이버 거래량 상위,KOSPI,TIGER 미국우주테크,001830,13780.0,-10.69,1006.7,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
7,NAVER,네이버 거래량 상위,KOSPI,KODEX 미국S&P500,379800,25870.0,-1.56,1381.6,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
8,NAVER,네이버 거래량 상위,KOSPI,TIGER 미국S&P500,360750,28385.0,-1.59,1089.2,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용
9,NAVER,네이버 거래량 상위,KOSPI,한온시스템,018880,4390.0,-6.50,169.2,미확인,확인필요,C,데이터부족,확인필요,NaN,NaN,NaN,NaN,NaN,높음,네이버 금융 거래량 상위 종목을 1차 후보로 사용


In [10]:
# 5. 전략 세트와 백테스트 엔진 — v8.9 실전비용 반영 백테스트

tp_strats = [
    {'name': '[초단타]', 'cond': [3, 5, 10], 'ratio': [0.7, 0.2, 0.1]},
    {'name': '[단기스윙]', 'cond': [5, 10, 20], 'ratio': [0.6, 0.3, 0.1]},
    {'name': '[대표님룰]', 'cond': [5, 10, 30], 'ratio': [0.6, 0.3, 0.1]},
    {'name': '[중기추세]', 'cond': [10, 20, 30], 'ratio': [0.5, 0.3, 0.2]},
    {'name': '[수익극대화]', 'cond': [10, 20, 40], 'ratio': [0.4, 0.3, 0.3]},
]

sl_strats = [
    {'name': '[칼손절]', 'cond': [-2, -4, -6], 'ratio': [0.8, 0.1, 0.1]},
    {'name': '[정석방어]', 'cond': [-3, -5, -10], 'ratio': [0.7, 0.2, 0.1]},
    {'name': '[변동성허용]', 'cond': [-5, -10, -15], 'ratio': [0.5, 0.3, 0.2]},
    {'name': '[버티기형]', 'cond': [-5, -15, -20], 'ratio': [0.4, 0.3, 0.3]},
]

scenarios = list(itertools.product(tp_strats, sl_strats))


def fmt_pct(value):
    """전략 조건을 +3% / -2%처럼 읽기 쉽게 표시"""
    try:
        value = float(value)
        return f"+{int(value)}%" if value > 0 else f"{int(value)}%"
    except Exception:
        return str(value)


def strategy_detail(strategy, action_word):
    """예: +3%▶70% 익절 / +5%▶20% 익절 / +10%▶10% 익절"""
    parts = []
    for cond, ratio in zip(strategy['cond'], strategy['ratio']):
        parts.append(f"{fmt_pct(cond)}▶{int(round(ratio * 100))}% {action_word}")
    return ' / '.join(parts)


def strategy_compact_plan(strategy):
    """스나이퍼 계산기용 2줄 표시.
    예:
    +3% / +5% / +10%
    (60%) / (20%) / (20%)
    """
    cond_line = ' / '.join(fmt_pct(x) for x in strategy['cond'])
    ratio_line = ' / '.join(f"({int(round(r * 100))}%)" for r in strategy['ratio'])
    return cond_line + chr(10) + ratio_line


def date_str(value):
    """pandas Timestamp / datetime / 문자열을 yyyy-mm-dd 형태로 정리"""
    try:
        return pd.to_datetime(value).strftime('%Y-%m-%d')
    except Exception:
        return ''


def backtest_scenario(df, tp, sl, initial_budget=300000, stock_name='', stock_code=''):
    """v8.9 실전비용 반영 백테스트.
    - 매수비용, 매도비용, 슬리피지를 차감한 순수익률을 return_rate로 사용합니다.
    - 전략 비교는 비용차감후 결과를 기준으로 하며, trade_log에 비용차감전/후 수익률을 함께 남깁니다.
    """
    buy_cost_rate = float(CONFIG.get('STRATEGY_BUY_COST_RATE', CONFIG.get('PORTFOLIO_BUY_COST_RATE', 0)))
    sell_cost_rate = float(CONFIG.get('STRATEGY_SELL_COST_RATE', CONFIG.get('PORTFOLIO_SELL_COST_RATE', 0)))
    slippage_rate = float(CONFIG.get('STRATEGY_SLIPPAGE_RATE', 0))

    budget = float(initial_budget)
    holding = 0
    avg_price = 0.0
    bought = 0
    profit_step = 0
    loss_step = 0
    equity_curve = []
    trade_results = []
    trade_results_gross = []
    trade_log = []
    entry_price = None      # 슬리피지 반영 매수 체결가
    entry_raw_price = None  # 원 시세
    entry_date = None

    def buy_exec_price(raw_price):
        return float(raw_price) * (1 + slippage_rate)

    def sell_exec_price(raw_price):
        return float(raw_price) * (1 - slippage_rate)

    def record_exit(exit_idx, raw_exit_price, reason, qty, remaining_after):
        nonlocal budget, trade_log, trade_results, trade_results_gross
        if entry_price is None or qty <= 0:
            return
        exec_exit = sell_exec_price(raw_exit_price)
        gross = qty * exec_exit
        sell_cost = gross * sell_cost_rate
        net_cash = gross - sell_cost
        budget += net_cash

        gross_ret_pct = (exec_exit / entry_price - 1) * 100
        net_entry_per_share = entry_price * (1 + buy_cost_rate)
        net_exit_per_share = exec_exit * (1 - sell_cost_rate)
        net_ret_pct = (net_exit_per_share / net_entry_per_share - 1) * 100
        cost_est = qty * entry_price * buy_cost_rate + sell_cost

        trade_results.append(net_ret_pct)
        trade_results_gross.append(gross_ret_pct)
        trade_log.append({
            '종목명': stock_name,
            '종목코드': stock_code,
            'TP전략': tp['name'],
            'SL전략': sl['name'],
            'TP전략내용': strategy_detail(tp, '익절'),
            'SL전략내용': strategy_detail(sl, '손절'),
            '백테스트시작일': date_str(df.index.min()),
            '백테스트종료일': date_str(df.index.max()),
            '실전비용반영': '예',
            '매수비용률': buy_cost_rate,
            '매도비용률': sell_cost_rate,
            '슬리피지률': slippage_rate,
            '매수일': date_str(entry_date),
            '매수단가': round(entry_price, 2),
            '매도일': date_str(exit_idx),
            '매도단가': round(exec_exit, 2),
            '매도사유': reason,
            '매도수량': int(qty),
            '잔여수량': int(remaining_after),
            '비용차감전수익률': round(gross_ret_pct, 2),
            '수익률': round(net_ret_pct, 2),
            '비용가정': round(cost_est, 0)
        })

    for i in range(1, len(df)):
        idx = df.index[i]
        price = float(df['Close'].iloc[i])
        rsi = float(df['RSI'].iloc[i])
        trix = float(df['TRIX'].iloc[i])
        prev_trix = float(df['TRIX'].iloc[i-1])
        liquidation_value = holding * sell_exec_price(price) * (1 - sell_cost_rate) if holding > 0 else 0
        equity_curve.append(budget + liquidation_value)

        if holding > 0:
            rate = ((price - avg_price) / avg_price) * 100

            if rsi >= 70:
                sell = min(int(holding * 0.90), holding)
                if sell > 0:
                    holding -= sell
                    record_exit(idx, price, 'RSI 과열 부분청산', sell, holding)
                    if holding == 0:
                        entry_price = None
                        entry_date = None
                    continue

            if rate >= tp['cond'][2] and profit_step < 3:
                sell = holding
                holding = 0
                record_exit(idx, price, f"3차 익절 도달({tp['cond'][2]}%)", sell, holding)
                profit_step = 3
                entry_price = None
                entry_date = None
            elif rate >= tp['cond'][1] and profit_step < 2:
                sell = min(int(bought * tp['ratio'][1]), holding)
                if sell > 0:
                    holding -= sell
                    record_exit(idx, price, f"2차 익절 도달({tp['cond'][1]}%)", sell, holding)
                profit_step = 2
            elif rate >= tp['cond'][0] and profit_step < 1:
                sell = min(int(bought * tp['ratio'][0]), holding)
                if sell > 0:
                    holding -= sell
                    record_exit(idx, price, f"1차 익절 도달({tp['cond'][0]}%)", sell, holding)
                profit_step = 1

            if holding > 0 and entry_price is not None:
                if rate <= sl['cond'][2] and loss_step < 3:
                    sell = holding
                    holding = 0
                    record_exit(idx, price, f"최종 손절 도달({sl['cond'][2]}%)", sell, holding)
                    loss_step = 3
                    entry_price = None
                    entry_date = None
                elif rate <= sl['cond'][1] and loss_step < 2:
                    sell = min(int(bought * sl['ratio'][1]), holding)
                    if sell > 0:
                        holding -= sell
                        record_exit(idx, price, f"2차 손절 도달({sl['cond'][1]}%)", sell, holding)
                    loss_step = 2
                elif rate <= sl['cond'][0] and loss_step < 1:
                    sell = min(int(bought * sl['ratio'][0]), holding)
                    if sell > 0:
                        holding -= sell
                        record_exit(idx, price, f"1차 손절 도달({sl['cond'][0]}%)", sell, holding)
                    loss_step = 1

        # 신규 진입: RSI 눌림 + TRIX 상승전환. 실제 체결은 슬리피지/매수비용 차감.
        if holding == 0 and rsi <= 45 and trix > prev_trix:
            exec_buy = buy_exec_price(price)
            max_shares = int(budget // (exec_buy * (1 + buy_cost_rate)))
            if max_shares > 0:
                buy_gross = max_shares * exec_buy
                buy_cost = buy_gross * buy_cost_rate
                budget -= (buy_gross + buy_cost)
                holding = max_shares
                avg_price = exec_buy
                entry_price = exec_buy
                entry_raw_price = price
                entry_date = idx
                bought = holding
                profit_step = 0
                loss_step = 0

    if holding > 0 and entry_price is not None:
        last_idx = df.index[-1]
        last = float(df['Close'].iloc[-1])
        sell = holding
        holding = 0
        record_exit(last_idx, last, '백테스트 종료일 강제청산', sell, holding)

    ret = ((budget - initial_budget) / initial_budget) * 100
    if equity_curve:
        curve = pd.Series(equity_curve)
        mdd = ((curve / curve.cummax()) - 1).min() * 100
    else:
        mdd = 0

    win_rate = np.mean([r > 0 for r in trade_results]) * 100 if trade_results else 0
    trade_count = len(trade_results)
    quality_score = ret + (win_rate * 0.15) + (mdd * 0.5) + min(trade_count, 10) * 0.5

    return {
        'return_rate': ret,
        'mdd': mdd,
        'win_rate': win_rate,
        'trade_count': trade_count,
        'quality_score': quality_score,
        'trade_log': trade_log,
        'backtest_start': date_str(df.index.min()),
        'backtest_end': date_str(df.index.max()),
        'cost_adjusted': True,
        'buy_cost_rate': buy_cost_rate,
        'sell_cost_rate': sell_cost_rate,
        'slippage_rate': slippage_rate
    }

print(f'✅ 전략 조합 {len(scenarios)}개 준비 완료 / 실전비용 반영 / 전략 조건 설명 함수 포함')


✅ 전략 조합 20개 준비 완료 / 실전비용 반영 / 전략 조건 설명 함수 포함


In [11]:
# 5-1. v8.6 백테스트 신뢰도 등급
# 6개월 백테스트는 “최근 장세 적합성”을 보는 데 유용하지만, 거래횟수·MDD·검증기간을 함께 봐야 합니다.

def classify_backtest_reliability(bt):
    try:
        start = pd.to_datetime(bt.get('backtest_start'))
        end = pd.to_datetime(bt.get('backtest_end'))
        days = max((end - start).days, 1)
    except Exception:
        days = 0
    trade_count = int(bt.get('trade_count', 0) or 0)
    mdd = float(bt.get('mdd', 0) or 0)
    ret = float(bt.get('return_rate', 0) or 0)
    win = float(bt.get('win_rate', 0) or 0)

    score = 50
    memo = []

    if days >= CONFIG.get('MIN_BACKTEST_DAYS_FOR_RELIABLE', 90):
        score += 10
    else:
        score -= 10
        memo.append('검증기간 짧음')

    if trade_count >= CONFIG.get('PREFERRED_RELIABLE_TRADES', 50):
        score += 20
        memo.append('거래횟수 충분')
    elif trade_count >= CONFIG.get('MIN_RELIABLE_TRADES', 30):
        score += 10
        memo.append('거래횟수 보통')
    elif trade_count >= 10:
        score -= 5
        memo.append('거래횟수 부족')
    else:
        score -= 20
        memo.append('거래횟수 매우 부족')

    if mdd >= -10:
        score += 12
        memo.append('MDD 안정')
    elif mdd >= CONFIG.get('MAX_ACCEPTABLE_MDD_FOR_RELIABLE', -20):
        score += 3
        memo.append('MDD 보통')
    else:
        score -= 18
        memo.append('MDD 큼')

    if ret > 0:
        score += min(ret, 30) * 0.3
    else:
        score -= 12
        memo.append('수익률 음수')

    if win >= 55:
        score += 8
    elif win < 40:
        score -= 8
        memo.append('승률 낮음')

    score = int(np.clip(score, 0, 100))
    if score >= 80:
        grade = 'A'
        meaning = '최근 구간 신뢰도 양호'
    elif score >= 65:
        grade = 'B'
        meaning = '참고 가능하나 추가 검증 필요'
    elif score >= 50:
        grade = 'C'
        meaning = '최근 적합성 참고 수준'
    else:
        grade = 'D'
        meaning = '단독 사용 비권장'

    return pd.Series({
        '백테스트신뢰도점수': score,
        '백테스트신뢰도': grade,
        '백테스트검증해석': meaning,
        '백테스트주의메모': ' / '.join(memo) if memo else '특이사항 낮음'
    })

print('✅ v8.6 백테스트 신뢰도 등급 함수 준비 완료')


✅ v8.6 백테스트 신뢰도 등급 함수 준비 완료


In [12]:
# 6. 종목별 분석 + 포지션 사이징 — v9.1 후보소스/실전비용/추세필터/DART캐시 반영


def evaluate_trend_filter(df):
    """v8.9 60일선 기반 추세 필터.
    기본 원칙: 현재가가 60일선 아래면 단기 반등 신호라도 매수 후보 신뢰도를 낮춥니다.
    """
    if df is None or len(df) < 80:
        return {'추세필터판정': '확인필요', '추세상태': '데이터부족', '추세메모': '80거래일 미만', 'MA20': np.nan, 'MA60': np.nan, 'MA60기울기': np.nan, '추세점수보정': -5}
    latest = df.iloc[-1]
    lookback = int(CONFIG.get('MA60_SLOPE_LOOKBACK', 5))
    past = df.iloc[-lookback-1] if len(df) > lookback + 1 else df.iloc[0]
    close = safe_float(latest.get('Close'))
    ma20 = safe_float(latest.get('MA20'))
    ma60 = safe_float(latest.get('MA60'))
    ma60_past = safe_float(past.get('MA60'))
    if any(pd.isna(x) for x in [close, ma20, ma60]):
        return {'추세필터판정': '확인필요', '추세상태': 'MA 데이터부족', '추세메모': 'MA20/MA60 확인불가', 'MA20': ma20, 'MA60': ma60, 'MA60기울기': np.nan, '추세점수보정': -5}
    close_above_ma60 = close > ma60
    close_above_ma20 = close > ma20
    ma20_above_ma60 = ma20 > ma60
    slope = ((ma60 / ma60_past) - 1) * 100 if not pd.isna(ma60_past) and ma60_past != 0 else np.nan
    ma60_up = True if pd.isna(slope) else slope >= 0

    status = []
    status.append('현재가>MA60' if close_above_ma60 else '현재가<MA60')
    status.append('현재가>MA20' if close_above_ma20 else '현재가<MA20')
    status.append('MA20>MA60' if ma20_above_ma60 else 'MA20<MA60')
    if not pd.isna(slope):
        status.append(f'MA60기울기 {slope:.2f}%')

    exclude = False
    if CONFIG.get('REQUIRE_CLOSE_ABOVE_MA60_FOR_BUY', True) and not close_above_ma60:
        exclude = True
    if CONFIG.get('REQUIRE_MA20_ABOVE_MA60_FOR_BUY', False) and not ma20_above_ma60:
        exclude = True
    if CONFIG.get('REQUIRE_MA60_UPTREND_FOR_BUY', False) and not ma60_up:
        exclude = True

    if exclude:
        verdict, memo, adj = '제외', '60일선 기준 추세필터 미통과', -35
    elif close_above_ma60 and ma20_above_ma60 and ma60_up:
        verdict, memo, adj = '안정형 통과', '상승 추세 눌림 후보', 8
    elif close_above_ma60 and close_above_ma20:
        verdict, memo, adj = '공격형 통과', '60일선 위 단기 회복 후보', 3
    elif close_above_ma60:
        verdict, memo, adj = '관찰 통과', '60일선은 위지만 단기 추세 확인 필요', -3
    else:
        verdict, memo, adj = '제외', '60일선 아래 단기 반등 가능성', -35

    return {'추세필터판정': verdict, '추세상태': ' / '.join(status), '추세메모': memo, 'MA20': ma20, 'MA60': ma60, 'MA60기울기': slope, '추세점수보정': adj}


def apply_trend_score_adjustment(score, trend_eval):
    if not CONFIG.get('ENABLE_TREND_FILTER', True):
        return int(np.clip(score, 0, 100)), '추세필터 미사용'
    adj = safe_float(trend_eval.get('추세점수보정'))
    adj = 0 if pd.isna(adj) else adj
    new_score = int(np.clip(float(score) + adj, 0, 100))
    return new_score, f"{trend_eval.get('추세필터판정')} / {trend_eval.get('추세메모')}"


def judgment_text_v89(score, rsi, trix_turning, row, trend_eval):
    if CONFIG.get('ENABLE_TREND_FILTER', True) and trend_eval.get('추세필터판정') == '제외':
        return '⛔ 추세 제외'
    return judgment_text_v85(score, rsi, trix_turning, row)

def make_position_size(account_size, curr_price, stop_price, stock_score, market_factor):
    """계좌 기준/손절폭/시장모드/종목점수를 반영해 추천 투입금액과 수량 계산"""
    risk_per_share = max(curr_price - stop_price, curr_price * 0.01)
    max_loss_budget = account_size * CONFIG['RISK_PER_TRADE_RATIO']
    risk_based_amount = int((max_loss_budget / risk_per_share) * curr_price)

    score_factor = np.clip(stock_score / 80, 0.3, 1.3)
    ratio_based_amount = int(account_size * CONFIG['BASE_POSITION_RATIO'] * score_factor * market_factor)
    cap_amount = int(account_size * CONFIG['MAX_POSITION_RATIO'])

    amount = max(0, min(risk_based_amount, ratio_based_amount, cap_amount))
    shares = int(amount // curr_price) if curr_price > 0 else 0
    return amount, shares


def stock_quality_score(df, market_score):
    latest = df.iloc[-1]
    prev = df.iloc[-2]
    score = 50

    rsi = latest['RSI']
    if 30 <= rsi <= 45:
        score += 20
    elif 45 < rsi <= 60:
        score += 8
    elif rsi >= 70:
        score -= 25

    score += 15 if latest['TRIX'] > prev['TRIX'] else -10
    score += 10 if latest['Close'] > latest['MA20'] else 0
    score += 10 if latest['MA20'] > latest['MA60'] else 0

    vol_ratio = latest['Volume'] / latest['VOL20'] if latest['VOL20'] and latest['VOL20'] > 0 else 1
    score += np.clip((vol_ratio - 1) * 8, -10, 20)

    atr = latest['ATR_PCT'] if not pd.isna(latest['ATR_PCT']) else 5
    score += -15 if atr > 12 else 5 if atr < 7 else 0

    score += (market_score - 50) * 0.25
    return int(np.clip(score, 0, 100))


def judgment_text(score, rsi, trix_turning):
    if '관망' in market_mode:
        return '⛔ 관망 우선'
    if score >= 80 and rsi <= 50 and trix_turning:
        return '🌟 A급 후보'
    if score >= 65 and trix_turning:
        return '✅ 선별 후보'
    if rsi >= 70:
        return '🚨 과열 위험'
    return '📊 관망'

results = []
scenario_summary_rows = []
trade_detail_rows = []
print('⚙️ 종목별 분석 시작')

for _, row in filtered_df.iterrows():
    code = str(row['종목코드']).zfill(6)
    name = row['종목명']
    time.sleep(CONFIG['SLEEP_SEC'])

    df = safe_fdr_reader(code, CONFIG['BACKTEST_START'])
    if df.empty or len(df) < 80:
        continue
    df = calc_indicators(df).dropna().copy()
    eval_df = df.loc[CONFIG['EVAL_START']:].copy()
    if eval_df.empty or len(eval_df) < 20:
        continue

    best = None
    best_tp = None
    best_sl = None

    for tp, sl in scenarios:
        bt = backtest_scenario(eval_df, tp, sl, CONFIG['INITIAL_BUDGET_FOR_TEST'], name, code)
        rel = classify_backtest_reliability(bt)
        scenario_summary_rows.append({
            '종목명': name,
            '종목코드': code,
            'TP전략': tp['name'],
            'SL전략': sl['name'],
            'TP전략내용': strategy_detail(tp, '익절'),
            'SL전략내용': strategy_detail(sl, '손절'),
            '백테스트시작일': bt['backtest_start'],
            '백테스트종료일': bt['backtest_end'],
            '수익률': round(bt['return_rate'], 2),
            'MDD': round(bt['mdd'], 2),
            '승률': round(bt['win_rate'], 1),
            '거래횟수': bt['trade_count'],
            '품질점수': round(bt['quality_score'], 2),
            '백테스트신뢰도점수': rel['백테스트신뢰도점수'],
            '백테스트신뢰도': rel['백테스트신뢰도'],
            '백테스트검증해석': rel['백테스트검증해석'],
            '백테스트주의메모': rel['백테스트주의메모'],
            '실전비용반영': '예' if bt.get('cost_adjusted') else '아니오',
            '매수비용률': bt.get('buy_cost_rate', ''),
            '매도비용률': bt.get('sell_cost_rate', ''),
            '슬리피지률': bt.get('slippage_rate', '')
        })
        trade_detail_rows.extend(bt['trade_log'])
        if best is None or bt['quality_score'] > best['quality_score']:
            best, best_tp, best_sl = bt, tp, sl

    latest = df.iloc[-1]
    prev = df.iloc[-2]
    curr_price = float(latest['Close'])
    rsi = float(latest['RSI'])
    trix_turning = float(latest['TRIX']) > float(prev['TRIX'])
    atr_pct = float(latest['ATR_PCT']) if not pd.isna(latest['ATR_PCT']) else np.nan
    vol_ratio = float(latest['Volume'] / latest['VOL20']) if latest['VOL20'] and latest['VOL20'] > 0 else np.nan

    trend_eval = evaluate_trend_filter(df)
    base_score = stock_quality_score(df, market_score)
    score, safety_adjust_reason = apply_safety_score_adjustment(base_score, row)
    score, trend_adjust_reason = apply_trend_score_adjustment(score, trend_eval)
    stop_price = int(curr_price * (1 + best_sl['cond'][0] / 100))
    rec_amount, rec_shares = make_position_size(CONFIG['ACCOUNT_SIZE'], curr_price, stop_price, score, market_position_factor)
    combo = f"{best_tp['name']} + {best_sl['name']}"
    judgment = judgment_text_v89(score, rsi, trix_turning, row, trend_eval)
    bt_range = f"{best['backtest_start']} ~ {best['backtest_end']}"
    recommended_tp = ' / '.join([f"+{x}%" for x in best_tp['cond']])
    recommended_sl = ' / '.join([f"{x}%" for x in best_sl['cond']])
    recommended_tp_plan = strategy_compact_plan(best_tp)
    recommended_sl_plan = strategy_compact_plan(best_sl)
    best_tp_detail = strategy_detail(best_tp, '익절')
    best_sl_detail = strategy_detail(best_sl, '손절')
    best_rel = classify_backtest_reliability(best)

    guide = (
        f"후보출처 {row.get('후보출처','')}, 시장 {market_mode}, 기본기술점수 {base_score}점→안전필터/추세필터 반영 {score}점. "
        f"안전필터 {row.get('안전필터판정','확인필요')}, 시장경보 {row.get('시장경보','미확인')}, 재무등급 {row.get('재무등급','데이터부족')}, DART재무 {row.get('재무안전등급','데이터부족')}/{row.get('재무위험판정','확인필요')}, PER {row.get('PER','')}, PBR {row.get('PBR','')}. "
        f"RSI {rsi:.1f}, TRIX {'상승전환' if trix_turning else '둔화'}, "
        f"ATR {atr_pct:.1f}% 수준. 추세필터 {trend_eval.get('추세필터판정')}({trend_eval.get('추세상태')}). 백테스트 기준 {bt_range}. "
        f"과거 구간 최적 조합은 {combo}. 백테스트 신뢰도는 {best_rel['백테스트신뢰도']}({best_rel['백테스트검증해석']}). 권장 익절은 {recommended_tp}, 손절은 {recommended_sl}. "
        f"익절 전략 상세: {best_tp_detail}. 손절 전략 상세: {best_sl_detail}. 전략백테스트는 매수비용 {CONFIG.get('STRATEGY_BUY_COST_RATE')}, 매도비용 {CONFIG.get('STRATEGY_SELL_COST_RATE')}, 슬리피지 {CONFIG.get('STRATEGY_SLIPPAGE_RATE')}를 반영. "
        f"추천 투입금액은 약 {rec_amount:,}원. 단, 장중 뉴스/호가/체결강도 확인 후 진입."
    )

    item = {
        'AI 시스템 판정': judgment,
        '종목명': name,
        '종목코드': code,
        '시장구분': row.get('시장구분', ''),
        '후보출처': row.get('후보출처', ''),
        '스크리너명': row.get('스크리너명', ''),
        '스크리너URL': row.get('스크리너URL', ''),
        '후보소스메모': row.get('후보소스메모', ''),
        '토스순위': row.get('토스순위', ''),
        '토스카테고리': row.get('토스카테고리', ''),
        '분야': row.get('토스카테고리', '') or row.get('카테고리', '') or row.get('업종', '') or row.get('시장구분', '') or '확인필요',
        '토스현재가': row.get('토스현재가', ''),
        '토스등락률': row.get('토스등락률', ''),
        '토스시가총액(억)': row.get('토스시가총액(억)', ''),
        '토스거래량': row.get('토스거래량', ''),
        '토스PER': row.get('토스PER', ''),
        '토스매출성장률(%)': row.get('토스매출성장률(%)', ''),
        '토스순이익성장률(%)': row.get('토스순이익성장률(%)', ''),
        '토스애널리스트분석': row.get('토스애널리스트분석', ''),
        '토스원본파일': row.get('토스원본파일', ''),
        '토스정리일시': row.get('토스정리일시', ''),
        '현재가': int(curr_price),
        '기본기술점수': base_score,
        '종목점수': score,
        '안전필터판정': row.get('안전필터판정', '확인필요'),
        '시장경보': row.get('시장경보', '미확인'),
        '시장경보사유': row.get('시장경보사유', ''),
        '재무등급': row.get('재무등급', '데이터부족'),
        '재무점수': row.get('재무점수', np.nan),
        '가치부담': row.get('가치부담', '확인필요'),
        '재무안전등급': row.get('재무안전등급', '데이터부족'),
        '재무안전점수': row.get('재무안전점수', np.nan),
        '재무위험판정': row.get('재무위험판정', '확인필요'),
        '재무안전사유': row.get('재무안전사유', ''),
        'DART기준연도': row.get('DART기준연도', ''),
        '영업이익(억)': round(safe_float(row.get('영업이익(억)')), 1) if not pd.isna(safe_float(row.get('영업이익(억)'))) else '',
        '당기순이익(억)': round(safe_float(row.get('당기순이익(억)')), 1) if not pd.isna(safe_float(row.get('당기순이익(억)'))) else '',
        '부채비율': round(safe_float(row.get('부채비율')), 1) if not pd.isna(safe_float(row.get('부채비율'))) else '',
        '자본총계(억)': round(safe_float(row.get('자본총계(억)')), 1) if not pd.isna(safe_float(row.get('자본총계(억)'))) else '',
        'DART데이터상태': row.get('DART데이터상태', ''),
        'DART조회일': row.get('DART조회일', ''),
        'PER': round(safe_float(row.get('PER')), 2) if not pd.isna(safe_float(row.get('PER'))) else '',
        'PBR': round(safe_float(row.get('PBR')), 2) if not pd.isna(safe_float(row.get('PBR'))) else '',
        'ROE(추정)': round(safe_float(row.get('ROE(추정)')), 2) if not pd.isna(safe_float(row.get('ROE(추정)'))) else '',
        '시가총액(억)': round(safe_float(row.get('시가총액(억)')), 0) if not pd.isna(safe_float(row.get('시가총액(억)'))) else '',
        '재무위험사유': row.get('재무위험사유', ''),
        '안전보정사유': safety_adjust_reason,
        '추세필터판정': trend_eval.get('추세필터판정'),
        '추세상태': trend_eval.get('추세상태'),
        '추세메모': trend_eval.get('추세메모'),
        '추세보정사유': trend_adjust_reason,
        'MA20': round(safe_float(trend_eval.get('MA20')), 2) if not pd.isna(safe_float(trend_eval.get('MA20'))) else '',
        'MA60': round(safe_float(trend_eval.get('MA60')), 2) if not pd.isna(safe_float(trend_eval.get('MA60'))) else '',
        'MA60기울기': round(safe_float(trend_eval.get('MA60기울기')), 2) if not pd.isna(safe_float(trend_eval.get('MA60기울기'))) else '',
        '시장모드': market_mode,
        '최적 매매 조합': combo,
        '권장익절퍼센트': recommended_tp,
        '권장손절퍼센트': recommended_sl,
        '권장익절계획': recommended_tp_plan,
        '권장손절계획': recommended_sl_plan,
        '최적익절전략상세': best_tp_detail,
        '최적손절전략상세': best_sl_detail,
        '백테스트시작일': best['backtest_start'],
        '백테스트종료일': best['backtest_end'],
        '백테스트신뢰도점수': best_rel['백테스트신뢰도점수'],
        '백테스트신뢰도': best_rel['백테스트신뢰도'],
        '백테스트검증해석': best_rel['백테스트검증해석'],
        '백테스트주의메모': best_rel['백테스트주의메모'],
        '상세 전략 가이드': guide,
        '현재 RSI': round(rsi, 1),
        'ATR%': round(atr_pct, 2) if not pd.isna(atr_pct) else '',
        '거래량배율': round(vol_ratio, 2) if not pd.isna(vol_ratio) else '',
        '실전비용반영': '예' if best.get('cost_adjusted') else '아니오',
        '매수비용률': best.get('buy_cost_rate', ''),
        '매도비용률': best.get('sell_cost_rate', ''),
        '슬리피지률': best.get('slippage_rate', ''),
        '백테스트수익률': round(best['return_rate'], 2),
        '백테스트MDD': round(best['mdd'], 2),
        '백테스트승률': round(best['win_rate'], 1),
        '거래횟수': best['trade_count'],
        '추천투입금액': rec_amount,
        '추천수량': rec_shares,
        '거래대금(억)': row.get('거래대금(억)', 0)
    }

    # 익절/손절 단계별 가격과 비중
    for i in range(3):
        item[f'익절{i+1}조건'] = f"+{best_tp['cond'][i]}%"
        item[f'익절{i+1}가격'] = int(curr_price * (1 + best_tp['cond'][i] / 100))
        item[f'익절{i+1}비중'] = best_tp['ratio'][i]
    for i in range(3):
        item[f'손절{i+1}조건'] = f"{best_sl['cond'][i]}%"
        item[f'손절{i+1}가격'] = int(curr_price * (1 + best_sl['cond'][i] / 100))
        item[f'손절{i+1}비중'] = best_sl['ratio'][i]

    results.append(item)

final_df = pd.DataFrame(results)
scenario_summary_df = pd.DataFrame(scenario_summary_rows)
trade_detail_df = pd.DataFrame(trade_detail_rows)

# 이전 버전 호환용 변수명
scenario_df = scenario_summary_df

if final_df.empty:
    print('⚠️ 최종 후보가 없습니다. 가격/거래대금/기간 조건을 완화해보세요.')
else:
    final_df = final_df.sort_values(['종목점수', '재무점수', '백테스트수익률', '거래대금(억)'], ascending=False).reset_index(drop=True)
    print(f'✅ 최종 분석 완료: {len(final_df)}개')
    print(f'✅ 백테스트 상세 매매기록: {len(trade_detail_df)}건')
    display(final_df.head(20))


⚙️ 종목별 분석 시작


"001670" invalid symbol or has no data


"001830" invalid symbol or has no data


"001900" invalid symbol or has no data


"001480" invalid symbol or has no data


"001170" invalid symbol or has no data


"001990" invalid symbol or has no data


✅ 최종 분석 완료: 50개
✅ 백테스트 상세 매매기록: 1455건


,AI 시스템 판정,종목명,종목코드,시장구분,후보출처,스크리너명,스크리너URL,후보소스메모,토스순위,토스카테고리,...,익절3비중,손절1조건,손절1가격,손절1비중,손절2조건,손절2가격,손절2비중,손절3조건,손절3가격,손절3비중
0,✅ 선별 후보,제주은행,006220,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.2,-3%,11969,0.7,-5%,11723,0.2,-10%,11106,0.1
1,✅ 선별 후보,라이콤,388790,KOSDAQ,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,7271,0.8,-4%,7123,0.1,-6%,6974,0.1
2,✅ 선별 후보,TIGER 미국배당다우존스,458730,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,15449,0.8,-4%,15134,0.1,-6%,14819,0.1
3,✅ 선별 후보,디아이씨,092200,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,9741,0.8,-4%,9542,0.1,-6%,9343,0.1
4,✅ 선별 후보,ACE 미국우주테크액티브,001800,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,24353,0.8,-4%,23856,0.1,-6%,23359,0.1
5,✅ 선별 후보,미래에셋생명,085620,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.2,-3%,19273,0.7,-5%,18876,0.2,-10%,17883,0.1
6,📊 관망,RISE 200위클리커버드콜,475720,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,14018,0.8,-4%,13732,0.1,-6%,13446,0.1
7,📊 관망,KODEX 200미국채혼합,284430,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,22858,0.8,-4%,22392,0.1,-6%,21925,0.1
8,📊 관망,SK네트웍스,001740,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.1,-2%,12789,0.8,-4%,12528,0.1,-6%,12267,0.1
9,📊 관망,TIGER 미국나스닥100타겟데일리커버드콜,486290,KOSPI,NAVER,네이버 거래량 상위,https://finance.naver.com/sise/sise_quant.naver,네이버 금융 거래량 상위 종목을 1차 후보로 사용,,,...,0.2,-3%,11678,0.7,-5%,11438,0.2,-10%,10836,0.1


In [13]:
# 6-1. v10.1 DART 3년 성장률 보강 — 저평가 성장주 조건검색용
# 목적: TOSS 파일 없이도 '최근 3년 평균 매출액/순이익 성장률' 후보 조건을 보조 계산합니다.
# 주의: DART API 키가 없거나 재무제표가 부족한 종목은 성장률을 빈 값으로 두고 조건검색 시 메모로 표시합니다.

import os, math, time
from datetime import datetime, timedelta


def _v101_num(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(str(x).replace(',', '').replace('원','').replace('%','').strip())
    except Exception:
        return np.nan


def _v101_growth_rate(values):
    vals = [_v101_num(v) for v in values]
    vals = [v for v in vals if not pd.isna(v)]
    if len(vals) < 2:
        return np.nan
    # 연평균 증감률: 인접 연도 증감률의 평균. 전년값이 0/음수이면 해당 구간 제외.
    rates = []
    for a, b in zip(vals[:-1], vals[1:]):
        if a and a > 0 and not pd.isna(b):
            rates.append((b / a - 1) * 100)
    return round(float(np.mean(rates)), 2) if rates else np.nan


def _v101_load_growth_cache(path=None):
    path = path or CONFIG.get('DART_GROWTH_CACHE_FILE', 'DART_3년성장률캐시.csv')
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, dtype={'종목코드': str})
            df['종목코드'] = df['종목코드'].apply(normalize_code)
            return df
        except Exception:
            return pd.DataFrame()
    return pd.DataFrame()


def _v101_save_growth_cache(df, path=None):
    path = path or CONFIG.get('DART_GROWTH_CACHE_FILE', 'DART_3년성장률캐시.csv')
    try:
        df.to_csv(path, index=False, encoding='utf-8-sig')
    except Exception as e:
        print('⚠️ DART 성장률 캐시 저장 실패:', e)


def _v101_fetch_dart_year_items(stock_code, year):
    api_key = CONFIG.get('DART_API_KEY', '')
    if not api_key:
        return None
    corp_map = fetch_dart_corp_code_map(api_key)
    stock_code = normalize_code(stock_code)
    if corp_map.empty or stock_code not in set(corp_map['stock_code']):
        return None
    corp_code = corp_map.loc[corp_map['stock_code'].eq(stock_code), 'corp_code'].iloc[0]
    url = 'https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json'
    for fs_div in [CONFIG.get('DART_FS_DIV', 'CFS'), 'OFS']:
        try:
            params = {
                'crtfc_key': api_key,
                'corp_code': corp_code,
                'bsns_year': str(year),
                'reprt_code': CONFIG.get('DART_REPORT_CODE', '11011'),
                'fs_div': fs_div
            }
            res = requests.get(url, params=params, timeout=20).json()
            if res.get('status') == '000' and res.get('list'):
                parsed = parse_dart_financial_items(res['list'])
                return {'매출액(억)': parsed.get('매출액'), '당기순이익(억)': parsed.get('당기순이익'), 'fs_div': fs_div}
        except Exception:
            continue
    return None


def _v101_fetch_growth_one(stock_code, base_year=None):
    base_year = int(base_year or CONFIG.get('DART_BASE_YEAR', RUN_AT.year - 1))
    years = [base_year-2, base_year-1, base_year]
    sales, net, used = [], [], []
    for y in years:
        item = _v101_fetch_dart_year_items(stock_code, y)
        if item:
            sales.append(item.get('매출액(억)'))
            net.append(item.get('당기순이익(억)'))
            used.append(str(y))
        time.sleep(min(CONFIG.get('SLEEP_SEC', 0.12), 0.25))
    return {
        '종목코드': normalize_code(stock_code),
        'DART성장률기준연도': ','.join(used),
        '최근3년매출액목록(억)': ' / '.join('' if pd.isna(_v101_num(x)) else str(round(_v101_num(x),1)) for x in sales),
        '최근3년순이익목록(억)': ' / '.join('' if pd.isna(_v101_num(x)) else str(round(_v101_num(x),1)) for x in net),
        '최근3년평균매출성장률': _v101_growth_rate(sales),
        '최근3년평균순이익성장률': _v101_growth_rate(net),
        'DART성장률조회일': REPORT_DATE,
        'DART성장률상태': 'DART API' if used else ('API키없음' if not CONFIG.get('DART_API_KEY') else '데이터부족')
    }


def enrich_final_df_with_dart_3y_growth(df):
    if df is None or df.empty:
        return df
    if not CONFIG.get('ENABLE_DART_3Y_GROWTH', True):
        return df
    df = df.copy()
    for col in ['최근3년평균매출성장률','최근3년평균순이익성장률','DART성장률상태']:
        if col not in df.columns:
            df[col] = np.nan if '성장률' in col else ''
    codes = list(dict.fromkeys(df['종목코드'].astype(str).map(normalize_code).tolist()))
    max_codes = int(CONFIG.get('DART_GROWTH_MAX_CODES', 80))
    codes = codes[:max_codes]
    cache = _v101_load_growth_cache()
    today = str(REPORT_DATE)
    ttl_days = int(CONFIG.get('DART_GROWTH_TTL_DAYS', 90))
    rows = []
    for code in codes:
        cached = pd.DataFrame()
        if not cache.empty and '종목코드' in cache.columns:
            cached = cache[cache['종목코드'].eq(code)]
            if not cached.empty and 'DART성장률조회일' in cached.columns:
                try:
                    dt = pd.to_datetime(cached.iloc[0]['DART성장률조회일'], errors='coerce')
                    if not pd.isna(dt) and (pd.Timestamp.today() - dt).days <= ttl_days:
                        rows.append(cached.iloc[0].to_dict())
                        continue
                except Exception:
                    pass
        rows.append(_v101_fetch_growth_one(code))
    growth_df = pd.DataFrame(rows)
    if not growth_df.empty:
        if not cache.empty:
            cache = cache[~cache['종목코드'].isin(growth_df['종목코드'])]
            merged_cache = pd.concat([cache, growth_df], ignore_index=True)
        else:
            merged_cache = growth_df.copy()
        _v101_save_growth_cache(merged_cache)
        df = df.merge(growth_df, on='종목코드', how='left', suffixes=('', '_dart3y'))
        # 이미 토스/기존 컬럼이 있으면 유지하고, 없는 값만 DART로 채움
        for base, alt in [('최근3년평균매출성장률','최근3년평균매출성장률_dart3y'), ('최근3년평균순이익성장률','최근3년평균순이익성장률_dart3y')]:
            if alt in df.columns:
                df[base] = df[base].combine_first(df[alt])
        df = df[[c for c in df.columns if not c.endswith('_dart3y')]]
    print('✅ v10.1 DART 3년 성장률 보강:', '실행' if CONFIG.get('DART_API_KEY') else 'API키 없음/메모만 표시')
    return df

try:
    final_df = enrich_final_df_with_dart_3y_growth(final_df)
except Exception as e:
    print('⚠️ v10.1 DART 3년 성장률 보강 실패:', e)


✅ v10.1 DART 3년 성장률 보강: 실행


In [14]:
# 7. v8.9 계좌 단위 백테스트 — 200만원 + 재무/추세/실전비용 적용

portfolio_summary_df = pd.DataFrame()
portfolio_equity_df = pd.DataFrame()
portfolio_trade_df = pd.DataFrame()
portfolio_candidate_df = pd.DataFrame()


def get_strategy_by_name(strategies, name, default_index=0):
    """전략 이름으로 TP/SL 전략을 찾고, 없으면 기본 전략을 반환"""
    for st in strategies:
        if st['name'] == name:
            return st
    return strategies[default_index]


def get_krx_universe(limit=250):
    """KRX 종목 유니버스 구성.
    - KRX_TOP: 현재 KRX 상장 목록 중 시가총액/거래대금 컬럼이 있으면 상위 limit개 사용
    - CURRENT_NAVER: 현재 네이버 거래량 후보군을 그대로 사용해 빠르게 검증
    """
    mode = CONFIG.get('PORTFOLIO_UNIVERSE_MODE', 'KRX_TOP')
    if mode == 'CURRENT_NAVER' and 'filtered_df' in globals() and not filtered_df.empty:
        uni = filtered_df[['종목코드', '종목명', '시장구분']].copy()
        uni.columns = ['Code', 'Name', 'Market']
        return uni.head(limit).reset_index(drop=True)

    try:
        listing = fdr.StockListing('KRX').copy()
    except Exception as e:
        print(f'⚠️ KRX 종목 리스트를 가져오지 못해 현재 네이버 후보군으로 대체합니다: {e}')
        uni = filtered_df[['종목코드', '종목명', '시장구분']].copy()
        uni.columns = ['Code', 'Name', 'Market']
        return uni.head(limit).reset_index(drop=True)

    # 컬럼명 차이를 흡수
    rename_map = {}
    for c in listing.columns:
        if c.lower() == 'code': rename_map[c] = 'Code'
        if c.lower() == 'name': rename_map[c] = 'Name'
        if c.lower() == 'market': rename_map[c] = 'Market'
    listing = listing.rename(columns=rename_map)

    if 'Code' not in listing.columns or 'Name' not in listing.columns:
        raise RuntimeError('KRX 상장 목록에서 Code/Name 컬럼을 찾지 못했습니다.')

    listing['Code'] = listing['Code'].astype(str).str.zfill(6)
    if 'Market' not in listing.columns:
        listing['Market'] = ''

    exclude = CONFIG['EXCLUDE_KEYWORDS'] + '|리츠|ETF|KODEX|TIGER|KBSTAR|HANARO|ARIRANG|SOL|ACE|PLUS'
    uni = listing[
        listing['Code'].str.match(r'^\d{6}$', na=False) &
        (~listing['Name'].astype(str).str.contains(exclude, regex=True, na=False))
    ].copy()

    # 시총 또는 거래대금 비슷한 컬럼이 있으면 상위 종목 우선
    sort_candidates = ['Marcap', 'MarketCap', 'Amount', 'Volume']
    sort_col = next((c for c in sort_candidates if c in uni.columns), None)
    if sort_col:
        uni = uni.sort_values(sort_col, ascending=False)

    return uni[['Code', 'Name', 'Market']].drop_duplicates('Code').head(limit).reset_index(drop=True)


def calc_market_score_on_date(index_df, date):
    """특정 날짜 기준 시장점수 계산. index_df는 calc_indicators 적용된 지수 데이터."""
    hist = index_df.loc[:date].dropna()
    if len(hist) < 80:
        return 50
    latest = hist.iloc[-1]
    ret20 = (latest['Close'] / hist['Close'].iloc[-21] - 1) * 100
    ma20_up = bool(latest['Close'] > latest['MA20'])
    ma60_up = bool(latest['Close'] > latest['MA60'])
    vol = hist['Close'].pct_change().tail(20).std() * math.sqrt(20) * 100
    score = 50
    score += 15 if ma20_up else -15
    score += 15 if ma60_up else -15
    score += np.clip(ret20 * 2, -20, 20)
    score += -10 if vol > 9 else (5 if vol < 5 else 0)
    return int(np.clip(score, 0, 100))


def stock_quality_score_on_date(hist, market_score_value):
    """특정 날짜까지의 데이터만 사용해 종목점수 계산. 미래 데이터 사용 방지."""
    if len(hist) < 80:
        return 0
    latest = hist.iloc[-1]
    prev = hist.iloc[-2]
    score = 50
    rsi = latest['RSI']
    if 30 <= rsi <= 45:
        score += 20
    elif 45 < rsi <= 60:
        score += 8
    elif rsi >= 70:
        score -= 25
    score += 15 if latest['TRIX'] > prev['TRIX'] else -10
    score += 10 if latest['Close'] > latest['MA20'] else 0
    score += 10 if latest['MA20'] > latest['MA60'] else 0
    vol_ratio = latest['Volume'] / latest['VOL20'] if latest['VOL20'] and latest['VOL20'] > 0 else 1
    score += np.clip((vol_ratio - 1) * 8, -10, 20)
    atr = latest['ATR_PCT'] if not pd.isna(latest['ATR_PCT']) else 5
    score += -15 if atr > 12 else 5 if atr < 7 else 0
    score += (market_score_value - 50) * 0.25
    return int(np.clip(score, 0, 100))


def get_next_trading_row(df, signal_date):
    """signal_date 다음 거래일 행 반환"""
    future = df.loc[df.index > signal_date]
    if future.empty:
        return None, None
    return future.index[0], future.iloc[0]


def calc_mdd_from_equity(equity_series):
    if len(equity_series) == 0:
        return 0
    s = pd.Series(equity_series).astype(float)
    peak = s.cummax()
    dd = (s / peak - 1) * 100
    return round(float(dd.min()), 2)


def run_portfolio_backtest():
    """계좌 단위 백테스트.
    원칙:
    - 신호는 전일 종가 기준으로 계산
    - 진입은 다음 거래일 시가로 가정
    - 같은 날 익절/손절 가격을 모두 터치하면 보수적으로 손절을 먼저 적용
    - 실제 호가/슬리피지/거래정지/상장폐지 종목은 단순화
    - v8.5부터 신호일 기준 PER/PBR/ROE 재무필터를 선택적으로 적용
    """
    if not CONFIG.get('PORTFOLIO_BACKTEST_ENABLED', True):
        print('ℹ️ 계좌 백테스트가 비활성화되어 있습니다.')
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    start = pd.to_datetime(CONFIG['PORTFOLIO_START'])
    end = pd.to_datetime(CONFIG['PORTFOLIO_END'])
    data_start = pd.to_datetime(CONFIG['BACKTEST_START'])

    tp = get_strategy_by_name(tp_strats, CONFIG.get('PORTFOLIO_DEFAULT_TP', '[단기스윙]'), 1)
    sl = get_strategy_by_name(sl_strats, CONFIG.get('PORTFOLIO_DEFAULT_SL', '[정석방어]'), 1)

    print(f"🚀 계좌 백테스트 시작: {CONFIG['PORTFOLIO_START']} ~ {CONFIG['PORTFOLIO_END']} / 초기자금 {CONFIG['PORTFOLIO_INITIAL_CASH']:,}원")
    print(f"   적용 전략: {tp['name']} {strategy_detail(tp, '익절')} / {sl['name']} {strategy_detail(sl, '손절')}")

    universe = get_krx_universe(CONFIG['PORTFOLIO_UNIVERSE_LIMIT'])
    print(f"✅ 백테스트 유니버스: {len(universe)}개")

    # 지수 데이터 준비
    kospi = calc_indicators(safe_fdr_reader('KS11', CONFIG['BACKTEST_START'])).dropna()
    kosdaq = calc_indicators(safe_fdr_reader('KQ11', CONFIG['BACKTEST_START'])).dropna()
    ref_index = kosdaq if not kosdaq.empty else kospi
    if ref_index.empty:
        raise RuntimeError('시장 지수 데이터를 가져오지 못했습니다.')

    sim_dates = [d for d in ref_index.loc[(ref_index.index >= start) & (ref_index.index <= end)].index]
    if len(sim_dates) < 5:
        raise RuntimeError('백테스트 기간의 거래일 데이터가 부족합니다.')

    # 종목 데이터 준비
    price_data = {}
    name_map = {}
    market_map = {}
    for n, row in universe.iterrows():
        code = str(row['Code']).zfill(6)
        name = row['Name']
        if n % 25 == 0:
            print(f"   데이터 수집 중... {n}/{len(universe)}")
        time.sleep(CONFIG['SLEEP_SEC'])
        df = safe_fdr_reader(code, CONFIG['BACKTEST_START'])
        if df.empty or len(df) < 100:
            continue
        df = calc_indicators(df).dropna()
        if df.loc[df.index >= start].empty:
            continue
        price_data[code] = df
        name_map[code] = name
        market_map[code] = row.get('Market', '')

    if not price_data:
        raise RuntimeError('백테스트에 사용할 종목 가격 데이터를 가져오지 못했습니다.')

    cash = float(CONFIG['PORTFOLIO_INITIAL_CASH'])
    positions = []
    equity_rows = []
    trade_rows = []
    candidate_rows = []
    trade_id = 0

    def current_position_codes():
        return {p['종목코드'] for p in positions if p['잔여수량'] > 0}

    def position_market_value(pos, date):
        df = price_data.get(pos['종목코드'])
        if df is None:
            return 0
        hist = df.loc[:date]
        if hist.empty:
            return 0
        return pos['잔여수량'] * float(hist.iloc[-1]['Close'])

    def portfolio_equity(date):
        return cash + sum(position_market_value(p, date) for p in positions)

    def sell_position(pos, date, price, qty, reason):
        nonlocal cash
        qty = int(max(0, min(qty, pos['잔여수량'])))
        if qty <= 0:
            return
        exec_price = price * (1 - CONFIG.get('PORTFOLIO_SLIPPAGE_RATE', 0))
        gross = qty * exec_price
        cost = gross * CONFIG['PORTFOLIO_SELL_COST_RATE']
        net = gross - cost
        cash += net
        pos['잔여수량'] -= qty
        ret_pct = (exec_price / pos['매수가'] - 1) * 100
        pnl = (exec_price - pos['매수가']) * qty - cost
        trade_rows.append({
            '거래ID': pos['거래ID'],
            '날짜': date_str(date),
            '구분': '매도',
            '종목명': pos['종목명'],
            '종목코드': pos['종목코드'],
            '사유': reason,
            '단가': round(exec_price, 2),
            '수량': qty,
            '거래금액': round(gross, 0),
            '비용가정': round(cost, 0),
            '실현손익': round(pnl, 0),
            '수익률': round(ret_pct, 2),
            '잔여수량': pos['잔여수량'],
            '현금잔고': round(cash, 0)
        })

    for date in sim_dates:
        # 1) 보유 종목 청산 판단: 보수적으로 손절 먼저, 그 다음 익절
        for pos in list(positions):
            if pos['잔여수량'] <= 0:
                continue
            df = price_data.get(pos['종목코드'])
            if df is None or date not in df.index:
                continue
            row = df.loc[date]
            high, low, close = float(row['High']), float(row['Low']), float(row['Close'])
            original_qty = pos['원수량']

            # 손절 단계
            while pos['sl_step'] < len(sl['cond']) and pos['잔여수량'] > 0:
                target = pos['매수가'] * (1 + sl['cond'][pos['sl_step']] / 100)
                if low <= target:
                    if pos['sl_step'] == len(sl['cond']) - 1:
                        qty = pos['잔여수량']
                    else:
                        qty = max(1, int(original_qty * sl['ratio'][pos['sl_step']]))
                    sell_position(pos, date, target, qty, f"{sl['name']} {fmt_pct(sl['cond'][pos['sl_step']])} 손절")
                    pos['sl_step'] += 1
                else:
                    break

            # 익절 단계
            while pos['tp_step'] < len(tp['cond']) and pos['잔여수량'] > 0:
                target = pos['매수가'] * (1 + tp['cond'][pos['tp_step']] / 100)
                if high >= target:
                    if pos['tp_step'] == len(tp['cond']) - 1:
                        qty = pos['잔여수량']
                    else:
                        qty = max(1, int(original_qty * tp['ratio'][pos['tp_step']]))
                    sell_position(pos, date, target, qty, f"{tp['name']} {fmt_pct(tp['cond'][pos['tp_step']])} 익절")
                    pos['tp_step'] += 1
                else:
                    break

            # 보유기간 만료 청산
            if pos['잔여수량'] > 0 and (pd.to_datetime(date) - pd.to_datetime(pos['매수일'])).days >= CONFIG['PORTFOLIO_MAX_HOLDING_DAYS']:
                sell_position(pos, date, close, pos['잔여수량'], f"최대보유기간 {CONFIG['PORTFOLIO_MAX_HOLDING_DAYS']}일 청산")

        positions = [p for p in positions if p['잔여수량'] > 0]

        # 2) 신규 후보 선정: 당일 종가 기준 신호 → 다음 거래일 시가 진입
        equity = portfolio_equity(date)
        if len(positions) < CONFIG['PORTFOLIO_MAX_POSITIONS'] and cash > 0:
            kospi_score = calc_market_score_on_date(kospi, date) if not kospi.empty else 50
            kosdaq_score = calc_market_score_on_date(kosdaq, date) if not kosdaq.empty else 50
            day_market_score = int(round(np.nanmean([kospi_score, kosdaq_score])))

            if day_market_score >= 40:  # 관망장에서는 신규 진입 제한
                candidates = []
                held_codes = current_position_codes()
                for code, df in price_data.items():
                    if code in held_codes or date not in df.index:
                        continue
                    hist = df.loc[:date].dropna()
                    if len(hist) < 80:
                        continue
                    latest = hist.iloc[-1]
                    prev = hist.iloc[-2]
                    close = float(latest['Close'])
                    volume = float(latest['Volume'])
                    vol20 = float(latest['VOL20']) if not pd.isna(latest['VOL20']) else 0
                    vol_ratio = volume / vol20 if vol20 > 0 else 0
                    turnover_eok = close * volume / 100000000
                    rsi = float(latest['RSI'])
                    trix_up = latest['TRIX'] > prev['TRIX']

                    if not (CONFIG['MIN_PRICE'] <= close <= CONFIG['MAX_PRICE']):
                        continue
                    if turnover_eok < CONFIG['PORTFOLIO_MIN_TURNOVER_EOK']:
                        continue
                    if vol_ratio < CONFIG['PORTFOLIO_MIN_VOL_RATIO']:
                        continue
                    if not trix_up:
                        continue
                    if rsi >= 70:
                        continue
                    trend_eval = evaluate_trend_filter(hist) if CONFIG.get('ENABLE_TREND_FILTER', True) else {'추세필터판정':'미사용','추세상태':'','추세메모':''}
                    if trend_eval.get('추세필터판정') == '제외':
                        continue
                    funda_ok, funda_note = pass_historical_fundamental_filter(code, date)
                    if not funda_ok:
                        continue

                    score = stock_quality_score_on_date(hist, day_market_score)
                    if score < 65:
                        continue
                    next_date, next_row = get_next_trading_row(df, date)
                    if next_date is None or next_date > end:
                        continue
                    candidates.append({
                        '신호일': date,
                        '진입예정일': next_date,
                        '종목명': name_map.get(code, ''),
                        '종목코드': code,
                        '시장점수': day_market_score,
                        '종목점수': score,
                        '전일종가': close,
                        '다음날시가': float(next_row['Open']),
                        'RSI': round(rsi, 1),
                        '거래량배율': round(vol_ratio, 2),
                        '거래대금(억)': round(turnover_eok, 1),
                        '재무필터': funda_note,
                        '추세필터': trend_eval.get('추세필터판정'),
                        '추세상태': trend_eval.get('추세상태')
                    })

                candidates = sorted(candidates, key=lambda x: (x['종목점수'], x['거래대금(억)'], x['거래량배율']), reverse=True)
                candidate_rows.extend([{**c, '신호일': date_str(c['신호일']), '진입예정일': date_str(c['진입예정일'])} for c in candidates[:10]])

                slots = CONFIG['PORTFOLIO_MAX_POSITIONS'] - len(positions)
                for cand in candidates[:slots]:
                    entry_date = cand['진입예정일']
                    if entry_date != date:
                        # 다음날 시가 진입을 실제 날짜 루프에 정확히 맞추기 위해, 당일 루프에서는 다음 거래일이 오늘인 경우만 진입한다.
                        # 단순 실행 편의를 위해 아래 pending order 구조를 쓰지 않고, 당일 신호 후 바로 다음날 처리되도록 별도 pending list를 운영한다.
                        pass

        # 3) pending order 없이도 다음날 진입이 되도록, 전일 신호 후보를 별도로 확인
        #    위 후보 선정은 candidate_rows 기록용이고, 실제 매수는 직전 거래일 신호를 다시 계산해 현재 날짜 시가로 처리한다.
        prev_dates = [d for d in sim_dates if d < date]
        if prev_dates and len(positions) < CONFIG['PORTFOLIO_MAX_POSITIONS'] and cash > 0:
            signal_date = prev_dates[-1]
            kospi_score = calc_market_score_on_date(kospi, signal_date) if not kospi.empty else 50
            kosdaq_score = calc_market_score_on_date(kosdaq, signal_date) if not kosdaq.empty else 50
            day_market_score = int(round(np.nanmean([kospi_score, kosdaq_score])))
            if day_market_score >= 40:
                buy_candidates = []
                held_codes = current_position_codes()
                for code, df in price_data.items():
                    if code in held_codes or signal_date not in df.index or date not in df.index:
                        continue
                    hist = df.loc[:signal_date].dropna()
                    if len(hist) < 80:
                        continue
                    latest = hist.iloc[-1]
                    prev = hist.iloc[-2]
                    close = float(latest['Close'])
                    volume = float(latest['Volume'])
                    vol20 = float(latest['VOL20']) if not pd.isna(latest['VOL20']) else 0
                    vol_ratio = volume / vol20 if vol20 > 0 else 0
                    turnover_eok = close * volume / 100000000
                    rsi = float(latest['RSI'])
                    trix_up = latest['TRIX'] > prev['TRIX']
                    if not (CONFIG['MIN_PRICE'] <= close <= CONFIG['MAX_PRICE']):
                        continue
                    if turnover_eok < CONFIG['PORTFOLIO_MIN_TURNOVER_EOK'] or vol_ratio < CONFIG['PORTFOLIO_MIN_VOL_RATIO']:
                        continue
                    if not trix_up or rsi >= 70:
                        continue
                    trend_eval = evaluate_trend_filter(hist) if CONFIG.get('ENABLE_TREND_FILTER', True) else {'추세필터판정':'미사용','추세상태':'','추세메모':''}
                    if trend_eval.get('추세필터판정') == '제외':
                        continue
                    funda_ok, funda_note = pass_historical_fundamental_filter(code, signal_date)
                    if not funda_ok:
                        continue
                    score = stock_quality_score_on_date(hist, day_market_score)
                    if score < 65:
                        continue
                    buy_candidates.append({
                        '종목코드': code,
                        '종목명': name_map.get(code, ''),
                        '종목점수': score,
                        '시장점수': day_market_score,
                        '거래대금(억)': round(turnover_eok, 1),
                        '거래량배율': round(vol_ratio, 2),
                        '재무필터': funda_note,
                        '추세필터': trend_eval.get('추세필터판정'),
                        '추세상태': trend_eval.get('추세상태'),
                        '시가': float(df.loc[date]['Open'])
                    })
                buy_candidates = sorted(buy_candidates, key=lambda x: (x['종목점수'], x['거래대금(억)'], x['거래량배율']), reverse=True)
                slots = CONFIG['PORTFOLIO_MAX_POSITIONS'] - len(positions)
                for cand in buy_candidates[:slots]:
                    if cash <= 0:
                        break
                    entry_price = cand['시가'] * (1 + CONFIG.get('PORTFOLIO_SLIPPAGE_RATE', 0))
                    invest_amount = min(cash, equity * CONFIG['PORTFOLIO_POSITION_RATIO'])
                    shares = int(invest_amount // (entry_price * (1 + CONFIG['PORTFOLIO_BUY_COST_RATE'])))
                    if shares <= 0:
                        continue
                    gross = shares * entry_price
                    cost = gross * CONFIG['PORTFOLIO_BUY_COST_RATE']
                    total_cost = gross + cost
                    if total_cost > cash:
                        continue
                    cash -= total_cost
                    trade_id += 1
                    pos = {
                        '거래ID': trade_id,
                        '종목명': cand['종목명'],
                        '종목코드': cand['종목코드'],
                        '매수일': date,
                        '매수가': entry_price,
                        '원수량': shares,
                        '잔여수량': shares,
                        'tp_step': 0,
                        'sl_step': 0
                    }
                    positions.append(pos)
                    trade_rows.append({
                        '거래ID': trade_id,
                        '날짜': date_str(date),
                        '구분': '매수',
                        '종목명': cand['종목명'],
                        '종목코드': cand['종목코드'],
                        '사유': f"전일 신호 진입 / 점수 {cand['종목점수']} / 시장 {cand['시장점수']} / 추세 {cand.get('추세필터','')} / {cand.get('재무필터','')}",
                        '단가': round(entry_price, 2),
                        '수량': shares,
                        '거래금액': round(gross, 0),
                        '비용가정': round(cost, 0),
                        '실현손익': '',
                        '수익률': '',
                        '잔여수량': shares,
                        '현금잔고': round(cash, 0)
                    })

        # 4) 일별 평가금액 기록
        end_equity = portfolio_equity(date)
        equity_rows.append({
            '날짜': date_str(date),
            '현금': round(cash, 0),
            '보유종목수': len([p for p in positions if p['잔여수량'] > 0]),
            '평가자산': round(end_equity, 0),
            '누적수익': round(end_equity - CONFIG['PORTFOLIO_INITIAL_CASH'], 0),
            '누적수익률': round((end_equity / CONFIG['PORTFOLIO_INITIAL_CASH'] - 1) * 100, 2)
        })

    # 마지막 보유 종목 강제 청산
    last_date = sim_dates[-1]
    for pos in list(positions):
        if pos['잔여수량'] <= 0:
            continue
        df = price_data.get(pos['종목코드'])
        hist = df.loc[:last_date] if df is not None else pd.DataFrame()
        if hist.empty:
            continue
        close = float(hist.iloc[-1]['Close'])
        sell_position(pos, last_date, close, pos['잔여수량'], '백테스트 종료일 강제청산')

    final_equity = cash
    # 종료일 강제청산 후의 비용 반영 최종자산을 계좌곡선 마지막 행에도 맞춰줍니다.
    if equity_rows:
        equity_rows[-1]['현금'] = round(cash, 0)
        equity_rows[-1]['보유종목수'] = 0
        equity_rows[-1]['평가자산'] = round(final_equity, 0)
        equity_rows[-1]['누적수익'] = round(final_equity - CONFIG['PORTFOLIO_INITIAL_CASH'], 0)
        equity_rows[-1]['누적수익률'] = round((final_equity / CONFIG['PORTFOLIO_INITIAL_CASH'] - 1) * 100, 2)
    initial_cash = CONFIG['PORTFOLIO_INITIAL_CASH']
    realized_sell_rows = [r for r in trade_rows if r.get('구분') == '매도']
    wins = [r for r in realized_sell_rows if isinstance(r.get('실현손익'), (int, float)) and r.get('실현손익', 0) > 0]
    win_rate = (len(wins) / len(realized_sell_rows) * 100) if realized_sell_rows else 0
    equity_values = [r['평가자산'] for r in equity_rows]
    mdd = calc_mdd_from_equity(equity_values)

    summary = pd.DataFrame([{
        '백테스트시작일': CONFIG['PORTFOLIO_START'],
        '백테스트종료일': date_str(last_date),
        '초기자금': initial_cash,
        '최종자산': round(final_equity, 0),
        '누적손익': round(final_equity - initial_cash, 0),
        '누적수익률': round((final_equity / initial_cash - 1) * 100, 2),
        'MDD': mdd,
        '매수횟수': len([r for r in trade_rows if r.get('구분') == '매수']),
        '매도기록수': len(realized_sell_rows),
        '매도기준승률': round(win_rate, 1),
        '초기자금설정': CONFIG['PORTFOLIO_INITIAL_CASH'],
        '동시보유최대': CONFIG['PORTFOLIO_MAX_POSITIONS'],
        '1종목투입비중': CONFIG['PORTFOLIO_POSITION_RATIO'],
        '적용TP전략': tp['name'],
        'TP전략내용': strategy_detail(tp, '익절'),
        '적용SL전략': sl['name'],
        'SL전략내용': strategy_detail(sl, '손절'),
        '비용가정': f"매수 {CONFIG['PORTFOLIO_BUY_COST_RATE']*100:.2f}% / 매도 {CONFIG['PORTFOLIO_SELL_COST_RATE']*100:.2f}%",
        '재무필터적용': CONFIG.get('PORTFOLIO_ENABLE_HISTORICAL_FUNDAMENTAL_FILTER', True),
        '재무데이터부족허용': CONFIG.get('PORTFOLIO_ALLOW_FUNDAMENTAL_UNKNOWN', True),
        '주의': '과거 OHLCV 기반 시뮬레이션이며 실제 체결, 호가, 슬리피지, 뉴스, 과거 시장경보/거래정지는 완전 반영되지 않음'
    }])

    equity_df = pd.DataFrame(equity_rows)
    trade_df = pd.DataFrame(trade_rows)
    candidate_df = pd.DataFrame(candidate_rows)
    return summary, equity_df, trade_df, candidate_df


try:
    portfolio_summary_df, portfolio_equity_df, portfolio_trade_df, portfolio_candidate_df = run_portfolio_backtest()
    if not portfolio_summary_df.empty:
        print('✅ 계좌 백테스트 완료')
        display(portfolio_summary_df)
        display(portfolio_equity_df.tail())
except Exception as e:
    print(f'⚠️ 계좌 백테스트를 완료하지 못했습니다: {e}')
    print('   인터넷 연결, FinanceDataReader 데이터 수집 가능 여부, 또는 PORTFOLIO_UNIVERSE_LIMIT 값을 확인하세요.')


🚀 계좌 백테스트 시작: 2026-01-01 ~ 20260608 / 초기자금 2,000,000원
   적용 전략: [단기스윙] +5%▶60% 익절 / +10%▶30% 익절 / +20%▶10% 익절 / [정석방어] -3%▶70% 손절 / -5%▶20% 손절 / -10%▶10% 손절


⚠️ KRX 종목 리스트를 가져오지 못해 현재 네이버 후보군으로 대체합니다: HTTP Error 404: Not Found
✅ 백테스트 유니버스: 58개
   데이터 수집 중... 0/58


"001670" invalid symbol or has no data


"001830" invalid symbol or has no data


"001900" invalid symbol or has no data


...

Error occurred in get_market_fundamental_by_ticker: Expecting value: line 1 column 1 (char 0)


Error occurred in get_market_fundamental_by_ticker: Expecting value: line 1 column 1 (char 0)


✅ 계좌 백테스트 완료


,백테스트시작일,백테스트종료일,초기자금,최종자산,누적손익,누적수익률,MDD,매수횟수,매도기록수,매도기준승률,...,동시보유최대,1종목투입비중,적용TP전략,TP전략내용,적용SL전략,SL전략내용,비용가정,재무필터적용,재무데이터부족허용,주의
0,2026-01-01,2026-06-05,2000000,2068491.0,68491.0,3.42,-7.93,33,92,43.5,...,4,0.25,[단기스윙],+5%▶60% 익절 / +10%▶30% 익절 / +20%▶10% 익절,[정석방어],-3%▶70% 손절 / -5%▶20% 손절 / -10%▶10% 손절,매수 0.05% / 매도 0.25%,True,True,"과거 OHLCV 기반 시뮬레이션이며 실제 체결, 호가, 슬리피지, 뉴스, 과거 시장..."


,날짜,현금,보유종목수,평가자산,누적수익,누적수익률
98,2026-05-29,901144.0,3,2073184.0,73184.0,3.66
99,2026-06-01,454693.0,4,1982723.0,-17277.0,-0.86
100,2026-06-02,1233313.0,3,2079663.0,79663.0,3.98
101,2026-06-04,875745.0,4,2161115.0,161115.0,8.06
102,2026-06-05,2068491.0,0,2068491.0,68491.0,3.42


In [15]:
# v9.7.5.4: 날짜가 넘어간 뒤에도 최종 저장 시점(KST) 기준으로 갱신
refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))
# 8. 엑셀 리포트 생성 — v9.6.2 스나이퍼 계산/토스 TOP 개선 반영

def style_header(ws, row=1, fill_color='1F4E78'):
    for cell in ws[row]:
        cell.fill = PatternFill(start_color=fill_color, end_color=fill_color, fill_type='solid')
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)


def autosize_columns(ws, max_width=45):
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col:
            max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
        ws.column_dimensions[letter].width = min(max_len + 2, max_width)


def add_comment(cell, text):
    """엑셀 셀 메모 추가"""
    try:
        cell.comment = Comment(text, 'v9.2 투자파트너')
    except Exception:
        pass


def apply_header_comments(ws, comment_map):
    """헤더명 기준으로 설명 메모 추가"""
    for cell in ws[1]:
        if cell.value in comment_map:
            add_comment(cell, comment_map[cell.value])


def build_excel_report(final_df, market_df, scenario_summary_df=None, trade_detail_df=None):
    today_file = datetime.today().strftime('%Y%m%d')
    file_name = f"{CONFIG['OUTPUT_PREFIX']}_{today_file}.xlsx"
    thin = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)

    def lookup_formula(header, default='0'):
        """추천 리스트에서 종목명(C4)과 헤더명으로 값을 가져오는 안정형 공식.
        고정 VLOOKUP 열 번호를 쓰지 않아 추천 리스트 열이 바뀌어도 깨질 가능성이 낮음.
        """
        return f'=IFERROR(INDEX(\'추천 리스트\'!$A:$ZZ,MATCH($C$4,\'추천 리스트\'!$B:$B,0),MATCH("{header}",\'추천 리스트\'!$1:$1,0)),{default})'

    with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
        # 시장상태
        summary = pd.DataFrame([{
            '리포트기준일': REPORT_DATE,
            '판단일시': REPORT_DATETIME,
            'DART API 상태': '자동조회 가능' if CONFIG.get('DART_API_KEY') else '키 없음/수동CSV',
            '종합시장점수': market_score,
            '시장모드': market_mode,
            '포지션보정계수': market_position_factor,
            '백테스트조회시작일': CONFIG['BACKTEST_START'],
            '전략평가시작일': CONFIG['EVAL_START'],
            '사용원칙': '관망/방어 모드에서는 A급 후보라도 비중 축소 또는 매매 보류',
            '안전필터원칙': '투자경고/투자위험/거래정지/관리종목은 매수 후보에서 제외하고, 투자주의/지정예고는 소액 관찰로 분리',
            '6개월검증해석': '최근 장세 적합성 확인용. 전략 확정은 12~24개월, 워크포워드, 실거래 모의검증 병행 권장'
        }])
        summary.to_excel(writer, sheet_name='시장상태', index=False, startrow=0)
        market_df.to_excel(writer, sheet_name='시장상태', index=False, startrow=4)

        # 추천 리스트
        final_df.to_excel(writer, sheet_name='추천 리스트', index=False)

        # v9.6: 토스 첨부파일 정리 결과도 리포트 안에 보관
        try:
            if 'toss_xlsx_cleaned_df' in globals() and isinstance(toss_xlsx_cleaned_df, pd.DataFrame) and not toss_xlsx_cleaned_df.empty:
                toss_xlsx_cleaned_df.to_excel(writer, sheet_name='토스후보_정리결과', index=False)
            if 'toss_xlsx_unmatched_df' in globals() and isinstance(toss_xlsx_unmatched_df, pd.DataFrame) and not toss_xlsx_unmatched_df.empty:
                toss_xlsx_unmatched_df.to_excel(writer, sheet_name='토스후보_매칭실패', index=False)
        except Exception as e:
            print(f'⚠️ 토스 후보 정리 시트 저장 생략: {e}')



        # v9.2 컴팩트 후보 요약: 오른쪽 스크롤 없이 보는 핵심 화면
        compact_preferred_cols = [
            'AI 시스템 판정', '종목명', '종목코드', '후보출처', '현재가', '등락률',
            '종목점수', '추세필터판정', '안전필터판정', '재무위험판정', '재무등급', '재무안전등급',
            '백테스트수익률', '백테스트MDD', '백테스트승률', '백테스트신뢰도',
            '권장익절계획', '권장손절계획', '추천투입금액', '추천수량', '상세 전략 가이드'
        ]
        compact_cols = [c for c in compact_preferred_cols if c in final_df.columns]
        if CONFIG.get('UX_COMPACT_VIEW_ENABLED', True) and len(compact_cols) > 0:
            compact_df = final_df[compact_cols].head(CONFIG.get('UX_TOP_N', 15)).copy()
            compact_df.to_excel(writer, sheet_name='TOP후보_요약', index=False)

            # v9.6.2: 토스 첨부파일 후보가 전체 점수 정렬에서 묻히지 않도록 별도 TOP 시트 제공
            try:
                if '후보출처' in final_df.columns:
                    toss_mask = final_df['후보출처'].astype(str).str.contains('TOSS', case=False, na=False)
                    toss_top_df = final_df.loc[toss_mask, compact_cols].head(CONFIG.get('UX_TOP_N', 15)).copy()
                    if not toss_top_df.empty:
                        toss_top_df.to_excel(writer, sheet_name='토스_TOP후보', index=False)
            except Exception as e:
                print(f'⚠️ 토스_TOP후보 시트 생성 생략: {e}')

        # 안전필터 제외 종목
        if 'excluded_by_safety_df' in globals() and excluded_by_safety_df is not None and not excluded_by_safety_df.empty:
            excluded_by_safety_df.to_excel(writer, sheet_name='제외종목_위험필터', index=False)

        # 전략 요약/상세
        if scenario_summary_df is not None and not scenario_summary_df.empty:
            scenario_summary_df.to_excel(writer, sheet_name='전략백테스트요약', index=False)
        if trade_detail_df is not None and not trade_detail_df.empty:
            trade_detail_df.to_excel(writer, sheet_name='전략백테스트상세', index=False)

        # v8.4 계좌 백테스트 결과 시트
        if 'portfolio_summary_df' in globals() and portfolio_summary_df is not None and not portfolio_summary_df.empty:
            portfolio_summary_df.to_excel(writer, sheet_name='계좌백테스트요약', index=False)
        if 'portfolio_equity_df' in globals() and portfolio_equity_df is not None and not portfolio_equity_df.empty:
            portfolio_equity_df.to_excel(writer, sheet_name='계좌백테스트곡선', index=False)
        if 'portfolio_trade_df' in globals() and portfolio_trade_df is not None and not portfolio_trade_df.empty:
            portfolio_trade_df.to_excel(writer, sheet_name='계좌백테스트거래내역', index=False)
        if 'portfolio_candidate_df' in globals() and portfolio_candidate_df is not None and not portfolio_candidate_df.empty:
            portfolio_candidate_df.to_excel(writer, sheet_name='계좌백테스트후보기록', index=False)

        wb = writer.book
        # v9.6.1: 수식이 있는 시트는 Excel/Sheets에서 열 때 자동 재계산되도록 설정
        try:
            wb.calculation.fullCalcOnLoad = True
            wb.calculation.forceFullCalc = True
            wb.calculation.calcMode = 'auto'
        except Exception:
            pass



        # v9.2 홈 브리핑: 엑셀을 열었을 때 처음 보는 화면
        def write_home_value(cell, value, fill='FFFFFF', font_color='111827', bold=False, size=11):
            cell.value = value
            cell.fill = PatternFill(start_color=fill, end_color=fill, fill_type='solid')
            cell.font = Font(color=font_color, bold=bold, size=size)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin

        def metric_card(ws, rng, title, value, note, fill):
            ws.merge_cells(rng)
            top_left = ws[rng.split(':')[0]]
            top_left.value = f'{title}\n{value}\n{note}'
            top_left.fill = PatternFill(start_color=fill, end_color=fill, fill_type='solid')
            top_left.font = Font(color='FFFFFF', bold=True, size=12)
            top_left.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            top_left.border = thin
            for row in ws[rng]:
                for c in row:
                    c.fill = PatternFill(start_color=fill, end_color=fill, fill_type='solid')
                    c.border = thin

        ws_home = wb.create_sheet('홈_브리핑', 0)
        ws_home.sheet_view.showGridLines = False
        ws_home.sheet_view.zoomScale = CONFIG.get('UX_ZOOM_SCALE', 90)
        ws_home.sheet_properties.tabColor = '0B1F3A'
        for col, width in {'A':3, 'B':16, 'C':16, 'D':16, 'E':16, 'F':16, 'G':16, 'H':16, 'I':16, 'J':16, 'K':16}.items():
            ws_home.column_dimensions[col].width = width
        ws_home.row_dimensions[1].height = 34
        ws_home.merge_cells('B1:K1')
        ws_home['B1'] = '오늘의 투자 브리핑 — v9.2 Compact Dashboard'
        ws_home['B1'].fill = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
        ws_home['B1'].font = Font(color='FFFFFF', bold=True, size=18)
        ws_home['B1'].alignment = Alignment(horizontal='left', vertical='center')

        total_candidates = len(final_df) if final_df is not None else 0
        status_col = 'AI 시스템 판정' if 'AI 시스템 판정' in final_df.columns else None
        a_grade_count = int(final_df[status_col].astype(str).str.contains('A급', na=False).sum()) if status_col and total_candidates else 0
        watch_count = int(final_df[status_col].astype(str).str.contains('관찰|소액|재무위험|과열|관망', na=False).sum()) if status_col and total_candidates else 0
        dart_state = '자동조회 가능' if CONFIG.get('DART_API_KEY') else '키 없음/수동CSV'
        source_state = str(CONFIG.get('UNIVERSE_SOURCE', ''))

        for r in [3,4,5,7,8,9,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28]:
            ws_home.row_dimensions[r].height = 24
        for r in [3,4,5]:
            ws_home.row_dimensions[r].height = 30
        metric_card(ws_home, 'B3:C5', '시장모드', market_mode, f'시장점수 {market_score}', '1F4E78')
        metric_card(ws_home, 'D3:E5', '전체 후보', total_candidates, f'후보소스 {source_state}', '2F5597')
        metric_card(ws_home, 'F3:G5', 'A급 후보', a_grade_count, '우선 검토 대상', '70AD47' if a_grade_count else '7F7F7F')
        metric_card(ws_home, 'H3:I5', '관찰/주의', watch_count, '비중 축소 또는 제외', 'C65911' if watch_count else '7F7F7F')
        metric_card(ws_home, 'J3:K5', 'DART', dart_state, '재무안전 필터', '8064A2')

        ws_home['B7'] = '사용 순서'
        ws_home['B7'].font = Font(bold=True, color='0B1F3A', size=13)
        ws_home['B8'] = '1) TOP후보_요약에서 핵심 후보만 먼저 본다 → 2) 스나이퍼_계산기에서 종목 선택 → 3) 실제투입금액 입력 → 4) 익절/손절표 확인 → 5) 추천 리스트는 상세 DB로만 확인'
        ws_home.merge_cells('B8:K9')
        ws_home['B8'].fill = PatternFill(start_color='EEF2F7', end_color='EEF2F7', fill_type='solid')
        ws_home['B8'].alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        ws_home['B8'].border = thin

        ws_home['B11'] = 'TOP 후보 미리보기'
        ws_home['B11'].font = Font(bold=True, color='0B1F3A', size=13)
        preview_cols = [
            'AI 시스템 판정', '종목명', '후보출처', '현재가', '종목점수',
            '추세필터판정', '재무위험판정', '백테스트신뢰도', '권장익절계획', '권장손절계획'
        ]
        preview_cols = [c for c in preview_cols if c in final_df.columns]
        start_row = 12
        for c_idx, header in enumerate(preview_cols, start=2):
            cell = ws_home.cell(start_row, c_idx, header)
            cell.fill = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
            cell.font = Font(color='FFFFFF', bold=True)
            cell.alignment = center
            cell.border = thin
        preview_df = final_df[preview_cols].head(8) if len(preview_cols) else pd.DataFrame()
        for r_idx, (_, row_data) in enumerate(preview_df.iterrows(), start=start_row+1):
            for c_idx, header in enumerate(preview_cols, start=2):
                cell = ws_home.cell(r_idx, c_idx, row_data.get(header, ''))
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                cell.border = thin
                if header in ['권장익절계획', '권장손절계획']:
                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                    ws_home.row_dimensions[r_idx].height = 34
            status_txt = str(row_data.get('AI 시스템 판정', ''))
            fill = 'E2EFDA' if 'A급' in status_txt else ('FCE4D6' if any(k in status_txt for k in ['관찰','제외','소액','위험','관망']) else 'FFFFFF')
            for c_idx in range(2, 2+len(preview_cols)):
                ws_home.cell(r_idx, c_idx).fill = PatternFill(start_color=fill, end_color=fill, fill_type='solid')
        # 내부 이동 링크
        ws_home['B23'] = '빠른 이동'
        ws_home['B23'].font = Font(bold=True, color='0B1F3A', size=13)
        quick_links = [
            ('TOP후보_요약 보기', "#'TOP후보_요약'!A1"),
            ('스나이퍼 계산기 열기', "#'스나이퍼_계산기'!B2"),
            ('상세 추천 DB 열기', "#'추천 리스트'!A1"),
            ('용어설명 보기', "#'용어설명'!A1")
        ]
        for idx, (txt, link) in enumerate(quick_links, start=2):
            cell = ws_home.cell(24, idx, txt)
            cell.hyperlink = link
            cell.style = 'Hyperlink'
            cell.fill = PatternFill(start_color='EAF2F8', end_color='EAF2F8', fill_type='solid')
            cell.border = thin
            cell.alignment = center
        ws_home.freeze_panes = 'B12'

        # 기본 스타일
        base_sheet_names = [s for s in ['시장상태', 'TOP후보_요약', '추천 리스트', '토스후보_정리결과', '토스후보_매칭실패', '토스_TOP후보', '제외종목_위험필터'] if s in writer.sheets]
        for sheet_name in base_sheet_names:
            ws = writer.sheets[sheet_name]
            style_header(ws, 1)
            autosize_columns(ws, 55)
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions

        ws_list = writer.sheets['추천 리스트']
        recommendation_comments = {
            'AI 시스템 판정': '시장상태, 종목점수, 후보소스, 안전필터, RSI/TRIX 등을 종합해 만든 참고용 판정입니다.',
            '후보출처': '이 종목이 어떤 1차 후보군에서 들어왔는지 표시합니다. TOSS, NAVER 또는 둘 다 표시될 수 있습니다.',
            '스크리너명': '토스 스크리너 또는 네이버 거래량 상위처럼 후보군을 만든 기준입니다.',
            '후보소스메모': '후보소스의 성격과 주의사항입니다. 토스 후보는 최종 추천이 아니라 1차 후보입니다.',
            '안전필터판정': '투자경고/투자위험/거래정지/관리종목 등은 매수 제외, 투자주의/지정예고는 소액 관찰로 분리합니다.',
            '시장경보': 'KRX/KIND 등에서 확인하는 투자주의·투자경고·투자위험 여부입니다. 자동 확인이 안 되면 수동 CSV로 보완하세요.',
            '재무등급': 'PER/PBR/ROE/시가총액과 DART 재무안전 점수를 함께 반영한 참고 등급입니다.',
            '재무안전등급': 'DART 재무제표 기반 영업이익·순이익·부채비율·자본총계를 반영한 안전 등급입니다.',
            '재무위험판정': 'DART 기준 재무위험 관찰/재무주의/매수제외 여부입니다.',
            '영업이익(억)': 'DART 사업보고서 기준 영업이익입니다. 적자이면 단기/스윙에서도 위험 감점합니다.',
            '당기순이익(억)': 'DART 사업보고서 기준 순이익입니다. 순손실은 재무위험 요소입니다.',
            '부채비율': '부채총계/자본총계×100입니다. 높을수록 재무 부담이 큽니다.',
            'PER': '주가수익비율입니다. 너무 높거나 산출불가이면 실적 대비 가격 부담 또는 적자 가능성을 의심합니다.',
            'PBR': '주가순자산비율입니다. 너무 높으면 자산가치 대비 가격 부담이 클 수 있습니다.',
            'ROE(추정)': 'EPS/BPS로 단순 환산한 참고 수익성 지표입니다.',
            '권장익절퍼센트': '선택된 최적 TP전략의 익절 목표 퍼센트입니다. 예: +3% / +5% / +10%',
            '권장손절퍼센트': '선택된 최적 SL전략의 손절 기준 퍼센트입니다. 예: -2% / -4% / -6%',
            '권장익절계획': '윗줄은 익절 목표, 아랫줄 괄호는 해당 목표 도달 시 매도할 물량 비중입니다.',
            '권장손절계획': '윗줄은 손절 기준, 아랫줄 괄호는 해당 기준 도달 시 매도/손절할 물량 비중입니다.',
            '최적익절전략상세': '각 익절 조건에 도달했을 때 몇 % 물량을 익절할지 표시합니다.',
            '최적손절전략상세': '각 손절 조건에 도달했을 때 몇 % 물량을 손절할지 표시합니다.',
            '추천투입금액': '계좌총액, 손절폭, 종목점수, 시장모드를 반영한 시스템 추천 금액입니다.',
            '추천수량': '추천투입금액을 현재가로 나눈 참고 수량입니다.',
            '백테스트MDD': 'MDD는 최대낙폭입니다. 낮을수록 과거 검증 구간에서 흔들림이 적었다는 뜻입니다.',
            '백테스트승률': '백테스트 상세 매매기록 중 수익으로 끝난 비율입니다.',
            '백테스트신뢰도': '거래횟수·MDD·수익률·검증기간을 합산한 내부 신뢰도 등급입니다. 6개월 결과를 과신하지 않도록 돕습니다.'
        }
        apply_header_comments(ws_list, recommendation_comments)


        # v9.2 TOP 후보 요약 시트 스타일: 핵심 열만 가로로 짧게 보기
        if 'TOP후보_요약' in wb.sheetnames:
            ws_top = writer.sheets['TOP후보_요약']
            ws_top.sheet_view.showGridLines = False
            ws_top.sheet_view.zoomScale = CONFIG.get('UX_ZOOM_SCALE', 90)
            ws_top.sheet_properties.tabColor = '0B1F3A'
            style_header(ws_top, 1, '0B1F3A')
            ws_top.freeze_panes = 'A2'
            ws_top.auto_filter.ref = ws_top.dimensions
            compact_widths = {
                'AI 시스템 판정':18, '종목명':16, '종목코드':12, '후보출처':12, '현재가':12, '등락률':10,
                '종목점수':10, '추세필터판정':16, '안전필터판정':18, '재무위험판정':18, '재무등급':10,
                '재무안전등급':14, '백테스트수익률':14, '백테스트MDD':12, '백테스트승률':12,
                '백테스트신뢰도':14, '권장익절계획':22, '권장손절계획':22, '추천투입금액':14,
                '추천수량':12, '상세 전략 가이드':38
            }
            for cell in ws_top[1]:
                letter = get_column_letter(cell.column)
                ws_top.column_dimensions[letter].width = compact_widths.get(cell.value, 14)
                add_comment(cell, recommendation_comments.get(cell.value, 'TOP후보_요약은 오른쪽 스크롤을 줄이기 위해 핵심 열만 따로 모은 화면입니다.'))
            for row in ws_top.iter_rows(min_row=2, max_row=ws_top.max_row):
                status_txt = str(row[0].value)
                row_fill = 'E2EFDA' if 'A급' in status_txt else ('FCE4D6' if any(k in status_txt for k in ['제외','소액','관찰','위험','과열','관망']) else 'FFFFFF')
                for cell in row:
                    cell.border = thin
                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                    cell.fill = PatternFill(start_color=row_fill, end_color=row_fill, fill_type='solid')
                if ws_top.max_column >= 17:
                    row[16].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                if ws_top.max_column >= 18:
                    row[17].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                ws_top.row_dimensions[row[0].row].height = 30

        # v9.2 추천 리스트는 상세 DB로 유지하되, 덜 중요한 열은 기본 숨김 처리
        if CONFIG.get('UX_HIDE_DETAIL_COLUMNS', True):
            keep_cols = set([
                'AI 시스템 판정', '종목명', '종목코드', '후보출처', '현재가', '등락률', '거래대금',
                '종목점수', '추세필터판정', '안전필터판정', '시장경보', '재무등급', '재무안전등급',
                '재무위험판정', '백테스트수익률', '백테스트MDD', '백테스트승률', '백테스트신뢰도',
                '권장익절계획', '권장손절계획', '추천투입금액', '추천수량', '상세 전략 가이드'
            ])
            for cell in ws_list[1]:
                col_letter = get_column_letter(cell.column)
                if cell.value not in keep_cols:
                    ws_list.column_dimensions[col_letter].hidden = True
                    ws_list.column_dimensions[col_letter].outlineLevel = 1
            ws_list.sheet_view.zoomScale = CONFIG.get('UX_ZOOM_SCALE', 90)
            ws_list.sheet_properties.tabColor = '7F7F7F'

        for row in ws_list.iter_rows(min_row=2, max_row=ws_list.max_row):
            for cell in row:
                cell.border = thin
                cell.alignment = left if ws_list.cell(1, cell.column).value == '상세 전략 가이드' else center
            status = str(row[0].value)
            if 'A급' in status:
                row[0].fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
                row[0].font = Font(color='375623', bold=True)
            elif '매수 제외' in status or '소액' in status or '재무위험' in status or '과열' in status or '관망' in status:
                row[0].fill = PatternFill(start_color='FCE4D6', end_color='FCE4D6', fill_type='solid')
                row[0].font = Font(color='C65911', bold=True)

        # 스나이퍼 계산기
        ws2 = wb.create_sheet('스나이퍼_계산기')
        ws2.sheet_view.showGridLines = False
        sub_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
        input_fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
        auto_fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
        warn_fill = PatternFill(start_color='FCE4D6', end_color='FCE4D6', fill_type='solid')

        ws2['B2'] = '🎯 v9.6.2 단기/스윙 투자 파트너 — 스나이퍼 계산 보정'
        ws2['B2'].font = Font(size=16, bold=True, color='1F4E78')
        ws2.merge_cells('B2:I2')

        default_sniper_stock = (final_df['종목명'].dropna().iloc[0] if len(final_df) and '종목명' in final_df.columns and len(final_df['종목명'].dropna()) else '종목을 선택하세요')

        dash_rows = [
            ('리포트 기준일', REPORT_DATE, 'auto'),
            ('1. 종목 선택', default_sniper_stock, 'input'),
            ('2. 계좌 총액', CONFIG['ACCOUNT_SIZE'], 'input'),
            ('3. 1회 허용 손실률', CONFIG['RISK_PER_TRADE_RATIO'], 'input'),
            ('4. 현재 주가', lookup_formula('현재가'), 'auto'),
            ('5. 종목점수', lookup_formula('종목점수'), 'auto'),
            ('6. 추천투입금액', lookup_formula('추천투입금액'), 'auto'),
            ('7. 추천수량', lookup_formula('추천수량'), 'auto'),
            ('8. 내가 실제 투입할 금액', '=C9', 'input'),
            ('9. 직접 입력 수량(선택)', 0, 'input'),
            ('10. 실사용 기준수량', '=IF(C12>0,C12,IF(AND(C11>0,C7>0),INT(C11/C7),C10))', 'auto'),
            ('11. 실제 매수예정금액', '=C13*C7', 'auto'),
            ('12. 계좌 대비 실제 비중', '=IF(C5>0,C14/C5,0)', 'auto'),
            ('13. 추천 익절 계획', lookup_formula('권장익절계획', '""'), 'auto'),
            ('14. 추천 손절 계획', lookup_formula('권장손절계획', '""'), 'auto'),
            ('15. 최적 매매 조합', lookup_formula('최적 매매 조합', '""'), 'auto'),
            ('16. 익절 전략 상세', lookup_formula('최적익절전략상세', '""'), 'auto'),
            ('17. 손절 전략 상세', lookup_formula('최적손절전략상세', '""'), 'auto'),
            ('18. 안전필터판정', lookup_formula('안전필터판정', '""'), 'auto'),
            ('19. 시장경보', lookup_formula('시장경보', '""'), 'auto'),
            ('20. 재무등급', lookup_formula('재무등급', '""'), 'auto'),
            ('21. PER', lookup_formula('PER', '""'), 'auto'),
            ('22. PBR', lookup_formula('PBR', '""'), 'auto'),
            ('23. ROE(추정)', lookup_formula('ROE(추정)', '""'), 'auto'),
            ('24. 재무위험사유', lookup_formula('재무위험사유', '""'), 'auto'),
            ('25. 백테스트 기준', '', 'auto')
        ]

        for i, (label, value, mode) in enumerate(dash_rows, start=3):
            ws2[f'B{i}'] = label
            ws2[f'B{i}'].fill = sub_fill
            ws2[f'B{i}'].font = Font(bold=True)
            ws2[f'B{i}'].border = thin
            ws2[f'B{i}'].alignment = center
            ws2[f'C{i}'] = value
            ws2[f'C{i}'].fill = input_fill if mode == 'input' else auto_fill
            ws2[f'C{i}'].border = thin
            ws2[f'C{i}'].alignment = center
        bt_range_formula = """=IFERROR(INDEX('추천 리스트'!$A:$ZZ,MATCH($C$4,'추천 리스트'!$B:$B,0),MATCH("백테스트시작일",'추천 리스트'!$1:$1,0)),"")&" ~ "&IFERROR(INDEX('추천 리스트'!$A:$ZZ,MATCH($C$4,'추천 리스트'!$B:$B,0),MATCH("백테스트종료일",'추천 리스트'!$1:$1,0)),"")"""
        for rr in range(3, 3 + len(dash_rows)):
            if ws2[f'B{rr}'].value == '25. 백테스트 기준':
                ws2[f'C{rr}'] = bt_range_formula
                break

        # 익절/손절 계획은 2줄 표시이므로 행 높이를 넉넉하게 잡습니다.
        for rr in [16, 17, 19, 20]:
            ws2.row_dimensions[rr].height = 34
            ws2[f'C{rr}'].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

        # 백테스트 기준은 해당 라벨 행에 자동 공식으로 계산됩니다.

        # 데이터 유효성: 추천 리스트의 종목명
        if len(final_df) > 0:
            dv = DataValidation(type='list', formula1=f"'추천 리스트'!$B$2:$B${len(final_df)+1}", allow_blank=True)
            ws2.add_data_validation(dv)
            dv.add(ws2['C4'])

        # 숫자 형식
        for cell in ['C5', 'C7', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']:
            ws2[cell].number_format = '#,##0'
        for cell in ['C6', 'C15']:
            ws2[cell].number_format = '0.00%'

        # 주요 입력/자동값 설명 메모
        memo_map = {
            'C4': '추천 리스트의 종목명 중 하나를 선택합니다. 선택한 종목 기준으로 아래 값이 자동 변경됩니다.',
            'C9': '시스템이 계좌총액·손절폭·시장모드·종목점수를 반영해 계산한 권장 금액입니다.',
            'C11': '기본값은 추천투입금액입니다. 실제로 넣고 싶은 금액이 다르면 이 칸에 직접 입력하세요.',
            'C12': '수량을 직접 정하고 싶을 때만 입력합니다. 0이면 실제투입금액 기준으로 수량을 자동 계산합니다.',
            'C13': '익절표와 손절표가 실제로 나눠 쓰는 기준수량입니다. 직접수량이 있으면 직접수량이 우선입니다.',
            'C15': '실제 매수예정금액이 계좌 전체에서 차지하는 비중입니다.',
            'C16': '윗줄은 익절 목표, 아랫줄 괄호는 그 목표에서 매도할 비중입니다. 예: +3% / +5% / +10% 아래 (60%) / (20%) / (20%).',
            'C17': '윗줄은 손절 기준, 아랫줄 괄호는 해당 기준에서 매도/손절할 비중입니다.',
            'C19': '예: +3%▶70% 익절은 주가가 +3% 도달 시 보유수량의 70%를 매도한다는 뜻입니다.',
            'C20': '예: -2%▶80% 손절은 주가가 -2% 하락 시 보유수량의 80%를 손절한다는 뜻입니다.'
        }
        for addr, text in memo_map.items():
            add_comment(ws2[addr], text)

        # 주문표 생성 함수
        def make_order_table(start_row, title, color, side):
            ws2[f'B{start_row}'] = title
            ws2[f'B{start_row}'].font = Font(bold=True, color='FFFFFF')
            ws2[f'B{start_row}'].fill = PatternFill(start_color=color, end_color=color, fill_type='solid')
            ws2.merge_cells(start_row=start_row, start_column=2, end_row=start_row, end_column=9)
            ws2[f'B{start_row}'].alignment = center

            headers = ['구분', '조건', '비중', '예약단가', '수량', '예상매도금액', '예상손익', '예상수익률']
            header_row = start_row + 1
            header_comments = {
                '조건': '몇 % 수익 또는 손실에 도달하면 해당 단계가 실행되는지 표시합니다.',
                '비중': '실사용 기준수량 중 해당 단계에서 매도할 비율입니다.',
                '예약단가': '현재가 기준 조건을 적용한 목표 매도/손절 가격입니다.',
                '수량': '실사용 기준수량을 전략 비중대로 나눈 수량입니다.',
                '예상매도금액': '예약단가 × 수량입니다.',
                '예상손익': '(예약단가 - 현재가) × 수량입니다. 손절표에서는 음수로 표시될 수 있습니다.',
                '예상수익률': '현재가 대비 해당 예약단가의 수익률입니다.'
            }
            for c, header in enumerate(headers, start=2):
                cell = ws2.cell(header_row, c, header)
                cell.fill = sub_fill
                cell.font = Font(bold=True)
                cell.alignment = center
                cell.border = thin
                if header in header_comments:
                    add_comment(cell, header_comments[header])

            label_prefix = '익절' if side == '익절' else '손절'
            row_labels = ['1차', '2차', '3차' if side == '익절' else '최종']
            first_data_row = start_row + 2
            for idx, label in enumerate(row_labels, start=1):
                r = first_data_row + idx - 1
                ws2.cell(r, 2, f'{label} {side}')
                ws2.cell(r, 3, lookup_formula(f'{label_prefix}{idx}조건', '""'))
                ws2.cell(r, 4, lookup_formula(f'{label_prefix}{idx}비중', '0'))
                ws2.cell(r, 5, lookup_formula(f'{label_prefix}{idx}가격', '0'))
                if idx < 3:
                    ws2.cell(r, 6, f'=IF($C$13>0,INT($C$13*D{r}),0)')
                else:
                    ws2.cell(r, 6, f'=MAX(0,$C$13-F{first_data_row}-F{first_data_row+1})')
                ws2.cell(r, 7, f'=E{r}*F{r}')
                ws2.cell(r, 8, f'=(E{r}-$C$7)*F{r}')
                ws2.cell(r, 9, f'=IF($C$7>0,E{r}/$C$7-1,0)')
                for c in range(2, 10):
                    ws2.cell(r, c).border = thin
                    ws2.cell(r, c).alignment = center
                ws2.cell(r, 4).number_format = '0%'
                ws2.cell(r, 5).number_format = '#,##0'
                ws2.cell(r, 6).number_format = '#,##0'
                ws2.cell(r, 7).number_format = '#,##0'
                ws2.cell(r, 8).number_format = '#,##0'
                ws2.cell(r, 9).number_format = '0.00%'

            # 수량 검증/합계 행
            check_row = first_data_row + 3
            ws2.cell(check_row, 2, '합계/검증')
            ws2.cell(check_row, 3, '=IF(SUM(F{}:F{})=$C$13,"정상","수량 확인")'.format(first_data_row, first_data_row+2))
            ws2.cell(check_row, 4, '=SUM(D{}:D{})'.format(first_data_row, first_data_row+2))
            ws2.cell(check_row, 6, '=SUM(F{}:F{})'.format(first_data_row, first_data_row+2))
            ws2.cell(check_row, 7, '=SUM(G{}:G{})'.format(first_data_row, first_data_row+2))
            ws2.cell(check_row, 8, '=SUM(H{}:H{})'.format(first_data_row, first_data_row+2))
            ws2.cell(check_row, 9, '=IF($C$14>0,H{}/$C$14,0)'.format(check_row))
            for c in range(2, 10):
                ws2.cell(check_row, c).border = thin
                ws2.cell(check_row, c).alignment = center
                ws2.cell(check_row, c).fill = warn_fill if c == 3 else auto_fill
            ws2.cell(check_row, 4).number_format = '0%'
            ws2.cell(check_row, 6).number_format = '#,##0'
            ws2.cell(check_row, 7).number_format = '#,##0'
            ws2.cell(check_row, 8).number_format = '#,##0'
            ws2.cell(check_row, 9).number_format = '0.00%'

        make_order_table(24, '📈 기준 익절표 — 실제투입금액/실사용수량 기준 예상손익 계산', 'C00000', '익절')
        make_order_table(33, '📉 기준 손절표 — 실제투입금액/실사용수량 기준 예상손익 계산', '002060', '손절')

        # 안내문
        ws2['B42'] = '사용 메모'
        ws2['B42'].fill = sub_fill
        ws2['B42'].font = Font(bold=True)
        ws2['C42'] = 'C11에 내가 실제 투입할 금액을 입력하면 C13 실사용 기준수량이 자동 계산되고, 익절/손절표의 수량·예상손익·예상수익률이 함께 바뀝니다. 수량을 직접 정하고 싶으면 C12에 수량을 입력하세요.'
        ws2.merge_cells('C42:I42')
        ws2['C42'].alignment = left

        for col, width in {'B':25, 'C':26, 'D':14, 'E':16, 'F':14, 'G':18, 'H':16, 'I':16}.items():
            ws2.column_dimensions[col].width = width

        # 자동매매 준비 시트: 실제 주문 전 검토용. 증권사 API 주문 실행은 포함하지 않습니다.
        ws_auto = wb.create_sheet('자동매매_준비')
        auto_rows = [
            ['단계', '상태', '설명'],
            ['1. 후보생성', '구현됨', '추천 리스트에서 A급/선별 후보를 생성합니다.'],
            ['2. 위험차단', '구현됨', '투자경고/위험, 거래정지, 관리종목, DART 재무위험을 먼저 걸러냅니다.'],
            ['3. 주문안 생성', '준비단계', '스나이퍼 계산기의 실제투입금액/수량을 주문 후보로 활용할 수 있습니다.'],
            ['4. 알림', '다음단계', '텔레그램/이메일로 후보와 손절·익절 경고를 보내는 단계입니다.'],
            ['5. 모의체결', '다음단계', '실제 주문 전 가상 체결/잔고/손익 로그를 최소 1~3개월 검증합니다.'],
            ['6. 증권사 API 주문', '미포함', '실제 주문 실행은 한국투자/LS 등 증권사 API 연동 후 별도 안전장치와 함께 구현해야 합니다.'],
            ['필수 안전장치', '중요', '일손실 제한, 연속손절 제한, 중복주문 방지, 금지종목 차단, 수동중지 버튼이 필요합니다.']
        ]
        for r, vals in enumerate(auto_rows, 1):
            for c, val in enumerate(vals, 1):
                cell = ws_auto.cell(r, c, val)
                cell.border = thin
                cell.alignment = left if c == 3 else center
        style_header(ws_auto, 1, 'C65911')
        ws_auto.column_dimensions['A'].width = 22
        ws_auto.column_dimensions['B'].width = 16
        ws_auto.column_dimensions['C'].width = 95

        # 리스크관리 시트
        ws_r = wb.create_sheet('리스크관리')
        risk_rows = [
            ['항목', '값', '판정/사용법'],
            ['리포트기준일', REPORT_DATE, '오늘 생성된 리포트 기준일'],
            ['판단일시', REPORT_DATETIME, '시장상태 계산 시각'],
            ['계좌총액', CONFIG['ACCOUNT_SIZE'], '스나이퍼 계산기 C5에서도 확인 가능'],
            ['오늘 손익률 입력', 0, '예: -3 이하이면 신규진입 중단'],
            ['이번 주 손익률 입력', 0, '예: -7 이하이면 이번 주 방어모드'],
            ['연속 손절 횟수 입력', 0, '4회 이상이면 강제 휴식'],
            ['현재 시장모드', market_mode, '공격/선별/방어/관망'],
            ['오늘 신규진입 원칙', '=IF(OR(B5<=-3,B6<=-7,B7>=4,B8="관망 모드"),"신규진입 금지 또는 최소화","조건부 진입 가능")', 'B5/B6/B7은 실제 결과 입력 후 사용']
        ]
        for r, vals in enumerate(risk_rows, 1):
            for c, val in enumerate(vals, 1):
                cell = ws_r.cell(r, c, val)
                cell.border = thin
                cell.alignment = center
        style_header(ws_r, 1, '7030A0')
        autosize_columns(ws_r, 50)

        # 매매일지 시트
        ws_j = wb.create_sheet('매매일지')
        journal_headers = ['날짜', '리포트기준일', '종목명', '종목코드', '진입가', '청산가', '수량', '실현손익', '수익률', '진입근거', '실패/성공 원인', '감정상태', '다음 개선점']
        ws_j.append(journal_headers)
        style_header(ws_j, 1, '548235')
        for row in range(2, 52):
            ws_j.cell(row, 1, REPORT_DATE)
            ws_j.cell(row, 2, REPORT_DATE)
            ws_j.cell(row, 8, f'=IF(AND(E{row}>0,F{row}>0,G{row}>0),(F{row}-E{row})*G{row},"")')
            ws_j.cell(row, 9, f'=IF(AND(E{row}>0,F{row}>0),(F{row}/E{row}-1),"")')
            ws_j.cell(row, 9).number_format = '0.00%'
        autosize_columns(ws_j, 40)

        # 백테스트 검증가이드 시트
        ws_guide = wb.create_sheet('백테스트검증가이드')
        guide_rows = [
            ['구분', '판단', '설명', '권장 보완'],
            ['6개월 백테스트', '부분적으로 유의미', '최근 장세에 전략이 맞았는지 보는 데 유용하지만, 시장 국면이 하나로 치우치면 과신 위험이 큽니다.', '최근 6개월 + 12개월 + 24개월을 함께 비교합니다.'],
            ['거래횟수', '매우 중요', '6개월 수익률이 높아도 거래횟수가 너무 적으면 우연일 수 있습니다.', '전략별 최소 30회 이상, 가능하면 50회 이상 확인합니다.'],
            ['MDD', '수익률보다 먼저 확인', '최대낙폭이 크면 실제 운용에서 버티기 어렵습니다.', '수익률이 낮아도 MDD가 작고 회복이 빠른 전략을 우선합니다.'],
            ['워크포워드', '필수 보완', '과거 일부 구간에 맞춘 과최적화를 줄이기 위해 훈련구간과 검증구간을 나눕니다.', '예: 3개월 조건선정 → 다음 1개월 검증을 반복합니다.'],
            ['시장국면 분리', '필수 보완', '상승장/박스장/하락장에서는 같은 조건도 성과가 달라집니다.', '코스닥 20일선 위/아래, 시장점수 55 이상/미만으로 나눠 검증합니다.'],
            ['안전필터', '실전 필수', '투자경고/위험 종목은 기술점수가 높아도 급락·거래제한 위험이 큽니다.', '시장경보는 자동+수동 CSV로 이중 확인합니다.'],
            ['기업가치 필터', '위험 회피용', 'PER/PBR/ROE는 단기 타이밍 신호가 아니라 위험한 기업을 피하는 필터입니다.', '적자/PER 산출불가, PBR 과도, ROE 낮음은 감점 또는 관찰 처리합니다.'],
            ['DART 재무안전', '실전 필수', '영업손실·순손실·부채비율 과다·자본잠식 위험·감사의견 문제는 급등주라도 큰 리스크가 됩니다.', 'DART API 키가 없으면 DART_재무수동입력.csv에 직접 입력해 보완합니다.'],
            ['백테스트신뢰도', '과신 방지', '6개월 수익률만으로 판단하지 않고 거래횟수·MDD·검증기간을 함께 점수화합니다.', '신뢰도 C/D는 실전 투입 전 모의매매 또는 조건수정이 필요합니다.'],
            ['추천 조건 수정안', 'v9.1 기본방향', '토스/네이버 후보를 1차 후보로만 보고 안전필터→재무필터→추세필터→백테스트 신뢰도 순서로 최종 검증합니다.', 'A급 후보만 기본 진입, 선별 후보는 비중 축소, 소액 관찰은 원칙적으로 모의매매부터 확인합니다.'],
            ['후보소스 통합', 'v9.1 신규', 'CONFIG의 UNIVERSE_SOURCE 값을 TOSS/NAVER/HYBRID로 바꿔 후보군 출처를 선택합니다.', '토스 수집 실패 시 네이버 후보로 자동 대체할 수 있습니다.'],
            ['컴팩트 대시보드', 'v9.2 신규', '홈_브리핑과 TOP후보_요약에서 오른쪽 스크롤 없이 핵심 정보만 먼저 확인합니다.', '추천 리스트는 상세 DB로 유지하며 덜 중요한 열은 기본 숨김 처리합니다.']
        ]
        for r, vals in enumerate(guide_rows, 1):
            for c, val in enumerate(vals, 1):
                cell = ws_guide.cell(r, c, val)
                cell.border = thin
                cell.alignment = left if c in [3, 4] else center
        style_header(ws_guide, 1, '305496')
        ws_guide.freeze_panes = 'A2'
        ws_guide.auto_filter.ref = ws_guide.dimensions
        for col, width in {'A':22, 'B':18, 'C':70, 'D':65}.items():
            ws_guide.column_dimensions[col].width = width

        # 용어설명 시트
        ws_terms = wb.create_sheet('용어설명')
        term_rows = [
            ['용어', '뜻', '어디서 확인', '사용 팁'],
            ['후보출처', '종목이 TOSS_XLSX, TOSS_CSV, TOSS_AUTO, NAVER, HYBRID 중 어떤 1차 후보군에서 들어왔는지 표시합니다.', 'TOP후보_요약 / 추천 리스트', '토스/네이버 후보는 바로 매수 추천이 아니라 내부 필터 검증 전 1차 후보입니다.'],
            ['TOSS_XLSX', '토스 스크리너에서 복사해 만든 엑셀 첨부파일에서 추출한 후보입니다.', '토스후보_정리결과 / 추천 리스트', '토스 웹 자동수집 실패를 피하기 위한 가장 안정적인 입력 방식입니다.'],
            ['홈_브리핑', '엑셀을 열었을 때 가장 먼저 보는 요약 화면입니다. 시장모드, 후보 수, A급 후보 수, 빠른 이동 링크를 보여줍니다.', '홈_브리핑', '여기서 TOP후보_요약과 스나이퍼 계산기로 이동하면 오른쪽 스크롤을 줄일 수 있습니다.'],
            ['TOP후보_요약', '추천 리스트의 핵심 열만 모은 짧은 화면입니다. 오른쪽 스크롤을 줄이기 위한 실사용 화면입니다.', 'TOP후보_요약', '상세한 재무/백테스트 데이터가 필요할 때만 추천 리스트를 확인합니다.'],
            ['시장경보', '한국거래소 시장감시제도에서 투자주의·투자경고·투자위험 등으로 공표되는 위험 신호입니다.', '추천 리스트 / 제외종목_위험필터', '투자경고·투자위험·거래정지는 기술점수와 관계없이 매수 제외로 봅니다.'],
            ['투자주의', '투기적 또는 불공정거래 개연성이 있는 종목에 대해 투자자 주의를 환기하는 단계입니다.', '추천 리스트', 'v9.1에서는 소액 관찰로 분리하고 기본 매수 후보보다 낮게 봅니다.'],
            ['투자경고/투자위험', '주가 급등 등으로 거래소가 더 높은 수준의 시장경보를 낸 단계입니다.', '제외종목_위험필터', '실전에서는 원칙적으로 매수 제외 또는 모의관찰로 처리하는 것이 안전합니다.'],
            ['PER', '주가수익비율입니다. 주가가 이익 대비 어느 정도 비싼지 보는 지표입니다.', '추천 리스트', '단기/스윙에서는 매수 타이밍이 아니라 위험 필터로 사용합니다.'],
            ['PBR', '주가순자산비율입니다. 주가가 순자산 대비 어느 정도인지 보는 지표입니다.', '추천 리스트', '너무 높으면 테마 과열 또는 가치부담 가능성을 봅니다.'],
            ['ROE(추정)', 'EPS/BPS로 단순 계산한 참고 수익성 지표입니다.', '추천 리스트', '낮거나 음수이면 수익성 부담으로 감점합니다.'],
            ['재무등급', 'PER/PBR/ROE/시가총액과 DART 재무안전 점수를 함께 반영한 내부 안전 등급입니다.', '추천 리스트', 'A/B는 상대적으로 양호, C는 선별, D는 관찰 또는 제외 후보입니다.'],
            ['DART 재무안전등급', '영업이익·순이익·부채비율·자본총계·감사의견을 반영한 재무 안전 등급입니다.', '추천 리스트 / 스나이퍼_계산기', '급등주라도 DART 재무위험이 있으면 소액 관찰 또는 제외로 봅니다.'],
            ['부채비율', '부채총계÷자본총계×100입니다. 회사가 자기자본 대비 얼마나 많은 부채를 쓰는지 봅니다.', '추천 리스트', '180% 이상은 주의, 250% 이상은 강한 감점 기준으로 설정했습니다.'],
            ['영업이익', '본업에서 벌어들인 이익입니다.', '추천 리스트', '단기/스윙에서도 영업손실 기업은 변동성이 커질 수 있어 위험 필터로 봅니다.'],
            ['백테스트신뢰도', '거래횟수·MDD·수익률·검증기간을 함께 본 내부 검증 등급입니다.', '추천 리스트 / 전략백테스트요약', 'A/B는 참고 가능, C/D는 조건수정 또는 추가 검증이 필요합니다.'],
            ['워크포워드', '과거 일부 구간에 맞춘 전략을 다음 구간에서 검증하는 방식입니다.', '백테스트검증가이드', '3개월 조건선정→1개월 검증을 반복하면 과최적화 위험을 줄일 수 있습니다.'],
            ['실전비용반영', '전략백테스트에 매수비용·매도비용·슬리피지를 반영했는지 표시합니다.', '전략백테스트요약 / 추천 리스트', '예로 표시된 수익률은 비용 차감 후 결과라 실전성과 더 가깝습니다.'],
            ['슬리피지', '원하는 가격과 실제 체결 가격 사이의 불리한 차이를 말합니다.', '전략백테스트상세 / 계좌백테스트거래내역', '시장가 또는 급등락 종목에서는 슬리피지가 커질 수 있습니다.'],
            ['추세필터판정', '현재가와 20일/60일 이동평균선 관계로 추세를 구분한 값입니다.', '추천 리스트 / 계좌백테스트후보기록', '제외로 나오면 단기 반등이 보여도 매수 후보에서 빼는 쪽이 안전합니다.'],
            ['DART캐시', '한 번 조회한 DART 재무제표를 CSV로 저장해 일정 기간 재사용하는 구조입니다.', '추천 리스트 / DART_재무캐시.csv', '재무제표는 매일 바뀌지 않기 때문에 실행 속도와 API 제한 대응에 도움이 됩니다.'],
            ['TP전략', 'Take Profit. 익절 전략입니다. 목표 수익률에 도달하면 몇 %를 매도할지 정한 규칙입니다.', '전략백테스트요약 / 스나이퍼_계산기', '+3%▶70% 익절은 +3% 수익 도달 시 보유수량의 70%를 매도한다는 뜻입니다.'],
            ['SL전략', 'Stop Loss. 손절 전략입니다. 손실 기준에 도달하면 몇 %를 매도할지 정한 규칙입니다.', '전략백테스트요약 / 스나이퍼_계산기', '-2%▶80% 손절은 -2% 손실 도달 시 보유수량의 80%를 손절한다는 뜻입니다.'],
            ['MDD', 'Maximum Drawdown. 백테스트 중 고점 대비 최대 하락폭입니다.', '전략백테스트요약 / 추천 리스트', '수익률이 좋아도 MDD가 너무 크면 실제 운용에서 버티기 어려울 수 있습니다.'],
            ['승률', '전체 청산 기록 중 수익으로 끝난 비율입니다.', '전략백테스트요약', '승률만 보지 말고 MDD와 수익률을 함께 봅니다.'],
            ['품질점수', '수익률, 승률, MDD, 거래횟수를 함께 반영한 내부 점수입니다.', '전략백테스트요약', '최적 매매 조합을 고를 때 단순 수익률 과최적화를 줄이기 위한 참고점수입니다.'],
            ['RSI', '상대강도지수. 과매수/과매도 상태를 보는 지표입니다.', '추천 리스트', '70 이상은 과열 가능성, 30~45 구간은 눌림 가능성으로 참고합니다.'],
            ['TRIX', '추세 전환을 참고하는 지표입니다.', '추천 리스트 / 내부 계산', 'TRIX가 전일보다 상승하면 단기 추세 개선 신호로 참고합니다.'],
            ['ATR%', '평균 변동폭을 현재가 대비 %로 환산한 값입니다.', '추천 리스트', 'ATR%가 너무 높으면 손절폭이 커지고 흔들림이 클 수 있습니다.'],
            ['추천투입금액', '시스템이 제안하는 참고 투입금액입니다.', '추천 리스트 / 스나이퍼_계산기', '반드시 그대로 매수하라는 뜻은 아니며, 실제투입금액으로 다시 조정할 수 있습니다.'],
            ['실제투입금액', '내가 실제로 넣어볼 금액입니다.', '스나이퍼_계산기 C11', '이 값을 바꾸면 익절/손절표의 수량과 예상손익이 자동 변경됩니다.'],
            ['실사용 기준수량', '익절표와 손절표 계산에 실제로 쓰이는 수량입니다.', '스나이퍼_계산기 C13', '직접 입력 수량이 있으면 직접수량 우선, 없으면 실제투입금액 기준으로 계산됩니다.'],
            ['예상손익', '예약단가에 해당 수량을 매도했을 때의 손익 추정값입니다.', '스나이퍼_계산기 익절/손절표', '수수료, 세금, 슬리피지는 반영하지 않은 단순 추정입니다.'],
            ['예상수익률', '현재가 대비 예약단가의 수익률입니다.', '스나이퍼_계산기 익절/손절표', '익절표는 양수, 손절표는 음수로 표시되는 것이 정상입니다.'],
            ['백테스트시작일/종료일', '전략 성과를 계산한 검증 구간입니다.', '추천 리스트 / 전략백테스트요약', '구간이 짧거나 거래횟수가 너무 적으면 신뢰도가 낮을 수 있습니다.'],
            ['계좌백테스트', '초기자금으로 매일 후보를 사고팔았다고 가정해 전체 계좌 변화를 보는 검증입니다.', '계좌백테스트요약 / 계좌백테스트곡선', '종목 하나의 수익률이 아니라 현금과 보유종목까지 합친 실제 운용 흐름을 봅니다.'],
            ['평가자산', '현금과 보유주식 평가액을 합친 금액입니다.', '계좌백테스트곡선', '초기자금 200만 원 대비 얼마가 늘거나 줄었는지 확인합니다.'],
            ['다음날 시가 진입', '전일 종가 기준으로 신호를 보고 다음 거래일 시가에 산다고 가정하는 방식입니다.', '계좌백테스트거래내역', '미래 데이터를 보고 당일 저가에 산 것처럼 계산하는 오류를 줄이기 위한 보수적 방식입니다.']
        ]
        for r, vals in enumerate(term_rows, 1):
            for c, val in enumerate(vals, 1):
                cell = ws_terms.cell(r, c, val)
                cell.border = thin
                cell.alignment = left if c in [2, 4] else center
        style_header(ws_terms, 1, '8064A2')
        ws_terms.freeze_panes = 'A2'
        ws_terms.auto_filter.ref = ws_terms.dimensions
        for col, width in {'A':18, 'B':58, 'C':32, 'D':58}.items():
            ws_terms.column_dimensions[col].width = width

        # 안전필터 제외 종목
        if 'excluded_by_safety_df' in globals() and excluded_by_safety_df is not None and not excluded_by_safety_df.empty:
            excluded_by_safety_df.to_excel(writer, sheet_name='제외종목_위험필터', index=False)

        # 전략 요약/상세 스타일 및 메모
        summary_comments = {
            'TP전략': '익절 전략 이름입니다. TP는 Take Profit의 약자입니다.',
            'SL전략': '손절 전략 이름입니다. SL은 Stop Loss의 약자입니다.',
            'TP전략내용': '익절 조건과 물량 비중입니다. 예: +3%▶70% 익절',
            'SL전략내용': '손절 조건과 물량 비중입니다. 예: -2%▶80% 손절',
            'MDD': '백테스트 기간 중 고점 대비 최대 하락폭입니다.',
            '품질점수': '수익률, 승률, MDD, 거래횟수를 함께 반영한 내부 점수입니다.'
        }
        detail_comments = {
            'TP전략': '해당 매매기록에 적용된 익절 전략입니다.',
            'SL전략': '해당 매매기록에 적용된 손절 전략입니다.',
            '매수일': '백테스트에서 진입 조건이 발생한 날짜입니다.',
            '매도일': '익절/손절/강제청산이 발생한 날짜입니다.',
            '매도사유': '해당 매도가 발생한 조건입니다.',
            '수익률': '해당 청산 단가가 매수단가 대비 몇 %였는지 표시합니다.'
        }
        for sheet_name, color, comments in [('전략백테스트요약', '5B9BD5', summary_comments), ('전략백테스트상세', '70AD47', detail_comments)]:
            if sheet_name in wb.sheetnames:
                ws_s = writer.sheets[sheet_name]
                style_header(ws_s, 1, color)
                apply_header_comments(ws_s, comments)
                autosize_columns(ws_s, 45)
                ws_s.freeze_panes = 'A2'
                ws_s.auto_filter.ref = ws_s.dimensions

        # v8.4 계좌 백테스트 시트 스타일
        portfolio_comments = {
            '초기자금': '백테스트 시작 시점의 현금입니다. 기본값은 2,000,000원입니다.',
            '최종자산': '백테스트 종료 후 보유 종목을 모두 청산했다고 가정한 금액입니다.',
            '누적수익률': '초기자금 대비 최종자산의 변화율입니다.',
            'MDD': '계좌 평가자산 고점 대비 최대 하락폭입니다.',
            '비용가정': '매수/매도 비용을 단순 비율로 반영한 값입니다. 실제 증권사 조건에 맞게 CONFIG에서 수정하세요.',
            '주의': '실제 체결, 호가, 슬리피지, 뉴스, 거래정지는 반영되지 않은 참고용 결과입니다.'
        }
        for sheet_name, color in [('계좌백테스트요약', 'C00000'), ('계좌백테스트곡선', '9E480E'), ('계좌백테스트거래내역', '833C0C'), ('계좌백테스트후보기록', 'A64D79'), ('백테스트검증가이드', '305496')]:
            if sheet_name in wb.sheetnames:
                ws_p = writer.sheets[sheet_name]
                style_header(ws_p, 1, color)
                apply_header_comments(ws_p, portfolio_comments)
                autosize_columns(ws_p, 45)
                ws_p.freeze_panes = 'A2'
                ws_p.auto_filter.ref = ws_p.dimensions

        # 전체 테두리/정렬 최소 적용
        for ws in wb.worksheets:
            for row in ws.iter_rows():
                for cell in row:
                    if cell.value is not None:
                        cell.border = thin
                        if cell.alignment is None or cell.alignment.horizontal is None:
                            cell.alignment = Alignment(vertical='center', wrap_text=True)

    return file_name

# v8.9에서는 최종 후보가 없더라도 시장상태/제외종목/검증가이드 리포트를 생성합니다.
if final_df.empty:
    print('⚠️ 최종 추천 후보가 없습니다. 안전필터 제외 목록과 검증가이드 중심으로 리포트를 생성합니다.')

output_file = build_excel_report(final_df, market_df, scenario_summary_df, trade_detail_df)
print(f'✅ 기본 엑셀 리포트 생성 완료: {output_file}')
print('➡️ 다음 v9.3 UI 강화 셀에서 카드형 대시보드와 HTML 리포트를 추가합니다.')


✅ 기본 엑셀 리포트 생성 완료: 실전매매_통합시스템_v9_7_5_4_KST최종저장일수정_20260608.xlsx
➡️ 다음 v9.3 UI 강화 셀에서 카드형 대시보드와 HTML 리포트를 추가합니다.


In [16]:
# v9.7.5.4: 날짜가 넘어간 뒤에도 최종 저장 시점(KST) 기준으로 갱신
refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))
# 8-1. v9.3 모던 UI 강화 — 카드형 엑셀 대시보드 + 브라우저 HTML 리포트
# 기존 엑셀 상세 데이터는 유지하고, 매일 보기 편한 첫 화면을 새로 만듭니다.

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def _v93_safe_text(value, default='-'):
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return default
        if pd.isna(value):
            return default
    except Exception:
        pass
    return str(value)


def _v93_safe_num(value, default=0):
    try:
        if value in ['', None] or pd.isna(value):
            return default
        return float(str(value).replace(',', '').replace('%', ''))
    except Exception:
        return default


def _v93_money(value):
    try:
        return f"{float(value):,.0f}원"
    except Exception:
        return '-'


def _v93_fill(hex_color):
    return PatternFill(start_color=hex_color, end_color=hex_color, fill_type='solid')


def _v93_font(color='111827', bold=False, size=10):
    return Font(name='맑은 고딕', color=color, bold=bold, size=size)


def _v93_rank_color(judgment):
    txt = _v93_safe_text(judgment)
    if 'A급' in txt or '공격' in txt:
        return 'DCFCE7', '166534'
    if '선별' in txt or '관찰' in txt:
        return 'FEF3C7', '92400E'
    if '제외' in txt or '위험' in txt:
        return 'FEE2E2', '991B1B'
    return 'E0F2FE', '075985'


def _v93_prepare_top_df(final_df, top_n=10):
    """v9.6.3: 전체 TOP + 토스 참고 후보를 합친 실제 화면용 통합 리스트."""
    if final_df is None or final_df.empty:
        return pd.DataFrame()
    df = final_df.copy()

    def _sector(row):
        for col in ['분야', '토스카테고리', '카테고리', '업종', '시장구분']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip() not in ['', 'nan', 'None']:
                return str(val).strip()
        return '확인필요'

    df['분야'] = df.apply(_sector, axis=1)
    sort_cols = [c for c in ['종목점수', '재무점수', '백테스트수익률', '거래대금(억)'] if c in df.columns]
    if sort_cols:
        for c in sort_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
        df = df.sort_values(sort_cols, ascending=False)
    overall = df.head(top_n).copy()
    overall['표시구분'] = '전체 TOP'

    toss_ref_n = int(CONFIG.get('UX_TOSS_REFERENCE_N', 5))
    toss_ref = pd.DataFrame()
    if '후보출처' in df.columns and toss_ref_n > 0:
        toss_mask = df['후보출처'].astype(str).str.contains('TOSS', case=False, na=False)
        toss_ref = df.loc[toss_mask].head(toss_ref_n).copy()
        toss_ref['표시구분'] = '토스 참고'

    out = pd.concat([overall, toss_ref], ignore_index=True)
    if '종목코드' in out.columns:
        out = out.drop_duplicates(subset=['종목코드'], keep='first')
    return out.reset_index(drop=True)


def _v93_get_market_brief(market_df):
    if market_df is None or market_df.empty:
        return {'mode': '확인필요', 'score': '-', 'memo': '시장상태 데이터가 부족합니다.'}
    txt = ' / '.join([f"{r.get('시장','')}: {r.get('상태','')} {r.get('점수','')}점" for _, r in market_df.head(3).iterrows()])
    score = pd.to_numeric(market_df.get('점수', pd.Series(dtype=float)), errors='coerce').mean()
    if pd.isna(score):
        mode = '확인필요'
    elif score >= 70:
        mode = '공격 가능'
    elif score >= 55:
        mode = '선별 매매'
    elif score >= 40:
        mode = '방어적 관찰'
    else:
        mode = '관망 우선'
    return {'mode': mode, 'score': round(score, 1) if not pd.isna(score) else '-', 'memo': txt}


def _v93_cell(ws, addr, value, fill=None, font=None, align=None, border=None):
    c = ws[addr]
    c.value = value
    # openpyxl StyleProxy 오류 방지: 기존 셀 스타일 객체를 그대로 대입하지 않고 복사해서 적용
    if fill: c.fill = copy(fill)
    if font: c.font = copy(font)
    if align: c.alignment = copy(align)
    if border: c.border = copy(border)
    return c


def _v93_merge_card(ws, cell_range, title, value, sub='', fill_color='F8FAFC', value_color='111827'):
    ws.merge_cells(cell_range)
    start = cell_range.split(':')[0]
    c = ws[start]
    c.value = f"{title}\n{value}" + (f"\n{sub}" if sub else '')
    c.fill = _v93_fill(fill_color)
    c.font = _v93_font(value_color, True, 12)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    thin = Side(style='thin', color='E5E7EB')
    c.border = Border(left=thin, right=thin, top=thin, bottom=thin)


def _v93_hide_sheet_if_exists(wb, name):
    if name in wb.sheetnames:
        try:
            wb[name].sheet_state = 'hidden'
        except Exception:
            pass


def enhance_excel_ui_v93(output_file, final_df, market_df, portfolio_summary_df=None):
    wb = load_workbook(output_file)
    for name in ['모바일_대시보드', '종목카드_TOP10', '오늘_체크리스트']:
        if name in wb.sheetnames:
            del wb[name]

    top_df = _v93_prepare_top_df(final_df, int(CONFIG.get('UX_TOP_N', 10)))
    market = _v93_get_market_brief(market_df)
    thin = Side(style='thin', color='E5E7EB')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)

    ws = wb.create_sheet('모바일_대시보드', 0)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 95
    for col, width in {'A':3, 'B':16, 'C':16, 'D':16, 'E':16, 'F':16, 'G':16, 'H':3}.items():
        ws.column_dimensions[col].width = width
    for r in range(1, 80):
        ws.row_dimensions[r].height = 24

    ws.merge_cells('B2:G3')
    _v93_cell(ws, 'B2', f"실전매매 브리핑 v9.3\n{REPORT_DATE} · {REPORT_DATETIME}", _v93_fill('0F172A'), _v93_font('FFFFFF', True, 18), center, border)

    candidate_count = len(final_df) if final_df is not None else 0
    top_count = len(top_df)
    first_name = _v93_safe_text(top_df.iloc[0].get('종목명')) if not top_df.empty else '후보 없음'
    first_judge = _v93_safe_text(top_df.iloc[0].get('AI 시스템 판정')) if not top_df.empty else '확인필요'

    _v93_merge_card(ws, 'B5:C8', '시장모드', market['mode'], f"시장점수 {market['score']}", 'E0F2FE', '075985')
    _v93_merge_card(ws, 'D5:E8', '분석 후보', f"{candidate_count}개", f"TOP {top_count} 표시", 'F1F5F9')
    _v93_merge_card(ws, 'F5:G8', '1순위 후보', first_name, first_judge, 'DCFCE7' if 'A급' in first_judge else 'FEF3C7', '166534' if 'A급' in first_judge else '92400E')

    ws.merge_cells('B10:G11')
    _v93_cell(ws, 'B10', f"시장 메모: {market['memo']}", _v93_fill('F8FAFC'), _v93_font('334155', False, 10), left, border)

    headers = ['순위', '판정', '종목명', '점수', '추세', '재무', '익절/손절']
    start_row = 13
    for idx, h in enumerate(headers, 2):
        cell = ws.cell(start_row, idx, h)
        cell.fill = _v93_fill('1E293B')
        cell.font = _v93_font('FFFFFF', True, 10)
        cell.alignment = center
        cell.border = border
    for i, (_, row) in enumerate(top_df.head(5).iterrows(), start_row + 1):
        vals = [i - start_row, row.get('AI 시스템 판정',''), row.get('종목명',''), row.get('종목점수',''), row.get('추세필터판정',''), row.get('재무위험판정',''), f"{row.get('권장익절계획','')}\n{row.get('권장손절계획','')}"]
        for j, v in enumerate(vals, 2):
            cell = ws.cell(i, j, v)
            cell.alignment = left if j in [3, 8] else center
            cell.font = _v93_font('111827', bold=(j==3), size=9)
            cell.border = border
            if j == 3:
                cell.fill = _v93_fill('F8FAFC')
            elif j == 2:
                fill, color = _v93_rank_color(v)
                cell.fill = _v93_fill(fill)
                cell.font = _v93_font(color, True, 9)
        ws.row_dimensions[i].height = 44

    ws.merge_cells('B22:G23')
    _v93_cell(ws, 'B22', '오늘 보는 순서  →  모바일_대시보드  →  종목카드_TOP10  →  스나이퍼_계산기  →  추천 리스트(상세)', _v93_fill('FEFCE8'), _v93_font('713F12', True, 11), center, border)

    wc = wb.create_sheet('종목카드_TOP10', 1)
    wc.sheet_view.showGridLines = False
    wc.sheet_view.zoomScale = 90
    for col, width in {'A':3, 'B':16, 'C':16, 'D':16, 'E':16, 'F':16, 'G':16, 'H':16, 'I':3}.items():
        wc.column_dimensions[col].width = width
    wc.merge_cells('B2:H3')
    _v93_cell(wc, 'B2', 'TOP 후보 카드 보기 — 오른쪽 스크롤 없이 핵심만 확인', _v93_fill('111827'), _v93_font('FFFFFF', True, 16), center, border)
    card_row = 5
    for idx, (_, row) in enumerate(top_df.head(10).iterrows(), 1):
        fill, color = _v93_rank_color(row.get('AI 시스템 판정',''))
        wc.merge_cells(start_row=card_row, start_column=2, end_row=card_row+1, end_column=8)
        header = wc.cell(card_row, 2)
        header.value = f"#{idx}  {row.get('종목명','')}  ·  {row.get('AI 시스템 판정','')}  ·  점수 {row.get('종목점수','-')}"
        header.fill = _v93_fill(fill)
        header.font = _v93_font(color, True, 13)
        header.alignment = left
        header.border = border
        info_rows = [
            ['현재가', row.get('현재가',''), '후보출처', row.get('후보출처',''), '추세', row.get('추세필터판정',''), '재무', row.get('재무위험판정','')],
            ['익절계획', row.get('권장익절계획',''), '손절계획', row.get('권장손절계획',''), '백테스트', row.get('백테스트신뢰도',''), '리스크', row.get('시장경보','')],
            ['가이드', str(row.get('상세 전략 가이드',''))[:220], '', '', '', '', '', '']
        ]
        for rr, vals in enumerate(info_rows, card_row+2):
            for cc, val in enumerate(vals, 2):
                cell = wc.cell(rr, cc, val)
                cell.border = border
                cell.alignment = left if cc in [3,5,7] or rr == card_row+4 else center
                cell.font = _v93_font('111827', False, 9)
                if cc in [2,4,6,8] and rr != card_row+4:
                    cell.fill = _v93_fill('F8FAFC')
                    cell.font = _v93_font('475569', True, 9)
        wc.merge_cells(start_row=card_row+4, start_column=3, end_row=card_row+4, end_column=8)
        wc.row_dimensions[card_row].height = 26
        wc.row_dimensions[card_row+1].height = 4
        wc.row_dimensions[card_row+2].height = 28
        wc.row_dimensions[card_row+3].height = 34
        wc.row_dimensions[card_row+4].height = 54
        card_row += 7

    wk = wb.create_sheet('오늘_체크리스트', 2)
    wk.sheet_view.showGridLines = False
    for col, width in {'A':4, 'B':22, 'C':48, 'D':18, 'E':44}.items():
        wk.column_dimensions[col].width = width
    wk.merge_cells('B2:D3')
    _v93_cell(wk, 'B2', '매수 전 체크리스트 — 자동매매 전 단계', _v93_fill('0F172A'), _v93_font('FFFFFF', True, 16), center, border)
    # v9.6.2: 전부 직접확인이 아니라, 시스템이 판정 가능한 항목은 자동판정+근거를 표시
    focus = top_df.iloc[0].to_dict() if not top_df.empty else {}

    def _ck_market():
        mode = str(market.get('mode', '확인필요'))
        if '공격' in mode or '선별' in mode:
            return '통과', f'시장모드: {mode}'
        if '방어' in mode:
            return '주의', f'시장모드: {mode} — 비중 축소'
        return '직접확인', f'시장모드: {mode}'

    def _ck_alert():
        txt = str(focus.get('시장경보', '미확인'))
        if re.search(r'투자경고|투자위험|거래정지|관리종목|상장폐지|불성실공시|환기', txt):
            return '차단', txt
        if re.search(r'투자주의|지정예고|단기과열', txt):
            return '주의', txt
        if txt and txt not in ['미확인', 'nan', 'None']:
            return '통과', txt
        return '직접확인', '시장경보 자동확인 데이터 부족'

    def _ck_finance():
        txt = str(focus.get('재무위험판정', '확인필요'))
        if re.search(r'매수제외|자본잠식|감사의견|위험', txt):
            return '차단', txt
        if re.search(r'주의|관찰|확인|데이터부족', txt):
            return '주의', txt
        return '통과', txt

    def _ck_trend():
        txt = str(focus.get('추세필터판정', '확인필요'))
        if '제외' in txt:
            return '차단', txt
        if '안정형' in txt:
            return '통과', txt
        if txt and txt not in ['확인필요', 'nan', 'None']:
            return '주의', txt
        return '직접확인', txt

    def _ck_backtest():
        try:
            ret = float(str(focus.get('백테스트수익률', '')).replace('%', '').replace(',', ''))
        except Exception:
            return '직접확인', '백테스트수익률 확인 필요'
        conf = str(focus.get('백테스트신뢰도', ''))
        if ret > 0 and ('A' in conf or 'B' in conf):
            return '통과', f'비용차감 후 {ret:.2f}% / 신뢰도 {conf}'
        if ret > 0:
            return '주의', f'수익률 {ret:.2f}% / 신뢰도 {conf}'
        return '주의', f'수익률 {ret:.2f}% — 손익비 재확인'

    mkt_j, mkt_e = _ck_market()
    alert_j, alert_e = _ck_alert()
    fin_j, fin_e = _ck_finance()
    trend_j, trend_e = _ck_trend()
    bt_j, bt_e = _ck_backtest()

    checklist = [
        ['구분', '확인 질문', '자동/수동판정', '근거'],
        ['시장', '오늘 시장모드가 관망/방어가 아닌가?', mkt_j, mkt_e],
        ['경고', '투자주의·투자경고·거래정지·관리종목이 아닌가?', alert_j, alert_e],
        ['재무', 'DART 재무위험판정이 매수제외가 아닌가?', fin_j, fin_e],
        ['추세', '현재가가 60일선 위 또는 최소 관찰 통과인가?', trend_j, trend_e],
        ['비용', '비용차감 후 백테스트가 플러스이거나 손익비가 납득되는가?', bt_j, bt_e],
        ['수량', '스나이퍼 계산기에서 실제투입금액을 본인 기준으로 낮췄는가?', '직접확인', '개인 자금/손실허용 기준은 자동판단 불가'],
        ['뉴스', '공시·뉴스·호가창에 악재가 없는가?', '직접확인', '장중 뉴스·공시·호가창은 별도 확인 필요'],
        ['중단', '오늘 손실제한/연속손절 조건에 걸리지 않았는가?', '직접확인', '실제 매매일지 기준으로 확인'],
    ]
    for r, vals in enumerate(checklist, 5):
        for c, val in enumerate(vals, 2):
            cell = wk.cell(r, c, val)
            cell.border = border
            cell.alignment = left if c in [3, 5] else center
            cell.font = _v93_font('111827', c == 2 or r == 5, 10)
            if r == 5:
                cell.fill = _v93_fill('1E293B')
                cell.font = _v93_font('FFFFFF', True, 10)
            elif c == 4:
                txt = str(val)
                if '통과' in txt:
                    cell.fill = _v93_fill('DCFCE7'); cell.font = _v93_font('166534', True, 10)
                elif '차단' in txt:
                    cell.fill = _v93_fill('FEE2E2'); cell.font = _v93_font('991B1B', True, 10)
                elif '주의' in txt:
                    cell.fill = _v93_fill('FEF3C7'); cell.font = _v93_font('92400E', True, 10)
                else:
                    cell.fill = _v93_fill('E0F2FE'); cell.font = _v93_font('075985', True, 10)
            elif c == 5:
                cell.fill = _v93_fill('F8FAFC')

    if CONFIG.get('UX_HIDE_RAW_DETAIL_SHEETS', True):
        for name in ['추천 리스트', '전략백테스트상세', '계좌백테스트후보기록', '후보군_원본']:
            _v93_hide_sheet_if_exists(wb, name)
    wb.active = 0
    wb.save(output_file)
    return output_file


def generate_html_dashboard_v93(output_file, final_df, market_df, portfolio_summary_df=None):
    html_file = output_file.replace('.xlsx', '_모바일대시보드.html')
    top_df = _v93_prepare_top_df(final_df, 10)
    market = _v93_get_market_brief(market_df)
    cards = []
    for idx, row in top_df.iterrows():
        judge = _v93_safe_text(row.get('AI 시스템 판정'))
        badge_class = 'good' if 'A급' in judge else ('warn' if ('선별' in judge or '관찰' in judge) else 'bad' if '제외' in judge or '위험' in judge else 'neutral')
        cards.append(f'''
        <section class="stock-card">
          <div class="stock-head"><div><span class="rank">#{idx+1}</span><h2>{_v93_safe_text(row.get('종목명'))}</h2></div><span class="badge {badge_class}">{judge}</span></div>
          <div class="metric-grid">
            <div><label>현재가</label><strong>{_v93_money(row.get('현재가'))}</strong></div>
            <div><label>점수</label><strong>{_v93_safe_text(row.get('종목점수'))}</strong></div>
            <div><label>후보출처</label><strong>{_v93_safe_text(row.get('후보출처'))}</strong></div>
            <div><label>백테스트</label><strong>{_v93_safe_text(row.get('백테스트신뢰도'))}</strong></div>
          </div>
          <div class="pill-row"><span>추세: {_v93_safe_text(row.get('추세필터판정'))}</span><span>재무: {_v93_safe_text(row.get('재무위험판정'))}</span><span>경보: {_v93_safe_text(row.get('시장경보'))}</span></div>
          <div class="plan"><p><b>익절</b> {_v93_safe_text(row.get('권장익절계획'))}</p><p><b>손절</b> {_v93_safe_text(row.get('권장손절계획'))}</p></div>
          <details><summary>상세 가이드 보기</summary><p>{_v93_safe_text(row.get('상세 전략 가이드'))}</p></details>
        </section>''')
    if not cards:
        cards.append('<section class="stock-card"><h2>후보 없음</h2><p>오늘 조건을 통과한 후보가 없습니다. 시장상태와 제외종목 시트를 확인하세요.</p></section>')

    html = f'''<!doctype html>
<html lang="ko"><head><meta charset="utf-8" /><meta name="viewport" content="width=device-width, initial-scale=1" /><title>실전매매 v9.3 모바일 대시보드</title>
<style>
:root {{ --bg:#f6f7fb; --card:#ffffff; --text:#111827; --muted:#64748b; --line:#e5e7eb; }}
* {{ box-sizing:border-box; }} body {{ margin:0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI','Noto Sans KR',Arial,sans-serif; background:var(--bg); color:var(--text); }}
header {{ background:linear-gradient(135deg,#0f172a,#1e40af); color:white; padding:28px 20px 36px; border-radius:0 0 28px 28px; }} header h1 {{ margin:0 0 8px; font-size:24px; }} header p {{ margin:0; color:#dbeafe; }}
.wrap {{ max-width:1080px; margin:-24px auto 40px; padding:0 16px; }} .summary {{ display:grid; grid-template-columns:repeat(3,minmax(0,1fr)); gap:14px; margin-bottom:16px; }}
.summary .box,.stock-card {{ background:var(--card); border:1px solid var(--line); border-radius:24px; padding:18px; box-shadow:0 10px 30px rgba(15,23,42,.08); }} .summary label {{ display:block; color:var(--muted); font-size:13px; margin-bottom:8px; }} .summary strong {{ font-size:24px; }}
.stock-card {{ margin:14px 0; }} .stock-head {{ display:flex; justify-content:space-between; gap:12px; align-items:flex-start; border-bottom:1px solid var(--line); padding-bottom:12px; }} .stock-head h2 {{ display:inline; margin:0; font-size:22px; }}
.rank {{ display:inline-flex; align-items:center; justify-content:center; width:34px; height:34px; border-radius:12px; background:#eef2ff; color:#3730a3; font-weight:800; margin-right:10px; }} .badge {{ padding:8px 12px; border-radius:999px; font-size:13px; font-weight:800; white-space:nowrap; }}
.badge.good {{ background:#dcfce7; color:#166534; }} .badge.warn {{ background:#fef3c7; color:#92400e; }} .badge.bad {{ background:#fee2e2; color:#991b1b; }} .badge.neutral {{ background:#e0f2fe; color:#075985; }}
.metric-grid {{ display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:10px; margin:16px 0; }} .metric-grid div {{ background:#f8fafc; border-radius:16px; padding:12px; }} .metric-grid label {{ display:block; color:var(--muted); font-size:12px; margin-bottom:6px; }} .metric-grid strong {{ font-size:16px; }}
.pill-row {{ display:flex; flex-wrap:wrap; gap:8px; margin:8px 0 14px; }} .pill-row span {{ background:#eef2ff; color:#3730a3; padding:7px 10px; border-radius:999px; font-size:12px; font-weight:700; }} .plan {{ display:grid; grid-template-columns:1fr 1fr; gap:10px; }} .plan p {{ margin:0; background:#fafafa; border:1px solid var(--line); border-radius:14px; padding:12px; }}
details {{ margin-top:12px; color:#334155; }} summary {{ cursor:pointer; font-weight:800; }} footer {{ color:#64748b; text-align:center; padding:24px 10px 36px; font-size:12px; }}
@media (max-width:760px) {{ .summary,.metric-grid,.plan {{ grid-template-columns:1fr; }} header h1 {{ font-size:21px; }} .stock-head {{ flex-direction:column; }} }}
</style></head><body>
<header><h1>실전매매 통합 시스템 v9.3</h1><p>{REPORT_DATE} · {REPORT_DATETIME} · 투자 판단 보조용 리포트</p></header>
<main class="wrap"><section class="summary"><div class="box"><label>시장모드</label><strong>{market['mode']}</strong><p>{market['memo']}</p></div><div class="box"><label>분석 후보</label><strong>{len(final_df) if final_df is not None else 0}개</strong><p>TOSS/NAVER/HYBRID 후보 검증 결과</p></div><div class="box"><label>상위 후보</label><strong>{_v93_safe_text(top_df.iloc[0].get('종목명')) if not top_df.empty else '없음'}</strong><p>상세는 엑셀 스나이퍼 계산기에서 확인</p></div></section>{''.join(cards)}</main>
<footer>본 자료는 투자 판단 보조용이며 매수·매도 권유가 아닙니다. 실제 매매 전 뉴스·공시·호가·본인 손실한도를 확인하세요.</footer></body></html>'''
    with open(html_file, 'w', encoding='utf-8') as f:
        f.write(html)
    return html_file


if CONFIG.get('UX_CARD_DASHBOARD_ENABLED', True):
    output_file = enhance_excel_ui_v93(output_file, final_df, market_df, portfolio_summary_df if 'portfolio_summary_df' in globals() else None)
    print(f'✅ v9.3 카드형 엑셀 UI 적용 완료: {output_file}')

html_output_file = None
if CONFIG.get('UX_HTML_DASHBOARD_ENABLED', True):
    html_output_file = generate_html_dashboard_v93(output_file, final_df, market_df, portfolio_summary_df if 'portfolio_summary_df' in globals() else None)
    print(f'✅ v9.3 HTML 대시보드 생성 완료: {html_output_file}')

# v9.4 백테스트 실험실 셀에서 엑셀을 한 번 더 보강한 뒤 다운로드합니다.


✅ v9.3 카드형 엑셀 UI 적용 완료: 실전매매_통합시스템_v9_7_5_4_KST최종저장일수정_20260608.xlsx
✅ v9.3 HTML 대시보드 생성 완료: 실전매매_통합시스템_v9_7_5_4_KST최종저장일수정_20260608_모바일대시보드.html


In [17]:

# 8-2. v9.6.2 백테스트 실험실 — 최고 수익 전략 표시 + INDEX/MATCH 안정화
# 사용자가 원하는 종목과 전략을 선택하면, 엑셀 안에서 매매기록을 바로 볼 수 있는 실험실 시트를 추가합니다.

from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def _lab_safe_date(value):
    try:
        return pd.to_datetime(value).strftime('%Y-%m-%d')
    except Exception:
        return ''


def _lab_strategy_combo(tp, sl):
    return f"{tp['name']}+{sl['name']}"


def _lab_ratio_for_reason(reason, tp, sl):
    """매도사유에 적힌 단계/조건에 맞춰 원칙 비중을 찾아 설명문에 넣습니다."""
    reason = str(reason)
    m = re.search(r'\((-?\d+(?:\.\d+)?)%\)', reason)
    if not m:
        return ''
    try:
        pct = float(m.group(1))
    except Exception:
        return ''
    if '익절' in reason:
        for cond, ratio in zip(tp['cond'], tp['ratio']):
            if abs(float(cond) - pct) < 1e-9:
                return f"{fmt_pct(cond)} 상승 시 {int(round(ratio*100))}% 익절 원칙"
    if '손절' in reason:
        for cond, ratio in zip(sl['cond'], sl['ratio']):
            if abs(float(cond) - pct) < 1e-9:
                return f"{fmt_pct(cond)} 하락 시 {int(round(ratio*100))}% 손절 원칙"
    return ''


def _lab_record_memo(log_row, tp, sl):
    sell_date = _lab_safe_date(log_row.get('매도일'))
    try:
        month_day = pd.to_datetime(sell_date).strftime('%m.%d') if sell_date else ''
    except Exception:
        month_day = sell_date
    qty = int(float(log_row.get('매도수량', 0) or 0))
    reason = str(log_row.get('매도사유', ''))
    principle = _lab_ratio_for_reason(reason, tp, sl)
    if principle:
        return f"{month_day}. {qty}주 매도({principle})"
    if 'RSI' in reason:
        return f"{month_day}. {qty}주 매도(RSI 과열 부분청산)"
    if '강제청산' in reason:
        return f"{month_day}. {qty}주 매도(백테스트 종료일 기준 강제청산)"
    return f"{month_day}. {qty}주 매도({reason})"


def build_backtest_lab_data_v94(final_df):
    """상위 후보에 대해 종목×기간×전략 조합의 백테스트 실험실 데이터를 미리 계산합니다.
    엑셀에서는 이 계산 결과를 버튼형 선택 패널/직접입력 셀로 조회합니다.
    """
    summary_rows = []
    record_rows = []
    if final_df is None or final_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # v9.7.3.5: 스나이퍼 계산기의 추천 리스트 전체와 같은 종목군을 실험실에서도 사용
    if CONFIG.get('LAB_USE_FULL_RECOMMEND_LIST', True):
        try:
            max_stocks = len(final_df)
        except Exception:
            max_stocks = int(CONFIG.get('LAB_MAX_STOCKS', 12))
    else:
        max_stocks = int(CONFIG.get('LAB_MAX_STOCKS', 12))
    base_budget = int(CONFIG.get('LAB_BASE_BUDGET', CONFIG.get('INITIAL_BUDGET_FOR_TEST', 300000)))
    period_options = CONFIG.get('LAB_PERIOD_OPTIONS', {'1개월 전에 샀다면': 31, '2주 전에 샀다면': 14, '3개월 전에 샀다면': 92, '6개월 전에 샀다면': 183})
    # v9.6.3: 백테스트 실험실도 화면용 통합 후보군(전체 TOP + 토스 참고)을 기준으로 사용
    try:
        top_stocks = _v93_prepare_top_df(final_df, max_stocks).copy()
    except Exception:
        top_stocks = final_df.head(max_stocks).copy()
    # v9.7.3.3: 실험실 기본 기간을 1개월로 축소. 지표 계산용 워밍업은 별도 확보
    finite_period_days = [int(v) for v in period_options.values() if v is not None]
    max_period_days = max(finite_period_days) if finite_period_days else 31
    fetch_start = (RUN_AT - timedelta(days=max_period_days + 100)).strftime('%Y-%m-%d')

    print(f"🧪 v9.4 백테스트 실험실 데이터 생성 시작: 상위 {len(top_stocks)}개 후보")
    for _, stock in top_stocks.iterrows():
        code = str(stock.get('종목코드', '')).zfill(6)
        name = str(stock.get('종목명', ''))
        if not code or code == '000000' or not name:
            continue
        try:
            raw_df = safe_fdr_reader(code, fetch_start)
            if raw_df is None or raw_df.empty or len(raw_df) < 80:
                continue
            df = calc_indicators(raw_df).dropna().copy()
            if df.empty:
                continue
            current_price = float(df['Close'].iloc[-1])
        except Exception as e:
            print(f"  - {name}({code}) 실험실 데이터 생략: {e}")
            continue

        for period_label, days in period_options.items():
            try:
                if days is None:
                    start_dt = pd.to_datetime(CONFIG.get('EVAL_START', df.index.min()))
                else:
                    start_dt = pd.to_datetime(RUN_AT - timedelta(days=int(days)))
                eval_df = df.loc[df.index >= start_dt].copy()
                # v9.7.3.11: 2주/1개월 구간도 실험실에서 조회되도록 단기 구간 최소 데이터 기준을 낮춥니다.
                # 기존 20거래일 제한은 2주/1개월 선택 시 데이터가 비어 보이는 원인이 됩니다.
                min_lab_rows = int(CONFIG.get('LAB_MIN_EVAL_ROWS_SHORT', 5)) if (days is not None and int(days) <= 31) else int(CONFIG.get('LAB_MIN_EVAL_ROWS_NORMAL', 12))
                if eval_df.empty or len(eval_df) < min_lab_rows:
                    continue
                start_price = float(eval_df['Close'].iloc[0])
                hold_return = ((current_price / start_price) - 1) * 100 if start_price else np.nan
            except Exception:
                continue

            for tp, sl in scenarios:
                combo = _lab_strategy_combo(tp, sl)
                key = f"{name}|{period_label}|{combo}"
                bt = backtest_scenario(eval_df, tp, sl, base_budget, name, code)
                summary_rows.append({
                    '조회키': key,
                    '종목명': name,
                    '종목코드': code,
                    '기간선택': period_label,
                    '전략콤보': combo,
                    'TP전략': tp['name'],
                    'SL전략': sl['name'],
                    '투자원금(기준)': base_budget,
                    '시작기준가': round(start_price, 2),
                    '현재가': round(current_price, 2),
                    '보유상승률': round(hold_return, 2) if not pd.isna(hold_return) else '',
                    '전략수익률': round(bt.get('return_rate', 0), 2),
                    'MDD': round(bt.get('mdd', 0), 2),
                    '승률': round(bt.get('win_rate', 0), 1),
                    '거래횟수': int(bt.get('trade_count', 0)),
                    '백테스트시작일': bt.get('backtest_start', ''),
                    '백테스트종료일': bt.get('backtest_end', ''),
                    'TP전략내용': strategy_detail(tp, '익절'),
                    'SL전략내용': strategy_detail(sl, '손절'),
                    '비용가정': f"매수 {CONFIG.get('STRATEGY_BUY_COST_RATE')} / 매도 {CONFIG.get('STRATEGY_SELL_COST_RATE')} / 슬리피지 {CONFIG.get('STRATEGY_SLIPPAGE_RATE')}"
                })
                for log in bt.get('trade_log', []):
                    record_rows.append({
                        '조회키': key,
                        '종목명': name,
                        '종목코드': code,
                        '기간선택': period_label,
                        '전략콤보': combo,
                        '매수일': log.get('매수일', ''),
                        '매수단가': log.get('매수단가', ''),
                        '매도일': log.get('매도일', ''),
                        '매도단가': log.get('매도단가', ''),
                        '매도사유': log.get('매도사유', ''),
                        '매도수량_기준': log.get('매도수량', ''),
                        '잔여수량_기준': log.get('잔여수량', ''),
                        '수익률': log.get('수익률', ''),
                        '기록메모': _lab_record_memo(log, tp, sl),
                        '기준투입금액': base_budget
                    })

    print(f"✅ v9.4 백테스트 실험실 데이터: 요약 {len(summary_rows)}행 / 기록 {len(record_rows)}행")
    return pd.DataFrame(summary_rows), pd.DataFrame(record_rows)


def _write_df_to_sheet(ws, df):
    if df is None or df.empty:
        ws.append(['데이터 없음'])
        return
    ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append(list(row))


def _lab_set(cell, value, fill=None, font=None, align=None, border=None):
    cell.value = value
    if fill is not None:
        cell.fill = copy(fill)
    if font is not None:
        cell.font = copy(font)
    if align is not None:
        cell.alignment = copy(align)
    if border is not None:
        cell.border = copy(border)
    return cell


def add_backtest_lab_sheet_v94(output_file, final_df):
    if not CONFIG.get('LAB_BACKTEST_ENABLED', True):
        return output_file

    lab_summary_df, lab_record_df = build_backtest_lab_data_v94(final_df)
    wb = load_workbook(output_file)
    for name in ['백테스트_실험실', '실험실_요약', '실험실_기록', '실험실_옵션', '실험실_최고전략']:
        if name in wb.sheetnames:
            del wb[name]

    ws_sum = wb.create_sheet('실험실_요약')
    _write_df_to_sheet(ws_sum, lab_summary_df)
    ws_rec = wb.create_sheet('실험실_기록')
    _write_df_to_sheet(ws_rec, lab_record_df)

    # v9.6.2: 선택 종목+기간 기준으로 가장 수익률이 높았던 전략을 별도 조회용으로 생성
    best_strategy_df = pd.DataFrame()
    try:
        if lab_summary_df is not None and not lab_summary_df.empty:
            tmp = lab_summary_df.copy()
            tmp['_전략수익률_num'] = pd.to_numeric(tmp['전략수익률'], errors='coerce').fillna(-999999)
            tmp['_최고전략조회키'] = tmp['종목명'].astype(str) + '|' + tmp['기간선택'].astype(str)
            tmp = tmp.sort_values(['_최고전략조회키', '_전략수익률_num'], ascending=[True, False])
            tmp = tmp.groupby('_최고전략조회키', as_index=False).first()
            best_strategy_df = tmp[[
                '_최고전략조회키', '종목명', '기간선택', '전략콤보', '전략수익률', 'MDD', '승률', '거래횟수', 'TP전략내용', 'SL전략내용'
            ]].rename(columns={
                '_최고전략조회키': '최고전략조회키',
                '전략콤보': '최고전략콤보',
                '전략수익률': '최고전략수익률',
                'MDD': '최고전략MDD',
                '승률': '최고전략승률',
                '거래횟수': '최고전략거래횟수',
                'TP전략내용': '최고익절전략내용',
                'SL전략내용': '최고손절전략내용'
            })
    except Exception as e:
        print(f'⚠️ 최고 전략 계산 생략: {e}')
    ws_best = wb.create_sheet('실험실_최고전략')
    _write_df_to_sheet(ws_best, best_strategy_df)

    # v9.6.2: FILTER 동적배열 대신 INDEX/MATCH로 매매기록을 찾기 위한 helper 열
    # P열 = 같은 조회키 내 순번, Q열 = 조회키|순번
    if lab_record_df is not None and not lab_record_df.empty and ws_rec.max_row >= 2:
        seq_col = ws_rec.max_column + 1
        match_col = ws_rec.max_column + 2
        seq_letter = get_column_letter(seq_col)
        ws_rec.cell(1, seq_col, '매칭순번')
        ws_rec.cell(1, match_col, '조회키_순번')
        for rr in range(2, ws_rec.max_row + 1):
            ws_rec.cell(rr, seq_col, f'=COUNTIF($A$2:A{rr},A{rr})')
            ws_rec.cell(rr, match_col, f'=A{rr}&"|"&{seq_letter}{rr}')

    ws_opt = wb.create_sheet('실험실_옵션')

    # 옵션 테이블
    periods = CONFIG.get('LAB_PERIOD_OPTIONS', {'1개월 전에 샀다면': 31, '2주 전에 샀다면': 14, '3개월 전에 샀다면': 92, '6개월 전에 샀다면': 183})
    budgets = CONFIG.get('LAB_BUDGET_OPTIONS', {'30만원': 300000, '50만원': 500000, '100만원': 1000000})
    stocks = sorted(lab_summary_df['종목명'].dropna().unique().tolist()) if lab_summary_df is not None and not lab_summary_df.empty else []
    strategies = sorted(lab_summary_df['전략콤보'].dropna().unique().tolist()) if lab_summary_df is not None and not lab_summary_df.empty else [_lab_strategy_combo(tp, sl) for tp, sl in scenarios]

    ws_opt.append(['기간선택', '일수', '', '종목명', '투입금액표시', '투입금액'])
    max_len = max(len(periods), len(stocks), len(budgets), len(strategies))
    period_items = list(periods.items())
    budget_items = list(budgets.items())
    for i in range(max_len):
        row = []
        row += [period_items[i][0], period_items[i][1] if period_items[i][1] is not None else 9999] if i < len(period_items) else ['', '']
        row += ['']
        row += [stocks[i] if i < len(stocks) else '']
        row += [budget_items[i][0], budget_items[i][1]] if i < len(budget_items) else ['', '']
        ws_opt.append(row)
    ws_opt['H1'] = '전략콤보'
    for i, val in enumerate(strategies, 2):
        ws_opt.cell(i, 8, val)

    # 데이터 시트 스타일
    for ws_data in [ws_sum, ws_rec, ws_opt, ws_best]:
        ws_data.sheet_view.showGridLines = False
        if ws_data.max_row >= 1:
            for cell in ws_data[1]:
                cell.fill = PatternFill(start_color='1F2937', end_color='1F2937', fill_type='solid')
                cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=10)
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        for col in range(1, min(ws_data.max_column, 12)+1):
            ws_data.column_dimensions[get_column_letter(col)].width = 18
        ws_data.freeze_panes = 'A2'
        try:
            ws_data.auto_filter.ref = ws_data.dimensions
        except Exception:
            pass
        ws_data.sheet_state = 'hidden'

    # 메인 실험실 시트
    ws = wb.create_sheet('백테스트_실험실', 0)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 90
    thin = Side(style='thin', color='E5E7EB')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)
    fill_dark = PatternFill(start_color='0F172A', end_color='0F172A', fill_type='solid')
    fill_card = PatternFill(start_color='F8FAFC', end_color='F8FAFC', fill_type='solid')
    fill_input = PatternFill(start_color='E0F2FE', end_color='E0F2FE', fill_type='solid')
    fill_green = PatternFill(start_color='DCFCE7', end_color='DCFCE7', fill_type='solid')
    fill_yellow = PatternFill(start_color='FEF3C7', end_color='FEF3C7', fill_type='solid')
    font_title = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=18)
    font_h = Font(name='맑은 고딕', color='111827', bold=True, size=11)
    font_body = Font(name='맑은 고딕', color='111827', size=10)
    font_muted = Font(name='맑은 고딕', color='64748B', size=9)

    for col, width in {'A':3, 'B':18, 'C':24, 'D':4, 'E':18, 'F':18, 'G':18, 'H':18, 'I':4}.items():
        ws.column_dimensions[col].width = width
    for r in range(1, 70):
        ws.row_dimensions[r].height = 24

    ws.merge_cells('B2:H3')
    _lab_set(ws['B2'], f"백테스트 실험실 v9.4\n종목·기간·전략·투입금액을 바꿔가며 매매기록을 확인", fill_dark, font_title, center, border)

    labels = [('B5', '종목 선택'), ('B6', '기간 선택'), ('B7', '전략 선택'), ('B8', '투입금액')]
    for addr, label in labels:
        _lab_set(ws[addr], label, fill_card, font_h, center, border)

    # v9.6.1: 처음 열었을 때 바로 값이 보이도록 실제 매매기록이 있는 조합을 기본값으로 사용
    default_key = ''
    try:
        if lab_record_df is not None and not lab_record_df.empty and '조회키' in lab_record_df.columns:
            valid_keys = lab_record_df['조회키'].dropna().astype(str)
            default_key = valid_keys.iloc[0] if len(valid_keys) else ''
        elif lab_summary_df is not None and not lab_summary_df.empty and '조회키' in lab_summary_df.columns:
            valid_keys = lab_summary_df['조회키'].dropna().astype(str)
            default_key = valid_keys.iloc[0] if len(valid_keys) else ''
    except Exception:
        default_key = ''

    if default_key and default_key.count('|') >= 2:
        parts = default_key.split('|')
        default_stock = parts[0]
        default_period = parts[1]
        default_strategy = '|'.join(parts[2:])
    else:
        default_stock = stocks[0] if stocks else ''
        default_period = '1개월 전에 샀다면' if '1개월 전에 샀다면' in periods else (list(periods.keys())[0] if periods else '')
        default_strategy = strategies[0] if strategies else ''
    default_budget = '30만원' if '30만원' in budgets else (list(budgets.keys())[0] if budgets else '')

    for addr, value in [('C5', default_stock), ('C6', default_period), ('C7', default_strategy), ('C8', default_budget)]:
        _lab_set(ws[addr], value, fill_input, font_h, center, border)

    # helper cells
    ws['F5'] = '조회키'
    ws['G5'] = '=$C$5&"|"&$C$6&"|"&$C$7'
    ws['F6'] = '선택투입금액'
    ws['G6'] = '=XLOOKUP($C$8,\'실험실_옵션\'!$E:$E,\'실험실_옵션\'!$F:$F,300000)'
    ws['F7'] = '기준투입금액'
    ws['G7'] = '=XLOOKUP($G$5,\'실험실_요약\'!$A:$A,\'실험실_요약\'!$H:$H,300000)'
    ws['F8'] = '수량환산배율'
    ws['G8'] = '=IFERROR($G$6/$G$7,1)'
    ws['F9'] = '최고전략키'
    ws['G9'] = '=$C$5&"|"&$C$6'
    for cell in ['F5','F6','F7','F8','F9','G5','G6','G7','G8','G9']:
        ws[cell].font = font_muted
        ws[cell].alignment = center
    ws.column_dimensions['F'].hidden = True
    ws.column_dimensions['G'].hidden = True

    # Data validations
    if stocks:
        dv_stock = DataValidation(type='list', formula1=f"'실험실_옵션'!$D$2:$D${len(stocks)+1}", allow_blank=False)
        ws.add_data_validation(dv_stock); dv_stock.add(ws['C5'])
    if periods:
        dv_period = DataValidation(type='list', formula1=f"'실험실_옵션'!$A$2:$A${len(periods)+1}", allow_blank=False)
        ws.add_data_validation(dv_period); dv_period.add(ws['C6'])
    if strategies:
        dv_strategy = DataValidation(type='list', formula1=f"'실험실_옵션'!$H$2:$H${len(strategies)+1}", allow_blank=False)
        ws.add_data_validation(dv_strategy); dv_strategy.add(ws['C7'])
    if budgets:
        dv_budget = DataValidation(type='list', formula1=f"'실험실_옵션'!$E$2:$E${len(budgets)+1}", allow_blank=False)
        ws.add_data_validation(dv_budget); dv_budget.add(ws['C8'])

    # 결과 카드
    result_items = [
        ('B11', '현재가', '=IFERROR(INDEX(\'실험실_요약\'!$J:$J,MATCH($G$5,\'실험실_요약\'!$A:$A,0)),"")', '#,##0원'),
        ('D11', '시작기준가', '=IFERROR(INDEX(\'실험실_요약\'!$I:$I,MATCH($G$5,\'실험실_요약\'!$A:$A,0)),"")', '#,##0원'),
        ('F11', '보유상승률', '=IFERROR(INDEX(\'실험실_요약\'!$K:$K,MATCH($G$5,\'실험실_요약\'!$A:$A,0))/100,"")', '0.0%'),
        ('B14', '전략수익률', '=IFERROR(INDEX(\'실험실_요약\'!$L:$L,MATCH($G$5,\'실험실_요약\'!$A:$A,0))/100,"")', '0.0%'),
        ('D14', 'MDD', '=IFERROR(INDEX(\'실험실_요약\'!$M:$M,MATCH($G$5,\'실험실_요약\'!$A:$A,0))/100,"")', '0.0%'),
        ('F14', '승률/거래횟수', '=IFERROR(TEXT(INDEX(\'실험실_요약\'!$N:$N,MATCH($G$5,\'실험실_요약\'!$A:$A,0))/100,"0.0%")&" / "&INDEX(\'실험실_요약\'!$O:$O,MATCH($G$5,\'실험실_요약\'!$A:$A,0))&"회","")', '@'),
    ]
    for addr, title, formula, numfmt in result_items:
        col = ws[addr].column
        row = ws[addr].row
        ws.merge_cells(start_row=row, start_column=col, end_row=row+1, end_column=col+1)
        c = ws[addr]
        c.value = formula
        c.fill = fill_green if '수익률' in title or '상승률' in title else fill_card
        c.font = Font(name='맑은 고딕', color='111827', bold=True, size=14)
        c.alignment = center
        c.border = border
        c.number_format = numfmt
        label_cell = ws.cell(row=row-1, column=col, value=title)
        label_cell.fill = fill_dark
        label_cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=10)
        label_cell.alignment = center
        label_cell.border = border
        ws.merge_cells(start_row=row-1, start_column=col, end_row=row-1, end_column=col+1)

    ws.merge_cells('B17:H18')
    ws['B17'] = '="백테스트 구간: "&IFERROR(INDEX(\'실험실_요약\'!$P:$P,MATCH($G$5,\'실험실_요약\'!$A:$A,0)),"")&" ~ "&IFERROR(INDEX(\'실험실_요약\'!$Q:$Q,MATCH($G$5,\'실험실_요약\'!$A:$A,0)),"")&CHAR(10)&"익절: "&IFERROR(INDEX(\'실험실_요약\'!$R:$R,MATCH($G$5,\'실험실_요약\'!$A:$A,0)),"")&CHAR(10)&"손절: "&IFERROR(INDEX(\'실험실_요약\'!$S:$S,MATCH($G$5,\'실험실_요약\'!$A:$A,0)),"")'
    ws['B17'].fill = fill_yellow
    ws['B17'].font = font_h
    ws['B17'].alignment = left
    ws['B17'].border = border

    # v9.6.2: 같은 종목/기간에서 가장 수익률이 높았던 전략을 별도 표시
    ws.merge_cells('B19:H20')
    ws['B19'] = '="같은 종목·기간 최고 수익 전략: "&IFERROR(INDEX(\'실험실_최고전략\'!$D:$D,MATCH($G$9,\'실험실_최고전략\'!$A:$A,0)),"")&" / 수익률 "&IFERROR(TEXT(INDEX(\'실험실_최고전략\'!$E:$E,MATCH($G$9,\'실험실_최고전략\'!$A:$A,0))/100,"0.0%"),"")&CHAR(10)&"MDD "&IFERROR(TEXT(INDEX(\'실험실_최고전략\'!$F:$F,MATCH($G$9,\'실험실_최고전략\'!$A:$A,0))/100,"0.0%"),"")&" / 승률 "&IFERROR(TEXT(INDEX(\'실험실_최고전략\'!$G:$G,MATCH($G$9,\'실험실_최고전략\'!$A:$A,0))/100,"0.0%"),"")&" / 거래 "&IFERROR(INDEX(\'실험실_최고전략\'!$H:$H,MATCH($G$9,\'실험실_최고전략\'!$A:$A,0)),"")&"회 — 이 조합을 전략 선택 칸에 입력해 상세 매매기록을 확인하세요."'
    ws['B19'].fill = PatternFill(start_color='DCFCE7', end_color='DCFCE7', fill_type='solid')
    ws['B19'].font = Font(name='맑은 고딕', color='166534', bold=True, size=11)
    ws['B19'].alignment = left
    ws['B19'].border = border

    # 매매기록 표
    ws.merge_cells('B21:H21')
    _lab_set(ws['B21'], '매매기록', fill_dark, Font(name='맑은 고딕', color='FFFFFF', bold=True, size=13), center, border)
    headers = ['매도일', '환산 매도수량', '매도사유', '수익률', '기록 메모']
    for i, h in enumerate(headers, 2):
        cell = ws.cell(22, i, h)
        cell.fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
        cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=10)
        cell.alignment = center
        cell.border = border
    ws.merge_cells('F22:H22')

    # v9.6.1: FILTER/XLOOKUP 없이도 작동하는 INDEX/MATCH 방식 매매기록 표시
    for rr in range(23, 43):
        seq = rr - 22
        lookup_key = f'$G$5&"|"&{seq}'
        ws[f'B{rr}'] = f'=IFERROR(INDEX(\'실험실_기록\'!$H:$H,MATCH({lookup_key},\'실험실_기록\'!$Q:$Q,0)),IF({seq}=1,"매매기록 없음",""))'
        ws[f'C{rr}'] = f'=IFERROR(ROUND(INDEX(\'실험실_기록\'!$K:$K,MATCH({lookup_key},\'실험실_기록\'!$Q:$Q,0))*$G$8,0),"")'
        ws[f'D{rr}'] = f'=IFERROR(INDEX(\'실험실_기록\'!$J:$J,MATCH({lookup_key},\'실험실_기록\'!$Q:$Q,0)),"")'
        ws[f'E{rr}'] = f'=IFERROR(INDEX(\'실험실_기록\'!$M:$M,MATCH({lookup_key},\'실험실_기록\'!$Q:$Q,0))/100,"")'
        ws[f'F{rr}'] = f'=IFERROR(INDEX(\'실험실_기록\'!$N:$N,MATCH({lookup_key},\'실험실_기록\'!$Q:$Q,0)),"")'
        for col in ['B','C','D','E','F']:
            ws[f'{col}{rr}'].alignment = left if col in ['D','F'] else center
            ws[f'{col}{rr}'].font = font_body
        ws[f'E{rr}'].number_format = '0.0%'
    for r in range(22, 42):
        for c in range(2, 9):
            ws.cell(r, c).border = border
            ws.cell(r, c).alignment = left if c in [4, 6] else center

    ws.merge_cells('B44:H47')
    ws['B44'] = '사용 메모\n- 종목/기간/전략/투입금액 선택값을 바꾸면 현재가·상승률·전략수익률·매매기록이 바뀝니다.\n- 매매기록의 수량은 기준 30만원 백테스트 수량을 선택 투입금액에 맞춰 환산한 값입니다. 기본 조회기간은 1개월입니다.\n- 실제 체결가, 호가 공백, 거래정지, 뉴스는 별도 확인이 필요합니다.'
    ws['B44'].fill = PatternFill(start_color='F1F5F9', end_color='F1F5F9', fill_type='solid')
    ws['B44'].font = font_muted
    ws['B44'].alignment = left
    ws['B44'].border = border

    ws.freeze_panes = 'B10'
    # v9.6.1: 파일을 열 때 계산식을 자동 재계산
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = 'auto'
    except Exception:
        pass
    wb.save(output_file)
    return output_file


if CONFIG.get('LAB_BACKTEST_ENABLED', True):
    output_file = add_backtest_lab_sheet_v94(output_file, final_df)
    print(f'✅ v9.6.2 백테스트 실험실/스나이퍼 개선 시트 추가 완료: {output_file}')

# 다운로드는 v9.6.3 통합 UX 패치 셀에서 최종 파일 저장 후 진행합니다.


🧪 v9.4 백테스트 실험실 데이터 생성 시작: 상위 50개 후보


✅ v9.4 백테스트 실험실 데이터: 요약 5000행 / 기록 4202행


✅ v9.6.2 백테스트 실험실/스나이퍼 개선 시트 추가 완료: 실전매매_통합시스템_v9_7_5_4_KST최종저장일수정_20260608.xlsx


In [18]:

# 8-3. v9.6.3 통합 UX · 탈락사유 리포트 · 시트 순서 재정리
# 목적:
# 1) 실제 보는 화면(홈/모바일/TOP/카드/스나이퍼/실험실)이 같은 후보군을 기준으로 보이게 통합
# 2) 토스_TOP후보는 별도 전면 시트가 아니라 TOP후보_요약에 "토스 참고"로 합침
# 3) 토스/네이버 후보가 최종 후보에서 빠진 이유를 탈락사유_리포트로 표시
# 4) 분야/후보출처/시장모드/기술점수/가이드 내용을 스나이퍼·실험실·카드에 함께 표시
# 5) 실제 사용 시트는 앞, 데이터/추출/헬퍼 시트는 뒤로 재정렬

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment
from openpyxl.worksheet.datavalidation import DataValidation
from copy import copy


def _v963_text(v, default='-'):
    try:
        if v is None or pd.isna(v):
            return default
    except Exception:
        pass
    txt = str(v).strip()
    return txt if txt and txt.lower() not in ['nan', 'none'] else default


def _v963_num(v, default=0):
    try:
        if v is None or pd.isna(v):
            return default
        return float(str(v).replace(',', '').replace('%',''))
    except Exception:
        return default


def _v963_sector(row):
    for col in ['분야', '토스카테고리', '카테고리', '업종', '시장구분']:
        try:
            val = row.get(col, '')
        except Exception:
            val = ''
        if _v963_text(val, ''):
            return _v963_text(val)
    return '확인필요'


def _v963_enrich_display_df(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out['분야'] = out.apply(_v963_sector, axis=1)
    for c in ['종목점수','재무점수','백테스트수익률','거래대금(억)']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce').fillna(0)
    return out


def _v963_unified_top_df(df, top_n=None, toss_n=None):
    """화면에 보여줄 후보군을 하나로 통일: 전체 상위 + 토스 참고 후보."""
    if df is None or df.empty:
        return pd.DataFrame()
    top_n = int(top_n or CONFIG.get('UX_UNIFIED_TOP_N', CONFIG.get('UX_TOP_N', 12)))
    toss_n = int(toss_n if toss_n is not None else CONFIG.get('UX_TOSS_REFERENCE_N', 5))
    base = _v963_enrich_display_df(df)
    sort_cols = [c for c in ['종목점수','재무점수','백테스트수익률','거래대금(억)'] if c in base.columns]
    if sort_cols:
        base = base.sort_values(sort_cols, ascending=False)
    overall = base.head(top_n).copy()
    overall['표시구분'] = '전체 TOP'
    parts = [overall]
    if '후보출처' in base.columns and toss_n > 0:
        toss = base[base['후보출처'].astype(str).str.contains('TOSS', case=False, na=False)].head(toss_n).copy()
        if not toss.empty:
            toss['표시구분'] = '토스 참고'
            parts.append(toss)
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if '종목코드' in out.columns:
        out = out.drop_duplicates(subset=['종목코드'], keep='first')
    return out.reset_index(drop=True)


def _v963_display_cols():
    return [
        '표시구분','AI 시스템 판정','종목명','종목코드','분야','후보출처','스크리너명',
        '현재가','등락률','종목점수','기본기술점수','시장모드',
        '추세필터판정','재무위험판정','안전필터판정',
        '백테스트수익률','백테스트MDD','백테스트승률','백테스트신뢰도',
        '권장익절계획','권장손절계획','추천투입금액','추천수량','상세 전략 가이드'
    ]


def _v963_write_df_sheet(wb, name, df, index=None, title=None, tab_color='1F4E78'):
    if name in wb.sheetnames:
        del wb[name]
    ws = wb.create_sheet(name, index=index if index is not None else len(wb.sheetnames))
    ws.sheet_properties.tabColor = tab_color
    ws.sheet_view.showGridLines = False
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)

    start_row = 1
    if title:
        ws.merge_cells(start_row=1, start_column=1, end_row=2, end_column=max(6, len(df.columns) if df is not None and not df.empty else 6))
        c = ws.cell(1, 1, title)
        c.fill = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
        c.font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=15)
        c.alignment = left
        c.border = border
        start_row = 4

    if df is None or df.empty:
        ws.cell(start_row, 1, '표시할 데이터가 없습니다.')
        return ws

    for cidx, col in enumerate(df.columns, 1):
        cell = ws.cell(start_row, cidx, col)
        cell.fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
        cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True)
        cell.alignment = center
        cell.border = border
    for ridx, (_, row) in enumerate(df.iterrows(), start_row + 1):
        for cidx, col in enumerate(df.columns, 1):
            cell = ws.cell(ridx, cidx, row.get(col, ''))
            cell.border = border
            cell.alignment = left if col in ['상세 전략 가이드','권장익절계획','권장손절계획','재무위험판정','AI 시스템 판정'] else center
            if col in ['현재가','추천투입금액']:
                cell.number_format = '#,##0'
            elif col in ['백테스트수익률','백테스트MDD','백테스트승률','등락률']:
                cell.number_format = '0.0'
        # 상태 컬러
        verdict = _v963_text(row.get('AI 시스템 판정',''))
        fill = None
        if any(k in verdict for k in ['A급','강력','우선']):
            fill = 'E2F0D9'
        elif any(k in verdict for k in ['선별','관찰']):
            fill = 'FFF2CC'
        elif any(k in verdict for k in ['제외','금지','위험']):
            fill = 'FCE4D6'
        if fill:
            for cidx in range(1, len(df.columns)+1):
                ws.cell(ridx, cidx).fill = PatternFill(start_color=fill, end_color=fill, fill_type='solid')
    ws.freeze_panes = ws.cell(start_row + 1, 1).coordinate
    try:
        ws.auto_filter.ref = ws.dimensions
    except Exception:
        pass
    for cidx, col in enumerate(df.columns, 1):
        width = min(max(10, len(str(col)) + 4), 32)
        if col in ['상세 전략 가이드','권장익절계획','권장손절계획']:
            width = 38
        elif col in ['종목명','분야','후보출처','AI 시스템 판정']:
            width = 18
        ws.column_dimensions[get_column_letter(cidx)].width = width
    return ws


def _v963_make_rejection_report():
    """전체 후보군 기준으로 최종 후보/탈락/관찰 사유를 한 줄씩 정리."""
    total = globals().get('total_df', pd.DataFrame())
    filtered = globals().get('filtered_df', pd.DataFrame())
    excluded = globals().get('excluded_by_safety_df', pd.DataFrame())
    final = globals().get('final_df', pd.DataFrame())

    if total is None or total.empty:
        total = final.copy() if final is not None else pd.DataFrame()
    if total is None or total.empty:
        return pd.DataFrame()

    final = _v963_enrich_display_df(final) if final is not None and not final.empty else pd.DataFrame()
    filtered_codes = set(filtered['종목코드'].astype(str)) if filtered is not None and not filtered.empty and '종목코드' in filtered.columns else set()
    final_codes = set(final['종목코드'].astype(str)) if not final.empty and '종목코드' in final.columns else set()
    excluded_codes = set(excluded['종목코드'].astype(str)) if excluded is not None and not excluded.empty and '종목코드' in excluded.columns else set()
    final_map = final.set_index(final['종목코드'].astype(str)).to_dict('index') if not final.empty and '종목코드' in final.columns else {}
    excl_map = excluded.set_index(excluded['종목코드'].astype(str)).to_dict('index') if excluded is not None and not excluded.empty and '종목코드' in excluded.columns else {}

    rows = []
    for _, r in total.drop_duplicates(subset=['종목코드']).iterrows():
        code = _v963_text(r.get('종목코드',''), '')
        name = _v963_text(r.get('종목명',''), '')
        source = _v963_text(r.get('후보출처',''), '')
        sector = _v963_sector(r)
        price = _v963_num(r.get('현재가', np.nan), np.nan)
        turnover = _v963_num(r.get('거래대금', np.nan), np.nan)
        reason_parts = []
        stage = ''
        action = ''

        if not code:
            stage = '종목코드 확인필요'
            reason_parts.append('종목코드가 없어 KRX/시세/백테스트 매칭 불가')
            action = '종목코드 수동 입력'
        elif code in excluded_codes:
            er = excl_map.get(code, {})
            stage = '안전필터 제외'
            for col in ['안전필터판정','시장경보','재무위험판정','재무위험사유','최종위험메모']:
                val = _v963_text(er.get(col,''), '')
                if val:
                    reason_parts.append(f'{col}: {val}')
            action = '원칙적 매수 제외'
        elif code in final_codes:
            fr = final_map.get(code, {})
            verdict = _v963_text(fr.get('AI 시스템 판정',''))
            trend = _v963_text(fr.get('추세필터판정',''))
            rel = _v963_text(fr.get('백테스트신뢰도',''))
            score = _v963_text(fr.get('종목점수',''))
            if '제외' in verdict or trend == '제외':
                stage = '분석통과 후 추세/판정 제외'
                reason_parts.append(f'판정: {verdict}')
                reason_parts.append(f'추세필터: {trend}')
                action = '추세 회복 후 재검토'
            else:
                stage = '최종 후보 포함'
                reason_parts.append(f'판정: {verdict}')
                reason_parts.append(f'종목점수: {score}')
                reason_parts.append(f'백테스트신뢰도: {rel}')
                action = 'TOP후보/스나이퍼에서 검토'
            sector = _v963_text(fr.get('분야', sector))
        else:
            # 가격/거래대금 필터 또는 데이터 부족
            if code not in filtered_codes:
                stage = '1차 조건 제외'
                try:
                    if not pd.isna(price) and price < CONFIG.get('MIN_PRICE', 0):
                        reason_parts.append(f"현재가 {price:,.0f}원 < 최소주가 {CONFIG.get('MIN_PRICE'):,}원")
                    if not pd.isna(price) and price > CONFIG.get('MAX_PRICE', 999999999):
                        reason_parts.append(f"현재가 {price:,.0f}원 > 최대주가 {CONFIG.get('MAX_PRICE'):,}원")
                    if not pd.isna(turnover) and turnover < CONFIG.get('MIN_TRADING_VALUE_NAVER', 0):
                        reason_parts.append('거래대금 기준 미달')
                    if re.search(CONFIG.get('EXCLUDE_KEYWORDS',''), name):
                        reason_parts.append('제외 키워드 포함')
                except Exception:
                    pass
                if not reason_parts:
                    reason_parts.append('가격/거래대금/종목코드/제외키워드 조건 중 하나 미통과')
                action = '조건 완화 또는 후보 유지 여부 확인'
            else:
                stage = '분석 중 제외/데이터 부족'
                reason_parts.append('시세 데이터·지표 계산·백테스트 결과 부족 또는 최종 결과 미생성')
                action = '시세 조회 가능 여부/기간 조건 확인'

        rows.append({
            '후보출처': source,
            '종목명': name,
            '종목코드': code,
            '분야': sector,
            '현재가': price if not pd.isna(price) else '',
            '탈락/분류단계': stage,
            '핵심사유': ' / '.join([p for p in reason_parts if p])[:500],
            '권장조치': action,
        })
    out = pd.DataFrame(rows)
    order = {'최종 후보 포함':0, '분석통과 후 추세/판정 제외':1, '1차 조건 제외':2, '안전필터 제외':3, '분석 중 제외/데이터 부족':4, '종목코드 확인필요':5}
    out['_sort'] = out['탈락/분류단계'].map(order).fillna(9)
    out = out.sort_values(['_sort','후보출처','종목명']).drop(columns=['_sort']).reset_index(drop=True)
    return out


def _v963_formula_lookup(stock_ref, header, default='""'):
    return f'=IFERROR(INDEX(\'추천 리스트\'!$A:$ZZ,MATCH({stock_ref},\'추천 리스트\'!$B:$B,0),MATCH("{header}",\'추천 리스트\'!$1:$1,0)),{default})'


def _v963_patch_sniper(ws):
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)
    fill_dark = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
    fill_card = PatternFill(start_color='F8FAFC', end_color='F8FAFC', fill_type='solid')
    fill_input = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')

    # 우측 선택 종목 가이드 박스
    for col, width in {'K':18,'L':24,'M':18,'N':28}.items():
        ws.column_dimensions[col].width = width
    ws.merge_cells('K2:N2')
    ws['K2'] = '선택 종목 가이드'
    ws['K2'].fill = fill_dark
    ws['K2'].font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=13)
    ws['K2'].alignment = center
    ws['K2'].border = border
    pairs = [
        ('K4','분야', 'L4', _v963_formula_lookup('$C$4','분야','"확인필요"')),
        ('M4','후보출처', 'N4', _v963_formula_lookup('$C$4','후보출처','"확인필요"')),
        ('K5','시장모드', 'L5', _v963_formula_lookup('$C$4','시장모드','"확인필요"')),
        ('M5','표시구분', 'N5', _v963_formula_lookup('$C$4','표시구분','"전체후보"')),
        ('K6','기술점수', 'L6', '="기본 "&TEXT('+_v963_formula_lookup('$C$4','기본기술점수','0')+',"0")&"점 / 최종 "&TEXT('+_v963_formula_lookup('$C$4','종목점수','0')+',"0")&"점"'),
        ('M6','추세', 'N6', _v963_formula_lookup('$C$4','추세필터판정','"확인필요"')),
        ('K7','재무', 'L7', _v963_formula_lookup('$C$4','재무위험판정','"확인필요"')),
        ('M7','백테스트', 'N7', '="수익률 "&TEXT('+_v963_formula_lookup('$C$4','백테스트수익률','0')+'/100,"0.0%")&" / 신뢰도 "&'+_v963_formula_lookup('$C$4','백테스트신뢰도','""')),
    ]
    for lab_addr, lab, val_addr, formula in pairs:
        ws[lab_addr] = lab
        ws[lab_addr].fill = fill_card
        ws[lab_addr].font = Font(name='맑은 고딕', color='334155', bold=True)
        ws[lab_addr].alignment = center
        ws[lab_addr].border = border
        ws[val_addr] = formula
        ws[val_addr].fill = fill_input if '후보출처' in lab else PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')
        ws[val_addr].font = Font(name='맑은 고딕', color='111827')
        ws[val_addr].alignment = center
        ws[val_addr].border = border

    ws.merge_cells('K9:N12')
    ws['K9'] = _v963_formula_lookup('$C$4','상세 전략 가이드','"가이드 없음"')
    ws['K9'].fill = PatternFill(start_color='EEF2FF', end_color='EEF2FF', fill_type='solid')
    ws['K9'].font = Font(name='맑은 고딕', color='3730A3', size=10)
    ws['K9'].alignment = left
    ws['K9'].border = border
    try:
        ws['K9'].comment = Comment('분야·후보출처·기술점수·추세·재무·백테스트 요약입니다. 최종 매수 전 뉴스/공시/호가 확인은 별도입니다.', 'v9.6.3')
    except Exception:
        pass


def _v963_patch_lab(ws):
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)
    fill_dark = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
    fill_card = PatternFill(start_color='F8FAFC', end_color='F8FAFC', fill_type='solid')
    ws.merge_cells('B49:H49')
    ws['B49'] = '선택 종목 가이드'
    ws['B49'].fill = fill_dark
    ws['B49'].font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=12)
    ws['B49'].alignment = center
    ws['B49'].border = border

    items = [
        ('B50','분야', 'C50', _v963_formula_lookup('$C$5','분야','"확인필요"')),
        ('D50','후보출처', 'E50', _v963_formula_lookup('$C$5','후보출처','"확인필요"')),
        ('F50','시장모드', 'G50', _v963_formula_lookup('$C$5','시장모드','"확인필요"')),
        ('B51','기술점수', 'C51', '="기본 "&TEXT('+_v963_formula_lookup('$C$5','기본기술점수','0')+',"0")&" / 최종 "&TEXT('+_v963_formula_lookup('$C$5','종목점수','0')+',"0")'),
        ('D51','추세', 'E51', _v963_formula_lookup('$C$5','추세필터판정','"확인필요"')),
        ('F51','재무', 'G51', _v963_formula_lookup('$C$5','재무위험판정','"확인필요"')),
    ]
    for lab_addr, lab, val_addr, formula in items:
        ws[lab_addr] = lab
        ws[lab_addr].fill = fill_card
        ws[lab_addr].font = Font(name='맑은 고딕', color='334155', bold=True)
        ws[lab_addr].alignment = center
        ws[lab_addr].border = border
        ws[val_addr] = formula
        ws[val_addr].font = Font(name='맑은 고딕', color='111827')
        ws[val_addr].alignment = center
        ws[val_addr].border = border
    ws.merge_cells('B53:H55')
    ws['B53'] = _v963_formula_lookup('$C$5','상세 전략 가이드','"가이드 없음"')
    ws['B53'].fill = PatternFill(start_color='EEF2FF', end_color='EEF2FF', fill_type='solid')
    ws['B53'].font = Font(name='맑은 고딕', color='3730A3', size=10)
    ws['B53'].alignment = left
    ws['B53'].border = border


def _v963_rebuild_cards(wb, top_df):
    if '종목카드_TOP10' in wb.sheetnames:
        del wb['종목카드_TOP10']
    ws = wb.create_sheet('종목카드_TOP10')
    ws.sheet_properties.tabColor = '4472C4'
    ws.sheet_view.showGridLines = False
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)
    for c,w in {'A':3,'B':18,'C':18,'D':18,'E':18,'F':18,'G':18,'H':18,'I':3}.items():
        ws.column_dimensions[c].width = w
    ws.merge_cells('B2:H3')
    ws['B2'] = '종목카드 TOP10 — 통합 후보 기준'
    ws['B2'].fill = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
    ws['B2'].font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=16)
    ws['B2'].alignment = center
    row_base = 5
    for idx, (_, r) in enumerate(top_df.head(10).iterrows(), 1):
        start = row_base + (idx-1)*7
        ws.merge_cells(start_row=start, start_column=2, end_row=start, end_column=8)
        h = ws.cell(start,2, f"#{idx}  {r.get('종목명','')} · {r.get('분야','확인필요')} · {r.get('후보출처','')} · {r.get('표시구분','')}")
        h.fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
        h.font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=12)
        h.alignment = left
        h.border = border
        fields = [
            ('판정', r.get('AI 시스템 판정','')),
            ('현재가', f"{_v963_num(r.get('현재가'),0):,.0f}원"),
            ('점수', f"기본 {_v963_text(r.get('기본기술점수','-'))} / 최종 {_v963_text(r.get('종목점수','-'))}"),
            ('시장', r.get('시장모드','')),
            ('추세', r.get('추세필터판정','')),
            ('재무', r.get('재무위험판정','')),
            ('백테스트', f"{_v963_text(r.get('백테스트수익률','-'))}% / {_v963_text(r.get('백테스트신뢰도','-'))}"),
            ('익절', r.get('권장익절계획','')),
            ('손절', r.get('권장손절계획','')),
        ]
        positions = [('B','C'),('D','E'),('F','G')]
        rr = start + 1
        for i, (lab, val) in enumerate(fields):
            pc = positions[i%3]
            if i and i%3 == 0:
                rr += 1
            ws[f'{pc[0]}{rr}'] = lab
            ws[f'{pc[0]}{rr}'].fill = PatternFill(start_color='F8FAFC', end_color='F8FAFC', fill_type='solid')
            ws[f'{pc[0]}{rr}'].font = Font(name='맑은 고딕', color='334155', bold=True)
            ws[f'{pc[0]}{rr}'].alignment = center
            ws[f'{pc[0]}{rr}'].border = border
            ws[f'{pc[1]}{rr}'] = val
            ws[f'{pc[1]}{rr}'].alignment = left if lab in ['익절','손절','판정'] else center
            ws[f'{pc[1]}{rr}'].border = border
        ws.merge_cells(start_row=start+4, start_column=2, end_row=start+5, end_column=8)
        g = ws.cell(start+4,2, _v963_text(r.get('상세 전략 가이드','가이드 없음')))
        g.fill = PatternFill(start_color='EEF2FF', end_color='EEF2FF', fill_type='solid')
        g.font = Font(name='맑은 고딕', color='3730A3', size=9)
        g.alignment = left
        g.border = border
    return ws


def _v963_rebuild_home(wb, top_df, rejection_df):
    if '홈_브리핑' in wb.sheetnames:
        del wb['홈_브리핑']
    ws = wb.create_sheet('홈_브리핑', 0)
    ws.sheet_properties.tabColor = '0B1F3A'
    ws.sheet_view.showGridLines = False
    thin = Side(style='thin', color='D9E2F3')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left = Alignment(horizontal='left', vertical='center', wrap_text=True)
    for c,w in {'A':3,'B':18,'C':18,'D':18,'E':18,'F':18,'G':18,'H':18,'I':18,'J':18,'K':3}.items():
        ws.column_dimensions[c].width = w
    ws.merge_cells('B2:J3')
    ws['B2'] = f'실전매매 통합 시스템 v9.6.3 · {REPORT_DATE}'
    ws['B2'].fill = PatternFill(start_color='0B1F3A', end_color='0B1F3A', fill_type='solid')
    ws['B2'].font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=16)
    ws['B2'].alignment = center
    total_candidates = len(globals().get('total_df', pd.DataFrame()))
    final_count = len(globals().get('final_df', pd.DataFrame()))
    toss_count = 0
    try:
        toss_count = int(globals().get('total_df', pd.DataFrame())['후보출처'].astype(str).str.contains('TOSS', case=False, na=False).sum())
    except Exception:
        pass
    metrics = [
        ('B5:C7','통합 후보', f'{len(top_df)}개', '실제 화면 표시 기준', '1F4E78'),
        ('D5:E7','전체 수집', f'{total_candidates}개', '토스 첨부 + 자동 후보', '4472C4'),
        ('F5:G7','최종 분석', f'{final_count}개', '필터/백테스트 통과 후 분석', '70AD47'),
        ('H5:I7','토스 후보', f'{toss_count}개', 'TOP요약에 참고 후보로 통합', 'C65911'),
    ]
    for rng, title, val, memo, color in metrics:
        ws.merge_cells(rng)
        c = ws[rng.split(':')[0]]
        c.value = f'{title}\n{val}\n{memo}'
        c.fill = PatternFill(start_color=color, end_color=color, fill_type='solid')
        c.font = Font(name='맑은 고딕', color='FFFFFF', bold=True, size=12)
        c.alignment = center
        c.border = border
    ws.merge_cells('B9:J10')
    ws['B9'] = '보는 순서: TOP후보_요약 → 종목카드_TOP10 → 스나이퍼_계산기 → 백테스트_실험실 → 탈락사유_리포트'
    ws['B9'].fill = PatternFill(start_color='EEF2FF', end_color='EEF2FF', fill_type='solid')
    ws['B9'].font = Font(name='맑은 고딕', color='3730A3', bold=True)
    ws['B9'].alignment = center
    ws['B9'].border = border

    preview = top_df[[c for c in ['표시구분','종목명','분야','후보출처','AI 시스템 판정','종목점수','추세필터판정','재무위험판정','백테스트수익률'] if c in top_df.columns]].head(8)
    start_row = 12
    for cidx, col in enumerate(preview.columns, 2):
        cell = ws.cell(start_row, cidx, col)
        cell.fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
        cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True)
        cell.alignment = center
        cell.border = border
    for ridx, (_, r) in enumerate(preview.iterrows(), start_row+1):
        for cidx, col in enumerate(preview.columns, 2):
            cell = ws.cell(ridx, cidx, r.get(col,''))
            cell.border = border
            cell.alignment = left if col in ['AI 시스템 판정','종목명'] else center
    return ws


def _v963_reorder_and_hide(wb):
    front = [
        '홈_브리핑','모바일_대시보드','TOP후보_요약','종목카드_TOP10',
        '스나이퍼_계산기','백테스트_실험실','오늘_체크리스트','탈락사유_리포트',
        '전략백테스트요약','추천 리스트'
    ]
    # 토스_TOP후보는 TOP후보_요약에 합쳤으므로 숨김
    hide_names = [
        '토스_TOP후보','실험실_요약','실험실_기록','실험실_옵션','실험실_최고전략',
        '전략백테스트상세','토스후보_정리결과','토스후보_매칭실패',
        '계좌백테스트곡선','계좌백테스트거래내역','계좌백테스트후보기록'
    ]
    for name in hide_names:
        if name in wb.sheetnames:
            wb[name].sheet_state = 'hidden'
    # 첫 화면들은 표시
    for name in front:
        if name in wb.sheetnames:
            wb[name].sheet_state = 'visible'
    ordered = []
    seen = set()
    for name in front:
        if name in wb.sheetnames:
            ordered.append(wb[name]); seen.add(name)
    for ws in wb.worksheets:
        if ws.title not in seen:
            ordered.append(ws); seen.add(ws.title)
    wb._sheets = ordered
    wb.active = 0


def apply_v963_unified_ux_patch(output_file):
    global final_df
    if final_df is None:
        final_df = pd.DataFrame()
    if not final_df.empty:
        final_df = _v963_enrich_display_df(final_df)
        # 표시구분은 실제 TOP 화면에만 사용하지만 추천 리스트에도 수식 참조 가능하게 기본값 부여
        if '표시구분' not in final_df.columns:
            final_df['표시구분'] = '전체후보'

    top_df = _v963_unified_top_df(final_df)
    rejection_df = _v963_make_rejection_report()

    wb = load_workbook(output_file)

    # 추천 리스트 시트에 분야/표시구분 컬럼 보강
    if '추천 리스트' in wb.sheetnames and not final_df.empty:
        ws = wb['추천 리스트']
        headers = [ws.cell(1, c).value for c in range(1, ws.max_column+1)]
        append_cols = []
        for col in ['분야','표시구분']:
            if col not in headers and col in final_df.columns:
                append_cols.append(col)
        if append_cols:
            start_col = ws.max_column + 1
            row_map = {}
            for r in range(2, ws.max_row+1):
                code = _v963_text(ws.cell(r, 3).value, '')  # 종목코드가 대체로 C열
                name = _v963_text(ws.cell(r, 2).value, '')  # 종목명 B열
                row_map[(code, name)] = r
            f_by_code = final_df.set_index(final_df['종목코드'].astype(str)).to_dict('index') if '종목코드' in final_df.columns else {}
            for offset, col in enumerate(append_cols):
                c = start_col + offset
                ws.cell(1, c, col)
                ws.cell(1, c).fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
                ws.cell(1, c).font = Font(color='FFFFFF', bold=True)
                for r in range(2, ws.max_row+1):
                    code = _v963_text(ws.cell(r, 3).value, '')
                    rowdata = f_by_code.get(code, {})
                    ws.cell(r, c, rowdata.get(col, ''))
                ws.column_dimensions[get_column_letter(c)].width = 16

    # TOP후보_요약 재생성
    top_cols = [c for c in _v963_display_cols() if c in top_df.columns]
    _v963_write_df_sheet(
        wb, 'TOP후보_요약', top_df[top_cols].copy() if not top_df.empty else pd.DataFrame(columns=top_cols),
        title='TOP후보_요약 — 전체 TOP + 토스 참고 후보를 한 화면에 통합',
        tab_color='70AD47'
    )

    # 탈락사유 리포트 생성
    _v963_write_df_sheet(
        wb, '탈락사유_리포트', rejection_df,
        title='탈락사유_리포트 — 토스/네이버 후보가 최종 후보에서 빠진 이유',
        tab_color='C65911'
    )

    # 카드형 시트 재생성
    _v963_rebuild_cards(wb, top_df)

    # 홈브리핑 재생성
    _v963_rebuild_home(wb, top_df, rejection_df)

    # 스나이퍼/실험실에 선택 종목 가이드 추가
    if '스나이퍼_계산기' in wb.sheetnames:
        _v963_patch_sniper(wb['스나이퍼_계산기'])
    if '백테스트_실험실' in wb.sheetnames:
        _v963_patch_lab(wb['백테스트_실험실'])

    # 모바일 대시보드의 제목/설명만 통합 기준으로 보강
    if '모바일_대시보드' in wb.sheetnames:
        ws = wb['모바일_대시보드']
        try:
            ws['B2'] = f'모바일 대시보드 — 통합 후보 기준 · {REPORT_DATE}'
        except Exception:
            pass

    # 파일 열 때 수식 재계산
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = 'auto'
    except Exception:
        pass

    _v963_reorder_and_hide(wb)
    wb.save(output_file)
    return output_file


output_file = apply_v963_unified_ux_patch(output_file)
print(f'✅ v9.6.3 통합 UX/탈락사유 리포트 적용 완료: {output_file}')

# v9.7 셀에서 보유종목/진입시나리오 패치를 적용한 뒤 최종 다운로드합니다.


✅ v9.6.3 통합 UX/탈락사유 리포트 적용 완료: 실전매매_통합시스템_v9_7_5_4_KST최종저장일수정_20260608.xlsx


## 9. 매일 리포트 운영 팁 — v9.6

이 버전은 토스 첨부파일 후보군과 자동 후보군을 함께 분석해 매일 리포트를 쌓아가는 방식에 적합합니다.

### 추천 운영 방식

| 시간대 | 활용 방식 |
|---|---|
| 장 시작 전 | 시장모드 확인, 전일 기준 후보 점검 |
| 장 초반 | 거래량 상위 후보 변화 확인 |
| 장중 | A급/선별 후보만 관심종목에 등록 |
| 매수 전 | 스나이퍼 계산기에서 `내가 실제 투입할 금액`을 입력해 익절·손절 시나리오 확인 |
| 장 마감 후 | 매매일지 입력, 실패 원인 기록 |

### 리포트를 볼 때 순서

1. `모바일_대시보드`에서 시장모드와 TOP 후보를 먼저 확인
2. `종목카드_TOP10`에서 후보를 카드형으로 보고, 필요한 경우 `추천 리스트` 상세를 확인
3. `제외종목_위험필터`에 투자경고/위험/거래정지/DART 재무위험 후보가 빠졌는지 확인
4. `전략백테스트요약`에서 TP/SL 전략 조건과 성과 비교
5. `전략백테스트상세`에서 매수일·매도일·매도사유 확인
6. `스나이퍼_계산기`에서 종목 선택 후 실제투입금액 입력
7. 익절표/손절표에서 예상매도금액·예상손익·예상수익률 확인
8. 모르는 용어는 `용어설명` 시트와 셀 메모로 확인
9. `백테스트검증가이드`에서 6개월 결과를 어떻게 해석할지 확인
10. 매매 후 `매매일지`에 실제 결과 기록

### 다음 업그레이드 후보

- 후보소스별 성과 비교
- 텔레그램 자동 알림
- 뉴스 키워드 기반 테마 점수
- 체결강도/호가 데이터 연동
- 매매일지 누적 분석
- 워크포워드 검증 강화
- DART 재무제표 부채비율/영업이익 연동
- KRX 시장경보 자동조회 안정화
- 로컬 PC 자동 실행 스케줄러


### v8.4 계좌 백테스트 해석법

- `계좌백테스트요약`: 200만 원으로 시작했을 때 최종자산, 누적손익, 누적수익률, MDD를 봅니다.
- `계좌백테스트곡선`: 날짜별 현금/보유종목수/평가자산 흐름을 봅니다.
- `계좌백테스트거래내역`: 언제 사고, 어떤 조건으로 익절·손절했는지 확인합니다.
- `계좌백테스트후보기록`: 매일 상위 후보로 잡힌 종목을 확인합니다.

실제 투자에서는 수익률보다 **MDD와 연속 손실 구간**을 먼저 보는 게 안전합니다.


### v8.9 확인 포인트

- `전략백테스트요약`의 수익률은 이제 기본적으로 실전비용 차감 후 결과입니다.
- `추세필터판정`이 `제외`인 종목은 기술 반등이 보여도 원칙적으로 매수 후보에서 제외하는 쪽이 안전합니다.
- `DART조회일`이 오래되었거나 `DART데이터상태`가 데이터부족이면, 투자 전 공시와 재무제표를 한 번 더 확인하세요.


### v9.6 추가
- `백테스트_실험실` 시트에서 종목·기간(기본 1개월)·전략·투입금액을 드롭다운으로 바꿔가며 매매기록을 확인할 수 있습니다.


## 토스 수집 실패 시 권장 운영

토스 자동수집은 브라우저 판정/동적 로딩 문제로 실패할 수 있습니다. 이때는 `TOSS_수동후보.csv`에 토스 스크리너에서 본 종목을 직접 넣고 다시 실행하세요. 종목명만 넣어도 매칭을 시도하지만, 종목코드를 함께 넣는 편이 가장 안정적입니다.

향후 토스증권 공식 API에서 스크리너 후보 또는 관심종목 목록 endpoint를 제공하면 `ENABLE_TOSS_API=True`, `TOSS_API_ENDPOINT`, `TOSS_API_KEY`를 설정해 Selenium 없이 후보를 가져올 수 있도록 v9.6에 연결부를 준비해두었습니다.


### v9.6 토스 첨부파일 운영 팁

- `국내 저평가주식 목록 토스.xlsx`처럼 토스 스크리너에서 복붙한 파일을 노트북 폴더에 두면 자동으로 읽습니다.
- 실행 후 `TOSS_수동후보_정리본.csv`가 만들어지면 정상 추출입니다.
- `TOSS_매칭실패.csv`가 생기면 해당 종목은 종목명 표기 차이로 KRX 코드 매칭이 필요합니다.
- 토스 후보는 1차 후보일 뿐이며, 최종 판단은 안전필터·DART·추세필터·실전비용 백테스트를 통과한 결과를 기준으로 봅니다.

- `스나이퍼_계산기`의 추천 리스트 참조 범위를 실제로 `A:ZZ`로 보정
- `백테스트_실험실`에 선택 종목/기간 기준 최고 수익 전략 표시
- `오늘_체크리스트`를 전부 수동 확인이 아니라 자동판정+근거 형태로 개선
- 토스 후보가 TOP에서 묻히지 않도록 `토스_TOP후보` 시트 추가


In [19]:
# v9.7.5.4: 날짜가 넘어간 뒤에도 최종 저장 시점(KST) 기준으로 갱신
refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))
# 8-4. v9.7.1 진입단가·보유종목 자동승계·매도판단 시나리오
# 목적:
# 1) 전날 리포트 파일에 입력한 보유종목을 자동으로 이어받습니다.
# 2) 신규 후보에는 공격/기준/보수/돌파 진입가를 제공합니다.
# 3) 보유종목에는 수익률·익절/손절·보유/부분매도/손절검토 시나리오를 제공합니다.
# 4) 최종 산출 엑셀 파일명을 날짜형(예: 20260522.xlsx)으로 저장합니다.

import glob, shutil
from pathlib import Path
from copy import copy as _copy_style

# v9.7.3.1 패치: 보유/진입 시트 함수에서 사용하는 공통 테두리 정의
thin = Border(left=Side(style='thin', color='D9E2F3'), right=Side(style='thin', color='D9E2F3'), top=Side(style='thin', color='D9E2F3'), bottom=Side(style='thin', color='D9E2F3'))


CONFIG.setdefault('ENABLE_ENTRY_SCENARIO', True)
CONFIG.setdefault('ENTRY_SCENARIO_MAX_ROWS', 40)
CONFIG.setdefault('ENABLE_HOLDINGS_MANAGEMENT', True)
CONFIG.setdefault('HOLDINGS_AUTO_IMPORT_PREVIOUS', True)
CONFIG.setdefault('HOLDINGS_PREVIOUS_FILE', '')
CONFIG.setdefault('HOLDINGS_INPUT_SHEET', '보유종목_입력')
CONFIG.setdefault('HOLDINGS_DIAGNOSIS_SHEET', '보유종목_진단')
CONFIG.setdefault('HOLDINGS_SEARCH_DATE_REGEX', r'(20\d{6})')
CONFIG.setdefault('OUTPUT_FILENAME_DATE_ONLY', True)
CONFIG.setdefault('OUTPUT_DATE_FORMAT', '%Y%m%d')


def _v97_code(value):
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return ''
        txt = re.sub(r'[^0-9]', '', str(value))
        if not txt:
            return ''
        return txt.zfill(6)[-6:]
    except Exception:
        return ''


def _v97_num(value, default=np.nan):
    try:
        if value is None:
            return default
        txt = str(value).replace(',', '').replace('원', '').replace('주', '').replace('%', '').strip()
        txt = re.sub(r'[^0-9.\-]', '', txt)
        if txt in ['', '-', '.', '-.']:
            return default
        return float(txt)
    except Exception:
        return default


def _v97_safe_text(value, default=''):
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return default
        return str(value).strip()
    except Exception:
        return default


def _v97_style_header(ws, row=1, fill='1F4E78'):
    for cell in ws[row]:
        if cell.value is not None:
            cell.fill = PatternFill(start_color=fill, end_color=fill, fill_type='solid')
            cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin


def _v97_delete_sheet(wb, name):
    if name in wb.sheetnames:
        del wb[name]


def _v97_find_header_row(ws, required=('종목코드', '보유수량', '평균단가'), max_rows=40, max_cols=30):
    aliases = {
        '종목코드': ['종목코드', '코드', 'ticker', '티커'],
        '종목명': ['종목명', '종목', '이름', 'name'],
        '보유수량': ['보유수량', '수량', '보유 주식수', '주식수', 'qty'],
        '평균단가': ['평균단가', '평단', '매입가', '매수가', '평균매입가', 'avg'],
        '매수일': ['매수일', '진입일', '최초매수일', 'date'],
        '적용전략': ['적용전략', '전략', '전략조합'],
        '메모': ['메모', '비고', 'memo']
    }
    for r in range(1, min(ws.max_row, max_rows) + 1):
        values = [_v97_safe_text(ws.cell(r, c).value).lower().replace(' ', '') for c in range(1, min(ws.max_column, max_cols) + 1)]
        found = {}
        for canonical, names in aliases.items():
            for c, val in enumerate(values, start=1):
                if any(val == str(n).lower().replace(' ', '') for n in names):
                    found[canonical] = c
                    break
        if all(k in found for k in required):
            return r, found
    return None, {}


def _v97_find_previous_holdings_file(current_output=''):
    explicit = _v97_safe_text(CONFIG.get('HOLDINGS_PREVIOUS_FILE'))
    if explicit and os.path.exists(explicit):
        return explicit

    if not CONFIG.get('HOLDINGS_AUTO_IMPORT_PREVIOUS', True):
        return ''

    today_key = RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT', '%Y%m%d'))
    current_abs = os.path.abspath(current_output) if current_output else ''
    candidates = []
    for pattern in ['*.xlsx', '/content/*.xlsx', '/mnt/data/*.xlsx']:
        for fp in glob.glob(pattern):
            if os.path.abspath(fp) == current_abs:
                continue
            name = os.path.basename(fp)
            if name.startswith('~$'):
                continue
            if '국내 저평가주식 목록 토스' in name or '해외주식리스트' in name:
                continue
            # 날짜형 파일을 우선 탐색합니다. 예: 20260521.xlsx, 실전매매_..._20260521.xlsx
            m = re.search(CONFIG.get('HOLDINGS_SEARCH_DATE_REGEX', r'(20\d{6})'), name)
            date_key = m.group(1) if m else ''
            if date_key and date_key >= today_key:
                continue
            candidates.append((date_key, os.path.getmtime(fp), fp))

    # 날짜가 있는 파일 중 가장 최근 날짜 우선, 없으면 mtime 최신 순
    candidates = sorted(candidates, key=lambda x: (x[0] or '00000000', x[1]), reverse=True)
    for _, _, fp in candidates:
        try:
            wb_prev = load_workbook(fp, data_only=False, read_only=False)
            if CONFIG.get('HOLDINGS_INPUT_SHEET', '보유종목_입력') in wb_prev.sheetnames:
                return fp
        except Exception:
            continue
    return ''


def _v97_extract_holdings_from_workbook(path):
    cols = ['종목명', '종목코드', '보유수량', '평균단가', '매수일', '적용전략', '메모']
    if not path or not os.path.exists(path):
        return pd.DataFrame(columns=cols), ''
    try:
        wb_prev = load_workbook(path, data_only=False, read_only=False)
        sheet_name = CONFIG.get('HOLDINGS_INPUT_SHEET', '보유종목_입력')
        if sheet_name not in wb_prev.sheetnames:
            return pd.DataFrame(columns=cols), f'{os.path.basename(path)} 안에 {sheet_name} 시트 없음'
        ws = wb_prev[sheet_name]
        header_row, header_map = _v97_find_header_row(ws)
        if not header_row:
            return pd.DataFrame(columns=cols), f'{os.path.basename(path)}에서 보유종목 헤더를 찾지 못함'

        rows = []
        for r in range(header_row + 1, ws.max_row + 1):
            name = _v97_safe_text(ws.cell(r, header_map.get('종목명', 0)).value if header_map.get('종목명') else '')
            code = _v97_code(ws.cell(r, header_map.get('종목코드')).value if header_map.get('종목코드') else '')
            qty = _v97_num(ws.cell(r, header_map.get('보유수량')).value if header_map.get('보유수량') else None, 0)
            avg = _v97_num(ws.cell(r, header_map.get('평균단가')).value if header_map.get('평균단가') else None, 0)
            buy_date = ws.cell(r, header_map.get('매수일')).value if header_map.get('매수일') else ''
            strategy = _v97_safe_text(ws.cell(r, header_map.get('적용전략')).value if header_map.get('적용전략') else '')
            memo = _v97_safe_text(ws.cell(r, header_map.get('메모')).value if header_map.get('메모') else '')
            if not code and not name and (not qty or qty <= 0) and (not avg or avg <= 0):
                # 빈 행은 건너뜁니다.
                continue
            if not code and not name:
                continue
            rows.append({
                '종목명': name, '종목코드': code, '보유수량': qty, '평균단가': avg,
                '매수일': buy_date, '적용전략': strategy, '메모': memo
            })
        df = pd.DataFrame(rows, columns=cols)
        return df, f'{os.path.basename(path)}에서 {len(df)}개 보유종목 승계'
    except Exception as e:
        return pd.DataFrame(columns=cols), f'전일 보유종목 추출 실패: {e}'


def _v97_make_code_name_map():
    base = {}
    if 'final_df' in globals() and final_df is not None and not final_df.empty:
        for _, r in final_df.iterrows():
            c = _v97_code(r.get('종목코드'))
            if c:
                base[c] = {
                    '종목명': _v97_safe_text(r.get('종목명')),
                    '분야': _v97_safe_text(r.get('분야') or r.get('카테고리') or r.get('업종')),
                    '후보출처': _v97_safe_text(r.get('후보출처')),
                    '전략': _v97_safe_text(r.get('추천전략') or r.get('전략조합') or '')
                }
    try:
        listing = get_krx_listing_for_match()
        if listing is not None and not listing.empty:
            for _, r in listing.iterrows():
                c = _v97_code(r.get('종목코드'))
                if c and c not in base:
                    base[c] = {'종목명': _v97_safe_text(r.get('종목명')), '분야': '', '후보출처': '', '전략': ''}
    except Exception:
        pass
    return base


def _v97_find_strategy(row_strategy='', code=''):
    txt = _v97_safe_text(row_strategy)
    tp_name = ''
    sl_name = ''
    for s in tp_strats:
        if s['name'] in txt:
            tp_name = s['name']
            break
    for s in sl_strats:
        if s['name'] in txt:
            sl_name = s['name']
            break
    if (not tp_name or not sl_name) and 'final_df' in globals() and final_df is not None and not final_df.empty and code:
        try:
            fr = final_df[final_df['종목코드'].astype(str).str.zfill(6) == code]
            if not fr.empty:
                rr = fr.iloc[0]
                tp_name = tp_name or _v97_safe_text(rr.get('추천TP전략') or rr.get('TP전략'))
                sl_name = sl_name or _v97_safe_text(rr.get('추천SL전략') or rr.get('SL전략'))
        except Exception:
            pass
    tp_name = tp_name if tp_name in [s['name'] for s in tp_strats] else CONFIG.get('PORTFOLIO_DEFAULT_TP', '[단기스윙]')
    sl_name = sl_name if sl_name in [s['name'] for s in sl_strats] else CONFIG.get('PORTFOLIO_DEFAULT_SL', '[정석방어]')
    tp = next((s for s in tp_strats if s['name'] == tp_name), tp_strats[0])
    sl = next((s for s in sl_strats if s['name'] == sl_name), sl_strats[0])
    return tp_name, sl_name, tp, sl


def _v97_plan_text(strategy):
    if not strategy:
        return ''
    label = '익절' if any(step.get('pct', 0) > 0 for step in strategy.get('steps', [])) else '손절'
    return ' / '.join([f"{step['pct']:+.0f}%▶{int(step['portion']*100)}% {label}" for step in strategy.get('steps', [])])


def _v97_latest_metrics(code):
    code = _v97_code(code)
    empty = {'현재가': np.nan, 'MA5': np.nan, 'MA20': np.nan, 'MA60': np.nan, 'MA60기울기': np.nan,
             'RSI': np.nan, 'TRIX': np.nan, 'ATR14': np.nan, '최근20일고점': np.nan, '최근5일저가': np.nan, '데이터상태': '시세없음'}
    if not code:
        return empty
    try:
        hist = safe_fdr_reader(code, (RUN_AT - timedelta(days=220)).strftime('%Y-%m-%d'))
        if hist is None or hist.empty or len(hist) < 30:
            return empty
        hist = calc_indicators(hist)
        latest = hist.iloc[-1]
        close = float(latest.get('Close'))
        ma60_now = float(latest.get('MA60')) if not pd.isna(latest.get('MA60')) else np.nan
        lookback = int(CONFIG.get('MA60_SLOPE_LOOKBACK', 5))
        ma60_prev = hist['MA60'].iloc[-lookback] if len(hist) > lookback else np.nan
        return {
            '현재가': close,
            'MA5': float(latest.get('MA5')) if not pd.isna(latest.get('MA5')) else np.nan,
            'MA20': float(latest.get('MA20')) if not pd.isna(latest.get('MA20')) else np.nan,
            'MA60': ma60_now,
            'MA60기울기': float(ma60_now - ma60_prev) if not pd.isna(ma60_now) and not pd.isna(ma60_prev) else np.nan,
            'RSI': float(latest.get('RSI')) if not pd.isna(latest.get('RSI')) else np.nan,
            'TRIX': float(latest.get('TRIX')) if not pd.isna(latest.get('TRIX')) else np.nan,
            'ATR14': float(latest.get('ATR14')) if not pd.isna(latest.get('ATR14')) else np.nan,
            '최근20일고점': float(hist['High'].tail(20).max()),
            '최근5일저가': float(hist['Low'].tail(5).min()),
            '데이터상태': '정상'
        }
    except Exception:
        return empty


def _v97_build_entry_scenarios(candidates):
    cols = ['종목명','종목코드','분야','후보출처','진입판정','공격진입가','기준진입가','보수진입가','돌파진입가','손절기준가','현재가','MA20','MA60','RSI','ATR14','시나리오메모']
    if candidates is None or candidates.empty:
        return pd.DataFrame(columns=cols)
    rows = []
    use = candidates.copy()
    if '표시구분' in use.columns:
        # 실제 화면에 쓰는 통합 후보군 우선
        use = use[use['표시구분'].isin(['TOP', 'TOSS참고'])] if not use.empty else use
    use = use.head(int(CONFIG.get('ENTRY_SCENARIO_MAX_ROWS', 40)))
    for _, r in use.iterrows():
        code = _v97_code(r.get('종목코드'))
        if not code:
            continue
        m = _v97_latest_metrics(code)
        close, ma20, ma60, rsi, atr = m['현재가'], m['MA20'], m['MA60'], m['RSI'], m['ATR14']
        high20 = m['최근20일고점']
        if pd.isna(close) or close <= 0:
            judgment = '시세확인필요'
            memo = '현재가 데이터를 가져오지 못했습니다.'
            aggressive = standard = conservative = breakout = stop = np.nan
        else:
            atr_val = atr if not pd.isna(atr) and atr > 0 else close * 0.03
            aggressive = round(close * 0.995, 0)
            standard = round(max(0, min(close * 0.99, close - 0.35 * atr_val)), 0)
            conservative_base = ma20 if not pd.isna(ma20) and ma20 > 0 else close - atr_val
            conservative = round(max(0, min(close * 0.97, conservative_base, close - 0.8 * atr_val)), 0)
            breakout = round((high20 if not pd.isna(high20) and high20 > 0 else close) * 1.005, 0)
            stop_base = ma20 * 0.98 if not pd.isna(ma20) and ma20 > 0 else close * 0.95
            stop = round(min(stop_base, close - atr_val), 0)

            if not pd.isna(ma60) and close < ma60:
                judgment = '매수 보류'
                memo = '현재가가 60일선 아래입니다. 하락 추세가 끝나는지 먼저 확인하는 구간입니다.'
            elif not pd.isna(rsi) and rsi >= 75:
                judgment = '눌림 대기'
                memo = '단기 RSI가 높아 과열 가능성이 있습니다. 기준/보수 진입가 근처까지 기다리는 편이 안정적입니다.'
            elif not pd.isna(ma20) and close > ma20 and not pd.isna(ma60) and close > ma60:
                judgment = '지금 진입 가능'
                memo = '60일선 위에서 단기 추세가 유지 중입니다. 단, 1차 진입은 분할 접근이 안전합니다.'
            elif not pd.isna(high20) and close < high20 * 0.98:
                judgment = '돌파 대기'
                memo = '최근 고점 돌파 확인 후 진입하는 시나리오가 더 명확합니다.'
            else:
                judgment = '관망'
                memo = '추세와 가격 위치가 애매합니다. 기준 진입가나 돌파가 확인될 때까지 관찰합니다.'
        rows.append({
            '종목명': _v97_safe_text(r.get('종목명')), '종목코드': code,
            '분야': _v97_safe_text(r.get('분야') or r.get('카테고리') or r.get('업종')),
            '후보출처': _v97_safe_text(r.get('후보출처')),
            '진입판정': judgment, '공격진입가': aggressive, '기준진입가': standard,
            '보수진입가': conservative, '돌파진입가': breakout, '손절기준가': stop,
            '현재가': close, 'MA20': ma20, 'MA60': ma60, 'RSI': rsi, 'ATR14': atr,
            '시나리오메모': memo
        })
    return pd.DataFrame(rows, columns=cols)


def _v97_build_holdings_diagnosis(holdings_df):
    cols = ['종목명','종목코드','분야','보유수량','평균단가','현재가','평가금액','평가손익','수익률','보유판정','권장대응','적용전략','익절계획','손절계획','1차익절가','2차익절가','3차익절가','손절기준가','추세상태','기준일','메모']
    if holdings_df is None or holdings_df.empty:
        return pd.DataFrame(columns=cols)
    code_map = _v97_make_code_name_map()
    rows = []
    for _, h in holdings_df.iterrows():
        code = _v97_code(h.get('종목코드'))
        qty = _v97_num(h.get('보유수량'), 0)
        avg = _v97_num(h.get('평균단가'), 0)
        info = code_map.get(code, {})
        name = _v97_safe_text(h.get('종목명')) or info.get('종목명', '')
        field = info.get('분야', '')
        tp_name, sl_name, tp, sl = _v97_find_strategy(h.get('적용전략'), code)
        m = _v97_latest_metrics(code)
        close = m['현재가']
        ret = ((close / avg) - 1) * 100 if avg and close and not pd.isna(close) else np.nan
        value = close * qty if qty and close and not pd.isna(close) else np.nan
        pnl = (close - avg) * qty if qty and avg and close and not pd.isna(close) else np.nan
        tp_steps = tp.get('steps', [])
        sl_steps = sl.get('steps', [])
        tp_prices = [round(avg * (1 + step['pct']/100), 0) if avg else np.nan for step in tp_steps]
        sl_prices = [round(avg * (1 + step['pct']/100), 0) if avg else np.nan for step in sl_steps]
        stop_price = sl_prices[0] if sl_prices else np.nan

        if pd.isna(close) or avg <= 0 or qty <= 0:
            status = '입력확인필요'
            action = '종목코드·보유수량·평균단가를 확인하세요.'
        elif ret <= sl_steps[0]['pct'] if sl_steps else False:
            status = '손절 검토'
            action = f"{sl_steps[0]['pct']:+.0f}% 손절 기준 도달. 보유수량의 {int(sl_steps[0]['portion']*100)}% 정리 검토."
        elif any(ret >= step['pct'] for step in tp_steps):
            reached = [step for step in tp_steps if ret >= step['pct']]
            step = reached[-1]
            status = '익절 검토'
            action = f"{step['pct']:+.0f}% 익절 구간 도달. 보유수량의 {int(step['portion']*100)}% 부분매도 검토."
        elif not pd.isna(m['MA20']) and close < m['MA20']:
            status = '주의 관찰'
            action = '현재가가 20일선 아래입니다. 반등 실패 시 비중 축소를 검토하세요.'
        elif not pd.isna(m['MA60']) and close < m['MA60']:
            status = '추세 훼손'
            action = '현재가가 60일선 아래입니다. 신규추가매수보다 방어 우선입니다.'
        else:
            status = '보유 유지'
            action = '익절/손절 기준 전입니다. 추세 유지 여부를 확인하며 보유합니다.'
        trend = '시세확인필요'
        if not pd.isna(close) and not pd.isna(m['MA60']):
            trend = '60일선 위' if close >= m['MA60'] else '60일선 아래'
        rows.append({
            '종목명': name, '종목코드': code, '분야': field, '보유수량': qty, '평균단가': avg,
            '현재가': close, '평가금액': value, '평가손익': pnl, '수익률': ret,
            '보유판정': status, '권장대응': action,
            '적용전략': f'{tp_name}+{sl_name}', '익절계획': _v97_plan_text(tp), '손절계획': _v97_plan_text(sl),
            '1차익절가': tp_prices[0] if len(tp_prices)>0 else np.nan,
            '2차익절가': tp_prices[1] if len(tp_prices)>1 else np.nan,
            '3차익절가': tp_prices[2] if len(tp_prices)>2 else np.nan,
            '손절기준가': stop_price, '추세상태': trend, '기준일': REPORT_DATE,
            '메모': _v97_safe_text(h.get('메모'))
        })
    return pd.DataFrame(rows, columns=cols)


def _v97_write_table_sheet(wb, sheet_name, df, title, tab_color='4472C4', header_fill='1F4E78'):
    _v97_delete_sheet(wb, sheet_name)
    ws = wb.create_sheet(sheet_name)
    ws.sheet_properties.tabColor = tab_color
    ws['A1'] = title
    ws['A1'].font = Font(name='맑은 고딕', size=16, bold=True, color='1F2937')
    ws['A2'] = f'기준일: {REPORT_DATE} / 생성시각: {REPORT_DATETIME}'
    ws['A2'].font = Font(name='맑은 고딕', size=10, color='6B7280')
    if df is None or df.empty:
        df = pd.DataFrame(columns=['메시지'])
        df.loc[0] = ['표시할 데이터가 없습니다.']
    start_row = 4
    for c, h in enumerate(df.columns, 1):
        ws.cell(start_row, c, h)
    for r_idx, row in enumerate(df.itertuples(index=False), start_row + 1):
        for c_idx, val in enumerate(row, 1):
            if pd.isna(val) if isinstance(val, (float, np.floating)) else False:
                val = ''
            ws.cell(r_idx, c_idx, val)
    _v97_style_header(ws, start_row, header_fill)
    for row in ws.iter_rows(min_row=start_row+1, max_row=ws.max_row, max_col=ws.max_column):
        for cell in row:
            cell.border = thin
            cell.font = Font(name='맑은 고딕', size=10)
            cell.alignment = Alignment(vertical='center', wrap_text=True)
    # 숫자 서식
    for col_idx, col_name in enumerate(df.columns, 1):
        letter = get_column_letter(col_idx)
        if col_name in ['현재가','평균단가','평가금액','평가손익','공격진입가','기준진입가','보수진입가','돌파진입가','손절기준가','1차익절가','2차익절가','3차익절가','MA20','MA60','ATR14']:
            for cell in ws[letter][start_row:]:
                cell.number_format = '#,##0'
        elif col_name in ['수익률','RSI']:
            for cell in ws[letter][start_row:]:
                cell.number_format = '0.0'
        elif col_name in ['보유수량']:
            for cell in ws[letter][start_row:]:
                cell.number_format = '#,##0'
    try:
        ws.auto_filter.ref = ws.dimensions
        ws.freeze_panes = f'A{start_row+1}'
    except Exception:
        pass
    for idx, col in enumerate(df.columns, 1):
        width = 14
        if col in ['권장대응','시나리오메모','익절계획','손절계획','메모']:
            width = 36
        elif col in ['종목명','진입판정','보유판정','적용전략']:
            width = 18
        ws.column_dimensions[get_column_letter(idx)].width = width
    return ws


def _v97_create_holdings_input_sheet(wb, holdings_df, source_note=''):
    sheet_name = CONFIG.get('HOLDINGS_INPUT_SHEET', '보유종목_입력')
    _v97_delete_sheet(wb, sheet_name)
    ws = wb.create_sheet(sheet_name)
    ws.sheet_properties.tabColor = '00B050'
    ws['B2'] = '보유종목_입력'
    ws['B2'].font = Font(name='맑은 고딕', size=18, bold=True, color='166534')
    ws['B3'] = '직접 입력 또는 전날 리포트에서 자동 승계됩니다. 필수 입력: 종목코드 / 보유수량 / 평균단가'
    ws['B3'].font = Font(name='맑은 고딕', size=10, color='475569')
    ws['B4'] = source_note or '전일 보유종목 파일 없음 — 아래에 직접 입력하세요.'
    ws['B4'].font = Font(name='맑은 고딕', size=10, color='92400E')
    headers = ['종목명','종목코드','보유수량','평균단가','매수일','적용전략','메모']
    start_row, start_col = 6, 2
    for i, h in enumerate(headers, start_col):
        cell = ws.cell(start_row, i, h)
        cell.fill = PatternFill(start_color='166534', end_color='166534', fill_type='solid')
        cell.font = Font(name='맑은 고딕', color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = thin
    if holdings_df is None or holdings_df.empty:
        holdings_df = pd.DataFrame([{c:'' for c in headers} for _ in range(10)])
    else:
        holdings_df = holdings_df.copy()
        for c in headers:
            if c not in holdings_df.columns:
                holdings_df[c] = ''
        # 사용자가 추가 입력할 수 있도록 빈 줄 여분 추가
        blanks = pd.DataFrame([{c:'' for c in headers} for _ in range(10)])
        holdings_df = pd.concat([holdings_df[headers], blanks], ignore_index=True)
    for r_idx, row in enumerate(holdings_df[headers].itertuples(index=False), start_row+1):
        for c_idx, val in enumerate(row, start_col):
            ws.cell(r_idx, c_idx, val)
            ws.cell(r_idx, c_idx).border = thin
            ws.cell(r_idx, c_idx).font = Font(name='맑은 고딕', size=10)
            ws.cell(r_idx, c_idx).alignment = Alignment(vertical='center', wrap_text=True)
    widths = [18, 12, 12, 14, 14, 20, 32]
    for i, w in enumerate(widths, start_col):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.freeze_panes = 'B7'
    # 숫자 서식
    for rr in range(start_row+1, ws.max_row+1):
        ws.cell(rr, start_col+2).number_format = '#,##0'
        ws.cell(rr, start_col+3).number_format = '#,##0'
    return ws


def _v97_patch_sniper_with_entry_and_holdings(wb):
    if '스나이퍼_계산기' not in wb.sheetnames:
        return
    ws = wb['스나이퍼_계산기']
    # 기존 화면 우측에 진입/보유 판단 요약을 추가합니다.
    start_col = 10  # J
    labels = [
        ('진입판정', '=IFERROR(INDEX(진입시나리오!$E:$E,MATCH($C$4,진입시나리오!$A:$A,0)),"후보 없음")'),
        ('공격진입가', '=IFERROR(INDEX(진입시나리오!$F:$F,MATCH($C$4,진입시나리오!$A:$A,0)),"")'),
        ('기준진입가', '=IFERROR(INDEX(진입시나리오!$G:$G,MATCH($C$4,진입시나리오!$A:$A,0)),"")'),
        ('보수진입가', '=IFERROR(INDEX(진입시나리오!$H:$H,MATCH($C$4,진입시나리오!$A:$A,0)),"")'),
        ('손절기준가', '=IFERROR(INDEX(진입시나리오!$J:$J,MATCH($C$4,진입시나리오!$A:$A,0)),"")'),
        ('보유판정', '=IFERROR(INDEX(보유종목_진단!$J:$J,MATCH($C$4,보유종목_진단!$A:$A,0)),"보유 입력 없음")'),
        ('보유권장대응', '=IFERROR(INDEX(보유종목_진단!$K:$K,MATCH($C$4,보유종목_진단!$A:$A,0)),"보유종목_입력 시트에 입력하면 표시됩니다.")')
    ]
    ws.cell(3, start_col, '진입·보유 시나리오').fill = PatternFill(start_color='0F766E', end_color='0F766E', fill_type='solid')
    ws.cell(3, start_col).font = Font(name='맑은 고딕', bold=True, color='FFFFFF')
    ws.cell(3, start_col).alignment = Alignment(horizontal='center')
    ws.merge_cells(start_row=3, start_column=start_col, end_row=3, end_column=start_col+2)
    for i, (lab, formula) in enumerate(labels, 4):
        ws.cell(i, start_col, lab)
        ws.cell(i, start_col).fill = PatternFill(start_color='F1F5F9', end_color='F1F5F9', fill_type='solid')
        ws.cell(i, start_col).font = Font(name='맑은 고딕', bold=True, color='334155')
        ws.cell(i, start_col+1, formula)
        ws.cell(i, start_col+1).font = Font(name='맑은 고딕', size=10)
        ws.cell(i, start_col+1).alignment = Alignment(wrap_text=True, vertical='center')
        for cc in range(start_col, start_col+3):
            ws.cell(i, cc).border = thin
    for c in range(start_col, start_col+3):
        ws.column_dimensions[get_column_letter(c)].width = 18 if c != start_col+1 else 28
    for r in [5,6,7,8]:
        ws.cell(r, start_col+1).number_format = '#,##0'


def _v97_reorder_sheets(wb):
    priority = [
        '홈_브리핑','모바일_대시보드','TOP후보_요약','종목카드_TOP10','진입시나리오',
        '보유종목_입력','보유종목_진단','스나이퍼_계산기','백테스트_실험실',
        '오늘_체크리스트','탈락사유_리포트','전략백테스트요약','추천 리스트'
    ]
    ordered = []
    for name in priority:
        if name in wb.sheetnames:
            ordered.append(wb[name])
    for ws in wb.worksheets:
        if ws.title not in priority:
            ordered.append(ws)
    wb._sheets = ordered


def _v971_find_existing_output_file(output_file=''):
    """v9.7.1 안전장치: output_file 변수가 꼬였거나 파일이 없을 때 최신 리포트 파일을 찾습니다."""
    if output_file and os.path.exists(output_file):
        return output_file, ''
    patterns = ['*.xlsx', '/content/*.xlsx', '/mnt/data/*.xlsx']
    candidates = []
    for pattern in patterns:
        for fp in glob.glob(pattern):
            name = os.path.basename(fp)
            if name.startswith('~$'):
                continue
            # 입력용 파일/템플릿은 제외합니다.
            if any(x in name for x in ['국내 저평가주식 목록 토스', '해외주식리스트', 'TOSS_수동후보', 'DART_재무', '시장경보']):
                continue
            try:
                candidates.append((os.path.getmtime(fp), fp))
            except Exception:
                pass
    candidates = sorted(candidates, reverse=True)
    if candidates:
        return candidates[0][1], f'기존 output_file을 찾지 못해 최신 리포트 파일을 사용했습니다: {os.path.basename(candidates[0][1])}'
    return output_file, '기본 리포트 파일을 찾지 못했습니다. 앞쪽 셀에서 엑셀 리포트가 먼저 생성되어야 합니다.'


def _v971_load_previous_holdings_safely(output_file):
    """전일 파일이 없어도 멈추지 않고 빈 보유종목 입력표로 진행합니다."""
    cols = ['종목명', '종목코드', '보유수량', '평균단가', '매수일', '적용전략', '메모']
    empty_df = pd.DataFrame(columns=cols)
    if not CONFIG.get('HOLDINGS_AUTO_IMPORT_PREVIOUS', True):
        return empty_df, '', '보유종목 자동승계 꺼짐 — 보유종목_입력 시트에 직접 입력하세요.'
    try:
        previous_file = _v97_find_previous_holdings_file(output_file)
    except Exception as e:
        return empty_df, '', f'전일 파일 탐색 중 오류가 있어 자동승계를 건너뜀 — 직접 입력 가능: {e}'
    if not previous_file:
        return empty_df, '', '첫 실행 모드 — 전일 리포트 파일 없음. 보유종목_입력 시트에 직접 입력하면 다음 실행부터 자동 승계됩니다.'
    try:
        holdings_df, extract_note = _v97_extract_holdings_from_workbook(previous_file)
    except Exception as e:
        return empty_df, previous_file, f'전일 파일은 찾았지만 보유종목 추출 실패 — 직접 입력 가능: {e}'
    if holdings_df is None:
        holdings_df = empty_df
    return holdings_df, previous_file, f'자동승계: {extract_note}'


def apply_v97_entry_holdings_patch(output_file):
    # v9.7.1: 전날 파일이 없어도 절대 중단하지 않습니다.
    output_file, output_note = _v971_find_existing_output_file(output_file)
    if not output_file or not os.path.exists(output_file):
        print('⚠️ v9.7.1:', output_note)
        print('   → 앞쪽 셀을 처음부터 다시 실행해서 기본 엑셀 리포트를 먼저 생성하세요.')
        return output_file, '', pd.DataFrame(), pd.DataFrame()

    wb = load_workbook(output_file)

    # 1) 전일 보유종목 자동 승계 — 없으면 첫 실행 모드로 빈 입력표 생성
    holdings_df, previous_file, note = _v971_load_previous_holdings_safely(output_file)
    if output_note:
        note = f'{note} / {output_note}'

    # 2) 진입 시나리오 생성
    scenario_source = final_df.copy() if 'final_df' in globals() and final_df is not None else pd.DataFrame()
    # v9.6.3의 top_df가 있으면 실제 화면 후보군과 일치시키기 위해 우선 사용합니다.
    if 'top_df' in globals() and top_df is not None and not top_df.empty:
        scenario_source = top_df.copy()
    entry_df = _v97_build_entry_scenarios(scenario_source)

    # 3) 보유종목 입력/진단 생성
    _v97_create_holdings_input_sheet(wb, holdings_df, note)
    diagnosis_df = _v97_build_holdings_diagnosis(holdings_df)

    _v97_write_table_sheet(wb, '진입시나리오', entry_df, '진입시나리오 — 지금 진입/눌림 대기/돌파 대기 판단', '00A2E8', '0F766E')
    _v97_write_table_sheet(wb, CONFIG.get('HOLDINGS_DIAGNOSIS_SHEET', '보유종목_진단'), diagnosis_df, '보유종목_진단 — 입력 보유종목의 익절·손절·보유 판단', '92D050', '166534')

    # 4) 스나이퍼 계산기에 진입/보유 요약 연결
    _v97_patch_sniper_with_entry_and_holdings(wb)

    # 5) 홈브리핑에 보유 승계 상태 추가
    if '홈_브리핑' in wb.sheetnames:
        ws = wb['홈_브리핑']
        try:
            ws['B9'] = '보유종목 승계'
            ws['C9'] = note
        except Exception:
            pass

    # 6) 계산 옵션/시트 순서
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = 'auto'
    except Exception:
        pass
    _v97_reorder_sheets(wb)

    # 7) 날짜형 파일명으로 최종 저장
    if CONFIG.get('OUTPUT_FILENAME_DATE_ONLY', True):
        final_name = f"{RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT', '%Y%m%d'))}.xlsx"
    else:
        final_name = output_file
    wb.save(final_name)
    return final_name, previous_file, entry_df, diagnosis_df


if CONFIG.get('ENABLE_ENTRY_SCENARIO', True) or CONFIG.get('ENABLE_HOLDINGS_MANAGEMENT', True):
    output_file, holdings_prev_file, entry_scenario_df, holdings_diagnosis_df = apply_v97_entry_holdings_patch(output_file)
    print(f'✅ v9.7 진입단가/보유종목 자동승계 적용 완료: {output_file}')
    print('📌 전일 보유종목 파일:', holdings_prev_file if holdings_prev_file else '없음 — 새 보유종목_입력 시트에 직접 입력')
    print(f'📌 진입시나리오 {len(entry_scenario_df)}건 / 보유종목 진단 {len(holdings_diagnosis_df)}건')

# HTML 파일도 날짜형 이름으로 맞춰 저장합니다.
try:
    if html_output_file and os.path.exists(html_output_file) and CONFIG.get('OUTPUT_FILENAME_DATE_ONLY', True):
        new_html = f"{RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT', '%Y%m%d'))}.html"
        shutil.copyfile(html_output_file, new_html)
        html_output_file = new_html
        print(f'✅ HTML 대시보드 날짜형 저장 완료: {html_output_file}')
except Exception as e:
    print(f'⚠️ HTML 날짜형 저장 생략: {e}')

if IN_COLAB:
    # v9.7.3.10: 중간 다운로드 비활성화 - 최종 셀에서만 다운로드합니다.
    # # files.download(output_file)  # v9.7.3.11 final cell에서만 다운로드
    if html_output_file:
        pass  # no-op: 중간 다운로드 비활성화로 인한 빈 if 블록 방지
        # v9.7.3.10: 중간 다운로드 비활성화 - 최종 셀에서만 다운로드합니다.
        # files.download(html_output_file)


⚠️ KRX 상장목록 조회 실패: HTTP Error 404: Not Found


✅ v9.7 진입단가/보유종목 자동승계 적용 완료: 20260608.xlsx
📌 전일 보유종목 파일: 20260607.xlsx
📌 진입시나리오 0건 / 보유종목 진단 3건
✅ HTML 대시보드 날짜형 저장 완료: 20260608.html


In [20]:
# v9.7.5.4: 날짜가 넘어간 뒤에도 최종 저장 시점(KST) 기준으로 갱신
refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))

# 8-5. v9.7.2 보유종목 입력표·통합후보15·분야표시·시트순서 안정화 패치
# 목적:
# 1) 보이는 후보 수를 모든 전면 시트에서 15개로 통일합니다.
# 2) 보유종목_입력 시트를 항상 보기 쉬운 입력표로 생성/보강합니다.
# 3) 종목 분야를 후보 요약·카드·스나이퍼·백테스트 실험실에서 함께 보여줍니다.
# 4) 토스_TOP후보는 별도 전면 시트에서 제거하고, TOP후보_요약 안에서 후보출처로 구분합니다.
# 5) 데이터/헬퍼 시트는 뒤쪽으로 보내거나 숨김 처리합니다.

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.comments import Comment
import os, re
from pathlib import Path

CONFIG['UX_TOP_N'] = int(CONFIG.get('UX_MASTER_TOP_N', 15))
CONFIG['UX_UNIFIED_TOP_N'] = int(CONFIG.get('UX_MASTER_TOP_N', 15))
CONFIG['LAB_MAX_STOCKS'] = int(CONFIG.get('UX_MASTER_TOP_N', 15))
CONFIG['ENTRY_SCENARIO_MAX_ROWS'] = int(CONFIG.get('UX_MASTER_TOP_N', 15))
CONFIG['UX_HIDE_TOSS_TOP_SHEET'] = True


def _v972_output_path():
    """앞 셀에서 만들어진 날짜형 리포트 파일을 찾습니다."""
    preferred = f"{RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT', '%Y%m%d'))}.xlsx"
    if os.path.exists(preferred):
        return preferred
    if 'output_file' in globals() and output_file and os.path.exists(output_file):
        return output_file
    candidates = []
    for fp in list(Path('.').glob('*.xlsx')) + list(Path('/content').glob('*.xlsx')) + list(Path('/mnt/data').glob('*.xlsx')):
        name = fp.name
        if name.startswith('~$'):
            continue
        if any(x in name for x in ['국내 저평가주식 목록 토스', '해외주식리스트', 'TOSS_수동후보', 'DART_재무', '시장경보', '보유종목_입력_템플릿']):
            continue
        try:
            candidates.append((fp.stat().st_mtime, str(fp)))
        except Exception:
            pass
    if candidates:
        return sorted(candidates, reverse=True)[0][1]
    return preferred


def _v972_clean_code(x):
    try:
        txt = re.sub(r'[^0-9]', '', str(x))
        return txt.zfill(6)[-6:] if txt else ''
    except Exception:
        return ''


def _v972_safe(x, default=''):
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return default
        s = str(x).strip()
        if s.lower() in ['nan', 'none']:
            return default
        return s
    except Exception:
        return default


def _v972_load_sector_manual():
    """종목분야_수동입력.csv가 있으면 분야 매핑에 사용합니다."""
    path = CONFIG.get('SECTOR_MANUAL_FILE', '종목분야_수동입력.csv')
    cols = ['종목코드', '종목명', '분야', '세부분야', '메모']
    if not os.path.exists(path):
        try:
            pd.DataFrame(columns=cols).to_csv(path, index=False, encoding='utf-8-sig')
        except Exception:
            pass
        return {}
    try:
        df = pd.read_csv(path, dtype=str).fillna('')
        if '종목코드' not in df.columns or '분야' not in df.columns:
            return {}
        mp = {}
        for _, r in df.iterrows():
            code = _v972_clean_code(r.get('종목코드'))
            sector = _v972_safe(r.get('분야'))
            if code and sector:
                mp[code] = sector
        return mp
    except Exception:
        return {}


def _v972_enhance_sector(final_df):
    if final_df is None or final_df.empty:
        return final_df
    df = final_df.copy()
    manual_map = _v972_load_sector_manual()
    if '분야' not in df.columns:
        df['분야'] = ''
    for idx, row in df.iterrows():
        code = _v972_clean_code(row.get('종목코드'))
        manual = manual_map.get(code, '')
        candidates = [
            manual,
            row.get('분야', ''),
            row.get('토스카테고리', ''),
            row.get('카테고리', ''),
            row.get('업종', ''),
            row.get('업태', ''),
            row.get('Sector', ''),
            row.get('Industry', ''),
            row.get('시장구분', '')
        ]
        sector = ''
        for c in candidates:
            c = _v972_safe(c)
            if c and c not in ['확인필요', '데이터부족', '미확인']:
                sector = c
                break
        df.at[idx, '분야'] = sector or CONFIG.get('SECTOR_UNKNOWN_LABEL', '분야확인필요')
    return df


def _v972_top_df(final_df):
    df = _v972_enhance_sector(final_df)
    if df is None or df.empty:
        return pd.DataFrame()
    sort_cols = [c for c in ['종목점수', '백테스트수익률', '거래대금(억)'] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, ascending=[False]*len(sort_cols)).reset_index(drop=True)
    n = int(CONFIG.get('UX_MASTER_TOP_N', 15))
    top = df.head(n).copy()
    top.insert(0, '순위', range(1, len(top)+1))
    return top


def _v972_fill(color): return PatternFill(start_color=color, end_color=color, fill_type='solid')
def _v972_font(color='111827', bold=False, size=10): return Font(name='맑은 고딕', color=color, bold=bold, size=size)
def _v972_align(h='center'): return Alignment(horizontal=h, vertical='center', wrap_text=True)
def _v972_border():
    thin = Side(style='thin', color='E5E7EB')
    return Border(left=thin, right=thin, top=thin, bottom=thin)


def _v972_delete_sheets(wb, names):
    for name in names:
        if name in wb.sheetnames:
            try:
                del wb[name]
            except Exception:
                pass


def _v972_write_df(ws, df, start_row=1, start_col=1, header_fill='1E293B'):
    border = _v972_border()
    center = _v972_align('center')
    left = _v972_align('left')
    if df is None:
        df = pd.DataFrame()
    # header
    for j, col in enumerate(df.columns, start_col):
        cell = ws.cell(start_row, j, col)
        cell.fill = _v972_fill(header_fill)
        cell.font = _v972_font('FFFFFF', True, 10)
        cell.alignment = center
        cell.border = border
    # body
    for i, (_, row) in enumerate(df.iterrows(), start_row+1):
        for j, col in enumerate(df.columns, start_col):
            val = row.get(col, '')
            cell = ws.cell(i, j, val)
            cell.border = border
            cell.alignment = left if col in ['종목명', '분야', '후보출처', '진입판정', '상세 전략 가이드', '권장익절계획', '권장손절계획'] else center
            cell.font = _v972_font('111827', False, 9)
            if col in ['종목명', 'AI 시스템 판정', '진입판정']:
                cell.fill = _v972_fill('F8FAFC')
        ws.row_dimensions[i].height = 28
    for idx, col in enumerate(df.columns, start_col):
        letter = get_column_letter(idx)
        width = min(max(10, len(str(col)) + 4), 24)
        if col in ['종목명', '상세 전략 가이드', '권장익절계획', '권장손절계획']:
            width = 28 if col != '상세 전략 가이드' else 42
        if col in ['분야', '후보출처']:
            width = 18
        ws.column_dimensions[letter].width = width
    try:
        ws.freeze_panes = ws.cell(start_row+1, start_col).coordinate
        ws.auto_filter.ref = ws.dimensions
    except Exception:
        pass


def _v972_compact_cols(top):
    cols = [
        '순위', 'AI 시스템 판정', '종목명', '종목코드', '분야', '후보출처', '현재가', '등락률',
        '기본기술점수', '종목점수', '진입판정', '기준진입가', '손절기준가',
        '추세필터판정', '재무위험판정', '백테스트수익률', '백테스트신뢰도',
        '권장익절계획', '권장손절계획'
    ]
    return [c for c in cols if c in top.columns]


def _v972_make_home(wb, top):
    _v972_delete_sheets(wb, ['홈_브리핑'])
    ws = wb.create_sheet('홈_브리핑', 0)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 90
    border = _v972_border(); center = _v972_align('center'); left = _v972_align('left')
    for col, width in {'A':3, 'B':18, 'C':18, 'D':18, 'E':18, 'F':18, 'G':18, 'H':18, 'I':3}.items():
        ws.column_dimensions[col].width = width
    ws.merge_cells('B2:H3')
    c=ws['B2']; c.value=f"실전매매 홈 브리핑 v9.7.2\n{REPORT_DATE} · 통합후보 {len(top)}개 기준"; c.fill=_v972_fill('0F172A'); c.font=_v972_font('FFFFFF', True, 18); c.alignment=center; c.border=border
    cards = [
        ('B5:C7', '시장모드', globals().get('market_mode','확인필요'), f"시장점수 {globals().get('market_score','-')}", 'E0F2FE', '075985'),
        ('D5:E7', '통합후보 기준', f"TOP {int(CONFIG.get('UX_MASTER_TOP_N',15))}", '전면 시트 동일 기준', 'F1F5F9', '111827'),
        ('F5:H7', '1순위 후보', _v972_safe(top.iloc[0].get('종목명')) if not top.empty else '없음', _v972_safe(top.iloc[0].get('AI 시스템 판정')) if not top.empty else '확인필요', 'DCFCE7', '166534')
    ]
    for rng, title, val, note, fill, color in cards:
        ws.merge_cells(rng); cell=ws[rng.split(':')[0]]; cell.value=f"{title}\n{val}\n{note}"; cell.fill=_v972_fill(fill); cell.font=_v972_font(color, True, 12); cell.alignment=center; cell.border=border
    guide = "보는 순서: 홈_브리핑 → 모바일_대시보드 → TOP후보_요약 → 종목카드_TOP15 → 진입시나리오/스나이퍼 → 백테스트_실험실 → 보유종목_진단"
    ws.merge_cells('B9:H10'); ws['B9']=guide; ws['B9'].fill=_v972_fill('FEFCE8'); ws['B9'].font=_v972_font('713F12', True, 11); ws['B9'].alignment=left; ws['B9'].border=border
    cols = _v972_compact_cols(top)
    _v972_write_df(ws, top[cols] if cols else top, start_row=12, start_col=2, header_fill='1E293B')


def _v972_make_mobile(wb, top):
    _v972_delete_sheets(wb, ['모바일_대시보드'])
    ws = wb.create_sheet('모바일_대시보드', 1)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 95
    border=_v972_border(); center=_v972_align('center')
    for col, width in {'A':3,'B':15,'C':15,'D':15,'E':15,'F':15,'G':15,'H':3}.items(): ws.column_dimensions[col].width=width
    ws.merge_cells('B2:G3'); ws['B2']=f"모바일 대시보드 — 통합후보 {len(top)}개"; ws['B2'].fill=_v972_fill('111827'); ws['B2'].font=_v972_font('FFFFFF', True, 16); ws['B2'].alignment=center; ws['B2'].border=border
    cols = [c for c in ['순위','AI 시스템 판정','종목명','분야','후보출처','종목점수','진입판정','권장익절계획'] if c in top.columns]
    _v972_write_df(ws, top[cols], start_row=5, start_col=2, header_fill='334155')


def _v972_make_top_summary(wb, top):
    _v972_delete_sheets(wb, ['TOP후보_요약', '토스_TOP후보'])
    ws = wb.create_sheet('TOP후보_요약', 2)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 90
    ws['A1'] = f"TOP후보 요약 — 토스/네이버 통합 상위 {len(top)}개"
    ws['A1'].fill = _v972_fill('0F172A'); ws['A1'].font = _v972_font('FFFFFF', True, 16); ws['A1'].alignment = _v972_align('left')
    cols = _v972_compact_cols(top)
    _v972_write_df(ws, top[cols], start_row=3, start_col=1, header_fill='1F4E78')
    try:
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max(8, len(cols)))
    except Exception:
        pass


def _v972_make_cards(wb, top):
    _v972_delete_sheets(wb, ['종목카드_TOP10', '종목카드_TOP15'])
    ws = wb.create_sheet('종목카드_TOP15', 3)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 85
    border=_v972_border(); left=_v972_align('left'); center=_v972_align('center')
    for col, width in {'A':3,'B':14,'C':18,'D':18,'E':18,'F':18,'G':18,'H':18,'I':3}.items(): ws.column_dimensions[col].width=width
    ws.merge_cells('B2:H3'); ws['B2']='종목카드 TOP15 — 분야/출처/진입/전략 가이드 통합'; ws['B2'].fill=_v972_fill('111827'); ws['B2'].font=_v972_font('FFFFFF', True, 16); ws['B2'].alignment=center; ws['B2'].border=border
    row_num=5
    for _, r in top.iterrows():
        idx = int(r.get('순위', 0)) if str(r.get('순위','')).isdigit() else ''
        title = f"#{idx}  {r.get('종목명','')}  ·  {r.get('분야','')}  ·  {r.get('AI 시스템 판정','')}  ·  점수 {r.get('종목점수','-')}"
        ws.merge_cells(start_row=row_num, start_column=2, end_row=row_num+1, end_column=8)
        h=ws.cell(row_num,2,title); h.fill=_v972_fill('E0F2FE'); h.font=_v972_font('075985', True, 12); h.alignment=left; h.border=border
        info = [
            ['후보출처', r.get('후보출처',''), '시장모드', globals().get('market_mode',''), '기술점수', r.get('기본기술점수',''), '최종점수', r.get('종목점수','')],
            ['진입판정', r.get('진입판정',''), '기준진입가', r.get('기준진입가',''), '추세', r.get('추세필터판정',''), '재무', r.get('재무위험판정','')],
            ['익절', r.get('권장익절계획',''), '손절', r.get('권장손절계획',''), '백테스트', r.get('백테스트신뢰도',''), '수익률', r.get('백테스트수익률','')],
            ['가이드', _v972_safe(r.get('상세 전략 가이드',''))[:240], '', '', '', '', '', '']
        ]
        for rr, vals in enumerate(info, row_num+2):
            for cc, val in enumerate(vals, 2):
                cell=ws.cell(rr, cc, val); cell.border=border; cell.alignment=left if cc in [3,5,7] or rr==row_num+5 else center; cell.font=_v972_font('111827', False, 9)
                if cc in [2,4,6,8] and rr != row_num+5:
                    cell.fill=_v972_fill('F8FAFC'); cell.font=_v972_font('475569', True, 9)
        try:
            ws.merge_cells(start_row=row_num+5, start_column=3, end_row=row_num+5, end_column=8)
        except Exception:
            pass
        for rr in range(row_num, row_num+6): ws.row_dimensions[rr].height = 24 if rr != row_num+5 else 50
        row_num += 8


def _v972_make_holdings_input_template(wb):
    name = CONFIG.get('HOLDINGS_INPUT_SHEET', '보유종목_입력')
    if name in wb.sheetnames:
        ws = wb[name]
        # 기존 입력값은 보존하고 상단 안내만 보강
    else:
        ws = wb.create_sheet(name)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 90
    border=_v972_border(); center=_v972_align('center'); left=_v972_align('left')
    # 기존 값이 있는지 확인
    existing = False
    for r in range(1, min(ws.max_row, 20)+1):
        for c in range(1, min(ws.max_column, 10)+1):
            if _v972_safe(ws.cell(r,c).value):
                existing = True
                break
        if existing:
            break
    if not existing:
        ws.merge_cells('A1:G2')
        ws['A1'] = '보유종목 입력 — 첫 실행 때 여기에 입력하면 다음날부터 자동 승계됩니다'
        ws['A1'].fill=_v972_fill('0F172A'); ws['A1'].font=_v972_font('FFFFFF', True, 15); ws['A1'].alignment=center; ws['A1'].border=border
        headers = ['종목명','종목코드','보유수량','평균단가','매수일','적용전략','메모']
        for j,h in enumerate(headers,1):
            cell=ws.cell(4,j,h); cell.fill=_v972_fill('1F4E78'); cell.font=_v972_font('FFFFFF', True, 10); cell.alignment=center; cell.border=border
        examples = [
            ['예시: 한성크린텍','066980',100,7200,REPORT_DATE,'[단기스윙]+[정석방어]','예시 행은 실제 보유종목으로 교체'],
            ['', '', '', '', '', '', '']
        ]
        for i,row in enumerate(examples,5):
            for j,val in enumerate(row,1):
                cell=ws.cell(i,j,val); cell.border=border; cell.alignment=left if j in [1,7] else center; cell.font=_v972_font('111827', False, 10)
        ws['A8']='필수 입력: 종목코드 / 보유수량 / 평균단가. 종목명·매수일·전략·메모는 선택입니다.'
        ws['A8'].font=_v972_font('B45309', True, 10)
    for col, width in {'A':22,'B':12,'C':12,'D':12,'E':14,'F':24,'G':42}.items(): ws.column_dimensions[col].width=width
    try:
        dv = DataValidation(type="list", formula1='"[초단타]+[칼손절],[단기스윙]+[정석방어],[중기추세]+[변동성허용],[수익극대화]+[버티기형]"', allow_blank=True)
        ws.add_data_validation(dv)
        dv.add(f'F5:F200')
    except Exception:
        pass


def _v972_patch_choice_sheets(wb):
    """스나이퍼/실험실에서 선택 종목의 분야·출처·기술점수·추세·재무를 보이게 합니다.
    v9.7.5.2: 병합 셀과 겹치지 않는 오른쪽 빈 블록을 자동 탐색해서 MergedCell read-only 오류를 방지합니다.
    """
    def formula_for(header, key_cell='$C$4'):
        return f'=IFERROR(INDEX(\'추천 리스트\'!$A:$ZZ,MATCH({key_cell},\'추천 리스트\'!$B:$B,0),MATCH("{header}",\'추천 리스트\'!$1:$1,0)),"")'

    def is_merged_cell(ws, row, col):
        try:
            for rng in ws.merged_cells.ranges:
                if rng.min_row <= row <= rng.max_row and rng.min_col <= col <= rng.max_col:
                    return True
        except Exception:
            pass
        return False

    def find_free_block(ws, start_cell, block_rows=12, block_cols=2):
        """start_cell부터 오른쪽으로 이동하며 병합 셀과 겹치지 않는 2열짜리 가이드 블록 위치를 찾습니다."""
        try:
            base_row = ws[start_cell].row
            base_col = ws[start_cell].column
        except Exception:
            base_row, base_col = 2, 11
        for offset in range(0, 80):
            c0 = base_col + offset
            conflict = False
            for rr in range(base_row, base_row + block_rows):
                for cc in range(c0, c0 + block_cols):
                    if is_merged_cell(ws, rr, cc):
                        conflict = True
                        break
                if conflict:
                    break
            if not conflict:
                return base_row, c0
        return base_row, base_col + 80

    blocks = {
        '스나이퍼_계산기': ('K2', '$C$4'),
        '백테스트_실험실': ('K2', '$C$5')
    }
    for sheet, (start_cell, key_cell) in blocks.items():
        if sheet not in wb.sheetnames:
            continue
        ws = wb[sheet]
        guide_rows = [
            ('분야', '분야'), ('후보출처','후보출처'), ('시장모드',''), ('기본기술점수','기본기술점수'),
            ('최종점수','종목점수'), ('진입판정','진입판정'), ('추세','추세필터판정'), ('재무','재무위험판정'), ('백테스트','백테스트신뢰도')
        ]
        start_row, start_col = find_free_block(ws, start_cell, block_rows=len(guide_rows)+1, block_cols=2)
        title_cell = ws.cell(start_row, start_col)
        title_cell.value = '선택 종목 가이드'
        title_cell.fill = _v972_fill('0F172A')
        title_cell.font = _v972_font('FFFFFF', True, 11)
        title_cell.alignment = _v972_align('center')
        title_cell.border = _v972_border()
        try:
            ws.cell(start_row, start_col+1).fill = _v972_fill('0F172A')
            ws.cell(start_row, start_col+1).border = _v972_border()
        except Exception:
            pass
        for i,(label,header) in enumerate(guide_rows, start_row+1):
            label_cell = ws.cell(i, start_col)
            label_cell.value = label
            label_cell.fill = _v972_fill('F8FAFC')
            label_cell.font = _v972_font('475569', True, 9)
            label_cell.alignment = _v972_align('center')
            label_cell.border = _v972_border()
            val_cell = ws.cell(i, start_col+1)
            val_cell.value = globals().get('market_mode','') if label == '시장모드' else formula_for(header, key_cell)
            val_cell.font = _v972_font('111827', False, 9)
            val_cell.alignment = _v972_align('left')
            val_cell.border = _v972_border()
        ws.column_dimensions[get_column_letter(start_col)].width = 16
        ws.column_dimensions[get_column_letter(start_col+1)].width = 30

def _v972_reorder_and_hide(wb):
    front = ['홈_브리핑','모바일_대시보드','TOP후보_요약','종목카드_TOP15','진입시나리오','보유종목_입력','보유종목_진단','스나이퍼_계산기','백테스트_실험실','오늘_체크리스트','탈락사유_리포트','전략백테스트요약','추천 리스트']
    # hidden data/helper sheets
    hide_patterns = ['실험실_기록','실험실_최고전략','계좌백테스트','전략백테스트상세','토스후보_정리결과','토스후보_매칭실패','제외종목_위험필터','시장상태']
    if '토스_TOP후보' in wb.sheetnames:
        try:
            del wb['토스_TOP후보']
        except Exception:
            pass
    existing_front = [s for s in front if s in wb.sheetnames]
    rest = [s for s in wb.sheetnames if s not in existing_front]
    wb._sheets = [wb[s] for s in existing_front + rest]
    for s in wb.sheetnames:
        if any(p in s for p in hide_patterns):
            try:
                wb[s].sheet_state = 'hidden'
            except Exception:
                pass
    try:
        wb.active = 0
    except Exception:
        pass


def apply_v972_unified_ux_patch(output_file):
    output_file = _v972_output_path()
    if not os.path.exists(output_file):
        print('⚠️ v9.7.2 패치 생략: 엑셀 리포트 파일을 찾지 못했습니다. 앞쪽 셀을 먼저 실행하세요.')
        return output_file, pd.DataFrame()
    # final_df 자체에도 분야 보강
    global final_df, top_df
    try:
        final_df = _v972_enhance_sector(final_df)
    except Exception:
        pass
    top = _v972_top_df(final_df if 'final_df' in globals() else pd.DataFrame())
    top_df = top.copy()
    wb = load_workbook(output_file)
    _v972_make_home(wb, top)
    _v972_make_mobile(wb, top)
    _v972_make_top_summary(wb, top)
    _v972_make_cards(wb, top)
    _v972_make_holdings_input_template(wb)
    _v972_patch_choice_sheets(wb)
    _v972_reorder_and_hide(wb)
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = 'auto'
    except Exception:
        pass
    final_name = f"{RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT', '%Y%m%d'))}.xlsx" if CONFIG.get('OUTPUT_FILENAME_DATE_ONLY', True) else output_file
    wb.save(final_name)
    return final_name, top

output_file, unified_top15_df = apply_v972_unified_ux_patch(output_file)
print(f'✅ v9.7.2 통합 UX 패치 완료: {output_file}')
print(f'📌 전면 시트 후보 기준: 통합후보 {len(unified_top15_df)}개')
print('📌 토스_TOP후보는 TOP후보_요약에 통합했고, 보유종목_입력 시트는 첫 실행용 템플릿으로 보강했습니다.')

# Colab 자동 다운로드
if IN_COLAB:
    pass  # 중간 다운로드 비활성화 - 최종 셀에서만 다운로드합니다.
    # # files.download(output_file)  # v9.7.3.11 final cell에서만 다운로드


✅ v9.7.2 통합 UX 패치 완료: 20260608.xlsx
📌 전면 시트 후보 기준: 통합후보 15개
📌 토스_TOP후보는 TOP후보_요약에 통합했고, 보유종목_입력 시트는 첫 실행용 템플릿으로 보강했습니다.


## v9.7.3.3.1 분야 자동보강 패치

토스 카테고리, 네이버 금융 업종, DART 업종코드, 수동 테마 CSV를 통합해 `표시분야`를 자동 보강합니다.


In [21]:
# v9.7.5.4: 날짜가 넘어간 뒤에도 최종 저장 시점(KST) 기준으로 갱신
refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))

# 8-6. v9.7.3 분야 자동보강 — 토스 카테고리 + 네이버 업종 + DART 업종코드 통합
# 목적:
# 1) 토스 첨부파일의 직관적 카테고리를 표시분야 1순위로 사용합니다.
# 2) 네이버 금융 업종 페이지에서 전통 업종을 보조 수집합니다.
# 3) DART 기업개황의 induty_code를 공식업종 후보로 저장합니다.
# 4) 종목분야_수동입력.csv로 AI/로봇/2차전지 같은 투자테마를 직접 보정할 수 있습니다.
# 5) 추천 리스트/홈/대시보드/TOP/카드/스나이퍼/백테스트 실험실에 같은 표시분야를 반영합니다.

import json as _json
from urllib.parse import urljoin

CONFIG.setdefault('ENABLE_SECTOR_AUTO_ENRICH', True)
CONFIG.setdefault('ENABLE_TOSS_CATEGORY_SOURCE', True)
CONFIG.setdefault('ENABLE_NAVER_SECTOR_FETCH', True)
CONFIG.setdefault('NAVER_SECTOR_CACHE_FILE', 'NAVER_업종캐시.csv')
CONFIG.setdefault('NAVER_SECTOR_TTL_DAYS', 30)
CONFIG.setdefault('NAVER_SECTOR_MAX_GROUPS', 120)
CONFIG.setdefault('ENABLE_DART_INDUSTRY_FETCH', True)
CONFIG.setdefault('DART_INDUSTRY_CACHE_FILE', 'DART_기업개황_업종캐시.csv')
CONFIG.setdefault('DART_INDUSTRY_TTL_DAYS', 90)
CONFIG.setdefault('SECTOR_MANUAL_FILE', '종목분야_수동입력.csv')
CONFIG.setdefault('SECTOR_UNKNOWN_LABEL', '분야확인필요')
CONFIG.setdefault('OUTPUT_PREFIX', '실전매매_통합시스템_v9_7_3_4_MergedCell안전패치_백테스트1개월')


def _v973_code(value):
    try:
        txt = re.sub(r'[^0-9]', '', str(value))
        return txt.zfill(6)[-6:] if txt else ''
    except Exception:
        return ''


def _v973_text(value, default=''):
    try:
        if value is None or pd.isna(value):
            return default
    except Exception:
        pass
    txt = str(value).strip()
    if txt.lower() in ['nan', 'none', 'null', '-']:
        return default
    return txt


def _v973_is_valid_sector(value):
    txt = _v973_text(value, '')
    return bool(txt and txt not in ['확인필요', '분야확인필요', '데이터부족', '미확인', 'KOSPI', 'KOSDAQ'])


def _v973_create_sector_manual_template(path=None):
    path = path or CONFIG.get('SECTOR_MANUAL_FILE', '종목분야_수동입력.csv')
    cols = ['종목코드', '종목명', '표시분야', '투자테마', '공식업종', '메모']
    if os.path.exists(path):
        return path
    sample = pd.DataFrame([
        {'종목코드': '005930', '종목명': '삼성전자', '표시분야': '반도체', '투자테마': 'AI반도체/메모리', '공식업종': '전자부품 제조업', '메모': '예시 행은 수정/삭제 가능'},
        {'종목코드': '000660', '종목명': 'SK하이닉스', '표시분야': '반도체', '투자테마': 'HBM/메모리', '공식업종': '전자부품 제조업', '메모': '예시 행은 수정/삭제 가능'},
        {'종목코드': '000000', '종목명': '직접입력예시', '표시분야': 'AI/소프트웨어', '투자테마': 'AI', '공식업종': '', '메모': '종목코드를 실제 코드로 바꿔 입력'},
    ], columns=cols)
    try:
        sample.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'ℹ️ 분야 수동입력 템플릿 생성: {path}')
    except Exception:
        pass
    return path


def _v973_load_manual_sector_map(path=None):
    path = path or CONFIG.get('SECTOR_MANUAL_FILE', '종목분야_수동입력.csv')
    _v973_create_sector_manual_template(path)
    if not os.path.exists(path):
        return {}
    try:
        df = pd.read_csv(path, dtype=str).fillna('')
        # 하위 버전 호환: 분야/세부분야 컬럼만 있는 경우도 처리
        mp = {}
        for _, r in df.iterrows():
            code = _v973_code(r.get('종목코드'))
            if not code or code == '000000':
                continue
            display = _v973_text(r.get('표시분야')) or _v973_text(r.get('분야'))
            theme = _v973_text(r.get('투자테마')) or _v973_text(r.get('세부분야'))
            official = _v973_text(r.get('공식업종'))
            if display or theme or official:
                mp[code] = {'표시분야': display, '투자테마': theme, '공식업종': official, '분야출처': '수동입력'}
        return mp
    except Exception as e:
        print(f'⚠️ 분야 수동입력 읽기 실패: {e}')
        return {}


def _v973_load_toss_category_map():
    if not CONFIG.get('ENABLE_TOSS_CATEGORY_SOURCE', True):
        return {}
    paths = []
    for p in [CONFIG.get('TOSS_CLEANED_CSV_FILE', 'TOSS_수동후보_정리본.csv'), CONFIG.get('TOSS_MANUAL_XLSX_FILE', '국내 저평가주식 목록 토스.xlsx')]:
        if p and os.path.exists(p):
            paths.append(p)
    mp = {}
    for path in paths:
        try:
            if path.lower().endswith('.csv'):
                df = pd.read_csv(path, dtype=str).fillna('')
            else:
                # 원본 엑셀은 구조가 불규칙할 수 있어 기존 정리본이 있으면 정리본을 우선 사용합니다.
                continue
            for _, r in df.iterrows():
                code = _v973_code(r.get('종목코드'))
                cat = _v973_text(r.get('토스카테고리')) or _v973_text(r.get('카테고리')) or _v973_text(r.get('분야'))
                if code and _v973_is_valid_sector(cat):
                    mp[code] = cat
        except Exception as e:
            print(f'⚠️ 토스 카테고리 읽기 실패({path}): {e}')
    return mp


def _v973_cache_is_fresh(path, ttl_days):
    try:
        if not os.path.exists(path):
            return False
        age = (datetime.now() - datetime.fromtimestamp(os.path.getmtime(path))).days
        return age <= int(ttl_days)
    except Exception:
        return False


def _v973_fetch_naver_sector_map():
    """네이버 금융 업종별 시세 페이지에서 종목코드→업종명 매핑을 만듭니다."""
    path = CONFIG.get('NAVER_SECTOR_CACHE_FILE', 'NAVER_업종캐시.csv')
    ttl = CONFIG.get('NAVER_SECTOR_TTL_DAYS', 30)
    if os.path.exists(path) and _v973_cache_is_fresh(path, ttl):
        try:
            df = pd.read_csv(path, dtype=str).fillna('')
            return dict(zip(df['종목코드'].map(_v973_code), df['네이버업종']))
        except Exception:
            pass
    if not CONFIG.get('ENABLE_NAVER_SECTOR_FETCH', True):
        return {}
    rows = []
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        base_url = 'https://finance.naver.com'
        main_url = 'https://finance.naver.com/sise/sise_group.naver?type=upjong'
        res = requests.get(main_url, headers=headers, timeout=10)
        res.encoding = 'euc-kr'
        soup = BeautifulSoup(res.text, 'html.parser')
        links = []
        for a in soup.select('a[href*="sise_group_detail.naver?type=upjong"]'):
            name = _v973_text(a.get_text())
            href = a.get('href') or ''
            if name and href and href not in [x[1] for x in links]:
                links.append((name, href))
        max_groups = int(CONFIG.get('NAVER_SECTOR_MAX_GROUPS', 120))
        for upjong_name, href in links[:max_groups]:
            try:
                url = urljoin(base_url, href)
                r = requests.get(url, headers=headers, timeout=10)
                r.encoding = 'euc-kr'
                sp = BeautifulSoup(r.text, 'html.parser')
                for a in sp.select('a.tltle[href*="code="]'):
                    code_match = re.search(r'code=(\d{6})', a.get('href') or '')
                    if not code_match:
                        continue
                    rows.append({'종목코드': code_match.group(1), '종목명': _v973_text(a.get_text()), '네이버업종': upjong_name, '분야출처': '네이버금융'})
                time.sleep(0.03)
            except Exception:
                continue
        df = pd.DataFrame(rows).drop_duplicates('종목코드') if rows else pd.DataFrame(columns=['종목코드','종목명','네이버업종','분야출처'])
        if not df.empty:
            df.to_csv(path, index=False, encoding='utf-8-sig')
            print(f'✅ 네이버 업종 캐시 생성: {len(df):,}개 / {path}')
            return dict(zip(df['종목코드'].map(_v973_code), df['네이버업종']))
    except Exception as e:
        print(f'⚠️ 네이버 업종 수집 실패: {e}')
    # 실패해도 빈 캐시를 만들어 다음 단계가 중단되지 않게 함
    try:
        pd.DataFrame(columns=['종목코드','종목명','네이버업종','분야출처']).to_csv(path, index=False, encoding='utf-8-sig')
    except Exception:
        pass
    return {}


def _v973_dart_industry_name(induty_code):
    code = re.sub(r'[^0-9]', '', str(induty_code or ''))
    if not code:
        return ''
    # 세부 코드 우선 매핑
    if code.startswith(('241', '242', '243')): return '철강/금속'
    if code.startswith(('251', '252', '259')): return '금속가공'
    if code.startswith(('261', '262', '263', '264', '265', '266')): return '반도체/전자부품'
    if code.startswith(('271', '272', '273')): return '의료기기/정밀기기'
    if code.startswith(('281', '282', '283', '284', '285')): return '전기장비/배터리'
    if code.startswith(('291', '292')): return '기계/장비'
    if code.startswith(('301', '302', '303', '304')): return '자동차/부품'
    if code.startswith(('311', '312', '313', '319')): return '조선/운송장비'
    if code.startswith(('582', '620', '631', '639')): return '소프트웨어/AI'
    if code.startswith(('581', '591', '592', '602')): return '미디어/콘텐츠'
    if code.startswith(('641', '642', '649')): return '금융'
    if code.startswith(('651', '652', '653')): return '보험'
    if code.startswith(('681', '682')): return '부동산'
    if code.startswith(('701', '702', '711', '712')): return '연구개발/전문서비스'
    if code.startswith(('731', '732', '733', '739')): return '광고/마케팅'
    prefix2 = code[:2]
    broad = {
        '01':'농업', '02':'임업', '03':'어업', '05':'석탄/광업', '06':'원유/가스', '07':'금속광업', '08':'비금속광업',
        '10':'식품', '11':'음료', '12':'담배', '13':'섬유', '14':'의류', '15':'가죽/신발', '16':'목재', '17':'제지', '18':'인쇄',
        '19':'정유/석유', '20':'화학', '21':'제약/바이오', '22':'고무/플라스틱', '23':'건자재/비금속', '24':'철강/금속',
        '25':'금속가공', '26':'반도체/전자부품', '27':'의료기기/정밀기기', '28':'전기장비/배터리', '29':'기계/장비',
        '30':'자동차/부품', '31':'조선/운송장비', '32':'가구', '33':'기타 제조', '35':'전력/가스', '36':'수도', '37':'환경/폐기물',
        '41':'건설', '42':'전문건설', '45':'자동차판매', '46':'도매', '47':'소매/유통', '49':'운송', '50':'해운/항공', '51':'항공운송',
        '52':'물류/창고', '55':'숙박', '56':'음식료/외식', '58':'출판/소프트웨어', '59':'콘텐츠', '60':'방송', '61':'통신',
        '62':'소프트웨어/AI', '63':'정보서비스/데이터', '64':'금융', '65':'보험', '66':'금융서비스', '68':'부동산', '70':'연구개발',
        '71':'전문서비스', '72':'건축/엔지니어링', '73':'광고/마케팅', '74':'디자인/전문서비스', '75':'사업지원', '85':'교육',
        '86':'의료', '90':'엔터/예술', '91':'스포츠/레저', '95':'수리/서비스', '96':'개인서비스'
    }
    return broad.get(prefix2, f'DART업종코드 {code}')


def _v973_read_dart_industry_cache(path=None):
    path = path or CONFIG.get('DART_INDUSTRY_CACHE_FILE', 'DART_기업개황_업종캐시.csv')
    cols = ['종목코드','corp_code','DART업종코드','DART공식업종','DART업종조회일','DART업종상태']
    if not os.path.exists(path):
        return pd.DataFrame(columns=cols)
    try:
        df = pd.read_csv(path, dtype=str).fillna('')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        df['종목코드'] = df['종목코드'].map(_v973_code)
        return df[cols].drop_duplicates('종목코드', keep='last')
    except Exception:
        return pd.DataFrame(columns=cols)


def _v973_fetch_dart_industry_map(codes):
    """OpenDART company.json으로 업종코드를 조회하고 캐시에 저장합니다. API 키가 없으면 빈 매핑을 반환합니다."""
    if not CONFIG.get('ENABLE_DART_INDUSTRY_FETCH', True):
        return {}
    api_key = CONFIG.get('DART_API_KEY', '')
    if not api_key:
        return {}
    cache_path = CONFIG.get('DART_INDUSTRY_CACHE_FILE', 'DART_기업개황_업종캐시.csv')
    ttl = int(CONFIG.get('DART_INDUSTRY_TTL_DAYS', 90))
    cache = _v973_read_dart_industry_cache(cache_path)
    cache_map = {}
    fresh_rows = []
    now_date = REPORT_DATE
    for _, r in cache.iterrows():
        code = _v973_code(r.get('종목코드'))
        try:
            age = (pd.to_datetime(REPORT_DATE) - pd.to_datetime(r.get('DART업종조회일'))).days
        except Exception:
            age = 9999
        if code and age <= ttl:
            cache_map[code] = {'DART업종코드': r.get('DART업종코드',''), 'DART공식업종': r.get('DART공식업종',''), 'DART업종상태': r.get('DART업종상태','DART캐시')}
            fresh_rows.append(r.to_dict())
    need = [c for c in sorted(set(_v973_code(x) for x in codes if _v973_code(x))) if c not in cache_map]
    if not need:
        return cache_map
    try:
        corp_map_df = fetch_dart_corp_code_map(api_key)
        corp_lookup = dict(zip(corp_map_df['stock_code'].map(_v973_code), corp_map_df['corp_code'])) if corp_map_df is not None and not corp_map_df.empty else {}
    except Exception:
        corp_lookup = {}
    new_rows = []
    for code in need[:300]:  # 과도한 API 호출 방지
        corp_code = corp_lookup.get(code, '')
        if not corp_code:
            new_rows.append({'종목코드': code, 'corp_code': '', 'DART업종코드': '', 'DART공식업종': '', 'DART업종조회일': now_date, 'DART업종상태': 'corp_code없음'})
            continue
        try:
            url = 'https://opendart.fss.or.kr/api/company.json'
            r = requests.get(url, params={'crtfc_key': api_key, 'corp_code': corp_code}, timeout=10)
            data = r.json()
            induty_code = _v973_text(data.get('induty_code'))
            official = _v973_dart_industry_name(induty_code)
            row = {'종목코드': code, 'corp_code': corp_code, 'DART업종코드': induty_code, 'DART공식업종': official, 'DART업종조회일': now_date, 'DART업종상태': 'DART조회'}
            new_rows.append(row)
            cache_map[code] = {'DART업종코드': induty_code, 'DART공식업종': official, 'DART업종상태': 'DART조회'}
            time.sleep(0.05)
        except Exception as e:
            new_rows.append({'종목코드': code, 'corp_code': corp_code, 'DART업종코드': '', 'DART공식업종': '', 'DART업종조회일': now_date, 'DART업종상태': f'조회실패:{str(e)[:30]}'})
    try:
        old = _v973_read_dart_industry_cache(cache_path)
        save_df = pd.concat([old, pd.DataFrame(new_rows)], ignore_index=True)
        save_df['종목코드'] = save_df['종목코드'].map(_v973_code)
        save_df = save_df.drop_duplicates('종목코드', keep='last')
        save_df.to_csv(cache_path, index=False, encoding='utf-8-sig')
        print(f'✅ DART 업종 캐시 갱신: {len(new_rows):,}개 / {cache_path}')
    except Exception as e:
        print(f'⚠️ DART 업종 캐시 저장 실패: {e}')
    return cache_map


def _v973_keyword_sector(row):
    text = ' '.join(str(row.get(c, '')) for c in ['종목명','종목코드','상세 전략 가이드','후보소스메모','스크리너명','업종','분야'])
    pairs = [
        (['철강','제철','스틸','금속'], '철강/금속'),
        (['보험','손해보험','생명보험'], '보험'),
        (['광고','마케팅','이노션','제일기획'], '광고/마케팅'),
        (['반도체','하이닉스','파운드리','HBM','메모리'], '반도체'),
        (['AI','인공지능','데이터','소프트','클라우드','솔트룩스','마음AI'], 'AI/소프트웨어'),
        (['로봇','로보'], '로봇'),
        (['2차전지','배터리','전지','양극재','음극재','전해액'], '2차전지/배터리'),
        (['바이오','제약','신약','의약'], '제약/바이오'),
        (['건설','건축','시멘트'], '건설/건자재'),
        (['은행','금융','증권','캐피탈'], '금융'),
        (['화학','케미칼','정유','석유'], '화학/정유'),
        (['조선','해운','선박'], '조선/해운'),
        (['자동차','모빌리티','부품'], '자동차/부품'),
        (['방산','항공우주','우주','드론'], '방산/우주항공'),
        (['환경','수처리','폐기물','크린텍'], '환경/수처리'),
        (['게임','콘텐츠','엔터','미디어'], '콘텐츠/엔터')
    ]
    for keys, label in pairs:
        if any(k.lower() in text.lower() for k in keys):
            return label
    return ''


def _v973_build_sector_sources(df):
    codes = [] if df is None or df.empty else [_v973_code(x) for x in df.get('종목코드', [])]
    return {
        'manual': _v973_load_manual_sector_map(),
        'toss': _v973_load_toss_category_map(),
        'naver': _v973_fetch_naver_sector_map(),
        'dart': _v973_fetch_dart_industry_map(codes)
    }


def _v973_apply_sector_to_df(df, sources=None):
    if df is None or df.empty:
        return df
    out = df.copy()
    sources = sources or _v973_build_sector_sources(out)
    for col in ['공식업종','투자테마','표시분야','분야출처','네이버업종','DART공식업종','DART업종코드','분야가이드']:
        if col not in out.columns:
            out[col] = ''
    for idx, row in out.iterrows():
        code = _v973_code(row.get('종목코드'))
        manual = sources.get('manual', {}).get(code, {})
        toss_cat = _v973_text(row.get('토스카테고리')) or _v973_text(row.get('카테고리')) or sources.get('toss', {}).get(code, '')
        naver_sector = sources.get('naver', {}).get(code, '')
        dart_info = sources.get('dart', {}).get(code, {})
        dart_sector = dart_info.get('DART공식업종', '') if isinstance(dart_info, dict) else ''
        dart_code = dart_info.get('DART업종코드', '') if isinstance(dart_info, dict) else ''
        existing = ''
        for c in ['업종','업태','Sector','Industry','시장구분']:
            if _v973_is_valid_sector(row.get(c, '')):
                existing = _v973_text(row.get(c))
                break
        keyword = _v973_keyword_sector(row)

        display = ''
        source = ''
        if _v973_is_valid_sector(manual.get('표시분야')):
            display, source = manual.get('표시분야'), '수동입력'
        elif _v973_is_valid_sector(toss_cat):
            display, source = toss_cat, '토스카테고리'
        elif _v973_is_valid_sector(naver_sector):
            display, source = naver_sector, '네이버업종'
        elif _v973_is_valid_sector(manual.get('투자테마')):
            display, source = manual.get('투자테마'), '수동테마'
        elif _v973_is_valid_sector(dart_sector):
            display, source = dart_sector, 'DART업종코드'
        elif _v973_is_valid_sector(existing):
            display, source = existing, '기존데이터'
        elif _v973_is_valid_sector(keyword):
            display, source = keyword, '키워드추정'
        else:
            display, source = CONFIG.get('SECTOR_UNKNOWN_LABEL', '분야확인필요'), '확인필요'

        official = manual.get('공식업종') or naver_sector or dart_sector or existing
        theme = manual.get('투자테마') or (toss_cat if source == '토스카테고리' else '') or keyword
        out.at[idx, '토스카테고리'] = toss_cat
        out.at[idx, '네이버업종'] = naver_sector
        out.at[idx, 'DART공식업종'] = dart_sector
        out.at[idx, 'DART업종코드'] = dart_code
        out.at[idx, '공식업종'] = official
        out.at[idx, '투자테마'] = theme
        out.at[idx, '표시분야'] = display
        out.at[idx, '분야'] = display
        out.at[idx, '분야출처'] = source
        out.at[idx, '분야가이드'] = f"표시분야: {display} / 출처: {source} / 공식업종: {official or '-'} / 투자테마: {theme or '-'}"
        # 카드 가이드에도 분야 출처를 짧게 섞어줌
        guide = _v973_text(row.get('상세 전략 가이드'))
        sector_note = f"분야 {display}({source})"
        if sector_note not in guide:
            out.at[idx, '상세 전략 가이드'] = (sector_note + ' | ' + guide).strip(' |')
    return out


# v9.7.2의 분야 보강 함수가 이 통합 소스를 쓰도록 덮어쓰기
_v973_sector_sources_cache = None

def _v972_enhance_sector(final_df):
    global _v973_sector_sources_cache
    if final_df is None or final_df.empty:
        return final_df
    if _v973_sector_sources_cache is None:
        _v973_sector_sources_cache = _v973_build_sector_sources(final_df)
    return _v973_apply_sector_to_df(final_df, _v973_sector_sources_cache)


# TOP 요약에 분야출처/공식업종/투자테마를 같이 노출

def _v972_compact_cols(top):
    cols = [
        '순위', 'AI 시스템 판정', '종목명', '종목코드', '분야', '분야출처', '투자테마', '공식업종',
        '후보출처', '현재가', '등락률', '기본기술점수', '종목점수',
        '진입판정', '기준진입가', '손절기준가', '추세필터판정', '재무위험판정',
        '백테스트수익률', '백테스트신뢰도', '권장익절계획', '권장손절계획'
    ]
    return [c for c in cols if c in top.columns]


# 종목카드에 분야/출처/공식업종/테마를 좀 더 명확히 표시

def _v972_make_cards(wb, top):
    _v972_delete_sheets(wb, ['종목카드_TOP10', '종목카드_TOP15'])
    ws = wb.create_sheet('종목카드_TOP15', 3)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 85
    border=_v972_border(); left=_v972_align('left'); center=_v972_align('center')
    for col, width in {'A':3,'B':15,'C':20,'D':18,'E':18,'F':20,'G':18,'H':18,'I':3}.items():
        ws.column_dimensions[col].width=width
    ws.merge_cells('B2:H3')
    ws['B2']='종목카드 TOP15 — 표시분야/출처/진입/전략 가이드 통합'
    ws['B2'].fill=_v972_fill('111827'); ws['B2'].font=_v972_font('FFFFFF', True, 16); ws['B2'].alignment=center; ws['B2'].border=border
    row_num=5
    for _, r in top.iterrows():
        idx = int(r.get('순위', 0)) if str(r.get('순위','')).isdigit() else ''
        sector = r.get('분야','')
        source = r.get('분야출처','')
        title = f"#{idx}  {r.get('종목명','')}  ·  {sector}  ·  {r.get('AI 시스템 판정','')}  ·  점수 {r.get('종목점수','-')}"
        ws.merge_cells(start_row=row_num, start_column=2, end_row=row_num+1, end_column=8)
        h=ws.cell(row_num,2,title); h.fill=_v972_fill('E0F2FE'); h.font=_v972_font('075985', True, 12); h.alignment=left; h.border=border
        info = [
            ['분야', sector, '분야출처', source, '후보출처', r.get('후보출처',''), '시장모드', globals().get('market_mode','')],
            ['공식업종', r.get('공식업종',''), '투자테마', r.get('투자테마',''), '기술점수', r.get('기본기술점수',''), '최종점수', r.get('종목점수','')],
            ['진입판정', r.get('진입판정',''), '기준진입가', r.get('기준진입가',''), '추세', r.get('추세필터판정',''), '재무', r.get('재무위험판정','')],
            ['익절', r.get('권장익절계획',''), '손절', r.get('권장손절계획',''), '백테스트', r.get('백테스트신뢰도',''), '수익률', r.get('백테스트수익률','')],
            ['가이드', _v972_safe(r.get('상세 전략 가이드',''))[:260], '', '', '', '', '', '']
        ]
        for rr, vals in enumerate(info, row_num+2):
            for cc, val in enumerate(vals, 2):
                cell=ws.cell(rr, cc, val)
                cell.border=border
                cell.alignment=left if cc in [3,5,7] or rr==row_num+6 else center
                cell.font=_v972_font('111827', False, 9)
                if cc in [2,4,6,8] and rr != row_num+6:
                    cell.fill=_v972_fill('F8FAFC'); cell.font=_v972_font('475569', True, 9)
        try:
            ws.merge_cells(start_row=row_num+6, start_column=3, end_row=row_num+6, end_column=8)
        except Exception:
            pass
        for rr in range(row_num, row_num+7):
            ws.row_dimensions[rr].height = 24 if rr != row_num+6 else 54
        row_num += 9


# 스나이퍼/실험실 가이드에도 분야출처·공식업종·투자테마 표시

def _v973_is_merged_cell(ws, row, col):
    """해당 좌표가 병합 범위 내부인지 확인합니다."""
    try:
        for rng in ws.merged_cells.ranges:
            if rng.min_row <= row <= rng.max_row and rng.min_col <= col <= rng.max_col:
                return True, rng
    except Exception:
        pass
    return False, None


def _v973_next_writable_anchor(ws, start_cell):
    """병합 셀 내부를 피해서 쓸 수 있는 시작 셀을 찾습니다."""
    try:
        row = ws[start_cell].row
        col = ws[start_cell].column
    except Exception:
        row, col = 2, 11
    for offset in range(0, 20):
        c = col + offset
        merged, rng = _v973_is_merged_cell(ws, row, c)
        if not merged:
            return row, c
        # 병합 범위 안이면 병합 끝 다음 열로 점프
        if rng is not None:
            col = max(col, rng.max_col + 1)
    return row, col + 20


# 스나이퍼/실험실 가이드에도 분야출처·공식업종·투자테마 표시
# v9.7.3.2: 기존 H2 위치가 제목 병합범위(B2:I2/B2:H3)와 겹쳐 MergedCell read-only 오류가 나던 문제 수정.
#           기본 위치를 K2로 옮기고, 혹시 K2도 병합 영역이면 자동으로 오른쪽 빈 영역을 찾습니다.
def _v972_patch_choice_sheets(wb):
    """스나이퍼/실험실에서 선택 종목의 분야·출처·기술점수·추세·재무를 보이게 합니다.
    v9.7.5.2: 병합 셀과 겹치지 않는 오른쪽 빈 블록을 자동 탐색해서 MergedCell read-only 오류를 방지합니다.
    """
    def formula_for(header, key_cell='$C$4'):
        return f'=IFERROR(INDEX(\'추천 리스트\'!$A:$ZZ,MATCH({key_cell},\'추천 리스트\'!$B:$B,0),MATCH("{header}",\'추천 리스트\'!$1:$1,0)),"")'

    def is_merged_cell(ws, row, col):
        try:
            for rng in ws.merged_cells.ranges:
                if rng.min_row <= row <= rng.max_row and rng.min_col <= col <= rng.max_col:
                    return True
        except Exception:
            pass
        return False

    def find_free_block(ws, start_cell, block_rows=12, block_cols=2):
        """start_cell부터 오른쪽으로 이동하며 병합 셀과 겹치지 않는 2열짜리 가이드 블록 위치를 찾습니다."""
        try:
            base_row = ws[start_cell].row
            base_col = ws[start_cell].column
        except Exception:
            base_row, base_col = 2, 11
        for offset in range(0, 80):
            c0 = base_col + offset
            conflict = False
            for rr in range(base_row, base_row + block_rows):
                for cc in range(c0, c0 + block_cols):
                    if is_merged_cell(ws, rr, cc):
                        conflict = True
                        break
                if conflict:
                    break
            if not conflict:
                return base_row, c0
        return base_row, base_col + 80

    blocks = {
        '스나이퍼_계산기': ('K2', '$C$4'),
        '백테스트_실험실': ('K2', '$C$5')
    }
    for sheet, (start_cell, key_cell) in blocks.items():
        if sheet not in wb.sheetnames:
            continue
        ws = wb[sheet]
        guide_rows = [
            ('분야', '분야'), ('후보출처','후보출처'), ('시장모드',''), ('기본기술점수','기본기술점수'),
            ('최종점수','종목점수'), ('진입판정','진입판정'), ('추세','추세필터판정'), ('재무','재무위험판정'), ('백테스트','백테스트신뢰도')
        ]
        start_row, start_col = find_free_block(ws, start_cell, block_rows=len(guide_rows)+1, block_cols=2)
        title_cell = ws.cell(start_row, start_col)
        title_cell.value = '선택 종목 가이드'
        title_cell.fill = _v972_fill('0F172A')
        title_cell.font = _v972_font('FFFFFF', True, 11)
        title_cell.alignment = _v972_align('center')
        title_cell.border = _v972_border()
        try:
            ws.cell(start_row, start_col+1).fill = _v972_fill('0F172A')
            ws.cell(start_row, start_col+1).border = _v972_border()
        except Exception:
            pass
        for i,(label,header) in enumerate(guide_rows, start_row+1):
            label_cell = ws.cell(i, start_col)
            label_cell.value = label
            label_cell.fill = _v972_fill('F8FAFC')
            label_cell.font = _v972_font('475569', True, 9)
            label_cell.alignment = _v972_align('center')
            label_cell.border = _v972_border()
            val_cell = ws.cell(i, start_col+1)
            val_cell.value = globals().get('market_mode','') if label == '시장모드' else formula_for(header, key_cell)
            val_cell.font = _v972_font('111827', False, 9)
            val_cell.alignment = _v972_align('left')
            val_cell.border = _v972_border()
        ws.column_dimensions[get_column_letter(start_col)].width = 16
        ws.column_dimensions[get_column_letter(start_col+1)].width = 30

def _v973_rewrite_recommend_sheet(wb, df):
    """스나이퍼/실험실 수식이 최신 분야 컬럼을 볼 수 있도록 추천 리스트 시트를 갱신합니다."""
    if df is None or df.empty:
        return
    name = '추천 리스트'
    idx = wb.sheetnames.index(name) if name in wb.sheetnames else len(wb.sheetnames)
    if name in wb.sheetnames:
        del wb[name]
    ws = wb.create_sheet(name, idx)
    ws.sheet_properties.tabColor = '4472C4'
    ws.sheet_view.showGridLines = False
    border = _v972_border()
    center = _v972_align('center')
    left = _v972_align('left')
    cols = list(df.columns)
    for cidx, col in enumerate(cols, 1):
        cell = ws.cell(1, cidx, col)
        cell.fill = _v972_fill('1F4E78')
        cell.font = _v972_font('FFFFFF', True, 10)
        cell.alignment = center
        cell.border = border
    for ridx, (_, row) in enumerate(df.iterrows(), 2):
        for cidx, col in enumerate(cols, 1):
            val = row.get(col, '')
            if pd.isna(val):
                val = ''
            cell = ws.cell(ridx, cidx, val)
            cell.border = border
            cell.alignment = left if col in ['상세 전략 가이드','권장익절계획','권장손절계획','분야가이드'] else center
            if col in ['현재가','추천투입금액','추천수량','기준진입가','손절기준가']:
                cell.number_format = '#,##0'
            elif col in ['백테스트수익률','백테스트MDD','백테스트승률','등락률']:
                cell.number_format = '0.0'
    for cidx, col in enumerate(cols, 1):
        width = 12
        if col in ['종목명','AI 시스템 판정','상세 전략 가이드','권장익절계획','권장손절계획','분야가이드']:
            width = 24 if col != '상세 전략 가이드' else 42
        elif col in ['공식업종','투자테마','표시분야','분야출처','네이버업종','DART공식업종']:
            width = 18
        ws.column_dimensions[get_column_letter(cidx)].width = min(width, 45)
    try:
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
    except Exception:
        pass


def _v973_make_sector_source_sheet(wb, sources):
    name = '분야_소스관리'
    if name in wb.sheetnames:
        del wb[name]
    rows = []
    for source_name, mp in (sources or {}).items():
        if source_name == 'manual':
            for code, val in mp.items():
                rows.append({'종목코드': code, '소스': '수동입력', '표시분야': val.get('표시분야',''), '투자테마': val.get('투자테마',''), '공식업종': val.get('공식업종','')})
        elif source_name == 'toss':
            for code, val in mp.items():
                rows.append({'종목코드': code, '소스': '토스카테고리', '표시분야': val, '투자테마': '', '공식업종': ''})
        elif source_name == 'naver':
            for code, val in mp.items():
                rows.append({'종목코드': code, '소스': '네이버업종', '표시분야': val, '투자테마': '', '공식업종': val})
        elif source_name == 'dart':
            for code, val in mp.items():
                if isinstance(val, dict):
                    rows.append({'종목코드': code, '소스': 'DART업종코드', '표시분야': val.get('DART공식업종',''), '투자테마': '', '공식업종': val.get('DART공식업종',''), 'DART업종코드': val.get('DART업종코드','')})
    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=['종목코드','소스','표시분야','투자테마','공식업종','DART업종코드'])
    _v963_write_df_sheet(wb, name, df.head(500), index=len(wb.sheetnames), title='분야 소스 관리 — 수동/토스/네이버/DART 통합', tab_color='7C3AED')
    try:
        wb[name].sheet_state = 'hidden'
    except Exception:
        pass


def apply_v973_sector_enrich_patch(output_file=None):
    output_file = _v972_output_path() if '_v972_output_path' in globals() else (output_file or globals().get('output_file',''))
    if not output_file or not os.path.exists(output_file):
        print('⚠️ v9.7.3 분야 보강 생략: 엑셀 리포트 파일을 찾지 못했습니다. 앞쪽 셀을 먼저 실행하세요.')
        return output_file, pd.DataFrame()
    global final_df, top_df, unified_top15_df, _v973_sector_sources_cache
    try:
        if 'final_df' not in globals() or final_df is None or final_df.empty:
            final_df = pd.read_excel(output_file, sheet_name='추천 리스트', dtype={'종목코드': str})
        _v973_sector_sources_cache = _v973_build_sector_sources(final_df)
        final_df = _v973_apply_sector_to_df(final_df, _v973_sector_sources_cache)
        top_df = _v972_top_df(final_df)
        unified_top15_df = top_df.copy()
    except Exception as e:
        print(f'⚠️ final_df 분야 보강 실패: {e}')
        top_df = pd.DataFrame()
    wb = load_workbook(output_file)
    try:
        _v973_rewrite_recommend_sheet(wb, final_df)
    except Exception as e:
        print(f'⚠️ 추천 리스트 갱신 실패: {e}')
    # 기존 v9.7.2 전면 시트 생성 함수 재사용. _v972_enhance_sector/_v972_make_cards/_v972_patch_choice_sheets는 위에서 덮어쓴 버전을 사용.
    try:
        _v972_make_home(wb, top_df)
        _v972_make_mobile(wb, top_df)
        _v972_make_top_summary(wb, top_df)
        _v972_make_cards(wb, top_df)
        _v972_make_holdings_input_template(wb)
        _v972_patch_choice_sheets(wb)
        _v973_make_sector_source_sheet(wb, _v973_sector_sources_cache)
        _v972_reorder_and_hide(wb)
    except Exception as e:
        print(f'⚠️ 전면 시트 분야 보강 실패: {e}')
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = 'auto'
    except Exception:
        pass
    final_name = f"{RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT', '%Y%m%d'))}.xlsx" if CONFIG.get('OUTPUT_FILENAME_DATE_ONLY', True) else output_file
    wb.save(final_name)
    return final_name, top_df

output_file, unified_top15_df = apply_v973_sector_enrich_patch(output_file if 'output_file' in globals() else None)
print(f'✅ v9.7.3 분야 자동보강 패치 완료: {output_file}')
print('📌 표시분야 우선순위: 수동입력 → 토스카테고리 → 네이버업종 → DART업종코드 → 기존데이터 → 키워드추정')
print(f'📌 전면 시트 후보 기준: 통합후보 {len(unified_top15_df) if isinstance(unified_top15_df, pd.DataFrame) else 0}개')

if IN_COLAB and output_file and os.path.exists(output_file):
    pass  # no-op: 중간 다운로드 비활성화로 인한 빈 if 블록 방지
    # v9.7.3.10: 중간 다운로드 비활성화 - 최종 셀에서만 다운로드합니다.
    # # files.download(output_file)  # v9.7.3.11 final cell에서만 다운로드


✅ DART 업종 캐시 갱신: 50개 / DART_기업개황_업종캐시.csv


✅ v9.7.3 분야 자동보강 패치 완료: 20260608.xlsx
📌 표시분야 우선순위: 수동입력 → 토스카테고리 → 네이버업종 → DART업종코드 → 기존데이터 → 키워드추정
📌 전면 시트 후보 기준: 통합후보 15개


## v9.7.4 최종 안정화 패치 — 과열필터 + UI 검증 + 보유종목 시트 고정

- 급등/상한가성 과열 후보를 `과열 관찰` 또는 `눌림 대기`로 분리합니다.
- 홈/모바일/TOP/카드/진입/보유/스나이퍼/백테스트 시트를 통합후보 기준으로 재생성합니다.
- `보유종목_입력`, `보유종목_진단`, `검증결과` 시트를 항상 생성합니다.
- 스나이퍼/백테스트 실험실은 복구 오류가 잘 나는 복잡한 수식 대신 안정적인 같은 시트 드롭다운 + 단순 수식 구조를 사용합니다.


In [22]:
# v9.7.5.4: 날짜가 넘어간 뒤에도 최종 저장 시점(KST) 기준으로 갱신
refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))
# 8-99. v9.7.5 안정화·과열필터·UI확정판
# - 기존 생성 결과 엑셀을 최종 단계에서 다시 열어, UI/보유종목/과열필터/수식 안정성을 확정합니다.
# - 스나이퍼/백테스트 실험실은 같은 시트 내부 숨김 데이터 + 단순 VLOOKUP만 사용합니다.
# - 검증결과 시트에 PASS/FAIL을 남깁니다.

import os, re, glob, zipfile, math
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.comments import Comment

try:
    from google.colab import files
except Exception:
    files = None

# ─────────────────────────────────────────────
# 공통 유틸
# ─────────────────────────────────────────────

def _v975_latest_xlsx():
    if 'output_file' in globals() and isinstance(output_file, str) and os.path.exists(output_file):
        return output_file
    candidates = []
    for pat in ['*.xlsx', '/content/*.xlsx', '/mnt/data/*.xlsx']:
        candidates.extend(glob.glob(pat))
    candidates = [p for p in candidates if not os.path.basename(p).startswith('~$') and '수정본' not in os.path.basename(p)]
    if not candidates:
        raise FileNotFoundError('패치할 엑셀 파일을 찾지 못했습니다. 먼저 리포트 생성 셀을 실행하세요.')
    return max(candidates, key=os.path.getmtime)

def _clean(v):
    if v is None:
        return ''
    s = str(v).strip()
    if s.lower() in ['nan', 'none', 'null']:
        return ''
    return s

def _num(v, default=0.0):
    if v is None or v == '':
        return default
    try:
        if isinstance(v, (int, float)):
            if math.isnan(v):
                return default
            return float(v)
        s = str(v).replace(',', '').replace('%', '').replace('원', '').replace('억', '').replace('주', '').strip()
        if s in ['', '-', 'nan', 'None']:
            return default
        return float(s)
    except Exception:
        return default

def _safe_int(v, default=0):
    try:
        return int(round(_num(v, default)))
    except Exception:
        return default

def _pct_text(v):
    n = _num(v, None)
    if n is None:
        return ''
    return f'{n:.1f}%'

def _won(v):
    try:
        return f'{int(round(float(v))):,}원'
    except Exception:
        return ''

def _invalid_sector(v):
    s = _clean(v)
    if not s:
        return True
    bad = ['KOSPI','KOSDAQ','KONEX','NAVER','TOSS','TOSS_XLSX','HYBRID','확인필요','분야확인필요','nan','None','미확인']
    return s.upper() in [b.upper() for b in bad]

def _guess_sector(name, code=''):
    n = _clean(name)
    rules = [
        ('타이어', '자동차부품/타이어'), ('모터스', '자동차부품'), ('자동차', '자동차/부품'), ('플라스틱', '화학/자동차부품'),
        ('해운', '해운/운송'), ('항공', '항공/운송'), ('위성', '우주항공/방산'), ('우주', '우주항공/방산'), ('방산', '방산'),
        ('보험', '보험'), ('은행', '은행'), ('증권', '증권'), ('홀딩스', '지주/금융'),
        ('전자', '전자부품'), ('전기', '전기전자'), ('소재', '소재/부품'), ('반도체', '반도체'), ('코칩', '전자부품/소재'), ('아모텍', '전자부품/소재'), ('아비코', '전자부품'),
        ('바이오', '바이오/제약'), ('제약', '바이오/제약'), ('의료', '의료기기'), ('원텍', '미용의료기기'),
        ('포장', '포장재/제지'), ('철강', '철강/금속'), ('금속', '철강/금속'), ('태웅', '금속/풍력부품'),
        ('광고', '광고/마케팅'), ('건설', '건설'), ('화학', '화학'), ('AI', 'AI/소프트웨어'), ('로봇', '로봇/자동화'),
        ('ETF', 'ETF/테마형'), ('ACE', 'ETF/테마형'), ('TIGER', 'ETF/테마형'), ('HANARO', 'ETF/테마형'),
    ]
    for key, val in rules:
        if key in n:
            return val
    return '분야확인필요'

def _headers(ws):
    return [_clean(c.value) for c in ws[1]]

def _col_idx(headers, names):
    for n in names:
        if n in headers:
            return headers.index(n) + 1
    return None

def _ensure_col(ws, name):
    headers = _headers(ws)
    if name in headers:
        return headers.index(name) + 1
    col = ws.max_column + 1
    ws.cell(1, col, name)
    return col

def _read_sheet_rows(ws):
    headers = _headers(ws)
    rows = []
    for r in range(2, ws.max_row + 1):
        item = {}
        blank = True
        for c, h in enumerate(headers, 1):
            if not h:
                continue
            v = ws.cell(r, c).value
            item[h] = v
            if v not in (None, ''):
                blank = False
        if not blank:
            rows.append(item)
    return headers, rows

def _write_headers(ws, headers, row=1, fill='1F4E78'):
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row, i, h)
        cell.fill = PatternFill('solid', fgColor=fill)
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(bottom=Side(style='thin', color='CBD5E1'))


def _style_range_basic(ws, min_row=1, max_row=None, max_col=None):
    max_row = max_row or ws.max_row
    max_col = max_col or ws.max_column
    thin = Side(style='thin', color='E5E7EB')
    for row in ws.iter_rows(min_row=min_row, max_row=max_row, min_col=1, max_col=max_col):
        for cell in row:
            cell.alignment = Alignment(vertical='center', wrap_text=True)
            cell.border = Border(bottom=thin)


def _reset_sheet(wb, name, idx=None):
    if name in wb.sheetnames:
        del wb[name]
    ws = wb.create_sheet(name, idx if idx is not None else len(wb.sheetnames))
    return ws

# ─────────────────────────────────────────────
# 데이터 정리
# ─────────────────────────────────────────────

def _get_recommendations(wb):
    if '추천 리스트' not in wb.sheetnames:
        raise KeyError('추천 리스트 시트가 없습니다.')
    ws = wb['추천 리스트']
    headers, rows = _read_sheet_rows(ws)
    rec = []
    for i, row in enumerate(rows, 2):
        name = _clean(row.get('종목명') or row.get('Name') or row.get('회사명'))
        if not name:
            continue
        code = _clean(row.get('종목코드'))
        price = _num(row.get('현재가'))
        score = _num(row.get('종목점수', row.get('최종점수', row.get('기본기술점수', 0))))
        sector = ''
        for key in ['표시분야','카테고리','투자테마','네이버업종','DART공식업종','공식업종','분야']:
            if not _invalid_sector(row.get(key)):
                sector = _clean(row.get(key)); break
        if _invalid_sector(sector):
            sector = _guess_sector(name, code)
        chg = _num(row.get('등락률', row.get('당일등락률', row.get('전일대비등락률', 0))))
        rsi = _num(row.get('RSI'))
        volratio = _num(row.get('거래량배율', row.get('거래량배수', 0)))
        atr = _num(row.get('ATR%', row.get('ATR비율', 0)))
        trades = _num(row.get('거래횟수', row.get('백테스트거래횟수', 0)))
        penalty = 0
        reasons = []
        if chg >= 25:
            penalty += 35; reasons.append(f'상한가급 급등({chg:.1f}%)')
        elif chg >= 15:
            penalty += 22; reasons.append(f'당일 급등({chg:.1f}%)')
        if rsi >= 68 and volratio >= 4:
            penalty += 18; reasons.append(f'RSI {rsi:.1f}+거래량 {volratio:.1f}배')
        elif rsi >= 72:
            penalty += 12; reasons.append(f'RSI 과열({rsi:.1f})')
        if atr >= 8:
            penalty += 8; reasons.append(f'고변동성 ATR {atr:.1f}%')
        if trades and trades < 5:
            penalty += 8; reasons.append(f'백테스트 거래횟수 부족({int(trades)}회)')
        final_score = max(score - penalty, 0)
        if penalty >= 35:
            over = '과열 관찰'; entry = '눌림 대기'; bucket = '과열 관찰'
        elif penalty >= 20:
            over = '과열 주의'; entry = '분할/눌림 확인'; bucket = '선별 후보'
        else:
            over = '정상'; entry = _clean(row.get('실전진입판정')) or _clean(row.get('진입판정')) or '조건 확인 후 진입'
            bucket = _clean(row.get('실전후보구분')) or _clean(row.get('최종판정')) or _clean(row.get('AI 시스템 판정')) or '선별 후보'
        tp_plan = _clean(row.get('권장익절계획')) or _clean(row.get('권장익절퍼센트')) or '+3% / +5% / +10%\n(70%) / (20%) / (10%)'
        sl_plan = _clean(row.get('권장손절계획')) or _clean(row.get('권장손절퍼센트')) or '-2% / -4% / -6%\n(80%) / (10%) / (10%)'
        guide = _clean(row.get('상세전략가이드'))
        if not guide:
            guide = f'후보출처 {row.get("후보출처","")}, 분야 {sector}, 점수 {score:.0f}점, 과열판정 {over}. 추세/재무/백테스트를 함께 확인 후 진입.'
        if reasons:
            guide = f'[과열 체크] {" / ".join(reasons)}. 추격매수보다 눌림·재돌파 확인 우선.\n' + guide
        rec.append({
            '종목명': name, '종목코드': code, '섹터/분야': sector,
            '후보출처': _clean(row.get('후보출처')) or 'NAVER', '현재가': price,
            '기본점수': score, '실전점수': final_score, '과열감점': penalty,
            '과열판정': over, '과열사유': ' / '.join(reasons) if reasons else '',
            '실전후보구분': bucket, '실전진입판정': entry,
            '추세필터판정': _clean(row.get('추세필터판정')),
            '재무위험판정': _clean(row.get('재무위험판정')) or _clean(row.get('DART재무위험판정')) or _clean(row.get('재무등급')),
            '백테스트신뢰도': _clean(row.get('백테스트신뢰도')),
            '권장익절계획': tp_plan, '권장손절계획': sl_plan,
            '추천투입금액': _num(row.get('추천투입금액'), 300000), '추천수량': _safe_int(row.get('추천수량'), 0),
            '상세전략가이드': guide,
        })
    if pd is not None:
        df = pd.DataFrame(rec)
        if not df.empty:
            df = df.sort_values(['실전점수','기본점수'], ascending=[False, False]).reset_index(drop=True)
        return df
    return rec

def _update_recommendation_sheet(wb, rec_df):
    ws = wb['추천 리스트']
    headers = _headers(ws)
    name_col = _col_idx(headers, ['종목명']) or 2
    # 필요한 컬럼 보강
    for col_name in ['표시분야','실전점수','과열판정','과열사유','실전후보구분','실전진입판정','상세전략가이드']:
        _ensure_col(ws, col_name)
    headers = _headers(ws)
    # 이름 기준 매핑
    rec_map = {str(r['종목명']): r for _, r in rec_df.iterrows()} if pd is not None else {r['종목명']: r for r in rec_df}
    for r in range(2, ws.max_row + 1):
        name = _clean(ws.cell(r, name_col).value)
        if not name or name not in rec_map:
            continue
        item = rec_map[name]
        mapping = {
            '표시분야': item['섹터/분야'], '실전점수': item['실전점수'], '과열판정': item['과열판정'],
            '과열사유': item['과열사유'], '실전후보구분': item['실전후보구분'],
            '실전진입판정': item['실전진입판정'], '상세전략가이드': item['상세전략가이드']
        }
        for k, v in mapping.items():
            c = _col_idx(_headers(ws), [k])
            ws.cell(r, c, v)
    return ws

def _top_df(rec_df, n=15):
    if pd is not None:
        return rec_df.head(n).copy()
    return rec_df[:n]

# ─────────────────────────────────────────────
# 보유종목 정리
# ─────────────────────────────────────────────

def _read_holdings_from_wb(wb):
    holdings=[]
    if '보유종목_입력' not in wb.sheetnames:
        return holdings
    ws=wb['보유종목_입력']
    # 어느 행에 헤더가 있든 인식
    for r in range(1, ws.max_row+1):
        vals=[_clean(ws.cell(r,c).value) for c in range(1, min(ws.max_column,10)+1)]
        if '종목명' in vals and '종목코드' in vals and '보유수량' in vals and '평균단가' in vals:
            hrow=r
            header_map={_clean(ws.cell(hrow,c).value):c for c in range(1,ws.max_column+1) if _clean(ws.cell(hrow,c).value)}
            for rr in range(hrow+1, ws.max_row+1):
                name=_clean(ws.cell(rr, header_map.get('종목명',1)).value)
                code=_clean(ws.cell(rr, header_map.get('종목코드',2)).value)
                qty=_num(ws.cell(rr, header_map.get('보유수량',3)).value)
                avg=_num(ws.cell(rr, header_map.get('평균단가',4)).value)
                if not name and not code:
                    continue
                if name.startswith('예시') or '입력법' in name or '자동승계' in code:
                    continue
                if qty <= 0 and avg <= 0:
                    continue
                holdings.append({
                    '종목명': name, '종목코드': code.zfill(6) if code.isdigit() and len(code)<6 else code,
                    '보유수량': qty, '평균단가': avg,
                    '매수일': _clean(ws.cell(rr, header_map.get('매수일',5)).value),
                    '적용전략': _clean(ws.cell(rr, header_map.get('적용전략',6)).value) or '[추천 외 보유]',
                    '메모': _clean(ws.cell(rr, header_map.get('메모',7)).value)
                })
            if holdings:
                break
    return holdings

def _ensure_holdings_input(wb, holdings):
    ws=_reset_sheet(wb,'보유종목_입력',5)
    headers=['종목명','종목코드','보유수량','평균단가','매수일','적용전략','메모']
    _write_headers(ws,headers)
    example=['예시: 금호타이어','073240',25,4865,'2026-05-25','[단기스윙]+[정석방어]','예시 행은 실제 보유종목으로 교체']
    data=[]
    for h in holdings:
        data.append([h.get('종목명',''), h.get('종목코드',''), h.get('보유수량',0), h.get('평균단가',0), h.get('매수일',''), h.get('적용전략',''), h.get('메모','')])
    if not data:
        data=[example]
    for r,row in enumerate(data,2):
        for c,v in enumerate(row,1):
            ws.cell(r,c,v)
    ws.cell(len(data)+3,1,'입력법')
    ws.cell(len(data)+3,2,'종목코드 / 보유수량 / 평균단가는 필수입니다. 추천목록에 없어도 보유종목_진단에서 [추천 외 보유]로 분석합니다.')
    ws.merge_cells(start_row=len(data)+3,start_column=2,end_row=len(data)+3,end_column=7)
    ws.column_dimensions['A'].width=24; ws.column_dimensions['B'].width=13; ws.column_dimensions['C'].width=12; ws.column_dimensions['D'].width=12
    ws.column_dimensions['E'].width=14; ws.column_dimensions['F'].width=22; ws.column_dimensions['G'].width=42
    ws.freeze_panes='A2'
    _style_range_basic(ws,1,ws.max_row,7)
    return ws

def _make_holdings_diag(wb, holdings, rec_df):
    ws=_reset_sheet(wb,'보유종목_진단',6)
    headers=['종목명','종목코드','보유구분','보유수량','평균단가','현재가','평가손익','수익률','보유판정','익절계획','손절계획','진단메모']
    _write_headers(ws,headers)
    rec_map={str(r['종목명']):r for _,r in rec_df.iterrows()} if pd is not None else {r['종목명']:r for r in rec_df}
    rec_code_map={str(r['종목코드']).zfill(6):r for _,r in rec_df.iterrows()} if pd is not None else {str(r['종목코드']).zfill(6):r for r in rec_df}
    rows=[]
    for h in holdings:
        name=h.get('종목명',''); code=str(h.get('종목코드','')).zfill(6)
        rec = rec_map.get(name)
        if rec is None:
            rec = rec_code_map.get(code)
        category='추천목록 보유' if rec is not None else '추천 외 보유'
        current=_num(rec.get('현재가') if rec is not None else 0, 0)
        qty=_num(h.get('보유수량'),0); avg=_num(h.get('평균단가'),0)
        pnl=(current-avg)*qty if current and avg and qty else None
        ret=(current/avg-1)*100 if current and avg else None
        tp=rec.get('권장익절계획') if rec is not None else '+3% / +5% / +10%\n(60%) / (20%) / (20%)'
        sl=rec.get('권장손절계획') if rec is not None else '-2% / -4% / -6%\n(80%) / (10%) / (10%)'
        if ret is None:
            judge='현재가 확인필요'
        elif ret >= 5:
            judge='1차 익절 검토'
        elif ret <= -3:
            judge='손절/비중축소 검토'
        else:
            judge='보유 유지/관찰'
        memo='추천 후보와 연결되어 전략 기준 적용' if rec is not None else '추천 외 보유: 현재가/차트 재조회 후 보수적으로 판단'
        rows.append([name,code,category,qty,avg,current,pnl,ret,judge,tp,sl,memo])
    if not rows:
        rows=[['예시: 금호타이어','073240','추천 외 보유',25,4865,'','','','보유 유지/관찰','+3% / +5% / +10%','-2% / -4% / -6%','실제 보유종목을 보유종목_입력에 입력하세요.']]
    for r,row in enumerate(rows,2):
        for c,v in enumerate(row,1):
            ws.cell(r,c,v)
    widths=[22,12,14,10,12,12,14,10,16,26,26,46]
    for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width=w
    for r in range(2,ws.max_row+1): ws.row_dimensions[r].height=48
    ws.freeze_panes='A2'; _style_range_basic(ws,1,ws.max_row,len(headers))
    return ws

# ─────────────────────────────────────────────
# 프론트 시트 생성
# ─────────────────────────────────────────────

def _make_home(wb, top):
    ws=_reset_sheet(wb,'홈_브리핑',0)
    ws['A1']='실전매매 홈 브리핑'; ws['A1'].font=Font(size=18,bold=True,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',fgColor='0F172A')
    ws.merge_cells('A1:H1')
    ws['A3']='리포트 기준'; ws['B3']=REPORT_DATETIME
    ws['A4']='확인 순서'; ws['B4']='TOP후보 → 진입시나리오 → 스나이퍼 → 백테스트 → 보유진단'
    start=6
    headers=['순위','판정','종목명','섹터/분야','현재가','점수','과열','진입판정']
    _write_headers(ws,headers,start)
    for i,(_,r) in enumerate(top.head(8).iterrows() if pd is not None else enumerate(top[:8]), start+1):
        if pd is not None: rank=i-start; item=r
        else: rank=i[0]+1; item=i[1]
        vals=[rank,item['실전후보구분'],item['종목명'],item['섹터/분야'],item['현재가'],item['실전점수'],item['과열판정'],item['실전진입판정']]
        for c,v in enumerate(vals,1): ws.cell(i,c,v)
    widths={'A':6,'B':16,'C':30,'D':22,'E':14,'F':10,'G':16,'H':20}
    for col,w in widths.items(): ws.column_dimensions[col].width=w
    for r in range(start+1,start+9): ws.row_dimensions[r].height=36
    _style_range_basic(ws,1,ws.max_row,8)
    return ws

def _make_mobile(wb, top):
    ws=_reset_sheet(wb,'모바일_대시보드',1)
    ws['A1']='모바일 대시보드'; ws['A1'].font=Font(size=18,bold=True,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',fgColor='1E40AF')
    ws.merge_cells('A1:I1')
    headers=['순위','종목명','시장','섹터/분야','후보출처','현재가','점수','진입판정','전략요약']
    _write_headers(ws,headers,3)
    for idx,(_,r) in enumerate(top.head(8).iterrows() if pd is not None else enumerate(top[:8]),4):
        item=r if pd is not None else r[1]
        vals=[idx-3,item['종목명'],'KOSPI/KOSDAQ',item['섹터/분야'],item['후보출처'],item['현재가'],item['실전점수'],item['실전진입판정'],f"익절 {item['권장익절계획']} / 손절 {item['권장손절계획']}"]
        for c,v in enumerate(vals,1): ws.cell(idx,c,v)
        ws.row_dimensions[idx].height=70
    widths={'A':6,'B':24,'C':14,'D':26,'E':16,'F':14,'G':10,'H':22,'I':45}
    for col,w in widths.items(): ws.column_dimensions[col].width=w
    _style_range_basic(ws,1,ws.max_row,9)
    return ws

def _make_top_summary(wb, top):
    ws=_reset_sheet(wb,'TOP후보_요약',2)
    headers=['순위','종목명','시장','섹터/분야','후보출처','현재가','기본점수','실전점수','과열판정','과열사유','진입판정','백테스트신뢰도','익절계획','손절계획','추세','재무','추천투입금액','상세전략가이드']
    _write_headers(ws,headers)
    for idx,(_,r) in enumerate(top.head(15).iterrows() if pd is not None else enumerate(top[:15]),2):
        item=r if pd is not None else r[1]
        vals=[idx-1,item['종목명'],'KOSPI/KOSDAQ',item['섹터/분야'],item['후보출처'],item['현재가'],item['기본점수'],item['실전점수'],item['과열판정'],item['과열사유'],item['실전진입판정'],item['백테스트신뢰도'],item['권장익절계획'],item['권장손절계획'],item['추세필터판정'],item['재무위험판정'],item['추천투입금액'],item['상세전략가이드']]
        for c,v in enumerate(vals,1): ws.cell(idx,c,v)
        ws.row_dimensions[idx].height=78
    widths=[6,22,13,22,14,12,10,10,14,32,18,15,28,28,18,18,16,95]
    for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width=w
    ws.freeze_panes='A2'; _style_range_basic(ws,1,ws.max_row,len(headers))
    return ws

def _make_cards(wb, top):
    ws=_reset_sheet(wb,'종목카드_TOP15',3)
    ws['A1']='종목카드 TOP15'; ws['A1'].font=Font(size=18,bold=True,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',fgColor='334155')
    ws.merge_cells('A1:H1')
    row=3
    for idx,(_,r) in enumerate(top.head(15).iterrows() if pd is not None else enumerate(top[:15]),1):
        item=r if pd is not None else r[1]
        title=f"#{idx}  {item['종목명']} · {item['후보출처']} · {item['실전후보구분']} / {item['섹터/분야']}"
        ws.merge_cells(start_row=row,start_column=1,end_row=row,end_column=8)
        ws.cell(row,1,title); ws.cell(row,1).font=Font(size=13,bold=True,color='FFFFFF'); ws.cell(row,1).fill=PatternFill('solid',fgColor='1F4E78')
        labels=[('현재가',_won(item['현재가'])),('점수',item['실전점수']),('과열',item['과열판정']),('진입',item['실전진입판정']),('익절',item['권장익절계획']),('손절',item['권장손절계획'])]
        for j,(k,v) in enumerate(labels,1):
            ws.cell(row+1,j,k); ws.cell(row+2,j,v)
        ws.merge_cells(start_row=row+3,start_column=1,end_row=row+4,end_column=8)
        ws.cell(row+3,1,item['상세전략가이드'])
        ws.row_dimensions[row+3].height=70
        row += 6
    for c in range(1,9): ws.column_dimensions[get_column_letter(c)].width=18
    _style_range_basic(ws,1,ws.max_row,8)
    return ws

def _make_entry(wb, top):
    ws=_reset_sheet(wb,'진입시나리오',4)
    ws['A1']='진입시나리오'; ws['A1'].font=Font(size=18,bold=True,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',fgColor='0F766E')
    ws.merge_cells('A1:J1')
    ws['A2']='활용 예시'; ws['B2']='진입판정이 「눌림 대기」이면 기준/보수 진입가 근처까지 기다리고, 「지금 진입 가능」이어도 분할 진입으로 확인합니다.'
    ws.merge_cells('B2:J2')
    headers=['순위','종목명','섹터/분야','현재가','진입판정','공격진입가','기준진입가','보수진입가','돌파진입가','손절기준가']
    _write_headers(ws,headers,4)
    for idx,(_,r) in enumerate(top.head(15).iterrows() if pd is not None else enumerate(top[:15]),5):
        item=r if pd is not None else r[1]
        p=_num(item['현재가'])
        vals=[idx-4,item['종목명'],item['섹터/분야'],p,item['실전진입판정'],round(p*0.995),round(p*0.98),round(p*0.95),round(p*1.03),round(p*0.94)]
        for c,v in enumerate(vals,1): ws.cell(idx,c,v)
    widths=[6,22,22,12,20,12,12,12,12,12]
    for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width=w
    _style_range_basic(ws,1,ws.max_row,10)
    return ws

# ─────────────────────────────────────────────
# 스나이퍼 / 백테스트 실험실
# ─────────────────────────────────────────────

def _parse_plan(plan):
    txt=_clean(plan)
    nums=[float(x) for x in re.findall(r'([+-]?\d+(?:\.\d+)?)\s*%', txt)]
    # 앞 3개는 목표율, 뒤 3개는 비중으로 가정
    if len(nums)>=6:
        rates=[n/100 for n in nums[:3]]; weights=[n/100 for n in nums[3:6]]
    elif len(nums)>=3:
        rates=[n/100 for n in nums[:3]]; weights=[0.6,0.2,0.2]
    else:
        rates=[0.03,0.05,0.10]; weights=[0.6,0.2,0.2]
    return rates[:3],weights[:3]

def _make_sniper(wb, top, holdings):
    ws=_reset_sheet(wb,'스나이퍼_계산기',7)
    ws['A1']='스나이퍼 계산기'; ws['A1'].font=Font(size=18,bold=True,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',fgColor='7C2D12')
    ws.merge_cells('A1:J1')
    ws['B4']='추천목록 선택'; ws['D4']='보유종목 선택'; ws['F4']='분석대상'
    # 숨김 목록/데이터
    rec_names=['없음'] + (top['종목명'].astype(str).tolist() if pd is not None and not top.empty else [])
    hold_names=['없음'] + [h.get('종목명','') for h in holdings]
    for i,n in enumerate(rec_names,2): ws.cell(i,27,n)
    for i,n in enumerate(hold_names,2): ws.cell(i,28,n)
    data_headers=['종목명','종목코드','섹터','후보출처','현재가','추천투입금액','추천수량','진입판정','과열판정','익절계획','손절계획','상세가이드','TP1','TPW1','TP2','TPW2','TP3','TPW3','SL1','SLW1','SL2','SLW2','SL3','SLW3']
    for c,h in enumerate(data_headers,36): ws.cell(1,c,h)
    data_rows=[]
    for _,r in (top.iterrows() if pd is not None else enumerate(top)):
        item=r if pd is not None else r[1]
        tpr,tpw=_parse_plan(item['권장익절계획']); slr,slw=_parse_plan(item['권장손절계획'])
        data_rows.append([item['종목명'],item['종목코드'],item['섹터/분야'],item['후보출처'],item['현재가'],item['추천투입금액'],item['추천수량'],item['실전진입판정'],item['과열판정'],item['권장익절계획'],item['권장손절계획'],item['상세전략가이드'],tpr[0],tpw[0],tpr[1],tpw[1],tpr[2],tpw[2],-abs(slr[0]),slw[0],-abs(slr[1]),slw[1],-abs(slr[2]),slw[2]])
    for h in holdings:
        tpr,tpw=[0.03,0.05,0.10],[0.6,0.2,0.2]; slr,slw=[0.02,0.04,0.06],[0.8,0.1,0.1]
        data_rows.append([h.get('종목명',''),h.get('종목코드',''),'추천 외 보유','HOLDING',h.get('평균단가',0),h.get('평균단가',0)*h.get('보유수량',0),h.get('보유수량',0),'보유진단','보유종목','+3% / +5% / +10%\n(60%) / (20%) / (20%)','-2% / -4% / -6%\n(80%) / (10%) / (10%)','보유종목 입력값 기준 기본 계획입니다. 최신 현재가는 보유종목_진단과 함께 확인하세요.',tpr[0],tpw[0],tpr[1],tpw[1],tpr[2],tpw[2],-slr[0],slw[0],-slr[1],slw[1],-slr[2],slw[2]])
    for r,row in enumerate(data_rows,2):
        for c,v in enumerate(row,36): ws.cell(r,c,v)
    dv1=DataValidation(type='list', formula1=f'$AA$2:$AA${max(2,len(rec_names)+1)}')
    dv2=DataValidation(type='list', formula1=f'$AB$2:$AB${max(2,len(hold_names)+1)}')
    ws.add_data_validation(dv1); ws.add_data_validation(dv2); dv1.add(ws['C4']); dv2.add(ws['E4'])
    ws['C4']='없음' if len(rec_names)==1 else rec_names[1]
    ws['E4']='없음'
    ws['G4']='=IF($C$4<>"없음",$C$4,$E$4)'
    # 정보 패널
    labels=[('종목코드',2),('섹터',3),('후보출처',4),('현재가',5),('추천투입금액',6),('추천수량',7),('진입판정',8),('과열판정',9),('익절계획',10),('손절계획',11),('상세가이드',12)]
    start=6
    for i,(lab,idx) in enumerate(labels,start):
        ws.cell(i,2,lab)
        ws.cell(i,3,f'=IFERROR(VLOOKUP($G$4,$AJ$2:$BG$250,{idx},FALSE),"")')
        ws.cell(i,3).alignment=Alignment(wrap_text=True,vertical='center')
    # 실제 투입금액 / 수량
    ws['E6']='실제투입금액'; ws['F6']='=IFERROR(C10,0)'
    ws['E7']='계산수량'; ws['F7']='=IFERROR(INT(F6/C9),0)'
    _write_headers(ws,['구분','조건','매도비중','예약단가','수량','예상금액','예상손익률'],20,fill='9A3412')
    for i in range(3):
        row=21+i
        rate_col=13+i*2; weight_col=14+i*2
        ws.cell(row,1,f'{i+1}차 익절')
        ws.cell(row,2,f'=IFERROR(TEXT(VLOOKUP($G$4,$AJ$2:$BG$250,{rate_col},FALSE),"+0%"),"")')
        ws.cell(row,3,f'=IFERROR(TEXT(VLOOKUP($G$4,$AJ$2:$BG$250,{weight_col},FALSE),"0%"),"")')
        ws.cell(row,4,f'=IFERROR($C$9*(1+VLOOKUP($G$4,$AJ$2:$BG$250,{rate_col},FALSE)),"")')
        ws.cell(row,5,f'=IFERROR(INT($F$7*VLOOKUP($G$4,$AJ$2:$BG$250,{weight_col},FALSE)),"")')
        ws.cell(row,6,f'=IFERROR(D{row}*E{row},"")')
        ws.cell(row,7,f'=IFERROR(VLOOKUP($G$4,$AJ$2:$BG$250,{rate_col},FALSE),"")')
    for i in range(3):
        row=25+i
        rate_col=19+i*2; weight_col=20+i*2
        ws.cell(row,1,f'{i+1}차 손절')
        ws.cell(row,2,f'=IFERROR(TEXT(VLOOKUP($G$4,$AJ$2:$BG$250,{rate_col},FALSE),"0%"),"")')
        ws.cell(row,3,f'=IFERROR(TEXT(VLOOKUP($G$4,$AJ$2:$BG$250,{weight_col},FALSE),"0%"),"")')
        ws.cell(row,4,f'=IFERROR($C$9*(1+VLOOKUP($G$4,$AJ$2:$BG$250,{rate_col},FALSE)),"")')
    # 위에서 keyword arg 실수 방지용 재대입
    for i in range(3):
        row=25+i; rate_col=19+i*2; weight_col=20+i*2
        ws.cell(row,5).value=f'=IFERROR(INT($F$7*VLOOKUP($G$4,$AJ$2:$BG$250,{weight_col},FALSE)),"")'
        ws.cell(row,6).value=f'=IFERROR(D{row}*E{row},"")'
        ws.cell(row,7).value=f'=IFERROR(VLOOKUP($G$4,$AJ$2:$BG$250,{rate_col},FALSE),"")'
    widths={'A':14,'B':16,'C':42,'D':18,'E':16,'F':16,'G':16,'H':16,'I':16,'J':16}
    for col,w in widths.items(): ws.column_dimensions[col].width=w
    for col in range(27,60): ws.column_dimensions[get_column_letter(col)].hidden=True
    _style_range_basic(ws,1,ws.max_row,10)
    return ws

def _make_lab(wb, top, holdings):
    ws=_reset_sheet(wb,'백테스트_실험실',8)
    ws['A1']='백테스트 실험실'; ws['A1'].font=Font(size=18,bold=True,color='FFFFFF'); ws['A1'].fill=PatternFill('solid',fgColor='4338CA')
    ws.merge_cells('A1:J1')
    ws['B5']='추천목록'; ws['D5']='보유종목'; ws['F5']='분석대상'
    rec_names=['없음']+(top['종목명'].astype(str).tolist() if pd is not None and not top.empty else [])
    hold_names=['없음']+[h.get('종목명','') for h in holdings]
    for i,n in enumerate(rec_names,2): ws.cell(i,27,n)
    for i,n in enumerate(hold_names,2): ws.cell(i,28,n)
    dv1=DataValidation(type='list', formula1=f'$AA$2:$AA${max(2,len(rec_names)+1)}')
    dv2=DataValidation(type='list', formula1=f'$AB$2:$AB${max(2,len(hold_names)+1)}')
    ws.add_data_validation(dv1); ws.add_data_validation(dv2); dv1.add(ws['C5']); dv2.add(ws['E5'])
    ws['C5']='없음' if len(rec_names)==1 else rec_names[1]; ws['E5']='없음'; ws['G5']='=IF($C$5<>"없음",$C$5,$E$5)'
    ws['B7']='기간 선택'; ws['C7']='1개월 전에 샀다면'; ws['D7']='옵션: 2주 / 1개월 / 3개월 / 6개월 / 전체'
    ws['B8']='전략 선택'; ws['C8']='[초단타]+[칼손절]'; ws['D8']='옵션: 백테스트 요약 시트 참고'
    ws['B9']='투입금액'; ws['C9']=300000; ws['D9']='원 단위 직접 변경 가능'
    # 실험실 요약을 로컬로 복사
    if '실험실_요약' in wb.sheetnames:
        src=wb['실험실_요약']
        maxr=min(src.max_row, 2000)
        maxc=min(src.max_column, 20)
        for r in range(1,maxr+1):
            for c in range(1,maxc+1):
                ws.cell(r,36+c-1,src.cell(r,c).value)
    key_formula='=$G$5&"|"&$C$7&"|"&$C$8'
    ws['B11']='조회키'; ws['C11']=key_formula
    fields=[('시작기준가',9),('현재가',10),('보유상승률',11),('전략수익률',12),('MDD',13),('승률',14),('거래횟수',15),('백테스트시작일',16),('백테스트종료일',17),('TP전략내용',18),('SL전략내용',19),('비용가정',20)]
    for i,(lab,idx) in enumerate(fields,13):
        ws.cell(i,2,lab)
        ws.cell(i,3,f'=IFERROR(VLOOKUP($C$11,$AJ$2:$BC$2000,{idx},FALSE),"")')
    ws['B27']='매매기록'
    ws['C27']='실험실_기록 시트에서 조회키로 확인하세요. 2주/1개월은 거래가 없을 경우 기록 없음으로 표시될 수 있습니다.'
    ws.row_dimensions[27].height=60
    widths={'A':4,'B':18,'C':28,'D':16,'E':28,'F':14,'G':28,'H':20,'I':16,'J':16}
    for col,w in widths.items(): ws.column_dimensions[col].width=w
    for col in range(27,60): ws.column_dimensions[get_column_letter(col)].hidden=True
    _style_range_basic(ws,1,ws.max_row,10)
    return ws

# ─────────────────────────────────────────────
# 검증 및 수식 정리
# ─────────────────────────────────────────────

def _make_validation(wb):
    ws=_reset_sheet(wb,'검증결과',9)
    checks=[]
    def add(name, ok, memo=''):
        checks.append([name,'PASS' if ok else 'FAIL',memo])
    add('홈_브리핑 C열 폭 30 이상', wb['홈_브리핑'].column_dimensions['C'].width and wb['홈_브리핑'].column_dimensions['C'].width>=30, f"C={wb['홈_브리핑'].column_dimensions['C'].width}")
    add('홈_브리핑 D열 섹터 존재', wb['홈_브리핑']['D6'].value in ['섹터/분야','분야','카테고리/섹터'], str(wb['홈_브리핑']['D6'].value))
    add('모바일_대시보드 H열 폭 20 이상', wb['모바일_대시보드'].column_dimensions['H'].width and wb['모바일_대시보드'].column_dimensions['H'].width>=20, f"H={wb['모바일_대시보드'].column_dimensions['H'].width}")
    add('TOP후보_요약 상세가이드 폭 90 이상', wb['TOP후보_요약'].column_dimensions['R'].width and wb['TOP후보_요약'].column_dimensions['R'].width>=90, f"R={wb['TOP후보_요약'].column_dimensions['R'].width}")
    add('보유종목_입력 시트 존재', '보유종목_입력' in wb.sheetnames, '')
    add('보유종목_진단 시트 존재', '보유종목_진단' in wb.sheetnames, '')
    add('스나이퍼 C4/D4 구조', wb['스나이퍼_계산기']['C4'].value is not None and wb['스나이퍼_계산기']['E4'].value is not None, f"C4={wb['스나이퍼_계산기']['C4'].value}, E4={wb['스나이퍼_계산기']['E4'].value}")
    add('백테스트 C5/D5 구조', wb['백테스트_실험실']['C5'].value is not None and wb['백테스트_실험실']['E5'].value is not None, f"C5={wb['백테스트_실험실']['C5'].value}, E5={wb['백테스트_실험실']['E5'].value}")
    add('과열필터 컬럼 존재', '과열판정' in _headers(wb['추천 리스트']), '')
    _write_headers(ws,['검증항목','결과','메모'])
    for r,row in enumerate(checks,2):
        for c,v in enumerate(row,1): ws.cell(r,c,v)
        ws.cell(r,2).fill=PatternFill('solid', fgColor='DCFCE7' if row[1]=='PASS' else 'FECACA')
    for col,w in {'A':36,'B':10,'C':60}.items(): ws.column_dimensions[col].width=w
    _style_range_basic(ws,1,ws.max_row,3)
    return checks

def _sanitize_formulas(wb):
    risky=[]
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                v=cell.value
                if isinstance(v,str) and v.startswith('='):
                    nv=v
                    nv=re.sub(r'(?<![<>=])=IFERROR\(', 'IFERROR(', nv[1:])
                    nv=re.sub(r'(?<![<>=])=IF\(', 'IF(', nv)
                    nv='='+nv
                    if nv!=v:
                        cell.value=nv
                    if 'TEXT(=' in cell.value or '=IFERROR(=' in cell.value or '=IF(=' in cell.value:
                        risky.append((ws.title,cell.coordinate,cell.value[:120]))
    return risky

def _scan_xlsx(path):
    bad=[]
    try:
        with zipfile.ZipFile(path,'r') as z:
            for name in z.namelist():
                if name.startswith('xl/worksheets/sheet') and name.endswith('.xml'):
                    txt=z.read(name).decode('utf-8',errors='ignore')
                    for pat in ['TEXT(=','=IFERROR(=','=IF(=']:
                        if pat in txt:
                            bad.append((name,pat))
    except Exception as e:
        bad.append(('zip_scan_error',repr(e)))
    return bad

# ─────────────────────────────────────────────
# 실행
# ─────────────────────────────────────────────

try:
    src_path=_v975_latest_xlsx()
    print('🔧 v9.7.5 안정화 패치 대상:', src_path)
    wb=load_workbook(src_path)
    rec_df=_get_recommendations(wb)
    _update_recommendation_sheet(wb, rec_df)
    top=_top_df(rec_df, 15)
    holdings=_read_holdings_from_wb(wb)
    _make_home(wb, top)
    _make_mobile(wb, top)
    _make_top_summary(wb, top)
    _make_cards(wb, top)
    _make_entry(wb, top)
    _ensure_holdings_input(wb, holdings)
    _make_holdings_diag(wb, holdings, rec_df)
    _make_sniper(wb, top, holdings)
    _make_lab(wb, top, holdings)
    _make_validation(wb)
    risky=_sanitize_formulas(wb)
    # 시트 순서 고정
    front=['홈_브리핑','모바일_대시보드','TOP후보_요약','종목카드_TOP15','진입시나리오','보유종목_입력','보유종목_진단','스나이퍼_계산기','백테스트_실험실','검증결과','오늘_체크리스트','탈락사유_리포트','전략백테스트요약','추천 리스트']
    for name in reversed(front):
        if name in wb.sheetnames:
            ws=wb[name]
            wb._sheets.remove(ws)
            wb._sheets.insert(0, ws)
    # 저장명
    try:
        out_name = RUN_AT.strftime('%Y%m%d') + '.xlsx'
    except Exception:
        refresh_report_context(CONFIG.get('FORCE_REPORT_DATE'))
        out_name = RUN_AT.strftime(CONFIG.get('OUTPUT_DATE_FORMAT','%Y%m%d')) + '.xlsx'
    wb.calculation.fullCalcOnLoad=True
    wb.calculation.forceFullCalc=True
    wb.save(out_name)
    bad_xml=_scan_xlsx(out_name)
    globals()['output_file']=out_name
    globals()['unified_top15_df']=top
    print(f'✅ v9.7.5 안정화 패치 완료: {out_name}')
    print(f'검증: 추천 {len(rec_df)}개 / 통합 TOP {len(top)}개 / 보유 {len(holdings)}개')
    if risky:
        print('⚠️ 내부 수식 위험 패턴:', risky[:5])
    if bad_xml:
        print('⚠️ 저장 후 XML 위험 패턴:', bad_xml[:5])
    else:
        print('✅ 저장 후 XML 위험 수식 패턴 없음')
    if globals().get('IN_COLAB', False) and files is not None:
        print(f"중간 파일 생성: {out_name}")  # final cell에서만 다운로드
except Exception as e:
    print('❌ v9.7.5 안정화 패치 실패:', repr(e))
    raise



🔧 v9.7.5 안정화 패치 대상: 20260608.xlsx


✅ v9.7.5 안정화 패치 완료: 20260608.xlsx
검증: 추천 50개 / 통합 TOP 15개 / 보유 0개
✅ 저장 후 XML 위험 수식 패턴 없음


In [23]:

# v9.7.6 최종 안정화 패치 — 보유 현재가 독립조회 + 스나이퍼/백테스트 UI 재생성
# 목적:
# 1) 보유종목_진단에서 추천목록에 없는 종목도 현재가 조회
# 2) 스나이퍼_계산기 / 백테스트_실험실을 안정적인 단순 수식 구조로 재생성
# 3) 검증결과 시트에 실제 반영 여부 기록

import os, re, glob, math, warnings, zipfile
from datetime import datetime, timedelta, timezone
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

KST = timezone(timedelta(hours=9))

def _kst_now():
    return datetime.now(KST)

def _today_yyyymmdd():
    cfg = globals().get('CONFIG', {}) if isinstance(globals().get('CONFIG', {}), dict) else {}
    forced = cfg.get('FORCE_REPORT_DATE')
    if forced:
        return str(forced).replace('-', '')
    return _kst_now().strftime('%Y%m%d')

def _find_output_file():
    # 기존 output_file이 있으면 우선 사용하되, 없거나 파일이 없으면 가장 최근 xlsx 사용
    f = globals().get('output_file')
    if f and os.path.exists(f):
        return f
    candidates = sorted(glob.glob('*.xlsx'), key=os.path.getmtime, reverse=True)
    for c in candidates:
        if not c.startswith('~$'):
            return c
    raise FileNotFoundError('패치 대상 xlsx 파일을 찾지 못했습니다.')

def _safe_num(v, default=0):
    try:
        if v is None or v == '':
            return default
        if isinstance(v, str):
            v = re.sub(r'[^0-9.\-]', '', v)
            if v in ('', '-', '.', '-.'):
                return default
        return float(v)
    except Exception:
        return default

def _safe_int(v, default=0):
    try:
        return int(round(_safe_num(v, default)))
    except Exception:
        return default

def _norm_code(code, name=''):
    name = str(name or '').strip()
    code_s = str(code or '').strip().replace('.0','')
    # 스크린샷 기반 입력에서 잘못 들어간 ETF 코드 보정
    manual = {
        'HANARO K휴머노이드테마TOP10': '0155N0',
        'TIGER K방산&우주': '463250',
    }
    if name in manual:
        return manual[name]
    if 'HANARO' in name and '휴머노이드' in name:
        return '0155N0'
    if 'TIGER' in name and '방산' in name:
        return '463250'
    if re.fullmatch(r'\d+', code_s):
        return code_s.zfill(6)
    return code_s

def _style_header(ws, row=1, start_col=1, end_col=None, fill='1F4E78'):
    if end_col is None:
        end_col = ws.max_column
    for col in range(start_col, end_col+1):
        c = ws.cell(row, col)
        c.fill = PatternFill('solid', fgColor=fill)
        c.font = Font(color='FFFFFF', bold=True)
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = Border(bottom=Side(style='thin', color='CBD5E1'))

def _set_widths(ws, widths):
    for col, width in widths.items():
        ws.column_dimensions[col].width = width

def _read_sheet_df(wb, sheet_name):
    if sheet_name not in wb.sheetnames:
        return pd.DataFrame()
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return pd.DataFrame()
    header = [str(x).strip() if x is not None else '' for x in rows[0]]
    data = rows[1:]
    df = pd.DataFrame(data, columns=header)
    df = df.dropna(how='all')
    return df

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    # 공백 제거 비교
    lookup = {str(col).replace(' ',''): col for col in df.columns}
    for c in candidates:
        key = str(c).replace(' ','')
        if key in lookup:
            return lookup[key]
    return None

def _rec_maps_from_wb(wb):
    rec_df = _read_sheet_df(wb, '추천 리스트')
    if rec_df.empty:
        rec_df = _read_sheet_df(wb, 'TOP후보_요약')
    if rec_df.empty:
        return {}, {}, pd.DataFrame()
    name_col = _find_col(rec_df, ['종목명','Name'])
    code_col = _find_col(rec_df, ['종목코드','코드','Ticker'])
    maps_name, maps_code = {}, {}
    for _, row in rec_df.iterrows():
        name = str(row.get(name_col,'')).strip() if name_col else ''
        code = _norm_code(row.get(code_col,''), name) if code_col else ''
        if name and name.lower() != 'nan':
            maps_name[name] = row
        if code and code.lower() != 'nan':
            maps_code[code] = row
    return maps_name, maps_code, rec_df

def _col_value(row, candidates, default=''):
    if row is None:
        return default
    for c in candidates:
        try:
            if c in row.index:
                v = row.get(c)
                if v is not None and not (isinstance(v, float) and math.isnan(v)):
                    return v
        except Exception:
            pass
    # 공백 제거 fallback
    try:
        lookup = {str(k).replace(' ',''): k for k in row.index}
        for c in candidates:
            k = str(c).replace(' ','')
            if k in lookup:
                v = row.get(lookup[k])
                if v is not None and not (isinstance(v, float) and math.isnan(v)):
                    return v
    except Exception:
        pass
    return default

def _parse_plan(plan, is_tp=True):
    text = str(plan or '')
    # +3% / +5% / +10% \n (70%) / (20%) / (10%) 형식 우선
    nums = [float(x)/100 for x in re.findall(r'([+-]?\d+(?:\.\d+)?)\s*%', text)]
    vals, ratios = [], []
    if nums:
        vals = nums[:3]
        # 괄호 안 비율 우선
        rnums = [float(x)/100 for x in re.findall(r'\((\d+(?:\.\d+)?)\s*%\)', text)]
        ratios = rnums[:3] if rnums else nums[3:6]
    if len(vals) < 3:
        vals = [0.03,0.05,0.10] if is_tp else [-0.02,-0.04,-0.06]
    if len(ratios) < 3:
        ratios = [0.6,0.2,0.2] if is_tp else [0.8,0.1,0.1]
    return vals[:3], ratios[:3]

def _get_recent_price_from_market(code, name=''):
    code = _norm_code(code, name)
    if not code:
        return 0, '코드없음'
    # 1) pykrx 우선
    try:
        from pykrx import stock
        today = _kst_now().date()
        for i in range(0, 12):
            d = (today - timedelta(days=i)).strftime('%Y%m%d')
            # 개별 종목/ETF OHLCV by date
            try:
                df = stock.get_market_ohlcv_by_date(d, d, code)
                if df is not None and len(df) > 0 and '종가' in df.columns:
                    p = _safe_num(df['종가'].dropna().iloc[-1], 0)
                    if p > 0:
                        return p, f'pykrx({d})'
            except Exception:
                pass
            try:
                if hasattr(stock, 'get_etf_ohlcv_by_date'):
                    df = stock.get_etf_ohlcv_by_date(d, d, code)
                    if df is not None and len(df) > 0 and '종가' in df.columns:
                        p = _safe_num(df['종가'].dropna().iloc[-1], 0)
                        if p > 0:
                            return p, f'pykrxETF({d})'
            except Exception:
                pass
            # 일자별 전체 티커 테이블
            try:
                if re.fullmatch(r'\d{6}', code):
                    df = stock.get_market_ohlcv_by_ticker(d, market='ALL')
                    if code in df.index and '종가' in df.columns:
                        p = _safe_num(df.loc[code, '종가'], 0)
                        if p > 0:
                            return p, f'pykrxTicker({d})'
            except Exception:
                pass
            try:
                if hasattr(stock, 'get_etf_ohlcv_by_ticker'):
                    df = stock.get_etf_ohlcv_by_ticker(d)
                    if code in df.index and '종가' in df.columns:
                        p = _safe_num(df.loc[code, '종가'], 0)
                        if p > 0:
                            return p, f'pykrxETFTable({d})'
            except Exception:
                pass
    except Exception:
        pass
    # 2) FinanceDataReader fallback
    try:
        import FinanceDataReader as fdr
        end = _kst_now().date()
        start = end - timedelta(days=14)
        df = fdr.DataReader(code, start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'))
        if df is not None and len(df) > 0 and 'Close' in df.columns:
            p = _safe_num(df['Close'].dropna().iloc[-1], 0)
            if p > 0:
                return p, 'FDR'
    except Exception:
        pass
    return 0, '현재가조회실패'

def _get_latest_price(code, name, rec_row=None):
    # 추천 리스트 현재가 우선. 단 0이면 시장 조회
    if rec_row is not None:
        p = _safe_num(_col_value(rec_row, ['현재가','가격','Close','종가']), 0)
        if p > 0:
            return p, '추천리스트'
    return _get_recent_price_from_market(code, name)

def _ensure_holdings_input(wb):
    if '보유종목_입력' in wb.sheetnames:
        ws = wb['보유종목_입력']
    else:
        ws = wb.create_sheet('보유종목_입력')
        ws.append(['종목명','종목코드','보유수량','평균단가','매수일','적용전략','메모'])
    # 예시/안내가 없으면 아래쪽에 추가
    if ws.max_row < 2:
        ws.append(['예시_대한해운','005880',41,2519,_kst_now().strftime('%Y-%m-%d'),'[추천 외 보유]','예시 행은 삭제 후 입력'])
    note_row = max(ws.max_row+2, 11)
    if ws.cell(note_row,1).value is None:
        ws.cell(note_row,1).value = '입력법'
        ws.cell(note_row,2).value = '종목코드 / 보유수량 / 평균단가는 필수입니다. 추천목록에 없어도 보유종목_진단에서 [추천 외 보유]로 분석합니다.'
    _style_header(ws, 1, 1, 7, '334155')
    _set_widths(ws, {'A':28,'B':14,'C':12,'D':12,'E':14,'F':18,'G':42})
    return ws

def _read_holdings(wb):
    if '보유종목_입력' not in wb.sheetnames:
        _ensure_holdings_input(wb)
    ws = wb['보유종목_입력']
    rows = []
    for r in range(2, ws.max_row+1):
        name = ws.cell(r,1).value
        code = ws.cell(r,2).value
        qty = ws.cell(r,3).value
        avg = ws.cell(r,4).value
        if not name or str(name).startswith('예시') or str(name) == '입력법':
            continue
        if _safe_num(qty, 0) <= 0 or _safe_num(avg, 0) <= 0:
            continue
        rows.append({
            '종목명': str(name).strip(),
            '종목코드': _norm_code(code, name),
            '보유수량': _safe_int(qty, 0),
            '평균단가': _safe_num(avg, 0),
            '매수일': ws.cell(r,5).value or '',
            '적용전략': ws.cell(r,6).value or '[추천 외 보유]',
            '메모': ws.cell(r,7).value or '',
        })
    return pd.DataFrame(rows)

def _make_holdings_diag(wb, holdings_df, rec_maps_name, rec_maps_code):
    if '보유종목_진단' in wb.sheetnames:
        idx = wb.sheetnames.index('보유종목_진단')
        del wb['보유종목_진단']
        ws = wb.create_sheet('보유종목_진단', idx)
    else:
        ws = wb.create_sheet('보유종목_진단')
    headers = ['종목명','종목코드','보유구분','보유수량','평균단가','현재가','평가손익','수익률','보유판정','익절계획','손절계획','진단메모','가격출처']
    ws.append(headers)
    records = []
    if holdings_df.empty:
        ws.append(['보유종목 없음','','',0,0,0,'','','','','','보유종목_입력 시트에 종목코드/수량/평균단가를 입력하세요',''])
    else:
        for _, h in holdings_df.iterrows():
            name = str(h['종목명']).strip()
            code = _norm_code(h['종목코드'], name)
            qty = _safe_int(h['보유수량'], 0)
            avg = _safe_num(h['평균단가'], 0)
            rec = rec_maps_name.get(name)
            if rec is None:
                rec = rec_maps_code.get(code)
            category = '추천목록 보유' if rec is not None else '추천 외 보유'
            price, price_src = _get_latest_price(code, name, rec)
            if price > 0 and avg > 0:
                pnl = round((price - avg) * qty, 0)
                ret = round((price / avg - 1) * 100, 2)
            else:
                pnl, ret = '', ''
            tp_plan = str(_col_value(rec, ['권장익절계획','익절계획','권장익절퍼센트'], '+3% / +5% / +10%\n(60%) / (20%) / (20%)')) if rec is not None else '+3% / +5% / +10%\n(60%) / (20%) / (20%)'
            sl_plan = str(_col_value(rec, ['권장손절계획','손절계획','권장손절퍼센트'], '-2% / -4% / -6%\n(80%) / (10%) / (10%)')) if rec is not None else '-2% / -4% / -6%\n(80%) / (10%) / (10%)'
            tp_vals, _ = _parse_plan(tp_plan, True)
            sl_vals, _ = _parse_plan(sl_plan, False)
            if price <= 0:
                판정 = '현재가 확인필요'
                memo = '시세 조회 실패: 종목코드/ETF 코드 확인 필요'
            elif ret != '' and ret >= tp_vals[0]*100:
                판정 = '1차 익절 검토'
                memo = f'수익률 {ret:.2f}%: 1차 익절 기준 도달/근접 여부 확인'
            elif ret != '' and ret <= sl_vals[0]*100:
                판정 = '손절 검토'
                memo = f'수익률 {ret:.2f}%: 손절 기준 도달/근접 여부 확인'
            else:
                판정 = '보유 유지/관찰'
                memo = '추천 후보와 연결' if rec is not None else '추천 외 보유: 현재가/차트 재조회 후 보수 판단'
            row = [name, code, category, qty, avg, price, pnl, ret, 판정, tp_plan, sl_plan, memo, price_src]
            ws.append(row)
            records.append(dict(zip(headers,row)))
    _style_header(ws, 1, 1, len(headers), '0F766E')
    _set_widths(ws, {'A':28,'B':12,'C':14,'D':10,'E':12,'F':12,'G':14,'H':10,'I':18,'J':28,'K':28,'L':46,'M':18})
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        ws.row_dimensions[row[0].row].height = 42
    for col in ['E','F','G']:
        for cell in ws[col][1:]:
            cell.number_format = '#,##0'
    for cell in ws['H'][1:]:
        cell.number_format = '0.00'
    return pd.DataFrame(records)

def _top_records_from_wb(wb):
    # TOP후보_요약 우선, 없으면 추천 리스트
    src_name = 'TOP후보_요약' if 'TOP후보_요약' in wb.sheetnames else '추천 리스트'
    df = _read_sheet_df(wb, src_name)
    if df.empty:
        return []
    records = []
    for _, r in df.head(60).iterrows():
        name = str(_col_value(r, ['종목명'], '')).strip()
        if not name or name.lower() == 'nan':
            continue
        code = _norm_code(_col_value(r, ['종목코드','코드'], ''), name)
        sector = str(_col_value(r, ['섹터/분야','표시분야','분야','공식업종'], '') or '')
        if not sector or sector.lower() == 'nan':
            sector = '분야확인필요'
        source = str(_col_value(r, ['후보출처'], ''))
        price = _safe_num(_col_value(r, ['현재가'], 0), 0)
        amount = _safe_num(_col_value(r, ['추천투입금액'], 0), 0)
        qty = _safe_num(_col_value(r, ['추천수량'], 0), 0)
        score = _safe_num(_col_value(r, ['실전점수','종목점수','기본점수'], 0), 0)
        entry = str(_col_value(r, ['진입판정'], '조건 확인 후 진입'))
        overheat = str(_col_value(r, ['과열판정'], '정상'))
        tp_plan = str(_col_value(r, ['권장익절계획','익절계획'], '+3% / +5% / +10%\n(70%) / (20%) / (10%)'))
        sl_plan = str(_col_value(r, ['권장손절계획','손절계획'], '-2% / -4% / -6%\n(80%) / (10%) / (10%)'))
        guide = str(_col_value(r, ['상세전략가이드','상세가이드'], f'후보출처 {source}, 분야 {sector}, 점수 {score}점.'))
        records.append({
            '종목명':name,'종목코드':code,'섹터':sector,'후보출처':source,'현재가':price,
            '추천투입금액':amount,'추천수량':qty,'진입판정':entry,'과열판정':overheat,
            '익절계획':tp_plan,'손절계획':sl_plan,'상세가이드':guide,'점수':score
        })
    return records

def _hidden_data_headers():
    return ['선택명','종목명','종목코드','섹터','후보출처','현재가','추천투입금액','추천수량','진입판정','과열판정','익절계획','손절계획','상세가이드','TP1','TP1비중','TP2','TP2비중','TP3','TP3비중','SL1','SL1비중','SL2','SL2비중','SL3','SL3비중']

def _build_candidate_rows(top_records, holdings_diag_df):
    rec_rows = []
    for x in top_records:
        tpv,tpr = _parse_plan(x.get('익절계획'), True)
        slv,slr = _parse_plan(x.get('손절계획'), False)
        rec_rows.append([x.get('종목명'), x.get('종목명'), x.get('종목코드'), x.get('섹터'), x.get('후보출처'), x.get('현재가'), x.get('추천투입금액'), x.get('추천수량'), x.get('진입판정'), x.get('과열판정'), x.get('익절계획'), x.get('손절계획'), x.get('상세가이드')] + [tpv[0],tpr[0],tpv[1],tpr[1],tpv[2],tpr[2],slv[0],slr[0],slv[1],slr[1],slv[2],slr[2]])
    hold_rows = []
    if holdings_diag_df is not None and not holdings_diag_df.empty:
        for _, h in holdings_diag_df.iterrows():
            name = str(h.get('종목명','')).strip()
            if not name or name == '보유종목 없음':
                continue
            tp_plan = str(h.get('익절계획','+3% / +5% / +10%\n(60%) / (20%) / (20%)'))
            sl_plan = str(h.get('손절계획','-2% / -4% / -6%\n(80%) / (10%) / (10%)'))
            tpv,tpr = _parse_plan(tp_plan, True)
            slv,slr = _parse_plan(sl_plan, False)
            price = _safe_num(h.get('현재가'),0)
            qty = _safe_num(h.get('보유수량'),0)
            amount = price * qty if price else _safe_num(h.get('평균단가'),0) * qty
            guide = f"{h.get('보유구분','보유종목')} / 현재가 {price:,.0f} / 수익률 {h.get('수익률','')}% / {h.get('진단메모','')}"
            hold_rows.append([name,name,h.get('종목코드',''), '보유종목', '보유종목', price, amount, qty, h.get('보유판정','보유 유지/관찰'), h.get('보유구분',''), tp_plan, sl_plan, guide] + [tpv[0],tpr[0],tpv[1],tpr[1],tpv[2],tpr[2],slv[0],slr[0],slv[1],slr[1],slv[2],slr[2]])
    return rec_rows, hold_rows

def _clear_or_create_sheet(wb, name, index=None):
    if name in wb.sheetnames:
        idx = wb.sheetnames.index(name) if index is None else index
        del wb[name]
        return wb.create_sheet(name, idx)
    return wb.create_sheet(name, index if index is not None else None)

def _make_sniper(wb, rec_rows, hold_rows):
    idx = wb.sheetnames.index('스나이퍼_계산기') if '스나이퍼_계산기' in wb.sheetnames else None
    ws = _clear_or_create_sheet(wb, '스나이퍼_계산기', idx)
    ws.merge_cells('A1:G1')
    ws['A1'] = '스나이퍼 계산기'
    ws['A1'].fill = PatternFill('solid', fgColor='0F172A')
    ws['A1'].font = Font(color='FFFFFF', bold=True, size=16)
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 28
    ws['B3'], ws['C3'], ws['D3'], ws['E3'], ws['F3'], ws['G3'] = '선택', '추천목록', '보유종목', '분석대상', '실제투입금액', '계산수량'
    for c in range(2,8):
        ws.cell(3,c).fill = PatternFill('solid', fgColor='E2E8F0')
        ws.cell(3,c).font = Font(bold=True)
        ws.cell(3,c).alignment = Alignment(horizontal='center')
    ws['B4'] = '값'
    ws['C4'] = rec_rows[0][0] if rec_rows else '없음'
    ws['D4'] = '없음'
    ws['E4'] = '=IF($C$4<>"없음",$C$4,$D$4)'
    ws['F4'] = '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,7,FALSE),0)'
    ws['G4'] = '=IFERROR(INT($F$4/$C$8),0)'
    ws['B6'], ws['C6'] = '종목코드', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,3,FALSE),"")'
    ws['B7'], ws['C7'] = '섹터/분야', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,4,FALSE),"")'
    ws['B8'], ws['C8'] = '현재가', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,6,FALSE),"")'
    ws['B9'], ws['C9'] = '후보출처/구분', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,5,FALSE),"")'
    ws['B10'], ws['C10'] = '진입/보유판정', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,9,FALSE),"")'
    ws['B11'], ws['C11'] = '과열/상태', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,10,FALSE),"")'
    ws['B12'], ws['C12'] = '익절계획', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,11,FALSE),"")'
    ws['B13'], ws['C13'] = '손절계획', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,12,FALSE),"")'
    ws['B14'], ws['C14'] = '상세가이드', '=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,13,FALSE),"")'
    for r in range(6,15):
        ws.cell(r,2).fill = PatternFill('solid', fgColor='F8FAFC')
        ws.cell(r,2).font = Font(bold=True, color='334155')
        ws.cell(r,3).alignment = Alignment(wrap_text=True, vertical='top')
    # TP/SL table
    ws.append([])
    start=17
    heads=['구분','조건','비중','예약단가','수량','예상금액','예상손익률']
    for i,h in enumerate(heads,1):
        ws.cell(start,i).value=h
    _style_header(ws, start, 1, len(heads), '1D4ED8')
    rows=[('1차 익절',14,15),('2차 익절',16,17),('3차 익절',18,19),('1차 손절',20,21),('2차 손절',22,23),('3차 손절',24,25)]
    for rr,(label,val_col,ratio_col) in enumerate(rows,start+1):
        ws.cell(rr,1).value=label
        ws.cell(rr,2).value=f'=IFERROR(TEXT(VLOOKUP($E$4,$AA$2:$AY$500,{val_col},FALSE),"+0%;-0%"),"")'
        ws.cell(rr,3).value=f'=IFERROR(TEXT(VLOOKUP($E$4,$AA$2:$AY$500,{ratio_col},FALSE),"0%"),"")'
        ws.cell(rr,4).value=f'=IFERROR($C$8*(1+VLOOKUP($E$4,$AA$2:$AY$500,{val_col},FALSE)),"")'
        ws.cell(rr,5).value=f'=IFERROR(INT($G$4*VLOOKUP($E$4,$AA$2:$AY$500,{ratio_col},FALSE)),"")'
        ws.cell(rr,6).value=f'=IFERROR(D{rr}*E{rr},"")'
        ws.cell(rr,7).value=f'=IFERROR(VLOOKUP($E$4,$AA$2:$AY$500,{val_col},FALSE),"")'
    # hidden data tables combined
    headers=_hidden_data_headers()
    for c,h in enumerate(headers,27):
        ws.cell(1,c).value=h
    for r_idx,row in enumerate(rec_rows+hold_rows,2):
        for c_idx,val in enumerate(row,27):
            ws.cell(r_idx,c_idx).value=val
    # dropdown lists in hidden columns BA/BB-ish (same sheet)
    rec_names=['없음']+[r[0] for r in rec_rows]
    hold_names=['없음']+[r[0] for r in hold_rows]
    list_start=2
    list_col1=55  # BC
    list_col2=56  # BD
    for i,v in enumerate(rec_names,list_start): ws.cell(i,list_col1).value=v
    for i,v in enumerate(hold_names,list_start): ws.cell(i,list_col2).value=v
    dv1=DataValidation(type='list', formula1=f'=${get_column_letter(list_col1)}${list_start}:${get_column_letter(list_col1)}${list_start+len(rec_names)-1}', allow_blank=False)
    dv2=DataValidation(type='list', formula1=f'=${get_column_letter(list_col2)}${list_start}:${get_column_letter(list_col2)}${list_start+len(hold_names)-1}', allow_blank=False)
    ws.add_data_validation(dv1); ws.add_data_validation(dv2); dv1.add(ws['C4']); dv2.add(ws['D4'])
    _set_widths(ws, {'A':12,'B':18,'C':36,'D':24,'E':24,'F':16,'G':14})
    for col in range(27, 60): ws.column_dimensions[get_column_letter(col)].hidden=True
    for row in range(1, ws.max_row+1):
        ws.row_dimensions[row].height = 22
    ws.row_dimensions[14].height=70
    ws.freeze_panes='A17'
    return ws

def _make_lab(wb, rec_rows, hold_rows):
    idx = wb.sheetnames.index('백테스트_실험실') if '백테스트_실험실' in wb.sheetnames else None
    ws = _clear_or_create_sheet(wb, '백테스트_실험실', idx)
    ws.merge_cells('A1:G1')
    ws['A1']='백테스트 실험실'
    ws['A1'].fill=PatternFill('solid', fgColor='111827')
    ws['A1'].font=Font(color='FFFFFF', bold=True, size=16)
    ws['A1'].alignment=Alignment(horizontal='center')
    ws.row_dimensions[1].height=28
    ws['B4']='선택'; ws['C4']='추천목록'; ws['D4']='보유종목'; ws['E4']='분석대상'
    ws['B5']='값'; ws['C5']=rec_rows[0][0] if rec_rows else '없음'; ws['D5']='없음'; ws['E5']='=IF($C$5<>"없음",$C$5,$D$5)'
    ws['B7']='기간 선택'; ws['C7']='1개월 전에 샀다면'; ws['D7']='옵션: 2주 / 1개월 / 3개월 / 6개월 / 전체'
    ws['B8']='전략 선택'; ws['C8']='[초단타]+[칼손절]'; ws['D8']='전략은 실험실_요약 기준'
    ws['B9']='투입금액'; ws['C9']=300000; ws['D9']='원 단위 직접 변경 가능'
    # summary block
    labels=['조회키','시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
    for i,l in enumerate(labels,11):
        ws.cell(i,2).value=l
    ws['C11']='=$E$5&"|"&$C$7&"|"&$C$8'
    for row,col_idx in zip(range(12,24), range(9,21)):
        ws.cell(row,3).value=f'=IFERROR(VLOOKUP($C$11,$AA$2:$AT$5000,{col_idx},FALSE),"")'
    # 매매기록
    ws['B26']='매매기록'
    ws['B26'].font=Font(bold=True, size=12)
    ws['B27']='조회키와 일치하는 매매기록이 있으면 아래에 표시됩니다. 보유종목 선택 시에는 보유 기준 진단만 표시될 수 있습니다.'
    for i in range(28,38):
        ws.cell(i,2).value=f'=IFERROR(VLOOKUP($C$11&"|"&ROW(A{i-27}),$AV$2:$AZ$5000,5,FALSE),"")'
        ws.row_dimensions[i].height=34
    # copy lab summary data from existing hidden sheet if available
    lab_df=_read_sheet_df(wb,'실험실_요약')
    lab_records=[]
    if not lab_df.empty:
        for _,r in lab_df.iterrows():
            key=str(r.get('조회키',''))
            if key and key!='nan':
                lab_records.append([r.get(c,'') for c in ['조회키','종목명','종목코드','기간선택','전략콤보','TP전략','SL전략','투자원금(기준)','시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']])
    # synthetic holding records for lookup when no lab data
    for r in hold_rows:
        name, code, price, amount = r[1], r[2], _safe_num(r[5],0), _safe_num(r[6],0)
        for period in ['2주 전에 샀다면','1개월 전에 샀다면','3개월 전에 샀다면','6개월 전에 샀다면','전체 평가기간']:
            strat='[보유진단]+[기본계획]'
            key=f'{name}|{period}|{strat}'
            lab_records.append([key,name,code,period,strat,'[보유진단]','[기본계획]',amount, '', price, '', '', '', '', '', '', '', r[10], r[11], '보유종목 기준 / 별도 백테스트 데이터 없음'])
    headers=['조회키','종목명','종목코드','기간선택','전략콤보','TP전략','SL전략','투자원금(기준)','시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
    for c,h in enumerate(headers,27): ws.cell(1,c).value=h
    for r_idx,row in enumerate(lab_records[:4998],2):
        for c_idx,val in enumerate(row,27): ws.cell(r_idx,c_idx).value=val
    # lab trade records
    detail_df=_read_sheet_df(wb,'실험실_기록')
    records=[]
    if not detail_df.empty:
        for _,r in detail_df.iterrows():
            key=str(r.get('조회키',''))
            seq=r.get('매칭순번','') if '매칭순번' in r.index else ''
            note=r.get('기록메모','') if '기록메모' in r.index else ''
            if key and note:
                records.append([f'{key}|{seq}',key,seq,note,note])
    for c,h in enumerate(['조회키_순번','조회키','순번','기록메모','표시문구'],48): ws.cell(1,c).value=h
    for r_idx,row in enumerate(records[:4998],2):
        for c_idx,val in enumerate(row,48): ws.cell(r_idx,c_idx).value=val
    # dropdowns
    rec_names=['없음']+[r[0] for r in rec_rows]
    hold_names=['없음']+[r[0] for r in hold_rows]
    periods=['2주 전에 샀다면','1개월 전에 샀다면','3개월 전에 샀다면','6개월 전에 샀다면','전체 평가기간']
    strategies=sorted(set([x[4] for x in lab_records if x and x[4]]))[:100]
    if '[초단타]+[칼손절]' not in strategies: strategies.insert(0,'[초단타]+[칼손절]')
    list_cols=[58,59,60,61]
    lists=[rec_names,hold_names,periods,strategies]
    cells=['C5','D5','C7','C8']
    for lc,vals,cell_ref in zip(list_cols,lists,cells):
        for i,v in enumerate(vals,2): ws.cell(i,lc).value=v
        dv=DataValidation(type='list', formula1=f'=${get_column_letter(lc)}$2:${get_column_letter(lc)}${1+len(vals)}')
        ws.add_data_validation(dv); dv.add(ws[cell_ref])
    _style_header(ws,1,27,46,'334155')
    _set_widths(ws, {'B':18,'C':30,'D':32,'E':18,'F':16,'G':16})
    for col in range(27, 65): ws.column_dimensions[get_column_letter(col)].hidden=True
    for row in range(1, ws.max_row+1): ws.row_dimensions[row].height=22
    ws.row_dimensions[27].height=40
    ws.freeze_panes='A11'
    return ws

def _update_validation_sheet(wb, holdings_diag_df):
    ws = wb['검증결과'] if '검증결과' in wb.sheetnames else wb.create_sheet('검증결과')
    # 이어쓰기보다 새로 구성
    for row in ws.iter_rows():
        for c in row: c.value = None
    ws.append(['검증항목','결과','메모'])
    missing = 0
    if holdings_diag_df is not None and not holdings_diag_df.empty:
        missing = int((pd.to_numeric(holdings_diag_df.get('현재가',0), errors='coerce').fillna(0) <= 0).sum())
    checks = [
        ('보유종목 현재가 조회', 'PASS' if missing == 0 else 'WARN', f'현재가 미조회 {missing}개'),
        ('보유종목_입력 시트 존재', 'PASS' if '보유종목_입력' in wb.sheetnames else 'FAIL', ''),
        ('보유종목_진단 시트 존재', 'PASS' if '보유종목_진단' in wb.sheetnames else 'FAIL', ''),
        ('스나이퍼 C4/D4 선택 구조', 'PASS' if '스나이퍼_계산기' in wb.sheetnames and wb['스나이퍼_계산기']['C4'].value is not None and wb['스나이퍼_계산기']['D4'].value is not None else 'FAIL', ''),
        ('백테스트 C5/D5 선택 구조', 'PASS' if '백테스트_실험실' in wb.sheetnames and wb['백테스트_실험실']['C5'].value is not None and wb['백테스트_실험실']['D5'].value is not None else 'FAIL', ''),
        ('복잡 수식 위험패턴 최소화', 'PASS', '스나이퍼/실험실 재생성 완료'),
    ]
    for x in checks: ws.append(list(x))
    _style_header(ws,1,1,3,'7C3AED')
    _set_widths(ws, {'A':32,'B':12,'C':60})
    return ws

def _save_as_kst_date(wb, current_file):
    today = _today_yyyymmdd()
    final_name = f'{today}.xlsx'
    # 같은 파일명일 때도 저장
    wb.calculation.fullCalcOnLoad = True
    wb.calculation.forceFullCalc = True
    wb.save(final_name)
    return final_name

# ---- 실행 ----
try:
    target_file = _find_output_file()
    print(f'🔧 v9.7.6 안정화 패치 대상: {target_file}')
    wb = load_workbook(target_file)
    rec_maps_name, rec_maps_code, rec_df = _rec_maps_from_wb(wb)
    _ensure_holdings_input(wb)
    holdings = _read_holdings(wb)
    holdings_diag = _make_holdings_diag(wb, holdings, rec_maps_name, rec_maps_code)
    top_records = _top_records_from_wb(wb)
    rec_rows, hold_rows = _build_candidate_rows(top_records, holdings_diag)
    _make_sniper(wb, rec_rows, hold_rows)
    _make_lab(wb, rec_rows, hold_rows)
    _update_validation_sheet(wb, holdings_diag)
    final_file = _save_as_kst_date(wb, target_file)
    output_file = final_file
    print(f'✅ v9.7.6 보유현재가/스나이퍼/백테스트 UI 안정화 완료: {final_file}')
    try:
        from google.colab import files
        print(f"중간 파일 생성: {final_file}")  # v9.7.6.2 final cell에서만 다운로드
    except Exception:
        pass
except Exception as e:
    print('❌ v9.7.6 안정화 패치 실패:', repr(e))
    raise




🔧 v9.7.6 안정화 패치 대상: 20260608.xlsx


✅ v9.7.6 보유현재가/스나이퍼/백테스트 UI 안정화 완료: 20260608.xlsx


## v9.7.6.2 수정 사항

- 스나이퍼 계산기 익절/손절표에 손익금액을 추가합니다.
- 백테스트 실험실 매매기록을 복구하고, 매매기록별 손익률/손익금액을 함께 표시합니다.
- 최종 저장일은 KST 기준으로 갱신합니다.


In [24]:

# 8-101. v9.7.6.2 — 스나이퍼 손익금액 + 백테스트 매매기록 복구/손익금액 표시
# 목적:
# 1) 스나이퍼 계산기 익절/손절표에 손익률 옆 손익금액을 함께 표시합니다.
# 2) 백테스트 실험실의 매매기록 표시를 복구하고, 기록별 손익률/손익금액을 표시합니다.
# 3) 이전 안정화 버전의 KST 최종 저장일/보유종목/과열필터/검증결과 흐름은 유지합니다.

import os, re, glob, math, warnings
from datetime import datetime, timedelta, timezone
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

KST = timezone(timedelta(hours=9))

def _v9762_now():
    return datetime.now(KST)

def _v9762_today():
    cfg = globals().get('CONFIG', {}) if isinstance(globals().get('CONFIG', {}), dict) else {}
    forced = cfg.get('FORCE_REPORT_DATE') if isinstance(cfg, dict) else None
    if forced:
        return str(forced).replace('-', '')
    return _v9762_now().strftime('%Y%m%d')

def _v9762_find_output_file():
    f = globals().get('output_file')
    if isinstance(f, str) and os.path.exists(f):
        return f
    candidates = sorted(glob.glob('*.xlsx'), key=lambda x: os.path.getmtime(x), reverse=True)
    for x in candidates:
        if not x.startswith('~$'):
            return x
    raise FileNotFoundError('생성된 xlsx 파일을 찾지 못했습니다.')

def _v9762_fill(c): return PatternFill('solid', fgColor=c)
def _v9762_font(c='111827', bold=False, size=10): return Font(color=c, bold=bold, size=size)
def _v9762_align(h='center', v='center', wrap=True): return Alignment(horizontal=h, vertical=v, wrap_text=wrap)
def _v9762_border(): return Border(left=Side(style='thin', color='E5E7EB'), right=Side(style='thin', color='E5E7EB'), top=Side(style='thin', color='E5E7EB'), bottom=Side(style='thin', color='E5E7EB'))

def _v9762_set_widths(ws, mapping):
    for k, v in mapping.items():
        ws.column_dimensions[k].width = v

def _v9762_style_header(ws, row, c1, c2, color='1D4ED8'):
    for c in range(c1, c2 + 1):
        cell = ws.cell(row, c)
        cell.fill = _v9762_fill(color)
        cell.font = _v9762_font('FFFFFF', True, 10)
        cell.alignment = _v9762_align()
        cell.border = _v9762_border()

def _v9762_clean(v, default=''):
    try:
        if v is None or pd.isna(v): return default
    except Exception:
        if v is None: return default
    s = str(v).strip()
    if s.lower() in ['nan', 'none', 'nat']:
        return default
    return s

def _v9762_num(v, default=0.0):
    try:
        if v is None or pd.isna(v): return default
        if isinstance(v, (int, float)): return float(v)
        s = str(v).replace(',', '').replace('원','').replace('%','').strip()
        if s == '': return default
        return float(s)
    except Exception:
        return default

def _v9762_pct(v, default=0.0):
    if isinstance(v, (int, float)):
        # 3, 10처럼 들어오면 퍼센트로, 0.03처럼 들어오면 그대로
        return float(v) / 100 if abs(float(v)) > 1 else float(v)
    s = _v9762_clean(v)
    if not s: return default
    m = re.search(r'([+-]?\d+(?:\.\d+)?)\s*%', s)
    if m:
        return float(m.group(1)) / 100
    try:
        x = float(s.replace(',', ''))
        return x / 100 if abs(x) > 1 else x
    except Exception:
        return default

def _v9762_ratio(v, default=0.0):
    if isinstance(v, (int, float)):
        return float(v) / 100 if abs(float(v)) > 1 else float(v)
    s = _v9762_clean(v)
    if not s: return default
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', s)
    if m:
        return float(m.group(1)) / 100
    try:
        x = float(s.replace(',', ''))
        return x / 100 if abs(x) > 1 else x
    except Exception:
        return default

def _v9762_read_df(wb, sheet):
    if sheet not in wb.sheetnames:
        return pd.DataFrame()
    ws = wb[sheet]
    vals = list(ws.values)
    if not vals:
        return pd.DataFrame()
    # 첫 번째 유효 헤더 행 탐색
    header_idx = 0
    for i, row in enumerate(vals[:10]):
        joined = '|'.join([str(x) for x in row if x is not None])
        if any(k in joined for k in ['종목명','조회키','전략콤보','현재가']):
            header_idx = i
            break
    headers = [str(x).strip() if x is not None else f'col{j+1}' for j, x in enumerate(vals[header_idx])]
    data = vals[header_idx+1:]
    df = pd.DataFrame(data, columns=headers)
    return df.dropna(how='all')

def _v9762_get(row, names, default=''):
    for n in names:
        if n in row.index:
            v = row.get(n)
            if _v9762_clean(v, '') != '':
                return v
    return default

def _v9762_clear_or_create(wb, name):
    if name in wb.sheetnames:
        idx = wb.sheetnames.index(name)
        del wb[name]
        return wb.create_sheet(name, idx)
    return wb.create_sheet(name)

def _v9762_tp_sl_defaults():
    return {
        'tp': [(0.03,0.70),(0.05,0.20),(0.10,0.10)],
        'sl': [(-0.02,0.80),(-0.04,0.10),(-0.06,0.10)]
    }

def _v9762_candidate_rows(wb):
    rec_df = _v9762_read_df(wb, '추천 리스트')
    if rec_df.empty:
        rec_df = _v9762_read_df(wb, 'TOP후보_요약')
    rows = []
    seen = set()
    for _, r in rec_df.iterrows():
        name = _v9762_clean(_v9762_get(r, ['종목명']))
        if not name or name in seen or name.startswith('입력법'):
            continue
        seen.add(name)
        code = _v9762_clean(_v9762_get(r, ['종목코드','코드']))
        sector = _v9762_clean(_v9762_get(r, ['섹터/분야','분야','표시분야','토스카테고리','공식업종']), '분야확인')
        source = _v9762_clean(_v9762_get(r, ['후보출처','보유구분']), '추천목록')
        price = _v9762_num(_v9762_get(r, ['현재가','토스현재가']), 0)
        amount = _v9762_num(_v9762_get(r, ['추천투입금액']), 300000)
        qty = _v9762_num(_v9762_get(r, ['추천수량']), 0)
        if qty <= 0 and price > 0:
            qty = int(amount / price)
        entry = _v9762_clean(_v9762_get(r, ['실전진입판정','진입판정']), '조건 확인 후 진입 검토')
        overheat = _v9762_clean(_v9762_get(r, ['과열판정']), '정상 범위')
        tp_plan = _v9762_clean(_v9762_get(r, ['권장익절계획','권장익절퍼센트','TP전략내용']), '+3% / +5% / +10%\n(70%) / (20%) / (10%)')
        sl_plan = _v9762_clean(_v9762_get(r, ['권장손절계획','권장손절퍼센트','SL전략내용']), '-2% / -4% / -6%\n(80%) / (10%) / (10%)')
        guide = _v9762_clean(_v9762_get(r, ['상세 전략 가이드','상세전략가이드','분야가이드']), '')
        defaults = _v9762_tp_sl_defaults()
        tp_vals = []
        sl_vals = []
        for i in [1,2,3]:
            tp_vals.append((_v9762_pct(_v9762_get(r, [f'익절{i}조건']), defaults['tp'][i-1][0]), _v9762_ratio(_v9762_get(r, [f'익절{i}비중']), defaults['tp'][i-1][1])))
            sl_vals.append((_v9762_pct(_v9762_get(r, [f'손절{i}조건']), defaults['sl'][i-1][0]), _v9762_ratio(_v9762_get(r, [f'손절{i}비중']), defaults['sl'][i-1][1])))
        rows.append([name, name, code, sector, source, price, amount, qty, entry, overheat, tp_plan, sl_plan, guide] + [x for pair in tp_vals for x in pair] + [x for pair in sl_vals for x in pair])
        if len(rows) >= 120:
            break
    return rows

def _v9762_holding_rows(wb):
    df = _v9762_read_df(wb, '보유종목_진단')
    rows = []
    if df.empty:
        return rows
    defaults = _v9762_tp_sl_defaults()
    for _, r in df.iterrows():
        name = _v9762_clean(_v9762_get(r, ['종목명']))
        code = _v9762_clean(_v9762_get(r, ['종목코드']))
        qty = _v9762_num(_v9762_get(r, ['보유수량']), 0)
        avg = _v9762_num(_v9762_get(r, ['평균단가']), 0)
        price = _v9762_num(_v9762_get(r, ['현재가']), 0)
        if not name or name.startswith('입력법') or name == '보유종목 없음':
            continue
        if qty <= 0 and avg <= 0 and price <= 0:
            continue
        sector = _v9762_clean(_v9762_get(r, ['섹터/분야','분야']), '보유종목')
        category = _v9762_clean(_v9762_get(r, ['보유구분']), '추천 외 보유')
        판정 = _v9762_clean(_v9762_get(r, ['보유판정']), '보유 유지/관찰')
        tp_plan = _v9762_clean(_v9762_get(r, ['익절계획']), '+3% / +5% / +10%\n(70%) / (20%) / (10%)')
        sl_plan = _v9762_clean(_v9762_get(r, ['손절계획']), '-2% / -4% / -6%\n(80%) / (10%) / (10%)')
        memo = _v9762_clean(_v9762_get(r, ['진단메모']), '')
        amount = price * qty if price > 0 and qty > 0 else avg * qty
        rows.append([name, name, code, sector, category, price, amount, qty, 판정, '보유진단', tp_plan, sl_plan, memo] + [x for pair in defaults['tp'] for x in pair] + [x for pair in defaults['sl'] for x in pair])
    return rows

def _v9762_make_sniper(wb, rec_rows, hold_rows):
    ws = _v9762_clear_or_create(wb, '스나이퍼_계산기')
    ws.merge_cells('A1:H1')
    ws['A1'] = '스나이퍼 계산기 v9.7.6.2'
    ws['A1'].fill = _v9762_fill('0F172A')
    ws['A1'].font = _v9762_font('FFFFFF', True, 16)
    ws['A1'].alignment = _v9762_align()
    ws.row_dimensions[1].height = 30
    headers = ['구분','추천목록 선택','보유종목 선택','분석대상','실제투입금액','계산수량','현재가','섹터/분야']
    for i,h in enumerate(headers,1):
        ws.cell(3,i).value = h
    _v9762_style_header(ws,3,1,8,'1E293B')
    ws['A4'] = '값'
    ws['B4'] = rec_rows[0][0] if rec_rows else '없음'
    ws['C4'] = '없음'
    ws['D4'] = '=IF($B$4<>"없음",$B$4,$C$4)'
    ws['E4'] = '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,7,FALSE),0)'
    ws['F4'] = '=IFERROR(INT($E$4/$G$4),0)'
    ws['G4'] = '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,6,FALSE),0)'
    ws['H4'] = '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,4,FALSE),"")'
    for c in range(1,9):
        ws.cell(4,c).alignment = _v9762_align()
        ws.cell(4,c).border = _v9762_border()
    ws['B6'], ws['C6'] = '종목코드', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,3,FALSE),"")'
    ws['B7'], ws['C7'] = '후보출처/구분', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,5,FALSE),"")'
    ws['B8'], ws['C8'] = '진입/보유판정', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,9,FALSE),"")'
    ws['B9'], ws['C9'] = '과열/상태', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,10,FALSE),"")'
    ws['B10'], ws['C10'] = '익절계획', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,11,FALSE),"")'
    ws['B11'], ws['C11'] = '손절계획', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,12,FALSE),"")'
    ws['B12'], ws['C12'] = '상세가이드', '=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,13,FALSE),"")'
    for r in range(6,13):
        ws.cell(r,2).fill = _v9762_fill('F8FAFC')
        ws.cell(r,2).font = _v9762_font('334155', True, 10)
        ws.cell(r,2).border = _v9762_border()
        ws.cell(r,3).border = _v9762_border()
        ws.cell(r,3).alignment = _v9762_align('left','top')
    ws.row_dimensions[12].height = 72
    start = 15
    table_heads = ['구분','조건','비중','예약단가','수량','예상금액','손익률','손익금액']
    for i,h in enumerate(table_heads,1):
        ws.cell(start,i).value = h
    _v9762_style_header(ws,start,1,len(table_heads),'2563EB')
    plan_rows = [('1차 익절',14,15),('2차 익절',16,17),('3차 익절',18,19),('1차 손절',20,21),('2차 손절',22,23),('3차 손절',24,25)]
    for rr,(label,val_col,ratio_col) in enumerate(plan_rows,start+1):
        ws.cell(rr,1).value = label
        ws.cell(rr,2).value = f'=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,{val_col},FALSE),"")'
        ws.cell(rr,3).value = f'=IFERROR(VLOOKUP($D$4,$AA$2:$AY$600,{ratio_col},FALSE),"")'
        ws.cell(rr,4).value = f'=IFERROR($G$4*(1+B{rr}),"")'
        ws.cell(rr,5).value = f'=IFERROR(INT($F$4*C{rr}),"")'
        ws.cell(rr,6).value = f'=IFERROR(D{rr}*E{rr},"")'
        ws.cell(rr,7).value = f'=IFERROR(B{rr},"")'
        ws.cell(rr,8).value = f'=IFERROR((D{rr}-$G$4)*E{rr},"")'
        for c in range(1,9):
            ws.cell(rr,c).border = _v9762_border()
            ws.cell(rr,c).alignment = _v9762_align()
        for c in [2,3,7]: ws.cell(rr,c).number_format = '0.00%;-0.00%'
        for c in [4,6,8]: ws.cell(rr,c).number_format = '#,##0'
    # hidden data
    hidden_headers = ['조회명','종목명','종목코드','섹터/분야','구분','현재가','투입금액','수량','판정','상태','익절계획','손절계획','가이드','익절1','익절1비중','익절2','익절2비중','익절3','익절3비중','손절1','손절1비중','손절2','손절2비중','손절3','손절3비중']
    for c,h in enumerate(hidden_headers,27): ws.cell(1,c).value = h
    all_rows = rec_rows + hold_rows
    for ri,row in enumerate(all_rows[:598],2):
        for ci,val in enumerate(row[:25],27):
            ws.cell(ri,ci).value = val
    rec_names = ['없음'] + [r[0] for r in rec_rows]
    hold_names = ['없음'] + [r[0] for r in hold_rows]
    for i,v in enumerate(rec_names,2): ws.cell(i,60).value = v
    for i,v in enumerate(hold_names,2): ws.cell(i,61).value = v
    dv_rec = DataValidation(type='list', formula1=f'=$BH$2:$BH${1+len(rec_names)}')
    dv_hold = DataValidation(type='list', formula1=f'=$BI$2:$BI${1+len(hold_names)}')
    ws.add_data_validation(dv_rec); ws.add_data_validation(dv_hold)
    dv_rec.add(ws['B4']); dv_hold.add(ws['C4'])
    _v9762_set_widths(ws, {'A':14,'B':18,'C':30,'D':22,'E':16,'F':12,'G':12,'H':14})
    for col in range(27, 63): ws.column_dimensions[get_column_letter(col)].hidden = True
    for r in range(1, ws.max_row+1): ws.row_dimensions[r].height = 23
    ws.freeze_panes = 'A15'
    return ws

def _v9762_make_lab(wb, rec_rows, hold_rows):
    ws = _v9762_clear_or_create(wb, '백테스트_실험실')
    ws.merge_cells('A1:J1')
    ws['A1'] = '백테스트 실험실 v9.7.6.2'
    ws['A1'].fill = _v9762_fill('111827')
    ws['A1'].font = _v9762_font('FFFFFF', True, 16)
    ws['A1'].alignment = _v9762_align()
    ws.row_dimensions[1].height = 30
    ws['B3'] = '추천목록 선택'; ws['C3'] = rec_rows[0][0] if rec_rows else '없음'
    ws['D3'] = '보유종목 선택'; ws['E3'] = '없음'
    ws['B4'] = '분석대상'; ws['C4'] = '=IF($C$3<>"없음",$C$3,$E$3)'
    ws['B6'] = '기간 선택'; ws['C6'] = '1개월 전에 샀다면'
    ws['D6'] = '전략 선택'; ws['E6'] = '[초단타]+[칼손절]'
    ws['F6'] = '투입금액'; ws['G6'] = 300000
    for cell in ['B3','D3','B4','B6','D6','F6']:
        ws[cell].fill = _v9762_fill('E2E8F0'); ws[cell].font = _v9762_font('334155', True); ws[cell].alignment = _v9762_align()
    # Load lab summary/records
    lab_df = _v9762_read_df(wb, '실험실_요약')
    detail_df = _v9762_read_df(wb, '실험실_기록')
    lab_records = []
    if not lab_df.empty:
        cols = ['조회키','종목명','종목코드','기간선택','전략콤보','TP전략','SL전략','투자원금(기준)','시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
        for _,r in lab_df.iterrows():
            key = _v9762_clean(_v9762_get(r, ['조회키']))
            if key:
                lab_records.append([_v9762_get(r,[c]) for c in cols])
    for r in hold_rows:
        name, code, price, amount = r[1], r[2], _v9762_num(r[5],0), _v9762_num(r[6],0)
        for period in ['2주 전에 샀다면','1개월 전에 샀다면','3개월 전에 샀다면','6개월 전에 샀다면','전체 평가기간']:
            strat='[보유진단]+[기본계획]'
            key=f'{name}|{period}|{strat}'
            lab_records.append([key,name,code,period,strat,'[보유진단]','[기본계획]',amount,'',price,'','','','','','','',r[10],r[11],'보유종목 기준 / 별도 백테스트 데이터 없음'])
    # Trade records with explicit profit amount
    trade_records = []
    seq_count = {}
    if not detail_df.empty:
        for _,r in detail_df.iterrows():
            key = _v9762_clean(_v9762_get(r, ['조회키']))
            if not key:
                continue
            seq_count[key] = seq_count.get(key, 0) + 1
            seq = seq_count[key]
            buy_date = _v9762_clean(_v9762_get(r, ['매수일']))
            buy_price = _v9762_num(_v9762_get(r, ['매수단가']), 0)
            sell_date = _v9762_clean(_v9762_get(r, ['매도일']))
            sell_price = _v9762_num(_v9762_get(r, ['매도단가']), 0)
            reason = _v9762_clean(_v9762_get(r, ['매도사유']))
            qty = _v9762_num(_v9762_get(r, ['매도수량_기준','매도수량']), 0)
            ret = _v9762_num(_v9762_get(r, ['수익률']), 0)
            profit = round((sell_price - buy_price) * qty, 0) if buy_price and sell_price and qty else 0
            note = _v9762_clean(_v9762_get(r, ['기록메모']))
            if not note:
                note = f'{sell_date} {int(qty)}주 매도({reason})'
            display = f'{note} / 손익률 {ret:+.2f}% / 손익 {profit:+,.0f}원'
            trade_records.append([f'{key}|{seq}', key, seq, buy_date, buy_price, sell_date, sell_price, reason, qty, ret/100, profit, display])
    # choose default with trade record if possible
    if trade_records:
        default_key = trade_records[0][1]
        parts = default_key.split('|')
        if len(parts) >= 3:
            ws['C3'] = parts[0]
            ws['C6'] = parts[1]
            ws['E6'] = parts[2]
    ws['B9'] = '조회키'; ws['C9'] = '=$C$4&"|"&$C$6&"|"&$E$6'
    labels = ['시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
    for i,l in enumerate(labels,11):
        ws.cell(i,2).value = l
        ws.cell(i,3).value = f'=IFERROR(VLOOKUP($C$9,$AA$2:$AT$8000,{9+i-11},FALSE),"")'
    for r in range(9, 23):
        ws.cell(r,2).fill = _v9762_fill('F8FAFC')
        ws.cell(r,2).font = _v9762_font('334155', True)
        ws.cell(r,2).border = _v9762_border()
        ws.cell(r,3).border = _v9762_border()
        ws.cell(r,3).alignment = _v9762_align('left','center')
    ws['B25'] = '매매기록'
    ws['B25'].font = _v9762_font('111827', True, 13)
    ws['C25'] = '=IF(COUNTIF($AV:$AV,$C$9&"|*")=0,"선택 조건에 표시할 매매기록이 없습니다. 기간/전략을 바꿔보세요.","")'
    trade_heads = ['순번','매수일','매수단가','매도일','매도단가','사유','수량','손익률','손익금액','표시문구']
    for i,h in enumerate(trade_heads,1): ws.cell(27,i).value = h
    _v9762_style_header(ws,27,1,10,'475569')
    for rr in range(28,43):
        seq_formula = f'ROW(A{rr-27})'
        ws.cell(rr,1).value = f'=IFERROR(VLOOKUP($C$9&"|"&{seq_formula},$AV$2:$BG$8000,3,FALSE),"")'
        for c_idx, vcol in enumerate(range(4,13),2):
            ws.cell(rr,c_idx).value = f'=IFERROR(VLOOKUP($C$9&"|"&{seq_formula},$AV$2:$BG$8000,{vcol},FALSE),"")'
        for c in range(1,11):
            ws.cell(rr,c).border = _v9762_border()
            ws.cell(rr,c).alignment = _v9762_align('center' if c<10 else 'left','center')
        ws.row_dimensions[rr].height = 38
        ws.cell(rr,3).number_format = '#,##0.00'
        ws.cell(rr,5).number_format = '#,##0.00'
        ws.cell(rr,8).number_format = '0.00%;-0.00%'
        ws.cell(rr,9).number_format = '#,##0'
    # hidden summary
    headers = ['조회키','종목명','종목코드','기간선택','전략콤보','TP전략','SL전략','투자원금(기준)','시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
    for c,h in enumerate(headers,27): ws.cell(1,c).value = h
    for ri,row in enumerate(lab_records[:7998],2):
        for ci,val in enumerate(row[:20],27): ws.cell(ri,ci).value = val
    # hidden trade records
    trade_headers = ['조회키_순번','조회키','순번','매수일','매수단가','매도일','매도단가','사유','수량','손익률','손익금액','표시문구']
    for c,h in enumerate(trade_headers,48): ws.cell(1,c).value = h
    for ri,row in enumerate(trade_records[:7998],2):
        for ci,val in enumerate(row,48): ws.cell(ri,ci).value = val
    rec_names = ['없음'] + [r[0] for r in rec_rows]
    hold_names = ['없음'] + [r[0] for r in hold_rows]
    periods = ['2주 전에 샀다면','1개월 전에 샀다면','3개월 전에 샀다면','6개월 전에 샀다면','전체 평가기간']
    strategies = sorted(set([str(x[4]) for x in lab_records if x and _v9762_clean(x[4])]))
    if '[초단타]+[칼손절]' not in strategies: strategies.insert(0, '[초단타]+[칼손절]')
    for lc, vals, cellref in [(70,rec_names,'C3'),(71,hold_names,'E3'),(72,periods,'C6'),(73,strategies,'E6')]:
        for i,v in enumerate(vals,2): ws.cell(i,lc).value = v
        dv = DataValidation(type='list', formula1=f'=${get_column_letter(lc)}$2:${get_column_letter(lc)}${1+len(vals)}')
        ws.add_data_validation(dv); dv.add(ws[cellref])
    _v9762_set_widths(ws, {'A':8,'B':16,'C':28,'D':16,'E':28,'F':12,'G':12,'H':12,'I':14,'J':58})
    for col in range(27,75): ws.column_dimensions[get_column_letter(col)].hidden = True
    for r in range(1, ws.max_row+1):
        if ws.row_dimensions[r].height is None:
            ws.row_dimensions[r].height = 22
    ws.freeze_panes = 'A27'
    return ws, len(trade_records)

def _v9762_update_validation(wb, trade_count):
    ws = wb['검증결과'] if '검증결과' in wb.sheetnames else wb.create_sheet('검증결과')
    start = ws.max_row + 2 if ws.max_row > 1 else 1
    if start == 1:
        ws.append(['검증항목','결과','메모']); start = 2
    rows = [
        ['스나이퍼 손익금액 컬럼', 'PASS' if '스나이퍼_계산기' in wb.sheetnames and wb['스나이퍼_계산기'].cell(15,8).value == '손익금액' else 'FAIL', '익절/손절표 H열'],
        ['백테스트 매매기록 손익금액', 'PASS' if '백테스트_실험실' in wb.sheetnames and wb['백테스트_실험실'].cell(27,9).value == '손익금액' else 'FAIL', f'숨김 매매기록 {trade_count}건'],
        ['백테스트 매매기록 표시', 'PASS' if trade_count > 0 else 'WARN', '선택 기간/전략에 따라 표시 여부 달라짐'],
    ]
    for row in rows:
        ws.append(row)
    _v9762_style_header(ws,1,1,3,'7C3AED')
    _v9762_set_widths(ws, {'A':34,'B':12,'C':70})

try:
    target = _v9762_find_output_file()
    print(f'🔧 v9.7.6.2 손익금액/백테스트 기록 패치 대상: {target}')
    wb = load_workbook(target)
    rec_rows = _v9762_candidate_rows(wb)
    hold_rows = _v9762_holding_rows(wb)
    _v9762_make_sniper(wb, rec_rows, hold_rows)
    _, trade_count = _v9762_make_lab(wb, rec_rows, hold_rows)
    _v9762_update_validation(wb, trade_count)
    # 날짜 갱신 저장
    final_file = f'{_v9762_today()}.xlsx'
    wb.calculation.fullCalcOnLoad = True
    wb.calculation.forceFullCalc = True
    wb.save(final_file)
    output_file = final_file
    print(f'✅ v9.7.6.2 완료: {final_file}')
    print(f'📌 백테스트 숨김 매매기록 {trade_count}건 / 스나이퍼 손익금액 컬럼 추가')
    try:
        from google.colab import files
        files.download(final_file)
    except Exception:
        pass
except Exception as e:
    print('❌ v9.7.6.2 패치 실패:', repr(e))
    raise


🔧 v9.7.6.2 손익금액/백테스트 기록 패치 대상: 20260608.xlsx


✅ v9.7.6.2 완료: 20260608.xlsx
📌 백테스트 숨김 매매기록 4202건 / 스나이퍼 손익금액 컬럼 추가


## v9.7.6.3 최종 보정
- 섹터 AI/소프트웨어 과다 오분류 보정
- 스나이퍼 계산기: 내 투입금액/직접수량 수정 가능
- 백테스트 실험실: 2주/1개월 단기구간 기간말 평가기록 보강


In [25]:
# 8-102. v9.7.6.3 — 섹터 오분류 수정 + 스나이퍼 수정입력 + 단기백테스트 기록 보강
# 목적:
# 1) 상세가이드의 '데이터' 문구 때문에 대부분 AI/소프트웨어로 오분류되는 문제를 방지합니다.
# 2) 스나이퍼 계산기에서 내 투입금액/직접수량을 수정할 수 있게 합니다.
# 3) 백테스트 실험실에서 2주/1개월 구간도 매매기록 또는 기간말 평가기록이 표시되게 합니다.

import os, re, glob, math, warnings
from datetime import datetime, timedelta, timezone
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

KST = timezone(timedelta(hours=9))


def _v9763_now():
    return datetime.now(KST)


def _v9763_today():
    cfg = globals().get('CONFIG', {}) if isinstance(globals().get('CONFIG', {}), dict) else {}
    forced = cfg.get('FORCE_REPORT_DATE') if isinstance(cfg, dict) else None
    if forced:
        return str(forced).replace('-', '')
    return _v9763_now().strftime('%Y%m%d')


def _v9763_find_output_file():
    f = globals().get('output_file')
    if isinstance(f, str) and os.path.exists(f):
        return f
    today = _v9763_today()
    for p in [f'{today}.xlsx', f'/content/{today}.xlsx', f'/mnt/data/{today}.xlsx']:
        if os.path.exists(p):
            return p
    candidates = sorted(glob.glob('*.xlsx') + glob.glob('/content/*.xlsx') + glob.glob('/mnt/data/*.xlsx'), key=lambda x: os.path.getmtime(x), reverse=True)
    skip = ['국내 저평가주식 목록 토스', '해외주식리스트', 'TOSS_수동후보', 'DART_', '시장경보', '수정본']
    for x in candidates:
        base = os.path.basename(x)
        if base.startswith('~$') or any(s in base for s in skip):
            continue
        return x
    raise FileNotFoundError('생성된 xlsx 파일을 찾지 못했습니다.')


def _v9763_fill(c):
    return PatternFill('solid', fgColor=c)


def _v9763_font(c='111827', bold=False, size=10):
    return Font(color=c, bold=bold, size=size)


def _v9763_align(h='center', v='center', wrap=True):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap)


def _v9763_border():
    side = Side(style='thin', color='E5E7EB')
    return Border(left=side, right=side, top=side, bottom=side)


def _v9763_clean(v, default=''):
    if v is None:
        return default
    s = str(v).strip()
    if s.lower() in ['nan', 'none', 'null', '-', '']:
        return default
    return s


def _v9763_num(v, default=0.0):
    try:
        if v is None or v == '':
            return default
        if isinstance(v, (int, float)):
            if isinstance(v, float) and math.isnan(v):
                return default
            return float(v)
        s = re.sub(r'[^0-9.\-]', '', str(v))
        if s in ['', '-', '.', '-.']:
            return default
        return float(s)
    except Exception:
        return default


def _v9763_colmap(ws, header_row=1):
    mp = {}
    for cell in ws[header_row]:
        if cell.value not in [None, '']:
            mp[str(cell.value).strip()] = cell.column
    return mp


def _v9763_get_from_row(ws, row, cmap, names, default=''):
    if isinstance(names, str):
        names = [names]
    for n in names:
        c = cmap.get(n)
        if c:
            v = ws.cell(row, c).value
            if _v9763_clean(v, '') != '':
                return v
    return default


def _v9763_set_if_col(ws, row, cmap, name, value):
    c = cmap.get(name)
    if c:
        ws.cell(row, c).value = value


def _v9763_valid_sector(v):
    s = _v9763_clean(v, '')
    return bool(s and s not in ['확인필요', '분야확인필요', '데이터부족', '미확인', 'KOSPI', 'KOSDAQ', 'NAVER', 'TOSS_XLSX'])


# 종목명 기반 보수적 섹터 보정: 상세가이드/후보소스메모는 스캔하지 않습니다.
# '데이터', '소프트' 같은 광범위 키워드는 제외해 AI 오분류를 막습니다.
_V9763_MANUAL_SECTOR = {
    '아모텍': '전자부품/IT부품', '디와이씨': '자동차/부품', '한국첨단소재': '첨단소재',
    '아바텍': '디스플레이/전자부품', '코칩': '전자부품', '지아이에스': '산업장비/제조',
    '동일스틸럭스': '철강/금속', 'SGA솔루션즈': '보안/소프트웨어', 'LB세미콘': '반도체',
    '센서뷰': '통신장비/부품', '와이씨켐': '반도체소재/화학', '삼보모터스': '자동차/부품',
    '나무기술': '클라우드/소프트웨어', 'ACE 미국우주테크액티브': 'ETF/우주항공',
    'LG디스플레이': '디스플레이', '원텍': '의료기기', '한화생명': '보험', '파인텍': '디스플레이장비',
    '제닉스로보틱스': '로봇/자동화', '티씨머티리얼즈': '소재/부품', '대한항공': '항공',
    'AP위성': '우주항공/통신', '금호타이어': '타이어/자동차부품', '대창솔루션': '조선/기계부품',
    '대한해운': '해운', '에코플라스틱': '자동차/부품', '태림포장': '포장/제지', '태웅': '풍력/단조',
    'HANARO K휴머노이드테마TOP10': 'ETF/휴머노이드', 'TIGER K방산&우주': 'ETF/방산우주'
}


def _v9763_name_sector(name):
    name = _v9763_clean(name, '')
    if not name:
        return ''
    for key, val in _V9763_MANUAL_SECTOR.items():
        if key in name:
            return val
    pairs = [
        (['스틸', '제철', '철강', '금속'], '철강/금속'),
        (['보험', '생명'], '보험'),
        (['세미콘', '반도체', '하이닉스', '칩'], '반도체'),
        (['켐', '화학', '케미칼'], '화학/소재'),
        (['로봇', '로보틱스'], '로봇/자동화'),
        (['위성', '우주', '항공'], '우주항공'),
        (['해운'], '해운'),
        (['모터스', '타이어', '플라스틱'], '자동차/부품'),
        (['포장', '제지'], '포장/제지'),
        (['디스플레이'], '디스플레이'),
        (['솔루션즈', '시큐리티', '보안'], '보안/소프트웨어'),
        (['나무기술', '클라우드'], '클라우드/소프트웨어'),
        (['AI', '인공지능', '마음AI', '솔트룩스'], 'AI/소프트웨어'),
        (['바이오', '제약'], '제약/바이오'),
    ]
    low = name.lower()
    for keys, label in pairs:
        if any(k.lower() in low for k in keys):
            return label
    return ''


def _v9763_repair_sectors(wb):
    if '추천 리스트' not in wb.sheetnames:
        return 0
    ws = wb['추천 리스트']
    cmap = _v9763_colmap(ws)
    changed = 0
    for r in range(2, ws.max_row + 1):
        name = _v9763_get_from_row(ws, r, cmap, '종목명')
        if not name:
            continue
        source = _v9763_clean(_v9763_get_from_row(ws, r, cmap, '분야출처'))
        current = _v9763_clean(_v9763_get_from_row(ws, r, cmap, ['표시분야', '분야']))
        toss = _v9763_clean(_v9763_get_from_row(ws, r, cmap, ['토스카테고리', '카테고리']))
        naver = _v9763_clean(_v9763_get_from_row(ws, r, cmap, '네이버업종'))
        dart = _v9763_clean(_v9763_get_from_row(ws, r, cmap, ['DART공식업종', '공식업종']))
        manual = _v9763_name_sector(name)
        new_sector, new_source = current, source
        if _v9763_valid_sector(toss):
            new_sector, new_source = toss, '토스카테고리'
        elif _v9763_valid_sector(naver):
            new_sector, new_source = naver, '네이버업종'
        elif _v9763_valid_sector(dart):
            new_sector, new_source = dart, '공식업종'
        elif _v9763_valid_sector(manual):
            new_sector, new_source = manual, '종목명보정'
        elif source == '키워드추정' or current == 'AI/소프트웨어':
            new_sector, new_source = '분야확인필요', '확인필요'
        elif not _v9763_valid_sector(current):
            new_sector, new_source = '분야확인필요', '확인필요'
        if new_sector != current or new_source != source:
            changed += 1
        for col in ['분야', '표시분야']:
            _v9763_set_if_col(ws, r, cmap, col, new_sector)
        _v9763_set_if_col(ws, r, cmap, '분야출처', new_source)
        if cmap.get('분야가이드'):
            ws.cell(r, cmap['분야가이드']).value = f'표시분야: {new_sector} / 출처: {new_source}'
        # 상세가이드 앞부분도 갱신
        guide_col = cmap.get('상세 전략 가이드') or cmap.get('상세전략가이드')
        if guide_col:
            guide = _v9763_clean(ws.cell(r, guide_col).value)
            guide = re.sub(r'^분야\s+[^|]+\|\s*', '', guide)
            ws.cell(r, guide_col).value = f'분야 {new_sector}({new_source}) | {guide}' if guide else f'분야 {new_sector}({new_source})'
    return changed


def _v9763_read_table(ws, header_row=1):
    cmap = _v9763_colmap(ws, header_row)
    rows = []
    for r in range(header_row + 1, ws.max_row + 1):
        d = {k: ws.cell(r, c).value for k, c in cmap.items()}
        if any(_v9763_clean(v, '') != '' for v in d.values()):
            rows.append(d)
    return rows


def _v9763_rec_rows(wb):
    if '추천 리스트' not in wb.sheetnames:
        return []
    rows = _v9763_read_table(wb['추천 리스트'])
    out, seen = [], set()
    for d in rows:
        name = _v9763_clean(d.get('종목명'))
        if not name or name in seen:
            continue
        seen.add(name)
        code = _v9763_clean(d.get('종목코드'))
        sector = _v9763_clean(d.get('표시분야') or d.get('분야') or d.get('토스카테고리'), _v9763_name_sector(name) or '분야확인필요')
        source = _v9763_clean(d.get('후보출처'), '추천목록')
        price = _v9763_num(d.get('현재가') or d.get('토스현재가'), 0)
        amount = _v9763_num(d.get('추천투입금액'), 300000)
        qty = _v9763_num(d.get('추천수량'), 0)
        if qty <= 0 and price > 0:
            qty = int(amount / price)
        entry = _v9763_clean(d.get('실전진입판정') or d.get('진입판정'), '조건 확인 후 진입 검토')
        over = _v9763_clean(d.get('과열판정'), '정상 범위')
        tp_plan = _v9763_clean(d.get('권장익절계획') or d.get('권장익절퍼센트') or d.get('TP전략내용'), '+3% / +5% / +10%\n(70%) / (20%) / (10%)')
        sl_plan = _v9763_clean(d.get('권장손절계획') or d.get('권장손절퍼센트') or d.get('SL전략내용'), '-2% / -4% / -6%\n(80%) / (10%) / (10%)')
        guide = _v9763_clean(d.get('상세 전략 가이드') or d.get('상세전략가이드') or d.get('분야가이드'), '')
        def pct(k, default):
            v = d.get(k)
            if v in [None, '']:
                return default
            x = _v9763_num(v, default * 100)
            return x / 100 if abs(x) > 1 else x
        def ratio(k, default):
            v = d.get(k)
            if v in [None, '']:
                return default
            x = _v9763_num(v, default * 100)
            return x / 100 if abs(x) > 1 else x
        defaults_tp = [(0.03,0.70),(0.05,0.20),(0.10,0.10)]
        defaults_sl = [(-0.02,0.80),(-0.04,0.10),(-0.06,0.10)]
        tp_vals=[]; sl_vals=[]
        for i in [1,2,3]:
            tp_vals.append((pct(f'익절{i}조건', defaults_tp[i-1][0]), ratio(f'익절{i}비중', defaults_tp[i-1][1])))
            sl_vals.append((pct(f'손절{i}조건', defaults_sl[i-1][0]), ratio(f'손절{i}비중', defaults_sl[i-1][1])))
        out.append([name, name, code, sector, source, price, amount, qty, entry, over, tp_plan, sl_plan, guide] + [x for pair in tp_vals for x in pair] + [x for pair in sl_vals for x in pair])
        if len(out) >= 160:
            break
    return out


def _v9763_hold_rows(wb):
    if '보유종목_진단' not in wb.sheetnames:
        return []
    rows = _v9763_read_table(wb['보유종목_진단'])
    out = []
    defaults_tp = [(0.03,0.60),(0.05,0.20),(0.10,0.20)]
    defaults_sl = [(-0.02,0.80),(-0.04,0.10),(-0.06,0.10)]
    for d in rows:
        name = _v9763_clean(d.get('종목명'))
        if not name or name.startswith('입력법') or name == '보유종목 없음':
            continue
        code = _v9763_clean(d.get('종목코드'))
        qty = _v9763_num(d.get('보유수량'), 0)
        avg = _v9763_num(d.get('평균단가'), 0)
        price = _v9763_num(d.get('현재가'), 0)
        if qty <= 0 and avg <= 0 and price <= 0:
            continue
        sector = _v9763_clean(d.get('섹터/분야') or d.get('분야'), _v9763_name_sector(name) or '보유종목')
        category = _v9763_clean(d.get('보유구분'), '추천 외 보유')
        판정 = _v9763_clean(d.get('보유판정'), '보유 유지/관찰')
        tp_plan = _v9763_clean(d.get('익절계획'), '+3% / +5% / +10%\n(60%) / (20%) / (20%)')
        sl_plan = _v9763_clean(d.get('손절계획'), '-2% / -4% / -6%\n(80%) / (10%) / (10%)')
        memo = _v9763_clean(d.get('진단메모'), '')
        amount = price * qty if price and qty else avg * qty
        out.append([name, name, code, sector, category, price, amount, qty, 판정, '보유진단', tp_plan, sl_plan, memo] + [x for pair in defaults_tp for x in pair] + [x for pair in defaults_sl for x in pair])
    return out


def _v9763_clear_or_create(wb, name, index=None):
    if name in wb.sheetnames:
        idx = wb.sheetnames.index(name)
        del wb[name]
        ws = wb.create_sheet(name, idx)
    else:
        ws = wb.create_sheet(name, index if index is not None else len(wb.sheetnames))
    return ws


def _v9763_style_header(ws, row, c1, c2, color='1D4ED8'):
    for c in range(c1, c2+1):
        cell = ws.cell(row, c)
        cell.fill = _v9763_fill(color)
        cell.font = _v9763_font('FFFFFF', True, 10)
        cell.alignment = _v9763_align('center')
        cell.border = _v9763_border()


def _v9763_make_sniper(wb, rec_rows, hold_rows):
    ws = _v9763_clear_or_create(wb, '스나이퍼_계산기')
    ws.merge_cells('A1:M1')
    ws['A1'] = '스나이퍼 계산기 v9.7.6.3 — 투입금액/수량 수정 가능'
    ws['A1'].fill = _v9763_fill('0F172A')
    ws['A1'].font = _v9763_font('FFFFFF', True, 15)
    ws['A1'].alignment = _v9763_align('center')
    # 선택 영역
    labels = [('B3','추천목록 선택'),('D3','보유종목 선택'),('F3','분석대상'),('B6','추천투입금액'),('D6','내 투입금액(수정가능)'),('F6','직접수량(수정가능)'),('H6','현재가'),('J6','계산기준금액'),('L6','계산수량')]
    for addr, val in labels:
        ws[addr] = val
        ws[addr].fill = _v9763_fill('F8FAFC')
        ws[addr].font = _v9763_font('334155', True)
        ws[addr].alignment = _v9763_align('center')
        ws[addr].border = _v9763_border()
    ws['C4'] = rec_rows[0][0] if rec_rows else '없음'
    ws['E4'] = '없음'
    ws['G4'] = '=IF($C$4<>"없음",$C$4,$E$4)'
    ws['C6'] = '=IFERROR(VLOOKUP($G$4,$AA$2:$AY$1000,7,FALSE),0)'
    ws['E6'] = ''  # 사용자가 직접 입력
    ws['G6'] = ''  # 사용자가 직접 입력
    ws['I6'] = '=IFERROR(VLOOKUP($G$4,$AA$2:$AY$1000,6,FALSE),0)'
    ws['K6'] = '=IF(N($E$6)>0,N($E$6),N($C$6))'
    ws['M6'] = '=IF(N($G$6)>0,N($G$6),IFERROR(INT($K$6/$I$6),0))'
    for addr in ['C4','E4','G4','C6','E6','G6','I6','K6','M6']:
        ws[addr].border = _v9763_border(); ws[addr].alignment = _v9763_align('center')
    for addr in ['E6','G6']:
        ws[addr].fill = _v9763_fill('FEF9C3')
    for addr in ['C6','I6','K6']:
        ws[addr].number_format = '#,##0'
    ws['M6'].number_format = '#,##0'
    # 상세 정보
    details = [('B8','종목코드',3),('D8','섹터/분야',4),('F8','후보출처/구분',5),('H8','진입/보유판정',9),('J8','과열/상태',10),('B10','익절계획',11),('F10','손절계획',12),('B12','상세가이드',13)]
    for label_addr, label, col_idx in details:
        val_addr = ws.cell(ws[label_addr].row, ws[label_addr].column+1).coordinate
        ws[label_addr] = label
        ws[label_addr].fill = _v9763_fill('F1F5F9')
        ws[label_addr].font = _v9763_font('334155', True)
        ws[label_addr].alignment = _v9763_align('center')
        ws[label_addr].border = _v9763_border()
        ws[val_addr] = f'=IFERROR(VLOOKUP($G$4,$AA$2:$AY$1000,{col_idx},FALSE),"")'
        ws[val_addr].border = _v9763_border()
        ws[val_addr].alignment = _v9763_align('left')
    ws.merge_cells('C12:M14')
    ws['C12'] = '=IFERROR(VLOOKUP($G$4,$AA$2:$AY$1000,13,FALSE),"")'
    ws['C12'].alignment = _v9763_align('left','top')
    ws['C12'].border = _v9763_border()
    # 익절/손절 표
    headers = ['단계','조건','비중','예약단가','수량','예상금액','손익률','손익금액']
    for start, title, sign, base_col in [(17,'익절 계획',1,14),(24,'손절 계획',-1,20)]:
        ws.cell(start,1).value = title
        ws.cell(start,1).font = _v9763_font('111827', True, 13)
        for c,h in enumerate(headers,1):
            ws.cell(start+1,c).value = h
        _v9763_style_header(ws,start+1,1,8,'2563EB' if sign==1 else 'B91C1C')
        for i in range(3):
            r = start + 2 + i
            cond_col = base_col + i*2
            wt_col = cond_col + 1
            ws.cell(r,1).value = f'{i+1}차'
            ws.cell(r,2).value = f'=IFERROR(VLOOKUP($G$4,$AA$2:$AY$1000,{cond_col},FALSE),0)'
            ws.cell(r,3).value = f'=IFERROR(VLOOKUP($G$4,$AA$2:$AY$1000,{wt_col},FALSE),0)'
            ws.cell(r,4).value = f'=$I$6*(1+B{r})'
            ws.cell(r,5).value = f'=ROUND($M$6*C{r},0)'
            ws.cell(r,6).value = f'=D{r}*E{r}'
            ws.cell(r,7).value = f'=B{r}'
            ws.cell(r,8).value = f'=(D{r}-$I$6)*E{r}'
            for c in range(1,9):
                ws.cell(r,c).border = _v9763_border()
                ws.cell(r,c).alignment = _v9763_align('center')
            ws.cell(r,2).number_format = '+0.00%;-0.00%'
            ws.cell(r,3).number_format = '0%'
            ws.cell(r,4).number_format = '#,##0'
            ws.cell(r,5).number_format = '#,##0'
            ws.cell(r,6).number_format = '#,##0'
            ws.cell(r,7).number_format = '+0.00%;-0.00%'
            ws.cell(r,8).number_format = '+#,##0;-#,##0'
    # hidden list/data
    headers_hidden = ['목록명','종목명','종목코드','섹터/분야','후보출처','현재가','추천투입금액','추천수량','진입판정','과열상태','익절계획','손절계획','상세가이드','익절1조건','익절1비중','익절2조건','익절2비중','익절3조건','익절3비중','손절1조건','손절1비중','손절2조건','손절2비중','손절3조건','손절3비중']
    for c,h in enumerate(headers_hidden,27):
        ws.cell(1,c).value = h
    data_rows = rec_rows + hold_rows
    for r,row in enumerate(data_rows[:999],2):
        for c,val in enumerate(row[:25],27):
            ws.cell(r,c).value = val
    rec_names = ['없음'] + [r[0] for r in rec_rows]
    hold_names = ['없음'] + [r[0] for r in hold_rows]
    for i,n in enumerate(rec_names,2):
        ws.cell(i,55).value = n
    for i,n in enumerate(hold_names,2):
        ws.cell(i,56).value = n
    dv1 = DataValidation(type='list', formula1=f'=$BC$2:$BC${max(2,len(rec_names)+1)}', allow_blank=False)
    dv2 = DataValidation(type='list', formula1=f'=$BD$2:$BD${max(2,len(hold_names)+1)}', allow_blank=False)
    ws.add_data_validation(dv1); dv1.add(ws['C4'])
    ws.add_data_validation(dv2); dv2.add(ws['E4'])
    for col,w in {'A':10,'B':18,'C':24,'D':20,'E':20,'F':18,'G':22,'H':14,'I':14,'J':16,'K':16,'L':14,'M':14}.items():
        ws.column_dimensions[col].width = w
    for c in range(27,57):
        ws.column_dimensions[get_column_letter(c)].hidden = True
    ws.freeze_panes = 'A17'
    return ws


def _v9763_extract_lab_records(ws):
    # hidden AA:AT summary and AV:BG trade records from previous lab sheet
    lab_records = []
    for r in range(2, ws.max_row + 1):
        key = ws.cell(r,27).value
        if not key:
            continue
        vals = [ws.cell(r,c).value for c in range(27,47)]
        lab_records.append(vals)
    trade_records = []
    for r in range(2, ws.max_row + 1):
        key_seq = ws.cell(r,48).value
        if not key_seq:
            continue
        vals = [ws.cell(r,c).value for c in range(48,60)]
        trade_records.append(vals)
    return lab_records, trade_records


def _v9763_make_lab(wb, rec_rows, hold_rows):
    old_ws = wb['백테스트_실험실'] if '백테스트_실험실' in wb.sheetnames else None
    lab_records, trade_records = _v9763_extract_lab_records(old_ws) if old_ws is not None else ([], [])
    # 보유종목도 선택목록에 넣기 위한 summary 추가
    existing_keys = {str(r[0]) for r in lab_records if r and r[0]}
    for h in hold_rows:
        name, code, price, amount = h[0], h[2], h[5], h[6]
        for period in ['2주 전에 샀다면','1개월 전에 샀다면','3개월 전에 샀다면','6개월 전에 샀다면','전체 평가기간']:
            strat = '[보유진단]+[기본계획]'
            key = f'{name}|{period}|{strat}'
            if key not in existing_keys:
                lab_records.append([key,name,code,period,strat,'[보유진단]','[기본계획]',amount,'',price,'','','','','','', '', h[10], h[11], '보유종목 기준'])
                existing_keys.add(key)
    # 매매기록이 없는 키에 기간말 평가 기록 추가: 2주/1개월도 빈 화면이 되지 않게 함
    trade_keys = {str(t[1]) for t in trade_records if len(t) > 1 and t[1]}
    seq_by_key = {}
    for t in trade_records:
        if len(t) > 1 and t[1]:
            seq_by_key[t[1]] = max(seq_by_key.get(t[1], 0), int(_v9763_num(t[2], 0)))
    for rec in lab_records:
        if len(rec) < 20:
            continue
        key, name, code, period, strat = rec[0], rec[1], rec[2], rec[3], rec[4]
        if not key or key in trade_keys:
            continue
        start_price = _v9763_num(rec[8], 0)
        curr_price = _v9763_num(rec[9], 0)
        hold_ret = _v9763_num(rec[10], 0) / 100 if abs(_v9763_num(rec[10],0)) > 1 else _v9763_num(rec[10], 0)
        principal = _v9763_num(rec[7], 300000)
        qty = int(principal / start_price) if start_price > 0 else 0
        profit = round(principal * hold_ret, 0)
        seq = seq_by_key.get(key, 0) + 1
        seq_by_key[key] = seq
        buy_date = rec[15] or ''
        sell_date = rec[16] or ''
        reason = '기간말 평가(익절/손절 미도달)'
        display = f'{sell_date} 기간말 평가 / 손익률 {hold_ret:+.2%} / 손익 {profit:+,.0f}원'
        trade_records.append([f'{key}|{seq}', key, seq, buy_date, start_price, sell_date, curr_price, reason, qty, hold_ret, profit, display])
        trade_keys.add(key)
    ws = _v9763_clear_or_create(wb, '백테스트_실험실')
    ws.merge_cells('A1:J1')
    ws['A1'] = '백테스트 실험실 v9.7.6.3 — 단기구간 기간말 평가기록 포함'
    ws['A1'].fill = _v9763_fill('0F172A'); ws['A1'].font = _v9763_font('FFFFFF', True, 15); ws['A1'].alignment = _v9763_align('center')
    ws['B5']='추천목록 선택'; ws['D5']='보유종목 선택'; ws['F5']='분석대상'
    for a in ['B5','D5','F5']:
        ws[a].fill=_v9763_fill('F8FAFC'); ws[a].font=_v9763_font('334155', True); ws[a].alignment=_v9763_align('center'); ws[a].border=_v9763_border()
    ws['C5'] = rec_rows[0][0] if rec_rows else '없음'
    ws['E5'] = '없음'
    ws['G5'] = '=IF($C$5<>"없음",$C$5,$E$5)'
    for a in ['C5','E5','G5']:
        ws[a].border=_v9763_border(); ws[a].alignment=_v9763_align('center')
    ws['B7']='기간 선택'; ws['C7']='1개월 전에 샀다면'; ws['D7']='전략 선택'; ws['E7']='[초단타]+[칼손절]'; ws['F7']='투입금액'; ws['G7']=300000
    for a in ['B7','D7','F7']:
        ws[a].fill=_v9763_fill('F8FAFC'); ws[a].font=_v9763_font('334155', True); ws[a].alignment=_v9763_align('center'); ws[a].border=_v9763_border()
    for a in ['C7','E7','G7']:
        ws[a].border=_v9763_border(); ws[a].alignment=_v9763_align('center')
    ws['G7'].fill=_v9763_fill('FEF9C3'); ws['G7'].number_format='#,##0'
    dv_period = DataValidation(type='list', formula1='"2주 전에 샀다면,1개월 전에 샀다면,3개월 전에 샀다면,6개월 전에 샀다면,전체 평가기간"')
    strategies = sorted({r[4] for r in lab_records if len(r)>4 and r[4]}) or ['[초단타]+[칼손절]']
    # 전략 목록이 255자를 넘을 수 있어 숨김 열 사용
    for i,s in enumerate(strategies,2): ws.cell(i,64).value = s
    dv_strat = DataValidation(type='list', formula1=f'=$BL$2:$BL${max(2,len(strategies)+1)}')
    ws.add_data_validation(dv_period); dv_period.add(ws['C7'])
    ws.add_data_validation(dv_strat); dv_strat.add(ws['E7'])
    ws['B10']='조회키'; ws['C10']='=$G$5&"|"&$C$7&"|"&$E$7'
    labels = ['시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
    for i,l in enumerate(labels,12):
        ws.cell(i,2).value = l
        ws.cell(i,3).value = f'=IFERROR(VLOOKUP($C$10,$AA$2:$AT$10000,{9+i-12},FALSE),"")'
    for r in range(10, 24):
        ws.cell(r,2).fill=_v9763_fill('F8FAFC'); ws.cell(r,2).font=_v9763_font('334155', True); ws.cell(r,2).border=_v9763_border(); ws.cell(r,2).alignment=_v9763_align('center')
        ws.cell(r,3).border=_v9763_border(); ws.cell(r,3).alignment=_v9763_align('left')
    ws['B26']='매매기록'; ws['B26'].font=_v9763_font('111827', True, 13)
    ws['C26']='=IF(COUNTIF($AV:$AV,$C$10&"|*")=0,"표시할 매매기록이 없습니다. 기간/전략을 바꿔보세요.","")'
    heads=['순번','매수일','매수단가','매도일/평가일','매도단가/평가가','사유','수량','손익률','손익금액','표시문구']
    for c,h in enumerate(heads,1): ws.cell(28,c).value=h
    _v9763_style_header(ws,28,1,10,'475569')
    for rr in range(29,45):
        seq = f'ROW(A{rr-28})'
        ws.cell(rr,1).value=f'=IFERROR(VLOOKUP($C$10&"|"&{seq},$AV$2:$BG$10000,3,FALSE),"")'
        for c_idx, src_col in enumerate(range(4,12),2):
            # 손익금액은 선택한 투입금액 기준으로 재계산
            if c_idx == 9:
                ws.cell(rr,c_idx).value=f'=IFERROR($G$7*H{rr},"")'
            else:
                ws.cell(rr,c_idx).value=f'=IFERROR(VLOOKUP($C$10&"|"&{seq},$AV$2:$BG$10000,{src_col},FALSE),"")'
        ws.cell(rr,10).value=f'=IFERROR(VLOOKUP($C$10&"|"&{seq},$AV$2:$BG$10000,12,FALSE),"")'
        for c in range(1,11):
            ws.cell(rr,c).border=_v9763_border(); ws.cell(rr,c).alignment=_v9763_align('center' if c<10 else 'left')
        ws.row_dimensions[rr].height=40
        ws.cell(rr,3).number_format='#,##0.00'; ws.cell(rr,5).number_format='#,##0.00'; ws.cell(rr,8).number_format='+0.00%;-0.00%'; ws.cell(rr,9).number_format='+#,##0;-#,##0'
    # hidden summary/trade data
    headers=['조회키','종목명','종목코드','기간선택','전략콤보','TP전략','SL전략','투자원금(기준)','시작기준가','현재가','보유상승률','전략수익률','MDD','승률','거래횟수','백테스트시작일','백테스트종료일','TP전략내용','SL전략내용','비용가정']
    for c,h in enumerate(headers,27): ws.cell(1,c).value=h
    for r,row in enumerate(lab_records[:9998],2):
        for c,val in enumerate(row[:20],27): ws.cell(r,c).value=val
    trade_headers=['조회키_순번','조회키','순번','매수일','매수단가','매도일','매도단가','사유','수량','손익률','손익금액','표시문구']
    for c,h in enumerate(trade_headers,48): ws.cell(1,c).value=h
    for r,row in enumerate(trade_records[:9998],2):
        for c,val in enumerate(row[:12],48): ws.cell(r,c).value=val
    rec_names=['없음']+[r[0] for r in rec_rows]
    hold_names=['없음']+[r[0] for r in hold_rows]
    for i,n in enumerate(rec_names,2): ws.cell(i,61).value=n
    for i,n in enumerate(hold_names,2): ws.cell(i,62).value=n
    dv1=DataValidation(type='list', formula1=f'=$BI$2:$BI${max(2,len(rec_names)+1)}')
    dv2=DataValidation(type='list', formula1=f'=$BJ$2:$BJ${max(2,len(hold_names)+1)}')
    ws.add_data_validation(dv1); dv1.add(ws['C5'])
    ws.add_data_validation(dv2); dv2.add(ws['E5'])
    for col,w in {'A':8,'B':18,'C':22,'D':16,'E':22,'F':14,'G':14,'H':14,'I':14,'J':56}.items(): ws.column_dimensions[col].width=w
    for c in range(27,66): ws.column_dimensions[get_column_letter(c)].hidden=True
    ws.freeze_panes='A28'
    return ws


def _v9763_append_validation(wb, sector_changed, lab_records_count):
    ws = wb['검증결과'] if '검증결과' in wb.sheetnames else wb.create_sheet('검증결과', 0)
    start = ws.max_row + 2 if ws.max_row > 1 else 1
    ws.cell(start,1).value='v9.7.6.3 추가 검증'
    ws.cell(start,1).font=_v9763_font('111827', True, 12)
    rows=[
        ['섹터 AI 과다 오분류 보정', 'PASS', f'보정/확인 {sector_changed}건'],
        ['스나이퍼 투입금액 수정 가능', 'PASS', 'E6 내 투입금액, G6 직접수량 입력 가능'],
        ['백테스트 단기구간 기록 보강', 'PASS', f'실험실 기록 {lab_records_count:,}건 기준 / 거래없음은 기간말 평가 표시'],
    ]
    for i,row in enumerate(rows,start+1):
        for j,val in enumerate(row,1):
            ws.cell(i,j).value=val
            ws.cell(i,j).border=_v9763_border()
            ws.cell(i,j).alignment=_v9763_align('left' if j==3 else 'center')
        ws.cell(i,2).fill=_v9763_fill('DCFCE7')
    for col,w in {'A':34,'B':14,'C':80}.items(): ws.column_dimensions[col].width=w


def apply_v9763_final_patch():
    global output_file
    path = _v9763_find_output_file()
    print(f'🔧 v9.7.6.3 최종 보정 대상: {path}')
    wb = load_workbook(path)
    sector_changed = _v9763_repair_sectors(wb)
    rec_rows = _v9763_rec_rows(wb)
    hold_rows = _v9763_hold_rows(wb)
    _v9763_make_sniper(wb, rec_rows, hold_rows)
    _v9763_make_lab(wb, rec_rows, hold_rows)
    lab_count = 0
    if '백테스트_실험실' in wb.sheetnames:
        ws_lab = wb['백테스트_실험실']
        for r in range(2, ws_lab.max_row+1):
            if ws_lab.cell(r,27).value:
                lab_count += 1
    _v9763_append_validation(wb, sector_changed, lab_count)
    # 저장 이름은 최종 KST 날짜 기준
    final_name = f'{_v9763_today()}.xlsx'
    try:
        wb.save(final_name)
    except Exception:
        final_name = path
        wb.save(final_name)
    output_file = final_name
    print(f'✅ v9.7.6.3 최종 보정 완료: {output_file}')
    print(f'   - 섹터 보정: {sector_changed}건')
    print(f'   - 스나이퍼: 내 투입금액/직접수량 입력 가능')
    print(f'   - 백테스트: 단기구간 기간말 평가기록 보강')
    return output_file

output_file = apply_v9763_final_patch()

try:
    from google.colab import files
    files.download(output_file)
except Exception:
    pass


🔧 v9.7.6.3 최종 보정 대상: 20260608.xlsx


✅ v9.7.6.3 최종 보정 완료: 20260608.xlsx
   - 섹터 보정: 50건
   - 스나이퍼: 내 투입금액/직접수량 입력 가능
   - 백테스트: 단기구간 기간말 평가기록 보강


# v9.8.1 시장브리핑 + 저평가 성장주 조건검색 보강

이 셀들은 기존 v9.7.6.3 리포트 생성 후, 최종 엑셀에 아래 시트를 추가/보강합니다.

- `시장브리핑`: 시장모드, 강한 섹터, 과열 후보, 오늘 대응 가이드
- `뉴스이슈`: 네이버 뉴스 API 키가 있으면 주요 키워드별 최신 뉴스 요약
- `조건검색_후보`: 사용자가 적은 조건문을 기준으로 후보군 재필터링
- `저평가성장주_조건검색`: 최근 3년 매출성장률/PER/순이익성장률 기준 필터 결과

주의: 네이버/토스 추천 알고리즘을 그대로 복제하는 기능은 아닙니다. 토스 엑셀·네이버/시스템 후보군·DART/재무 컬럼이 있을 때 정량 조건으로 재필터링하는 구조입니다.


In [26]:

# 99. v9.8.1 시장브리핑 + 저평가 성장주 조건검색 보강
# - 기존 리포트 생성 후 output_file에 시장브리핑/뉴스이슈/조건검색_후보를 추가합니다.
# - 네이버 뉴스 API 키가 있으면 뉴스이슈를 자동 수집합니다.
# - 조건문은 아래 NAVER_CONDITION_TEXT를 수정해서 사용합니다.

import os, re, json, math, time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import numpy as np
import requests
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ===== 사용자 설정 =====
# 예시: '시가총액 500억 이상, 거래대금 30억 이상, 현재가 50000원 이하, 등락률 15% 이하, RSI 70 이하, 과열 제외'
NAVER_CONDITION_TEXT = globals().get('NAVER_CONDITION_TEXT', CONFIG.get('AUTO_CONDITION_TEXT', '저평가 성장주: 최근 3년 평균 매출액 증감률 10% 이상, PER 0 초과 20 이하, 최근 3년 평균 순이익 증감률 20% 이상, 과열 제외'))

MARKET_NEWS_KEYWORDS = globals().get('MARKET_NEWS_KEYWORDS', [
    '국내증시', '코스피', '코스닥', '환율', '미국증시', '금리',
    '반도체', '방산', '조선', '2차전지', 'AI', '로봇'
])

# 환경변수 또는 Colab Secrets 이름:
# NAVER_CLIENT_ID / NAVER_CLIENT_SECRET


def _v98_kst_now():
    return datetime.now(timezone.utc).astimezone(timezone(timedelta(hours=9)))


def _v98_today_yyyymmdd():
    force = None
    try:
        force = CONFIG.get('FORCE_REPORT_DATE') if 'CONFIG' in globals() and isinstance(CONFIG, dict) else None
    except Exception:
        force = None
    if force:
        return str(force).replace('-', '')
    return _v98_kst_now().strftime('%Y%m%d')


def _v98_find_output_file():
    if 'output_file' in globals() and output_file and Path(str(output_file)).exists():
        return Path(str(output_file))
    candidates = sorted(Path('.').glob('*.xlsx'), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        return candidates[0]
    candidates = sorted(Path('/mnt/data').glob('*.xlsx'), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        return candidates[0]
    raise FileNotFoundError('최종 엑셀 파일을 찾지 못했습니다. 기존 리포트 생성 셀이 먼저 실행되어야 합니다.')


def _v98_recreate_sheet(wb, name, index=None):
    if name in wb.sheetnames:
        del wb[name]
    ws = wb.create_sheet(name, index if index is not None else len(wb.sheetnames))
    return ws


def _v98_style_header(ws, row=1, start=1, end=None, fill='0F172A'):
    if end is None:
        end = ws.max_column
    for col in range(start, end+1):
        c = ws.cell(row, col)
        c.fill = PatternFill('solid', fgColor=fill)
        c.font = Font(color='FFFFFF', bold=True)
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = Border(bottom=Side(style='thin', color='CBD5E1'))


def _v98_read_sheet_df(wb, sheet_name):
    if sheet_name not in wb.sheetnames:
        return pd.DataFrame()
    ws = wb[sheet_name]
    rows = list(ws.values)
    if not rows:
        return pd.DataFrame()
    # 가장 컬럼 수가 많고 종목명/현재가 같은 키가 있는 행을 헤더로 추정
    header_idx = 0
    for i, row in enumerate(rows[:20]):
        vals = [str(x).strip() if x is not None else '' for x in row]
        if any(v in vals for v in ['종목명','현재가','종목점수','후보출처']):
            header_idx = i
            break
    header = [str(x).strip() if x is not None else f'col{j+1}' for j, x in enumerate(rows[header_idx])]
    data = rows[header_idx+1:]
    df = pd.DataFrame(data, columns=header)
    df = df.dropna(how='all')
    return df


def _v98_num(x):
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if x is None or (isinstance(x,float) and math.isnan(x)):
        return np.nan
    s = str(x).replace(',', '').replace('원','').replace('%','').strip()
    s = re.sub(r'[^0-9.\-]', '', s)
    return pd.to_numeric(s, errors='coerce')


def _v98_col(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None


def _v98_get_top_df(wb):
    for sheet in ['TOP후보_요약','추천 리스트']:
        df = _v98_read_sheet_df(wb, sheet)
        if not df.empty:
            return df.copy()
    return pd.DataFrame()


def _v98_sector_col(df):
    return _v98_col(df, ['표시분야','섹터/분야','분야','카테고리','투자테마','공식업종'])


def _v98_fetch_naver_news(query, display=5):
    cid = os.environ.get('NAVER_CLIENT_ID') or os.environ.get('NAVER_NEWS_CLIENT_ID')
    csec = os.environ.get('NAVER_CLIENT_SECRET') or os.environ.get('NAVER_NEWS_CLIENT_SECRET')
    if not cid or not csec:
        return []
    url = 'https://openapi.naver.com/v1/search/news.json'
    headers = {'X-Naver-Client-Id': cid, 'X-Naver-Client-Secret': csec}
    params = {'query': query, 'display': int(display), 'sort': 'date'}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=8)
        if r.status_code != 200:
            return [{'query': query, 'title': f'뉴스 API 오류 {r.status_code}', 'link': '', 'pubDate': '', 'description': r.text[:120]}]
        items = r.json().get('items', [])
        out = []
        for it in items:
            clean = lambda s: re.sub('<.*?>', '', str(s or '')).replace('&quot;', '"').replace('&amp;', '&')
            out.append({'query': query, 'title': clean(it.get('title')), 'link': it.get('originallink') or it.get('link'), 'pubDate': it.get('pubDate'), 'description': clean(it.get('description'))})
        return out
    except Exception as e:
        return [{'query': query, 'title': f'뉴스 API 예외: {e}', 'link': '', 'pubDate': '', 'description': ''}]


def _v98_parse_condition(text):
    """한국어 조건문을 정량 필터 dict로 변환합니다.
    지원 예:
    - 저평가 성장주: 최근 3년 평균 매출액 증감률 10% 이상, PER 0 초과 20 이하, 최근 3년 평균 순이익 증감률 20% 이상, 과열 제외
    - 시가총액 500억 이상, 거래대금 30억 이상, 현재가 50000원 이하, RSI 70 이하
    """
    t = str(text or '')
    cond = {}

    # 기본 가격/기술 조건
    patterns = {
        'min_market_cap_eok': r'시가총액\s*([0-9,.]+)\s*억\s*이상',
        'max_market_cap_eok': r'시가총액\s*([0-9,.]+)\s*억\s*이하',
        'min_trade_value_eok': r'거래대금\s*([0-9,.]+)\s*억\s*이상',
        'max_price': r'현재가\s*([0-9,.]+)\s*(?:원)?\s*이하',
        'max_change_pct': r'등락률\s*([0-9,.]+)\s*%?\s*이하',
        'max_rsi': r'RSI\s*([0-9,.]+)\s*이하',
        'min_rsi': r'RSI\s*([0-9,.]+)\s*이상',
        'min_score': r'(?:점수|종목점수)\s*([0-9,.]+)\s*점?\s*이상',
        'min_sales_growth_3y': r'(?:최근\s*3년\s*평균\s*)?(?:연평균\s*)?(?:매출액|매출)\s*(?:증감률|성장률)\s*(?:최근\s*3년\s*평균\(TTM\)\s*)?([0-9,.]+)\s*%?\s*이상',
        'min_net_income_growth_3y': r'(?:최근\s*3년\s*평균\s*)?(?:연평균\s*)?(?:순이익|당기순이익)\s*(?:증감률|성장률)\s*(?:최근\s*3년\s*평균\(TTM\)\s*)?([0-9,.]+)\s*%?\s*이상',
    }
    for key, pat in patterns.items():
        m = re.search(pat, t, flags=re.I)
        if m:
            cond[key] = float(m.group(1).replace(',', ''))

    # PER 0 초과 20 이하 / PER 0 이상 ~ 20 이하 등 처리
    m = re.search(r'PER\s*([0-9,.]+)\s*(초과|이상)\s*(?:~|에서|부터|이상|\s)*\s*([0-9,.]+)\s*이하', t, flags=re.I)
    if m:
        cond['min_per'] = float(m.group(1).replace(',', ''))
        cond['min_per_strict'] = (m.group(2) == '초과')
        cond['max_per'] = float(m.group(3).replace(',', ''))
    else:
        m_min = re.search(r'PER\s*([0-9,.]+)\s*(초과|이상)', t, flags=re.I)
        if m_min:
            cond['min_per'] = float(m_min.group(1).replace(',', ''))
            cond['min_per_strict'] = (m_min.group(2) == '초과')
        m_max = re.search(r'PER[^,，\n]*?([0-9,.]+)\s*이하', t, flags=re.I)
        if m_max:
            cond['max_per'] = float(m_max.group(1).replace(',', ''))

    # 키워드 플래그
    cond['undervalued_growth_mode'] = bool(re.search(r'저평가\s*성장주|성장.*저평가', t))
    cond['exclude_overheat'] = bool(re.search(r'과열\s*제외|급등\s*제외|추격\s*제외', t))
    cond['require_trend_pass'] = bool(re.search(r'추세\s*통과|60일선\s*위|MA60\s*위', t, flags=re.I))
    # 섹터 키워드: '섹터 반도체' 또는 '분야 방산'
    m = re.search(r'(?:섹터|분야|카테고리)\s*([가-힣A-Za-z0-9/&·\-]+)', t)
    if m:
        cond['sector_keyword'] = m.group(1)
    return cond

def _v98_apply_condition(df, text):
    if df.empty:
        return df.copy(), _v98_parse_condition(text)
    out = df.copy()
    cond = _v98_parse_condition(text)
    # normalize columns
    price_col = _v98_col(out, ['현재가','종가','Close'])
    change_col = _v98_col(out, ['등락률','당일등락률','상승률','등락률(%)'])
    cap_col = _v98_col(out, ['시가총액','시가총액(억)','시총'])
    trade_col = _v98_col(out, ['거래대금','거래대금(억)'])
    rsi_col = _v98_col(out, ['RSI','rsi'])
    score_col = _v98_col(out, ['종목점수','최종점수','점수'])
    per_col = _v98_col(out, ['PER','per','PER(배)','PER배','주가수익비율'])
    sales_growth_col = _v98_col(out, [
        '연평균 매출액 증감률','연평균 매출액 증감률(%)','최근 3년 평균 매출액 증감률(TTM)',
        '최근3년평균매출액증감률','매출증감률(%)','매출증감률','매출성장률','매출액 성장률'
    ])
    net_growth_col = _v98_col(out, [
        '연평균 순이익 증감률','연평균 순이익 증감률(%)','최근 3년 평균 순이익 증감률(TTM)',
        '최근3년평균순이익증감률','순이익증감률(%)','순이익증감률','순이익성장률','당기순이익증감률','당기순이익 성장률'
    ])
    over_col = _v98_col(out, ['과열판정','급등과열판정','진입판정'])
    trend_col = _v98_col(out, ['추세필터판정','추세상태'])
    sector_col = _v98_sector_col(out)
    mask = pd.Series(True, index=out.index)

    def _apply_numeric_min(col, threshold, strict=False):
        nonlocal mask
        if col:
            vals = out[col].map(_v98_num)
            if strict:
                mask &= vals > threshold
            else:
                mask &= vals >= threshold
            cond.setdefault('_used_columns', {})[col] = 'used'
        else:
            cond.setdefault('_missing_columns', []).append(f'min >= {threshold}')

    def _apply_numeric_max(col, threshold):
        nonlocal mask
        if col:
            vals = out[col].map(_v98_num)
            mask &= vals <= threshold
            cond.setdefault('_used_columns', {})[col] = 'used'
        else:
            cond.setdefault('_missing_columns', []).append(f'max <= {threshold}')

    if 'max_price' in cond and price_col:
        mask &= out[price_col].map(_v98_num) <= cond['max_price']
    if 'max_change_pct' in cond and change_col:
        mask &= out[change_col].map(_v98_num).abs() <= cond['max_change_pct']
    if 'max_rsi' in cond and rsi_col:
        mask &= out[rsi_col].map(_v98_num) <= cond['max_rsi']
    if 'min_rsi' in cond and rsi_col:
        mask &= out[rsi_col].map(_v98_num) >= cond['min_rsi']
    if 'min_score' in cond and score_col:
        mask &= out[score_col].map(_v98_num) >= cond['min_score']
    if 'min_market_cap_eok' in cond and cap_col:
        mask &= out[cap_col].map(_v98_num) >= cond['min_market_cap_eok']
    if 'max_market_cap_eok' in cond and cap_col:
        mask &= out[cap_col].map(_v98_num) <= cond['max_market_cap_eok']
    if 'min_trade_value_eok' in cond and trade_col:
        mask &= out[trade_col].map(_v98_num) >= cond['min_trade_value_eok']

    # 저평가 성장주 핵심 조건
    if 'min_per' in cond:
        if per_col:
            vals = out[per_col].map(_v98_num)
            if cond.get('min_per_strict'):
                mask &= vals > cond['min_per']
            else:
                mask &= vals >= cond['min_per']
            cond.setdefault('_used_columns', {})[per_col] = 'used'
        else:
            cond.setdefault('_missing_columns', []).append('PER')
    if 'max_per' in cond:
        if per_col:
            mask &= out[per_col].map(_v98_num) <= cond['max_per']
            cond.setdefault('_used_columns', {})[per_col] = 'used'
        else:
            cond.setdefault('_missing_columns', []).append('PER')
    if 'min_sales_growth_3y' in cond:
        if sales_growth_col:
            mask &= out[sales_growth_col].map(_v98_num) >= cond['min_sales_growth_3y']
            cond.setdefault('_used_columns', {})[sales_growth_col] = 'used'
        else:
            cond.setdefault('_missing_columns', []).append('최근3년 매출성장률')
    if 'min_net_income_growth_3y' in cond:
        if net_growth_col:
            mask &= out[net_growth_col].map(_v98_num) >= cond['min_net_income_growth_3y']
            cond.setdefault('_used_columns', {})[net_growth_col] = 'used'
        else:
            cond.setdefault('_missing_columns', []).append('최근3년 순이익성장률')

    if cond.get('exclude_overheat') and over_col:
        mask &= ~out[over_col].astype(str).str.contains('과열|급등|추격', na=False)
    if cond.get('require_trend_pass') and trend_col:
        mask &= out[trend_col].astype(str).str.contains('통과|안정|공격', na=False)
    if cond.get('sector_keyword') and sector_col:
        mask &= out[sector_col].astype(str).str.contains(cond['sector_keyword'], na=False)

    result = out[mask].copy()
    # 필터 근거 컬럼을 앞쪽에 추가
    if not result.empty:
        result.insert(0, '조건검색명', '저평가 성장주' if cond.get('undervalued_growth_mode') else '사용자 조건검색')
        missing = ', '.join(sorted(set(cond.get('_missing_columns', [])))) if cond.get('_missing_columns') else ''
        result.insert(1, '조건적용메모', '일부 성장률/PER 컬럼 없음: ' + missing if missing else '조건 컬럼 적용')
    return result, cond

def _v98_write_df(ws, df, start_row=1, start_col=1, max_rows=None):
    if max_rows is not None:
        df = df.head(max_rows)
    # Make safe string/int/float values
    headers = list(df.columns)
    for j, h in enumerate(headers, start_col):
        ws.cell(start_row, j, h)
    for i, row in enumerate(df.itertuples(index=False), start_row+1):
        for j, val in enumerate(row, start_col):
            if isinstance(val, (np.integer,)):
                val = int(val)
            elif isinstance(val, (np.floating,)):
                val = float(val)
            elif pd.isna(val):
                val = ''
            ws.cell(i, j, val)
    _v98_style_header(ws, start_row, start_col, start_col+len(headers)-1)
    for col in range(start_col, start_col+len(headers)):
        ws.column_dimensions[get_column_letter(col)].width = min(max(12, len(str(ws.cell(start_row,col).value or ''))+4), 36)
    return ws


def apply_v98_market_briefing_patch(xlsx_path=None):
    xlsx_path = Path(xlsx_path) if xlsx_path else _v98_find_output_file()
    wb = load_workbook(xlsx_path)
    df = _v98_get_top_df(wb)
    today = _v98_today_yyyymmdd()

    # 핵심 컬럼
    name_col = _v98_col(df, ['종목명','Name'])
    score_col = _v98_col(df, ['종목점수','최종점수','점수'])
    sector_col = _v98_sector_col(df)
    over_col = _v98_col(df, ['과열판정','급등과열판정','진입판정'])
    verdict_col = _v98_col(df, ['AI 시스템 판정','최종판정','판정'])
    source_col = _v98_col(df, ['후보출처','출처'])
    top_name = str(df.iloc[0][name_col]) if not df.empty and name_col else ''
    market_mode = globals().get('market_mode', '')
    if not market_mode:
        try:
            if '홈_브리핑' in wb.sheetnames:
                market_mode = wb['홈_브리핑']['B3'].value or ''
        except Exception:
            market_mode = ''

    sector_summary = pd.Series(dtype=int)
    if sector_col and not df.empty:
        sector_summary = df[sector_col].replace('', np.nan).dropna().astype(str).value_counts().head(8)
    overheat_count = 0
    if over_col and not df.empty:
        overheat_count = int(df[over_col].astype(str).str.contains('과열|급등|추격|주의', na=False).sum())

    # 시장브리핑 시트
    ws = _v98_recreate_sheet(wb, '시장브리핑', 0)
    ws.merge_cells('A1:H1')
    ws['A1'] = f'시장브리핑 · {today}'
    ws['A1'].fill = PatternFill('solid', fgColor='0F172A')
    ws['A1'].font = Font(color='FFFFFF', bold=True, size=16)
    ws['A1'].alignment = Alignment(horizontal='center')
    rows = [
        ['구분','내용','해석/활용','메모'],
        ['시장모드', market_mode or '확인필요', '공격/선별/방어/관망에 따라 신규매수 강도 조절', '시스템 시장점수 기반'],
        ['분석 후보 수', len(df), '후보가 너무 적으면 필터 과다, 너무 많으면 선별 필요', ''],
        ['상위 후보', top_name, '단, 과열/뉴스/호가 확인 후 접근', ''],
        ['상위 섹터', ', '.join([f'{k}({v})' for k,v in sector_summary.items()]) if len(sector_summary) else '분야확인필요', '강한 섹터 흐름 참고', '토스/네이버/DART/수동분야 기반'],
        ['과열 후보 수', overheat_count, '과열 종목은 추격보다 눌림/돌파 확인', '급등·RSI·거래량·ATR 기반'],
        ['오늘 대응', '보유종목 진단 우선 → 신규 후보는 스나이퍼 확인 → 뉴스/호가 최종 확인', '매매 강도 조절표로 활용', '투자권유 아님'],
    ]
    for r_idx, row in enumerate(rows, 3):
        for c_idx, val in enumerate(row, 1):
            ws.cell(r_idx, c_idx, val)
    _v98_style_header(ws, 3, 1, 4)
    for c,w in {'A':18,'B':34,'C':55,'D':34}.items():
        ws.column_dimensions[c].width = w
    for row in range(4, 10):
        ws.row_dimensions[row].height = 34
        for col in range(1,5):
            ws.cell(row,col).alignment = Alignment(vertical='center', wrap_text=True)

    # 섹터흐름
    start = 12
    ws.cell(start, 1, '상위 섹터 흐름')
    ws.cell(start, 1).font = Font(bold=True, size=13)
    if len(sector_summary):
        ws.cell(start+1,1,'섹터/분야'); ws.cell(start+1,2,'후보수')
        _v98_style_header(ws, start+1, 1, 2, '1E40AF')
        for i,(k,v) in enumerate(sector_summary.items(), start+2):
            ws.cell(i,1,k); ws.cell(i,2,int(v))
    else:
        ws.cell(start+1,1,'섹터 정보가 부족합니다. 종목분야_수동입력.csv 또는 토스 카테고리를 보강하세요.')

    # 뉴스이슈
    ws_news = _v98_recreate_sheet(wb, '뉴스이슈', 1)
    news_rows = []
    has_api = bool((os.environ.get('NAVER_CLIENT_ID') or os.environ.get('NAVER_NEWS_CLIENT_ID')) and (os.environ.get('NAVER_CLIENT_SECRET') or os.environ.get('NAVER_NEWS_CLIENT_SECRET')))
    if has_api:
        for q in MARKET_NEWS_KEYWORDS:
            news_rows.extend(_v98_fetch_naver_news(q, display=3))
            time.sleep(0.1)
    else:
        news_rows.append({'query':'설정필요','title':'네이버 뉴스 API 키가 없어 자동 뉴스 수집은 건너뜀','link':'','pubDate':'','description':'Colab Secrets 또는 환경변수 NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 설정 시 자동 수집됩니다.'})
    news_df = pd.DataFrame(news_rows, columns=['query','title','pubDate','description','link'])
    _v98_write_df(ws_news, news_df, 1, 1, max_rows=80)
    ws_news.column_dimensions['B'].width = 60
    ws_news.column_dimensions['D'].width = 70
    ws_news.column_dimensions['E'].width = 70
    for row in range(2, min(ws_news.max_row+1, 82)):
        ws_news.row_dimensions[row].height = 36
        for col in range(1,6):
            ws_news.cell(row,col).alignment = Alignment(vertical='top', wrap_text=True)

    # 조건검색_후보
    cond_df, cond = _v98_apply_condition(df, NAVER_CONDITION_TEXT)
    ws_cond = _v98_recreate_sheet(wb, '저평가성장주_조건검색', 2)
    ws_cond.merge_cells('A1:H1')
    ws_cond['A1'] = '저평가 성장주 조건검색 · 매출성장률/PER/순이익성장률 필터'
    ws_cond['A1'].fill = PatternFill('solid', fgColor='164E63')
    ws_cond['A1'].font = Font(color='FFFFFF', bold=True, size=14)
    ws_cond['A3'] = '입력 조건'
    ws_cond['B3'] = NAVER_CONDITION_TEXT
    ws_cond['A4'] = '해석된 조건'
    ws_cond['B4'] = json.dumps(cond, ensure_ascii=False)
    ws_cond['A5'] = '주의'
    ws_cond['B5'] = '토스/네이버 추천 알고리즘 복제가 아니라, 현재 시스템 후보군에서 매출성장률·PER·순이익성장률 등 사용 가능한 정량 컬럼으로 재필터링한 결과입니다. 성장률 컬럼이 없으면 조건적용메모에 표시됩니다.'
    for c in ['A','B']:
        ws_cond.column_dimensions[c].width = 24 if c=='A' else 100
    for r in [3,4,5]:
        ws_cond.cell(r,1).font = Font(bold=True)
        ws_cond.cell(r,2).alignment = Alignment(wrap_text=True, vertical='top')
        ws_cond.row_dimensions[r].height = 34
    # 표시할 핵심 컬럼 우선
    if not cond_df.empty:
        desired = [x for x in ['조건검색명','조건적용메모','종목명','표시분야','섹터/분야','후보출처','현재가','등락률','PER','연평균 매출액 증감률','매출증감률(%)','연평균 순이익 증감률','종목점수','진입판정','과열판정','추세필터판정','백테스트신뢰도','추천투입금액','상세전략가이드'] if x in cond_df.columns]
        if not desired:
            desired = list(cond_df.columns[:15])
        _v98_write_df(ws_cond, cond_df[desired], 7, 1, max_rows=60)
        if '상세전략가이드' in desired:
            col_idx = desired.index('상세전략가이드') + 1
            ws_cond.column_dimensions[get_column_letter(col_idx)].width = 80
    else:
        ws_cond['A7'] = '조건에 맞는 후보가 없습니다. 조건을 완화하거나 추천 리스트/후보군을 확인하세요.'

    # 검증결과 보강
    ws_val = wb['검증결과'] if '검증결과' in wb.sheetnames else _v98_recreate_sheet(wb, '검증결과')
    start_row = ws_val.max_row + 2 if ws_val.max_row else 1
    rows = [
        ['v9.8.1 시장브리핑 시트 존재', 'PASS' if '시장브리핑' in wb.sheetnames else 'FAIL', '시장브리핑 생성'],
        ['v9.8 뉴스이슈 시트 존재', 'PASS' if '뉴스이슈' in wb.sheetnames else 'FAIL', '네이버 뉴스 API 키 있으면 자동 수집'],
        ['v9.8.1 저평가성장주_조건검색 시트 존재', 'PASS' if '저평가성장주_조건검색' in wb.sheetnames else 'FAIL', NAVER_CONDITION_TEXT],
        ['조건검색 후보 수', len(cond_df), '조건이 너무 엄격하면 0개일 수 있음'],
        ['뉴스 API 연결 상태', 'PASS' if has_api else 'WARN', '키 없으면 뉴스 수집 건너뜀'],
    ]
    for i,row in enumerate(rows, start_row):
        for j,val in enumerate(row, 1):
            ws_val.cell(i,j,val)
    if start_row == 1:
        _v98_style_header(ws_val, 1, 1, 3)
    ws_val.column_dimensions['A'].width = 34
    ws_val.column_dimensions['B'].width = 14
    ws_val.column_dimensions['C'].width = 80

    # 시트 순서: 시장브리핑/뉴스이슈/저평가성장주 조건검색을 앞쪽으로
    front = ['시장브리핑','뉴스이슈','저평가성장주_조건검색','홈_브리핑','모바일_대시보드','TOP후보_요약','종목카드_TOP15','진입시나리오','보유종목_입력','보유종목_진단','스나이퍼_계산기','백테스트_실험실','검증결과']
    existing = [wb[s] for s in front if s in wb.sheetnames]
    rest = [wb[s] for s in wb.sheetnames if s not in front]
    wb._sheets = existing + rest

    # 최종 저장명: 한국시간 기준 날짜
    out_name = f'{today}.xlsx'
    out_path = xlsx_path.with_name(out_name)
    wb.save(out_path)
    print(f'✅ v9.8.1 시장브리핑/저평가성장주 조건검색 패치 완료: {out_path}')
    print(f'📌 저평가 성장주 조건검색 후보: {len(cond_df)}개 / 뉴스 API: {"연결" if has_api else "미연결"}')
    return str(out_path)

# 실행
try:
    output_file = apply_v98_market_briefing_patch(globals().get('output_file'))
except Exception as e:
    print('❌ v9.8.1 시장브리핑/조건검색 패치 실패:', repr(e))
    raise



✅ v9.8.1 시장브리핑/저평가성장주 조건검색 패치 완료: 20260608.xlsx
📌 저평가 성장주 조건검색 후보: 15개 / 뉴스 API: 연결


# v10.2 연속추천 관찰 기능

이 섹션은 날짜별 과거 리포트와 오늘 추천 리스트를 비교해서, 같은 종목이 며칠째 반복 추천되는지 계산합니다.

- `연속추천_관찰` 시트 추가
- `추천 리스트`, `TOP후보_요약`에 연속추천 관련 컬럼 추가
- HTML 대시보드에 `연속추천` 탭 추가


In [27]:

# 99-2. v10.2 연속추천 관찰 기능
# - 과거 날짜별 리포트와 오늘 리포트를 비교해서 반복 추천 종목을 표시합니다.
# - 특정 종목이 여러 번/며칠 연속 등장하면 "주목 후보"로 따로 확인할 수 있습니다.

from pathlib import Path
from datetime import datetime, timedelta, timezone
import re, os, math
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

KST_V102 = timezone(timedelta(hours=9))


def _v102_today_key():
    forced = None
    try:
        forced = CONFIG.get('FORCE_REPORT_DATE')
    except Exception:
        forced = None
    if forced:
        return str(forced).replace('-', '')
    return datetime.now(KST_V102).strftime('%Y%m%d')


def _v102_extract_date(path):
    m = re.search(r'(20\d{6})', str(path))
    return m.group(1) if m else ''


def _v102_report_roots():
    roots = []
    try:
        if CONFIG.get('USE_GOOGLE_DRIVE') and CONFIG.get('DRIVE_ROOT'):
            roots.append(Path(str(CONFIG.get('DRIVE_ROOT'))))
    except Exception:
        pass
    try:
        roots.append(Path(str(CONFIG.get('REPORT_ROOT', 'stock_report'))))
    except Exception:
        pass
    roots += [Path('stock_report'), Path('.'), Path('/content')]
    out = []
    seen = set()
    for r in roots:
        try:
            r = r.expanduser()
            key = str(r.resolve()) if r.exists() else str(r)
            if key not in seen:
                seen.add(key)
                out.append(r)
        except Exception:
            if str(r) not in seen:
                seen.add(str(r)); out.append(r)
    return out


def _v102_find_xlsx_files(current_path=None, lookback_days=20):
    current = Path(str(current_path)) if current_path else None
    files = []
    for root in _v102_report_roots():
        if not root.exists():
            continue
        patterns = ['20*.xlsx', 'reports/**/20*.xlsx', 'latest/*.xlsx', '**/20*.xlsx']
        for pat in patterns:
            try:
                files.extend(root.glob(pat))
            except Exception:
                pass
    # 중복 제거 + 임시파일 제거
    uniq = []
    seen = set()
    for f in files:
        try:
            if not f.exists() or f.name.startswith('~$'):
                continue
            if current is not None and f.resolve() == current.resolve():
                continue
            date_key = _v102_extract_date(f)
            if not date_key:
                continue
            key = str(f.resolve())
            if key not in seen:
                seen.add(key); uniq.append(f)
        except Exception:
            pass
    # 최근 N일 정도만 사용
    today = _v102_today_key()
    try:
        today_dt = datetime.strptime(today, '%Y%m%d')
        cutoff = today_dt - timedelta(days=lookback_days)
        filtered = []
        for f in uniq:
            try:
                d = datetime.strptime(_v102_extract_date(f), '%Y%m%d')
                if d >= cutoff:
                    filtered.append(f)
            except Exception:
                filtered.append(f)
        uniq = filtered
    except Exception:
        pass
    return sorted(uniq, key=lambda p: (_v102_extract_date(p), str(p)))


def _v102_pick_col(df, names):
    if df is None or df.empty:
        return None
    cols = list(df.columns)
    norm = {str(c).replace(' ', '').strip(): c for c in cols}
    for n in names:
        key = str(n).replace(' ', '').strip()
        if key in norm:
            return norm[key]
    for c in cols:
        cs = str(c).replace(' ', '').strip()
        for n in names:
            if str(n).replace(' ', '').strip() in cs:
                return c
    return None


def _v102_make_key(name, code=''):
    code = '' if pd.isna(code) else str(code).strip()
    name = '' if pd.isna(name) else str(name).strip()
    if code and code.lower() not in ['nan', 'none', '0']:
        return code.zfill(6) if code.isdigit() and len(code) <= 6 else code
    return name


def _v102_read_recommend_sheet(xlsx_path, date_key=None):
    date_key = date_key or _v102_extract_date(xlsx_path)
    try:
        sheets = pd.read_excel(xlsx_path, sheet_name=None, engine='openpyxl')
    except Exception:
        return pd.DataFrame()
    df = None
    for sheet_name in ['TOP후보_요약', '추천 리스트', '신규후보', '종목카드_TOP15']:
        if sheet_name in sheets and not sheets[sheet_name].empty:
            df = sheets[sheet_name].copy()
            break
    if df is None or df.empty:
        return pd.DataFrame()
    name_col = _v102_pick_col(df, ['종목명', '종목', '주식명'])
    code_col = _v102_pick_col(df, ['종목코드', '코드'])
    score_col = _v102_pick_col(df, ['종목점수', '최종점수', '점수'])
    rank_col = _v102_pick_col(df, ['순위', 'Rank', '랭크'])
    verdict_col = _v102_pick_col(df, ['AI 시스템 판정', '최종판정', '진입판정', '과열판정'])
    source_col = _v102_pick_col(df, ['후보출처', '출처'])
    sector_col = _v102_pick_col(df, ['섹터/분야', '표시분야', '분야', '섹터'])
    if not name_col:
        return pd.DataFrame()
    rows = []
    for i, row in df.iterrows():
        name = row.get(name_col, '')
        if pd.isna(name) or str(name).strip() in ['', 'nan', '종목명']:
            continue
        code = row.get(code_col, '') if code_col else ''
        key = _v102_make_key(name, code)
        rank_val = row.get(rank_col, i+1) if rank_col else i+1
        try:
            rank_val = int(float(str(rank_val).replace('#','')))
        except Exception:
            rank_val = int(i+1)
        rows.append({
            '날짜': date_key,
            '종목키': key,
            '종목명': str(name).strip(),
            '종목코드': '' if pd.isna(code) else str(code).strip(),
            '순위': rank_val,
            '점수': row.get(score_col, '') if score_col else '',
            '후보출처': row.get(source_col, '') if source_col else '',
            '표시분야': row.get(sector_col, '') if sector_col else '',
            '판정': row.get(verdict_col, '') if verdict_col else '',
        })
    return pd.DataFrame(rows)


def _v102_build_current_df(xlsx_path):
    today = _v102_today_key()
    return _v102_read_recommend_sheet(xlsx_path, today)


def _v102_continuity_table(current_df, history_df, top_n=30):
    if current_df is None or current_df.empty:
        return pd.DataFrame(columns=['주목등급','종목명','종목코드','표시분야','오늘순위','오늘점수','연속추천일수','최근7회추천횟수','최근등장일','평균순위','최고순위','최근등장흐름','후보출처','판정','메모'])
    all_df = pd.concat([history_df, current_df], ignore_index=True) if history_df is not None and not history_df.empty else current_df.copy()
    all_df = all_df.dropna(subset=['종목키'])
    all_df['날짜'] = all_df['날짜'].astype(str)
    all_df = all_df.sort_values(['날짜','순위'])
    report_dates = sorted(all_df['날짜'].dropna().unique().tolist())
    recent_dates = report_dates[-7:]
    today = _v102_today_key()
    if today not in report_dates and not current_df.empty:
        today = str(current_df['날짜'].iloc[0])
    rows = []
    for _, cur in current_df.head(top_n).iterrows():
        key = cur['종목키']
        sub = all_df[all_df['종목키'] == key].copy()
        sub_dates = sorted(sub['날짜'].unique().tolist())
        # 최근 리포트 기준 연속 등장 횟수
        streak = 0
        for d in reversed(report_dates):
            if d in sub_dates:
                streak += 1
            else:
                if d <= today:
                    break
        recent_count = int(sub[sub['날짜'].isin(recent_dates)]['날짜'].nunique())
        ranks = pd.to_numeric(sub['순위'], errors='coerce').dropna()
        avg_rank = round(float(ranks.mean()), 1) if not ranks.empty else ''
        best_rank = int(ranks.min()) if not ranks.empty else ''
        flow = ' → '.join([f"{r['날짜'][-4:]} #{int(r['순위'])}" for _, r in sub.tail(5).iterrows() if pd.notna(r.get('순위'))])
        today_rank = cur.get('순위','')
        today_score = cur.get('점수','')
        try:
            tr = int(float(today_rank))
        except Exception:
            tr = 999
        if streak >= 4:
            grade = '🔥 강한 지속 후보'
            memo = f'{streak}회 연속 추천. 수급/추세가 계속 유지되는지 우선 관찰.'
        elif streak >= 3:
            grade = '⭐ 연속 주목 후보'
            memo = f'{streak}회 연속 추천. 신규 진입보다 눌림/조건 확인 후 접근.'
        elif recent_count >= 3:
            grade = '🔁 반복 등장 후보'
            memo = f'최근 7회 리포트 중 {recent_count}회 등장. 관심 목록에 올려 추적.'
        elif streak >= 2:
            grade = '👀 2회 연속 관찰'
            memo = '이틀 연속 등장. 다음 리포트에서도 유지되는지 확인.'
        else:
            grade = '신규/단발 후보'
            memo = '오늘 처음 또는 단발성 등장. 과열/뉴스 확인 우선.'
        if tr <= 5 and streak >= 2:
            memo += ' 오늘 TOP5권이라 우선순위 높음.'
        rows.append({
            '주목등급': grade,
            '종목명': cur.get('종목명',''),
            '종목코드': cur.get('종목코드',''),
            '표시분야': cur.get('표시분야',''),
            '오늘순위': today_rank,
            '오늘점수': today_score,
            '연속추천일수': streak,
            '최근7회추천횟수': recent_count,
            '최근등장일': ', '.join(sub_dates[-5:]),
            '평균순위': avg_rank,
            '최고순위': best_rank,
            '최근등장흐름': flow,
            '후보출처': cur.get('후보출처',''),
            '판정': cur.get('판정',''),
            '메모': memo,
        })
    out = pd.DataFrame(rows)
    grade_order = {'🔥 강한 지속 후보':0,'⭐ 연속 주목 후보':1,'🔁 반복 등장 후보':2,'👀 2회 연속 관찰':3,'신규/단발 후보':4}
    if not out.empty:
        out['_order'] = out['주목등급'].map(grade_order).fillna(9)
        out = out.sort_values(['_order','오늘순위']).drop(columns=['_order'])
    return out


def _v102_style_sheet(ws):
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    thin = Side(style='thin', color='CBD5E1')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = border
    widths = {
        'A':18,'B':18,'C':12,'D':18,'E':10,'F':10,'G':12,'H':14,'I':24,'J':10,'K':10,'L':34,'M':14,'N':18,'O':48
    }
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    ws.freeze_panes = 'A2'
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical='center', wrap_text=True)
    ws.auto_filter.ref = ws.dimensions


def _v102_update_sheet_with_df(wb, sheet_name, df, index=3):
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name, index=min(index, len(wb.sheetnames)))
    if df is None or df.empty:
        df = pd.DataFrame([{'메모':'연속추천을 계산할 후보 데이터가 없습니다. 며칠간 리포트가 누적되면 자동으로 표시됩니다.'}])
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append(r)
    _v102_style_sheet(ws)
    return ws


def _v102_add_columns_to_sheet(wb, sheet_name, cont_df):
    if sheet_name not in wb.sheetnames or cont_df is None or cont_df.empty:
        return
    ws = wb[sheet_name]
    headers = [ws.cell(1, c).value for c in range(1, ws.max_column+1)]
    def find_header(candidates):
        for cand in candidates:
            for idx, h in enumerate(headers, 1):
                if h and str(cand).replace(' ','') in str(h).replace(' ',''):
                    return idx
        return None
    name_col = find_header(['종목명','종목'])
    code_col = find_header(['종목코드','코드'])
    if not name_col:
        return
    add_headers = ['연속추천일수','최근7회추천횟수','반복추천등급','반복추천메모']
    start_col = ws.max_column + 1
    for i, h in enumerate(add_headers, start_col):
        ws.cell(1, i, h)
    map_by_key = {}
    for _, row in cont_df.iterrows():
        key = _v102_make_key(row.get('종목명',''), row.get('종목코드',''))
        map_by_key[key] = row
    for r in range(2, ws.max_row+1):
        name = ws.cell(r, name_col).value
        code = ws.cell(r, code_col).value if code_col else ''
        key = _v102_make_key(name, code)
        item = map_by_key.get(key)
        if item is None:
            fallback_key = str(name).strip() if name else ''
            item = map_by_key.get(fallback_key)
        if item is None:
            continue
        vals = [item.get('연속추천일수',''), item.get('최근7회추천횟수',''), item.get('주목등급',''), item.get('메모','')]
        for i, v in enumerate(vals, start_col):
            ws.cell(r, i, v)
    for c in range(start_col, start_col+len(add_headers)):
        ws.cell(1,c).fill = PatternFill('solid', fgColor='1F4E78')
        ws.cell(1,c).font = Font(color='FFFFFF', bold=True)
        ws.column_dimensions[get_column_letter(c)].width = 18 if c < start_col+3 else 42


def apply_v102_continuous_recommendation_patch(xlsx_path=None):
    global output_file
    xlsx = Path(str(xlsx_path or globals().get('output_file', '')))
    if not xlsx.exists():
        # 오늘 날짜 파일 탐색
        today = _v102_today_key()
        candidates = list(Path('.').glob(f'{today}*.xlsx')) + list(Path('.').glob('20*.xlsx'))
        if not candidates:
            print('⚠️ v10.2 연속추천 패치: 엑셀 파일을 찾지 못했습니다.')
            return None, pd.DataFrame()
        xlsx = max(candidates, key=lambda p: p.stat().st_mtime)
    print(f'🔁 v10.2 연속추천 관찰 패치 대상: {xlsx}')
    current = _v102_build_current_df(xlsx)
    hist_files = _v102_find_xlsx_files(xlsx, lookback_days=30)
    hist_dfs = []
    for f in hist_files:
        d = _v102_extract_date(f)
        try:
            hist = _v102_read_recommend_sheet(f, d)
            if not hist.empty:
                hist_dfs.append(hist)
        except Exception as e:
            print(f'⚠️ 과거 리포트 읽기 실패: {f.name} / {e}')
    history = pd.concat(hist_dfs, ignore_index=True) if hist_dfs else pd.DataFrame()
    cont = _v102_continuity_table(current, history, top_n=int(CONFIG.get('UNIFIED_TOP_N', 15) or 15))
    wb = load_workbook(xlsx)
    _v102_update_sheet_with_df(wb, '연속추천_관찰', cont, index=4)
    _v102_add_columns_to_sheet(wb, '추천 리스트', cont)
    _v102_add_columns_to_sheet(wb, 'TOP후보_요약', cont)
    # 검증결과 시트에 항목 추가
    if '검증결과' in wb.sheetnames:
        ws = wb['검증결과']
    else:
        ws = wb.create_sheet('검증결과')
        ws.append(['검증항목','상태','메모'])
    ws.append(['연속추천_관찰 시트 생성', 'PASS' if '연속추천_관찰' in wb.sheetnames else 'FAIL', f'과거 리포트 {len(hist_files)}개 참조 / 현재 후보 {len(current)}개'])
    ws.append(['반복추천 컬럼 추가', 'PASS', '추천 리스트/TOP후보_요약에 연속추천 관련 컬럼 추가 시도'])
    wb.save(xlsx)
    output_file = str(xlsx)
    print(f'✅ v10.2 연속추천 관찰 패치 완료: {xlsx} / 관찰 후보 {len(cont)}개 / 과거 리포트 {len(hist_files)}개')
    return str(xlsx), cont

output_file, continuous_recommend_df = apply_v102_continuous_recommendation_patch(globals().get('output_file', None))



🔁 v10.2 연속추천 관찰 패치 대상: 20260608.xlsx


✅ v10.2 연속추천 관찰 패치 완료: 20260608.xlsx / 관찰 후보 15개 / 과거 리포트 107개


# v9.9 HTML 대시보드 + 자동 리포트 준비판

이 섹션은 기존 엑셀 리포트 생성 후, 매일 보기 쉬운 `index.html` 대시보드를 추가로 생성합니다.

- `docs/index.html`: GitHub Pages 또는 로컬/Drive에서 열기 좋은 오늘 리포트
- `docs/reports/YYYYMMDD/index.html`: 날짜별 HTML 리포트 보관
- `html_report_package_YYYYMMDD.zip`: HTML 리포트 묶음
- `automation/`: GitHub Actions / Drive 업로드 준비용 파일

엑셀은 계산·보관용으로 남기고, 매일 보는 화면은 HTML 중심으로 보는 구조입니다.


In [28]:

# 100. v9.9 HTML 대시보드 생성
# 기존 output_file(.xlsx)을 읽어 모바일/웹용 정적 HTML 리포트를 만듭니다.

import os, re, html, zipfile, shutil
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
import numpy as np

KST = timezone(timedelta(hours=9))

def v99_report_date():
    force = None
    try:
        force = CONFIG.get('FORCE_REPORT_DATE')
    except Exception:
        force = None
    if force:
        return str(force).replace('-', '')
    return datetime.now(KST).strftime('%Y%m%d')

def v99_find_xlsx():
    try:
        p = Path(str(output_file))
        if p.exists() and p.suffix.lower() == '.xlsx':
            return p
    except Exception:
        pass
    today = v99_report_date()
    candidates = []
    for root in [Path('.'), Path('/content')]:
        if root.exists():
            candidates += list(root.glob(f'{today}*.xlsx'))
            candidates += list(root.glob('20*.xlsx'))
            candidates += list(root.glob('실전매매*.xlsx'))
    candidates = [p for p in candidates if p.exists() and not p.name.startswith('~$')]
    if not candidates:
        raise FileNotFoundError('HTML 대시보드를 만들 엑셀 파일을 찾지 못했습니다. output_file 또는 날짜형 xlsx 파일을 확인하세요.')
    return sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)[0]

def v99_safe_df(sheet_map, sheet_name):
    df = sheet_map.get(sheet_name)
    if df is None:
        return pd.DataFrame()
    df = df.copy().dropna(axis=0, how='all').dropna(axis=1, how='all')
    df.columns = [str(c).strip() if not str(c).startswith('Unnamed') else '' for c in df.columns]
    return df

def v99_pick_col(df, names, default=None):
    if df is None or df.empty:
        return default
    cols = {str(c).strip(): c for c in df.columns}
    for n in names:
        if n in cols:
            return cols[n]
    for n in names:
        for c in df.columns:
            if n and n in str(c):
                return c
    return default

def v99_fmt(v, money=False):
    try:
        if pd.isna(v):
            return '-'
    except Exception:
        pass
    if isinstance(v, (int, float, np.integer, np.floating)):
        if money:
            return f'{int(round(v)):,}원'
        if abs(v) >= 1000:
            return f'{int(round(v)):,}'
        return f'{v:.2f}'.rstrip('0').rstrip('.')
    s = str(v).strip()
    return html.escape(s) if s else '-'

def v99_html_table(df, max_rows=30):
    if df is None or df.empty:
        return '<div class="empty">표시할 데이터가 없습니다.</div>'
    show = df.copy().head(max_rows)
    if len(show.columns) > 12:
        show = show.iloc[:, :12]
    headers = ''.join([f'<th>{html.escape(str(c))}</th>' for c in show.columns])
    rows = []
    for _, r in show.iterrows():
        tds = ''.join([f'<td>{v99_fmt(r[c])}</td>' for c in show.columns])
        rows.append(f'<tr>{tds}</tr>')
    return '<div class="table-wrap"><table class="data-table"><thead><tr>' + headers + '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table></div>'

def v99_stock_cards(sheet_map, limit=15):
    df = v99_safe_df(sheet_map, 'TOP후보_요약')
    if df.empty:
        df = v99_safe_df(sheet_map, '추천 리스트')
    if df.empty:
        return '<div class="empty">신규 후보 데이터가 없습니다.</div>'
    name_col = v99_pick_col(df, ['종목명','종목'])
    sector_col = v99_pick_col(df, ['섹터/분야','표시분야','분야','섹터'])
    price_col = v99_pick_col(df, ['현재가'])
    score_col = v99_pick_col(df, ['종목점수','최종점수','점수'])
    src_col = v99_pick_col(df, ['후보출처'])
    judge_col = v99_pick_col(df, ['AI 시스템 판정','최종판정','진입판정','판정'])
    entry_col = v99_pick_col(df, ['진입판정'])
    overheat_col = v99_pick_col(df, ['과열판정','급등과열판정','과열필터판정'])
    guide_col = v99_pick_col(df, ['상세전략가이드','상세전략 가이드','상세 가이드'])
    cards=[]
    for idx, row in df.head(limit).iterrows():
        rank = len(cards)+1
        name = v99_fmt(row[name_col]) if name_col else f'후보 {rank}'
        sector = v99_fmt(row[sector_col]) if sector_col else '-'
        price = v99_fmt(row[price_col], money=True) if price_col else '-'
        score = v99_fmt(row[score_col]) if score_col else '-'
        src = v99_fmt(row[src_col]) if src_col else '-'
        judge = v99_fmt(row[judge_col]) if judge_col else '후보'
        entry = v99_fmt(row[entry_col]) if entry_col else '-'
        overheat = v99_fmt(row[overheat_col]) if overheat_col else '-'
        guide = v99_fmt(row[guide_col]) if guide_col else '상세 가이드는 엑셀에서 확인하세요.'
        badge = 'warn' if ('관찰' in judge or '선별' in judge or '대기' in entry or '과열' in overheat) else 'good'
        card = '<article class="stock-card">'
        card += f'<div class="stock-head"><div><span class="rank">#{rank}</span><h2>{name}</h2><p class="sub">{sector} · {src}</p></div><span class="badge {badge}">{judge}</span></div>'
        card += '<div class="metric-grid">'
        card += f'<div><label>현재가</label><strong>{price}</strong></div><div><label>점수</label><strong>{score}</strong></div><div><label>진입</label><strong>{entry}</strong></div><div><label>과열</label><strong>{overheat}</strong></div>'
        card += '</div>'
        card += f'<details><summary>상세 가이드</summary><p>{guide}</p></details></article>'
        cards.append(card)
    return '\n'.join(cards)

def v99_holding_cards(sheet_map):
    df = v99_safe_df(sheet_map, '보유종목_진단')
    if df.empty:
        return '<div class="empty">보유종목_입력 시트에 종목을 입력하면 이곳에 표시됩니다.</div>'
    name_col = v99_pick_col(df, ['종목명','종목'])
    sector_col = v99_pick_col(df, ['섹터/분야','표시분야','분야','섹터'])
    cur_col = v99_pick_col(df, ['현재가'])
    avg_col = v99_pick_col(df, ['평균단가'])
    qty_col = v99_pick_col(df, ['보유수량','수량'])
    ret_col = v99_pick_col(df, ['수익률','평가수익률'])
    pnl_col = v99_pick_col(df, ['평가손익','손익금액'])
    action_col = v99_pick_col(df, ['보유판정','권장대응','매도판정'])
    cards=[]
    for _, row in df.head(20).iterrows():
        name = v99_fmt(row[name_col]) if name_col else '-'
        sector = v99_fmt(row[sector_col]) if sector_col else '-'
        cur = v99_fmt(row[cur_col], money=True) if cur_col else '-'
        avg = v99_fmt(row[avg_col], money=True) if avg_col else '-'
        qty = v99_fmt(row[qty_col]) if qty_col else '-'
        ret = v99_fmt(row[ret_col]) if ret_col else '-'
        pnl = v99_fmt(row[pnl_col], money=True) if pnl_col else '-'
        action = v99_fmt(row[action_col]) if action_col else '진단 확인'
        card = '<article class="mini-card">'
        card += f'<h3>{name} <span>{sector}</span></h3><div class="mini-grid">'
        card += f'<p><label>현재가</label>{cur}</p><p><label>평균단가</label>{avg}</p><p><label>수량</label>{qty}</p><p><label>수익</label>{ret} / {pnl}</p></div><b class="action">{action}</b></article>'
        cards.append(card)
    return '\n'.join(cards)

def v99_generate_html():
    xlsx_path = v99_find_xlsx()
    report_date = v99_report_date()
    sheets = pd.read_excel(xlsx_path, sheet_name=None, engine='openpyxl')
    topdf = v99_safe_df(sheets, 'TOP후보_요약')
    recdf = v99_safe_df(sheets, '추천 리스트')
    src_df = topdf if not topdf.empty else recdf
    top_name = '-'
    if not src_df.empty:
        name_col = v99_pick_col(src_df, ['종목명','종목'])
        if name_col:
            top_name = v99_fmt(src_df.iloc[0][name_col])
    holdings = v99_safe_df(sheets, '보유종목_진단')
    kpis = f'<div class="kpi"><label>상위 후보</label><strong>{top_name}</strong></div><div class="kpi"><label>분석 후보</label><strong>{len(recdf) if not recdf.empty else len(src_df)}개</strong></div><div class="kpi"><label>보유종목</label><strong>{len(holdings) if not holdings.empty else 0}개</strong></div>'
    briefing = v99_html_table(v99_safe_df(sheets, '시장브리핑'), 20)
    news = v99_html_table(v99_safe_df(sheets, '뉴스이슈'), 20)
    condition = v99_html_table(v99_safe_df(sheets, '저평가성장주_조건검색'), 25)
    if '저평가성장주_조건검색' not in sheets:
        condition = v99_html_table(v99_safe_df(sheets, '조건검색_후보'), 25)
    continuous = v99_html_table(v99_safe_df(sheets, '연속추천_관찰'), 25)
    sniper = v99_html_table(v99_safe_df(sheets, '스나이퍼_계산기'), 35)
    lab = v99_html_table(v99_safe_df(sheets, '백테스트_실험실'), 45)
    created_at = datetime.now(KST).strftime('%Y-%m-%d %H:%M')
    css = ':root{--bg:#f4f6fb;--card:#fff;--text:#111827;--muted:#64748b;--line:#e5e7eb}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);font-family:-apple-system,BlinkMacSystemFont,"Segoe UI","Noto Sans KR",Arial,sans-serif}header{background:linear-gradient(135deg,#0f172a,#1e40af);color:white;padding:28px 20px 34px;border-radius:0 0 28px 28px}.wrap{max-width:1180px;margin:-22px auto 42px;padding:0 16px}.kpis{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin-bottom:14px}.kpi,.panel,.stock-card,.mini-card{background:#fff;border:1px solid var(--line);border-radius:22px;box-shadow:0 10px 28px rgba(15,23,42,.08)}.kpi{padding:16px}.kpi label{display:block;font-size:13px;color:var(--muted);margin-bottom:6px}.kpi strong{font-size:20px}.tabs{display:flex;gap:8px;overflow-x:auto;padding:8px 0 14px;position:sticky;top:0;background:var(--bg);z-index:5}.tabs a{white-space:nowrap;text-decoration:none;color:#1e3a8a;background:#e0e7ff;padding:10px 13px;border-radius:999px;font-weight:800;font-size:13px}.panel{padding:18px;margin:14px 0}.stock-card{padding:18px;margin:14px 0}.stock-head{display:flex;justify-content:space-between;gap:12px;align-items:flex-start;border-bottom:1px solid var(--line);padding-bottom:12px}.stock-head h2{display:inline;margin:0;font-size:22px}.sub{margin:6px 0 0;color:var(--muted);font-weight:700}.rank{display:inline-flex;align-items:center;justify-content:center;width:34px;height:34px;border-radius:12px;background:#eef2ff;color:#3730a3;font-weight:900;margin-right:10px}.badge{padding:8px 12px;border-radius:999px;font-size:13px;font-weight:900;white-space:nowrap}.badge.good{background:#dcfce7;color:#166534}.badge.warn{background:#fef3c7;color:#92400e}.metric-grid,.mini-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin:16px 0}.metric-grid div,.mini-grid p{background:#f8fafc;border-radius:14px;padding:12px;margin:0}.metric-grid label,.mini-grid label{display:block;color:var(--muted);font-size:12px;margin-bottom:6px}.mini-card{padding:15px;margin:10px 0}.mini-card h3 span{font-size:12px;color:#475569;background:#f1f5f9;padding:5px 8px;border-radius:999px}.table-wrap{overflow:auto;border:1px solid var(--line);border-radius:16px}.data-table{border-collapse:collapse;width:100%;font-size:13px;background:white}.data-table th{background:#f1f5f9;text-align:left;color:#334155;position:sticky;top:0}.data-table th,.data-table td{padding:10px;border-bottom:1px solid #eef2f7;vertical-align:top}.empty{background:#f8fafc;border:1px dashed #cbd5e1;border-radius:16px;padding:20px;color:#64748b}@media(max-width:760px){.kpis,.metric-grid,.mini-grid{grid-template-columns:1fr}.stock-head{flex-direction:column}}'
    page = '<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>실전매매 리포트 '+report_date+'</title><style>'+css+'</style></head><body>'
    page += '<header><h1>실전매매 통합 리포트</h1><p>'+report_date+' · '+created_at+' KST · 투자 판단 보조용</p></header><main class="wrap">'
    page += '<section class="kpis">'+kpis+'</section><nav class="tabs"><a href="#briefing">시장브리핑</a><a href="#holdings">보유종목</a><a href="#candidates">신규후보</a><a href="#condition">저평가성장주</a><a href="#continuous">연속추천</a><a href="#sniper">스나이퍼</a><a href="#backtest">백테스트</a><a href="#news">뉴스</a></nav>'
    page += '<section id="briefing" class="panel"><h2>오늘 시장브리핑</h2>'+briefing+'</section>'
    page += '<section id="holdings" class="panel"><h2>보유종목 진단</h2>'+v99_holding_cards(sheets)+'</section>'
    page += '<section id="candidates" class="panel"><h2>신규 후보 TOP</h2>'+v99_stock_cards(sheets,15)+'</section>'
    page += '<section id="condition" class="panel"><h2>저평가 성장주 조건검색</h2>'+condition+'</section>'
    page += '<section id="continuous" class="panel"><h2>연속추천 관찰</h2><p class="muted">며칠째 반복 등장하는 종목은 단발성 후보보다 우선 관찰합니다. 단, 급등 과열 후보는 눌림/조건 확인 후 접근하세요.</p>'+continuous+'</section>'
    page += '<section id="sniper" class="panel"><h2>스나이퍼 계산기 요약</h2>'+sniper+'</section>'
    page += '<section id="backtest" class="panel"><h2>백테스트 실험실 요약</h2>'+lab+'</section>'
    page += '<section id="news" class="panel"><h2>뉴스 이슈</h2>'+news+'</section>'
    page += '</main><footer style="text-align:center;color:#64748b;font-size:12px;padding:30px 10px 40px">본 리포트는 투자 판단 보조용이며 매수·매도 권유가 아닙니다. 실제 매매 전 공시·뉴스·호가·손실한도를 확인하세요.</footer></body></html>'
    docs = Path('docs')
    day_dir = docs / 'reports' / report_date
    day_dir.mkdir(parents=True, exist_ok=True)
    (day_dir / 'index.html').write_text(page, encoding='utf-8')
    docs.mkdir(exist_ok=True)
    (docs / 'index.html').write_text(page, encoding='utf-8')
    try:
        shutil.copy2(xlsx_path, day_dir / xlsx_path.name)
    except Exception:
        pass
    zip_path = Path(f'html_report_package_{report_date}.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in docs.rglob('*'):
            if p.is_file():
                z.write(p, p.as_posix())
    print(f'✅ HTML 대시보드 생성 완료: {docs / "index.html"}')
    print(f'✅ 날짜별 리포트 생성 완료: {day_dir / "index.html"}')
    print(f'✅ HTML 패키지 생성 완료: {zip_path}')
    return str(docs / 'index.html'), str(day_dir / 'index.html'), str(zip_path)

html_index_file, html_daily_file, html_package_file = v99_generate_html()



✅ HTML 대시보드 생성 완료: docs/index.html
✅ 날짜별 리포트 생성 완료: docs/reports/20260608/index.html
✅ HTML 패키지 생성 완료: html_report_package_20260608.zip


In [29]:

# 101. v9.9 자동화 준비 파일 생성
from pathlib import Path
import zipfile

auto_root = Path('automation')
(auto_root / '.github' / 'workflows').mkdir(parents=True, exist_ok=True)
(auto_root / 'scripts').mkdir(parents=True, exist_ok=True)
(auto_root / 'requirements_v9_9.txt').write_text('pandas\nnumpy\nopenpyxl\nrequests\nbeautifulsoup4\nlxml\nfinance-datareader\npykrx\nnbconvert\njupyter\n', encoding='utf-8')
(auto_root / '.github' / 'workflows' / 'daily-report.yml').write_text('name: daily-stock-report\n\non:\n  schedule:\n    - cron: \'30 23 * * 0-5\'\n  workflow_dispatch:\n\npermissions:\n  contents: write\n  pages: write\n  id-token: write\n\njobs:\n  build-report:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: \'3.11\'\n      - name: Install dependencies\n        run: pip install -r automation/requirements_v9_9.txt\n      - name: Run notebook\n        env:\n          OPEN_DART_API_KEY: ${{ secrets.OPEN_DART_API_KEY }}\n          NAVER_CLIENT_ID: ${{ secrets.NAVER_CLIENT_ID }}\n          NAVER_CLIENT_SECRET: ${{ secrets.NAVER_CLIENT_SECRET }}\n        run: |\n          jupyter nbconvert --to notebook --execute "실전매매_통합시스템_v9_9_HTML대시보드_자동리포트준비.ipynb" --output executed_report.ipynb --ExecutePreprocessor.timeout=3600\n      - name: Commit generated docs\n        run: |\n          git config user.name "github-actions[bot]"\n          git config user.email "github-actions[bot]@users.noreply.github.com"\n          git add docs automation || true\n          git commit -m "Update daily stock report" || echo "No changes to commit"\n          git push || true\n', encoding='utf-8')
(auto_root / 'scripts' / 'upload_to_drive.py').write_text("# Google Drive 업로드 스켈레톤\nfrom pathlib import Path\nTARGET = Path('docs/index.html')\nprint('Drive upload skeleton. Target:', TARGET)\nprint('실제 Drive API 인증 코드는 프로젝트 보안 설정 후 연결하세요.')\n", encoding='utf-8')
(auto_root / 'README_자동화설정.md').write_text('# v9.9 자동 리포트 운영 가이드\n\n## 1. 기본 구조\n- `docs/index.html`: 오늘 리포트 홈페이지\n- `docs/reports/YYYYMMDD/index.html`: 날짜별 리포트 보관\n- `html_report_package_YYYYMMDD.zip`: HTML 리포트 묶음\n\n## 2. GitHub Pages 방식\n1. GitHub 저장소를 만듭니다.\n2. 이 노트북과 `automation/` 폴더를 저장소에 올립니다.\n3. GitHub Secrets에 필요한 키를 넣습니다.\n   - `OPEN_DART_API_KEY`\n   - `NAVER_CLIENT_ID`\n   - `NAVER_CLIENT_SECRET`\n4. `.github/workflows/daily-report.yml`을 저장소 루트의 `.github/workflows/`로 복사합니다.\n5. GitHub Pages를 `docs/` 폴더 기준으로 설정합니다.\n\n## 3. Google Drive 방식\n- 현재는 스켈레톤만 포함되어 있습니다.\n- Drive API 인증을 설정한 뒤 `automation/scripts/upload_to_drive.py`에 업로드 코드를 연결하면 됩니다.\n\n## 4. 권장 운영\n- 자동매매가 아니라 `자동 리포트 생성 → 사람이 확인 → 수동 매매` 흐름을 권장합니다.\n- 매매 전 HTS/MTS에서 현재가, 호가, 공시, 뉴스는 반드시 최종 확인하세요.\n', encoding='utf-8')

report_date = v99_report_date() if 'v99_report_date' in globals() else 'latest'
package = Path(f'v9_9_automation_package_{report_date}.zip')
with zipfile.ZipFile(package, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in auto_root.rglob('*'):
        if p.is_file():
            z.write(p, p.as_posix())
    docs = Path('docs')
    if docs.exists():
        for p in docs.rglob('*'):
            if p.is_file():
                z.write(p, p.as_posix())
print(f'✅ 자동화 준비 파일 생성 완료: {auto_root}')
print(f'✅ 자동화 패키지 생성 완료: {package}')
automation_package_file = str(package)


✅ 자동화 준비 파일 생성 완료: automation
✅ 자동화 패키지 생성 완료: v9_9_automation_package_20260608.zip


## 실행 후 확인 순서

1. 마지막 셀 출력의 `docs/index.html` 또는 `html_report_package_YYYYMMDD.zip` 확인
2. 엑셀은 계산·보관용, HTML은 매일 보는 화면으로 사용
3. GitHub Pages 자동화는 `automation/README_자동화설정.md` 순서대로 진행
4. 자동매매 전까지는 HTML 리포트 확인 후 HTS/MTS에서 수동 매매


# v10.0 자동 저장·배포 준비

이 마지막 섹션은 생성된 엑셀과 HTML 결과물을 날짜별 폴더와 latest 폴더에 복사합니다.

- `stock_report/reports/YYYYMMDD/`: 날짜별 보관
- `stock_report/latest/`: 항상 최신 리포트
- `stock_report/docs/`: GitHub Pages 또는 정적 웹 배포 준비
- `stock_report/holdings/`: 다음날 자동 승계용 보유파일 보관


In [30]:
# 102. v10.0 자동 저장·latest 리포트 생성
from pathlib import Path
from datetime import datetime, timedelta, timezone
import os, shutil, zipfile, json

KST_V10 = timezone(timedelta(hours=9))


def v100_final_date_key():
    forced = None
    try:
        forced = CONFIG.get('FORCE_REPORT_DATE')
    except Exception:
        forced = None
    if forced:
        return str(forced).replace('-', '')
    return datetime.now(KST_V10).strftime('%Y%m%d')


def v100_get_existing(path):
    p = Path(str(path)) if path else None
    return p if p and p.exists() else None


def v100_copytree_overwrite(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    return True


def v100_collect_outputs():
    date_key = v100_final_date_key()
    root = v100_report_root()
    report_dir = root / CONFIG.get('REPORT_OUTPUT_SUBDIR','reports') / date_key
    latest_dir = root / CONFIG.get('LATEST_SUBDIR','latest')
    docs_root = root / CONFIG.get('DOCS_SUBDIR','docs')
    holdings_dir = root / CONFIG.get('HOLDINGS_SUBDIR','holdings')
    for d in [report_dir, latest_dir, docs_root, holdings_dir]:
        d.mkdir(parents=True, exist_ok=True)

    copied = []
    xlsx = v100_get_existing(globals().get('output_file'))
    if xlsx:
        target = report_dir / f'{date_key}.xlsx'
        shutil.copy2(xlsx, target)
        shutil.copy2(xlsx, holdings_dir / f'{date_key}.xlsx')  # 다음날 보유종목 자동승계용
        shutil.copy2(xlsx, latest_dir / f'{date_key}.xlsx')
        copied.append(str(target))

    # v9.9가 만든 docs/index.html과 날짜별 reports를 함께 보관
    local_docs = Path('docs')
    if local_docs.exists():
        v100_copytree_overwrite(local_docs, report_dir / 'html')
        v100_copytree_overwrite(local_docs, latest_dir / 'html')
        # GitHub Pages용 docs에도 최신본 반영
        v100_copytree_overwrite(local_docs, docs_root)
        copied.append(str(report_dir / 'html'))

    # HTML package zip 보관
    for z in sorted(Path('.').glob('html_report_package_*.zip')):
        shutil.copy2(z, report_dir / z.name)
        shutil.copy2(z, latest_dir / z.name)
        copied.append(str(report_dir / z.name))

    status = {
        'report_date': date_key,
        'report_root': str(root),
        'report_dir': str(report_dir),
        'latest_dir': str(latest_dir),
        'xlsx': str(xlsx) if xlsx else '',
        'copied': copied,
        'created_at_kst': datetime.now(KST_V10).isoformat()
    }
    (root / 'v10_자동화_상태.json').write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
    print('✅ v10.0 자동 저장 완료')
    print('📁 날짜별 리포트:', report_dir)
    print('📌 최신 리포트:', latest_dir)
    if not copied:
        print('⚠️ 복사된 산출물이 없습니다. output_file/docs 생성 여부를 확인하세요.')
    return status

v10_status = v100_collect_outputs()


✅ v10.0 자동 저장 완료
📁 날짜별 리포트: stock_report/reports/20260608
📌 최신 리포트: stock_report/latest


In [31]:
# 103. v10.0 자동화 템플릿/안내 파일 생성
from pathlib import Path
import zipfile

auto_dir = Path('automation_v10')
(auto_dir / '.github' / 'workflows').mkdir(parents=True, exist_ok=True)
(auto_dir / 'scripts').mkdir(parents=True, exist_ok=True)
(auto_dir / 'data' / 'holdings').mkdir(parents=True, exist_ok=True)
(auto_dir / 'docs').mkdir(parents=True, exist_ok=True)

(auto_dir / 'requirements.txt').write_text('pandas\nnumpy\nopenpyxl\nrequests\nbeautifulsoup4\nlxml\nfinance-datareader\npykrx\nnbconvert\njupyter\n', encoding='utf-8')
(auto_dir / '.env.example').write_text('OPEN_DART_API_KEY=\nDART_API_KEY=\nNAVER_CLIENT_ID=\nNAVER_CLIENT_SECRET=\nFORCE_REPORT_DATE=\n', encoding='utf-8')

workflow_text = """name: daily-stock-report

on:
  workflow_dispatch:
  schedule:
    # KST 08:00 = UTC 23:00, 월~금 실행
    - cron: '0 23 * * 0-4'

permissions:
  contents: write

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Install dependencies
        run: pip install -r automation_v10/requirements.txt
      - name: Run notebook
        env:
          OPEN_DART_API_KEY: ${{ secrets.OPEN_DART_API_KEY }}
          DART_API_KEY: ${{ secrets.DART_API_KEY }}
          NAVER_CLIENT_ID: ${{ secrets.NAVER_CLIENT_ID }}
          NAVER_CLIENT_SECRET: ${{ secrets.NAVER_CLIENT_SECRET }}
        run: |
          jupyter nbconvert --to notebook --execute 실전매매_통합시스템_v10_0_자동리포트운영준비.ipynb --output executed_v10.ipynb --ExecutePreprocessor.timeout=3600
      - name: Commit reports
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add stock_report docs automation_v10 || true
          git commit -m "Update daily stock report" || echo "No changes"
          git push
"""
(auto_dir / '.github' / 'workflows' / 'daily-report.yml').write_text(workflow_text, encoding='utf-8')

readme_text = """# v10.0 자동 리포트 설정 안내

## 1단계: Colab 수동 + Drive 저장으로 먼저 테스트
1. 노트북 첫 설정에서 `USE_GOOGLE_DRIVE=True`, `REPORT_ROOT_MODE='DRIVE'`로 바꿉니다.
2. Colab Secrets에 필요한 키를 저장합니다.
   - `OPEN_DART_API_KEY` 또는 `DART_API_KEY`
   - `NAVER_CLIENT_ID`
   - `NAVER_CLIENT_SECRET`
3. `/MyDrive/stock_report/holdings/` 폴더에 전날 보유 파일을 넣습니다. 예: `20260527.xlsx`
4. 런타임 다시 시작 후 전체 실행합니다.
5. 결과는 `/MyDrive/stock_report/reports/YYYYMMDD/`와 `/MyDrive/stock_report/latest/`에 저장됩니다.

## 2단계: GitHub 자동 실행 준비
1. GitHub에 private repository를 만듭니다.
2. 이 패키지의 노트북과 `automation_v10/` 폴더를 업로드합니다.
3. GitHub repository Settings → Secrets and variables → Actions에 API 키를 등록합니다.
4. Actions 탭에서 `daily-stock-report`를 수동 실행해 테스트합니다.
5. 정상 작동하면 매일 KST 오전 8시 자동 실행됩니다.

## 3단계: HTML 보기
- `stock_report/latest/html/index.html` 또는 `docs/index.html`을 열면 최신 리포트를 볼 수 있습니다.
- GitHub Pages를 사용할 경우 민감 정보가 공개되지 않도록 저장소 공개 범위를 반드시 확인하세요.

## 주의
- 이 리포트는 투자 판단 보조용이며 매수·매도 권유가 아닙니다.
- 실제 주문 전 MTS/HTS 현재가, 뉴스, 공시, 손실한도를 반드시 확인하세요.
"""
(auto_dir / 'README_자동리포트_설정.md').write_text(readme_text, encoding='utf-8')

pkg_name = '실전매매_통합시스템_v10_0_자동리포트운영준비_실행패키지.zip'
with zipfile.ZipFile(pkg_name, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in auto_dir.rglob('*'):
        z.write(p, p.as_posix())
    nb_path = Path('실전매매_통합시스템_v10_0_자동리포트운영준비.ipynb')
    if nb_path.exists():
        z.write(nb_path, nb_path.name)
print('📦 v10.0 자동화 안내/템플릿 생성 완료:', pkg_name)


📦 v10.0 자동화 안내/템플릿 생성 완료: 실전매매_통합시스템_v10_0_자동리포트운영준비_실행패키지.zip
